# FinGuard360 — Credit Risk Analysis

End-to-end portfolio analysis, credit-risk modeling, SHAP explainability, risk segmentation, and Power BI export workflow.

**Reproducibility note:** Download the Home Credit Default Risk files from Kaggle and place them under `data/raw/`. The notebook resolves the project root from the current working directory and does not depend on a personal Windows path. The raw dataset is intentionally excluded from the public repository.

In [1]:
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

print("Python:", sys.version)
print("Python path:", sys.executable)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Kurulum başarılı!")

Python: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
Python path: <PYTHON_EXECUTABLE>
Pandas: 2.2.2
NumPy: 1.26.4
Scikit-learn: 1.5.1
Kurulum başarılı!


In [ ]:
from pathlib import Path

project_path = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
raw_data_path = project_path / "data" / "raw"

zip_files = list(raw_data_path.glob("*.zip"))

print("Veri klasörü:", raw_data_path)
print("Bulunan ZIP sayısı:", len(zip_files))

for zip_file in zip_files:
    size_mb = zip_file.stat().st_size / (1024 ** 2)
    print(f"Dosya: {zip_file.name}")
    print(f"Boyut: {size_mb:.2f} MB")

In [3]:
import shutil
import zipfile

zip_path = raw_data_path / "home-credit-default-risk.zip"
extract_path = raw_data_path / "home_credit"

with zipfile.ZipFile(zip_path, "r") as archive:
    extracted_size_gb = sum(
        file.file_size for file in archive.infolist()
    ) / (1024 ** 3)

free_space_gb = shutil.disk_usage(raw_data_path).free / (1024 ** 3)

print(f"Açılmış tahmini veri boyutu: {extracted_size_gb:.2f} GB")
print(f"Boş disk alanı: {free_space_gb:.2f} GB")

if free_space_gb < extracted_size_gb + 1:
    print("Yetersiz disk alanı! Dosyalar açılmadı.")
else:
    extract_path.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(extract_path)

    csv_files = list(extract_path.glob("*.csv"))

    print("ZIP başarıyla açıldı.")
    print("Bulunan CSV sayısı:", len(csv_files))

    for csv_file in sorted(csv_files):
        print(csv_file.name)

Açılmış tahmini veri boyutu: 2.50 GB
Boş disk alanı: 3.31 GB
Yetersiz disk alanı! Dosyalar açılmadı.


In [4]:
import pandas as pd

inventory_records = []

for csv_file in sorted(csv_files):
    try:
        header = pd.read_csv(csv_file, nrows=0)
    except UnicodeDecodeError:
        header = pd.read_csv(csv_file, nrows=0, encoding="latin1")

    inventory_records.append(
        {
            "table_name": csv_file.stem,
            "file_size_mb": round(csv_file.stat().st_size / (1024 ** 2), 2),
            "column_count": len(header.columns),
        }
    )

inventory_df = (
    pd.DataFrame(inventory_records)
    .sort_values("file_size_mb", ascending=False)
    .reset_index(drop=True)
)

inventory_df

NameError: name 'csv_files' is not defined

In [ ]:
train_path = extract_path / "application_train.csv"

application_train = pd.read_csv(
    train_path,
    low_memory=False
)

memory_mb = (
    application_train.memory_usage(deep=True).sum()
    / (1024 ** 2)
)

print("Satır sayısı:", application_train.shape[0])
print("Sütun sayısı:", application_train.shape[1])
print(f"Bellek kullanımı: {memory_mb:.2f} MB")

print("\nİlk 20 sütun:")
print(application_train.columns[:20].tolist())

application_train.iloc[:5, :10]

In [ ]:
quality_summary = pd.DataFrame(
    {
        "metric": [
            "Total rows",
            "Unique application IDs",
            "Duplicate application IDs",
            "Null application IDs",
            "Fully duplicated rows",
            "Null TARGET values",
        ],
        "value": [
            len(application_train),
            application_train["SK_ID_CURR"].nunique(),
            application_train["SK_ID_CURR"].duplicated().sum(),
            application_train["SK_ID_CURR"].isna().sum(),
            application_train.duplicated().sum(),
            application_train["TARGET"].isna().sum(),
        ],
    }
)

quality_summary


In [ ]:
target_distribution = (
    application_train["TARGET"]
    .value_counts(dropna=False)
    .rename_axis("TARGET")
    .reset_index(name="customer_count")
)

target_distribution["percentage"] = (
    target_distribution["customer_count"]
    / len(application_train)
    * 100
).round(2)

target_distribution

In [ ]:
quality_summary = pd.DataFrame(
    {
        "metric": [
            "Total rows",
            "Unique application IDs",
            "Duplicate application IDs",
            "Null application IDs",
            "Fully duplicated rows",
            "Null TARGET values",
        ],
        "value": [
            len(application_train),
            application_train["SK_ID_CURR"].nunique(),
            application_train["SK_ID_CURR"].duplicated().sum(),
            application_train["SK_ID_CURR"].isna().sum(),
            application_train.duplicated().sum(),
            application_train["TARGET"].isna().sum(),
        ],
    }
)

quality_summary

In [ ]:
target_distribution = (
    application_train["TARGET"]
    .value_counts()
    .rename_axis("TARGET")
    .reset_index(name="customer_count")
)

target_distribution["percentage"] = (
    target_distribution["customer_count"]
    / len(application_train)
    * 100
).round(2)

target_distribution

### Key Finding

The target variable is highly imbalanced. Only 8.07% of applications are associated with payment difficulty, while 91.93% belong to the non-default class. Therefore, accuracy alone will not be sufficient for model evaluation. Recall, precision, PR-AUC, ROC-AUC and probability calibration will also be evaluated.



In [ ]:
target_plot = target_distribution.copy()

target_plot["customer_status"] = target_plot["TARGET"].map(
    {
        0: "No payment difficulty",
        1: "Payment difficulty",
    }
)

majority_baseline = target_plot["percentage"].max()
imbalance_ratio = 282686 / 24825

print(f"Majority-class baseline accuracy: {majority_baseline:.2f}%")
print(f"Class imbalance ratio: {imbalance_ratio:.2f} : 1")

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    target_plot["customer_status"],
    target_plot["percentage"],
    color=["#2F80ED", "#EB5757"],
)

ax.bar_label(
    bars,
    labels=[f"{value:.2f}%" for value in target_plot["percentage"]],
    padding=4,
)

ax.set_title("Distribution of Credit Payment Difficulty")
ax.set_xlabel("Customer Status")
ax.set_ylabel("Percentage of Applications")
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()


### Key Finding

The target variable is highly imbalanced. Only 8.07% of applications are associated with payment difficulty, while 91.93% belong to the non-default class. Therefore, accuracy alone will not be sufficient for model evaluation. Recall, precision, PR-AUC, ROC-AUC and probability calibration will also be evaluated.



In [ ]:
missing_summary = (
    application_train.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"]
    / len(application_train)
    * 100
).round(2)

missing_summary = (
    missing_summary[missing_summary["missing_count"] > 0]
    .sort_values("missing_percentage", ascending=False)
)

print("Toplam sütun sayısı:", application_train.shape[1])
print("Eksik değer bulunan sütun sayısı:", len(missing_summary))
print(
    "Eksik değer bulunmayan sütun sayısı:",
    application_train.shape[1] - len(missing_summary),
)

missing_summary.head(20)

In [ ]:
top_missing = (
    missing_summary.head(20)
    .sort_values("missing_percentage")
)

fig, ax = plt.subplots(figsize=(10, 8))

bars = ax.barh(
    top_missing.index,
    top_missing["missing_percentage"],
    color="#F2994A",
)

ax.bar_label(
    bars,
    labels=[
        f"{value:.1f}%"
        for value in top_missing["missing_percentage"]
    ],
    padding=4,
)

ax.axvline(
    50,
    color="#EB5757",
    linestyle="--",
    linewidth=1.5,
    label="50% missing",
)

ax.set_title("Top 20 Features by Missing Value Percentage")
ax.set_xlabel("Missing Values (%)")
ax.set_ylabel("Feature")
ax.set_xlim(0, 80)
ax.legend()

plt.tight_layout()
plt.show()

### Missing Data Observation

Missing values are concentrated in housing, building and asset-related features. Some missingness may be conditional rather than random. For example, `OWN_CAR_AGE` is expected to be unavailable for applicants who do not own a car. Therefore, high-missingness features will not be removed automatically before their business meaning and relationship with the target are evaluated.

In [ ]:
description_path = (
    extract_path
    / "HomeCredit_columns_description.csv"
)

column_description = pd.read_csv(
    description_path,
    encoding="latin1"
)

print("Satır sayısı:", column_description.shape[0])
print("Sütun sayısı:", column_description.shape[1])
print("Sütun isimleri:")
print(column_description.columns.tolist())

column_description.head(10)

In [ ]:
from pathlib import Path
import pandas as pd

project_path = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
extract_path = project_path / "data" / "raw" / "home_credit"

# Ana tablo bellekte yoksa yeniden yükle
if "application_train" not in globals():
    application_train = pd.read_csv(
        extract_path / "application_train.csv",
        low_memory=False,
    )

# Eksik değer özetini yeniden oluştur
missing_summary = (
    application_train.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"]
    / len(application_train)
    * 100
).round(2)

missing_summary = (
    missing_summary[missing_summary["missing_count"] > 0]
    .sort_values("missing_percentage", ascending=False)
)

# Veri sözlüğünü yeniden oku
column_description = pd.read_csv(
    extract_path / "HomeCredit_columns_description.csv",
    encoding="latin1",
)

column_description = column_description.drop(
    columns=["Unnamed: 0"],
    errors="ignore",
)

application_dictionary = column_description[
    column_description["Table"]
    .str.contains("application", case=False, na=False)
].copy()

top_missing_features = (
    missing_summary.head(20)
    .reset_index()
    .rename(columns={"index": "feature"})
)

top_missing_with_description = top_missing_features.merge(
    application_dictionary[["Row", "Description", "Special"]],
    left_on="feature",
    right_on="Row",
    how="left",
)

top_missing_with_description[
    [
        "feature",
        "missing_percentage",
        "Description",
        "Special",
    ]
]

### Interpretation of High-Missingness Features

Nineteen of the twenty features with the highest missingness are normalized building or housing attributes. This concentration indicates that missingness is structural rather than uniformly random. `OWN_CAR_AGE` is also conditionally applicable because it is expected to be missing for applicants who do not own a car. These features require conditional validation before imputation or removal.

In [ ]:
car_age_analysis = (
    application_train.assign(
        OWN_CAR_AGE_MISSING=application_train[
            "OWN_CAR_AGE"
        ].isna()
    )
    .groupby("FLAG_OWN_CAR")
    .agg(
        applicant_count=("SK_ID_CURR", "count"),
        missing_car_age=("OWN_CAR_AGE_MISSING", "sum"),
        default_rate=("TARGET", "mean"),
    )
    .reset_index()
)

car_age_analysis["missing_car_age_percentage"] = (
    car_age_analysis["missing_car_age"]
    / car_age_analysis["applicant_count"]
    * 100
).round(2)

car_age_analysis["default_rate_percentage"] = (
    car_age_analysis["default_rate"] * 100
).round(2)

car_age_analysis[
    [
        "FLAG_OWN_CAR",
        "applicant_count",
        "missing_car_age",
        "missing_car_age_percentage",
        "default_rate_percentage",
    ]
]

In [ ]:
car_owners_with_missing_age = application_train.loc[
    (application_train["FLAG_OWN_CAR"] == "Y")
    & (application_train["OWN_CAR_AGE"].isna()),
    [
        "SK_ID_CURR",
        "TARGET",
        "CODE_GENDER",
        "AMT_INCOME_TOTAL",
        "AMT_CREDIT",
        "NAME_INCOME_TYPE",
        "NAME_HOUSING_TYPE",
        "OWN_CAR_AGE",
    ],
]

print(
    "Araç sahibi olup araç yaşı eksik kayıt sayısı:",
    len(car_owners_with_missing_age),
)

car_owners_with_missing_age

In [ ]:
missingness_impact_records = []

for feature in missing_summary.index:
    missing_mask = application_train[feature].isna()

    missing_count = int(missing_mask.sum())
    present_count = int((~missing_mask).sum())

    missing_default_rate = application_train.loc[
        missing_mask, "TARGET"
    ].mean()

    present_default_rate = application_train.loc[
        ~missing_mask, "TARGET"
    ].mean()

    difference_percentage_points = (
        missing_default_rate - present_default_rate
    ) * 100

    missingness_impact_records.append(
        {
            "feature": feature,
            "missing_count": missing_count,
            "present_count": present_count,
            "missing_default_rate_pct": round(
                missing_default_rate * 100, 2
            ),
            "present_default_rate_pct": round(
                present_default_rate * 100, 2
            ),
            "difference_pp": round(
                difference_percentage_points, 2
            ),
        }
    )

missingness_impact = pd.DataFrame(
    missingness_impact_records
)

missingness_impact["absolute_difference_pp"] = (
    missingness_impact["difference_pp"].abs()
)

reliable_missingness_impact = (
    missingness_impact[
        (missingness_impact["missing_count"] >= 1000)
        & (missingness_impact["present_count"] >= 1000)
    ]
    .sort_values(
        "absolute_difference_pp",
        ascending=False,
    )
    .reset_index(drop=True)
)

reliable_missingness_impact.head(15)

### Missingness as a Risk Signal

Missingness is associated with different default rates across several feature groups. Credit bureau request features are jointly missing for 41,519 applications, and this group has a 10.34% default rate compared with 7.72% for applications with available information. Building-related missingness is also associated with higher default rates. Missingness indicators may therefore provide predictive information, but these relationships should not be interpreted as causal.

In [ ]:
missingness_groups = {
    "social_circle": [
        "OBS_30_CNT_SOCIAL_CIRCLE",
        "DEF_30_CNT_SOCIAL_CIRCLE",
        "OBS_60_CNT_SOCIAL_CIRCLE",
        "DEF_60_CNT_SOCIAL_CIRCLE",
    ],
    "credit_bureau_requests": [
        "AMT_REQ_CREDIT_BUREAU_HOUR",
        "AMT_REQ_CREDIT_BUREAU_DAY",
        "AMT_REQ_CREDIT_BUREAU_WEEK",
        "AMT_REQ_CREDIT_BUREAU_MON",
        "AMT_REQ_CREDIT_BUREAU_QRT",
        "AMT_REQ_CREDIT_BUREAU_YEAR",
    ],
}

group_missingness_records = []

for group_name, feature_list in missingness_groups.items():
    missing_matrix = application_train[
        feature_list
    ].isna()

    group_missingness_records.append(
        {
            "feature_group": group_name,
            "any_feature_missing": int(
                missing_matrix.any(axis=1).sum()
            ),
            "all_features_missing": int(
                missing_matrix.all(axis=1).sum()
            ),
            "mixed_missing_pattern": int(
                (
                    missing_matrix.any(axis=1)
                    & ~missing_matrix.all(axis=1)
                ).sum()
            ),
        }
    )

group_missingness_summary = pd.DataFrame(
    group_missingness_records
)

group_missingness_summary

In [ ]:
data_type_summary = (
    application_train.dtypes
    .astype(str)
    .value_counts()
    .rename_axis("data_type")
    .reset_index(name="column_count")
)

print("Veri tipi özeti:")
display(data_type_summary)

categorical_columns = application_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_columns = application_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Kategorik sütun sayısı:", len(categorical_columns))
print("Sayısal sütun sayısı:", len(numeric_columns))

categorical_summary = pd.DataFrame(
    {
        "feature": categorical_columns,
        "unique_value_count": [
            application_train[column].nunique(
                dropna=False
            )
            for column in categorical_columns
        ],
        "missing_count": [
            application_train[column].isna().sum()
            for column in categorical_columns
        ],
    }
).sort_values(
    "unique_value_count",
    ascending=False,
)

categorical_summary

In [ ]:
categorical_value_records = []

for feature in categorical_columns:
    feature_values = (
        application_train[feature]
        .astype("object")
        .where(
            application_train[feature].notna(),
            "<MISSING>",
        )
    )

    value_counts = feature_values.value_counts(
        dropna=False
    )

    for category, count in value_counts.items():
        categorical_value_records.append(
            {
                "feature": feature,
                "category": str(category),
                "customer_count": int(count),
                "percentage": round(
                    count
                    / len(application_train)
                    * 100,
                    4,
                ),
            }
        )

categorical_value_summary = pd.DataFrame(
    categorical_value_records
)

rare_categories = (
    categorical_value_summary[
        categorical_value_summary[
            "customer_count"
        ] < 100
    ]
    .sort_values(
        ["customer_count", "feature"],
        ascending=[True, True],
    )
    .reset_index(drop=True)
)

print(
    "100 kayıttan az görülen kategori sayısı:",
    len(rare_categories),
)

rare_categories.head(30)

In [ ]:
important_numeric_features = [
    "CNT_CHILDREN",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",
    "OWN_CAR_AGE",
]

numeric_profile = (
    application_train[
        important_numeric_features
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.50,
            0.95,
            0.99,
        ]
    )
    .T
)

numeric_profile[
    [
        "count",
        "mean",
        "std",
        "min",
        "1%",
        "5%",
        "50%",
        "95%",
        "99%",
        "max",
    ]
].round(2)

In [ ]:
income_99_threshold = application_train[
    "AMT_INCOME_TOTAL"
].quantile(0.99)

anomaly_masks = {
    "DAYS_EMPLOYED sentinel (365243)": (
        application_train["DAYS_EMPLOYED"]
        == 365243
    ),
    "Income above 99th percentile": (
        application_train["AMT_INCOME_TOTAL"]
        > income_99_threshold
    ),
    "More than 5 children": (
        application_train["CNT_CHILDREN"] > 5
    ),
    "Car age above 50": (
        application_train["OWN_CAR_AGE"] > 50
    ),
    "Missing annuity amount": (
        application_train["AMT_ANNUITY"].isna()
    ),
    "Missing goods price": (
        application_train["AMT_GOODS_PRICE"].isna()
    ),
}

anomaly_records = []

for anomaly_name, anomaly_mask in anomaly_masks.items():
    anomaly_count = int(anomaly_mask.sum())

    anomaly_records.append(
        {
            "anomaly": anomaly_name,
            "record_count": anomaly_count,
            "percentage": round(
                anomaly_count
                / len(application_train)
                * 100,
                3,
            ),
            "default_rate_pct": round(
                application_train.loc[
                    anomaly_mask, "TARGET"
                ].mean()
                * 100,
                2,
            )
            if anomaly_count > 0
            else None,
        }
    )

anomaly_summary = pd.DataFrame(anomaly_records)

print(
    f"Gelir için 99. yüzdelik sınırı: "
    f"{income_99_threshold:,.2f}"
)

anomaly_summary

In [ ]:
employment_analysis_df = application_train.assign(
    EMPLOYMENT_SENTINEL=(
        application_train["DAYS_EMPLOYED"]
        == 365243
    ),
    AGE_YEARS=(
        -application_train["DAYS_BIRTH"]
        / 365.25
    ),
)

employment_group_summary = (
    employment_analysis_df
    .groupby("EMPLOYMENT_SENTINEL")
    .agg(
        applicant_count=("SK_ID_CURR", "count"),
        median_age=("AGE_YEARS", "median"),
        default_rate=("TARGET", "mean"),
    )
    .reset_index()
)

employment_group_summary["default_rate_pct"] = (
    employment_group_summary["default_rate"]
    * 100
).round(2)

employment_group_summary[
    [
        "EMPLOYMENT_SENTINEL",
        "applicant_count",
        "median_age",
        "default_rate_pct",
    ]
]

In [ ]:
sentinel_income_profile = (
    employment_analysis_df.loc[
        employment_analysis_df[
            "EMPLOYMENT_SENTINEL"
        ],
        ["NAME_INCOME_TYPE", "TARGET"],
    ]
    .groupby("NAME_INCOME_TYPE")
    .agg(
        applicant_count=("TARGET", "size"),
        default_rate=("TARGET", "mean"),
    )
    .reset_index()
    .sort_values(
        "applicant_count",
        ascending=False,
    )
)

sentinel_income_profile["group_percentage"] = (
    sentinel_income_profile["applicant_count"]
    / sentinel_income_profile[
        "applicant_count"
    ].sum()
    * 100
).round(2)

sentinel_income_profile["default_rate_pct"] = (
    sentinel_income_profile["default_rate"]
    * 100
).round(2)

sentinel_income_profile[
    [
        "NAME_INCOME_TYPE",
        "applicant_count",
        "group_percentage",
        "default_rate_pct",
    ]
]

### DAYS_EMPLOYED Special Value

The value `365243` in `DAYS_EMPLOYED` is a structural sentinel rather than a valid employment duration. Of the 55,374 applicants with this value, 99.96% are pensioners. The sentinel will be replaced with a missing value for numerical processing, while a separate indicator will preserve its categorical information.

In [ ]:
employment_features = application_train[
    [
        "SK_ID_CURR",
        "TARGET",
        "DAYS_BIRTH",
        "DAYS_EMPLOYED",
        "NAME_INCOME_TYPE",
    ]
].copy()

employment_features[
    "AGE_YEARS"
] = (
    -employment_features["DAYS_BIRTH"]
    / 365.25
)

employment_features[
    "DAYS_EMPLOYED_ANOMALY"
] = (
    employment_features["DAYS_EMPLOYED"]
    == 365243
)

employment_features[
    "EMPLOYMENT_YEARS"
] = np.where(
    employment_features[
        "DAYS_EMPLOYED_ANOMALY"
    ],
    np.nan,
    -employment_features["DAYS_EMPLOYED"]
    / 365.25,
)

employment_features[
    [
        "AGE_YEARS",
        "EMPLOYMENT_YEARS",
    ]
].describe().round(2)

In [ ]:
age_risk_analysis = employment_features.assign(
    AGE_GROUP=pd.cut(
        employment_features["AGE_YEARS"],
        bins=[20, 30, 40, 50, 60, 70],
        labels=[
            "20–29",
            "30–39",
            "40–49",
            "50–59",
            "60–69",
        ],
        right=False,
    )
)

age_risk_summary = (
    age_risk_analysis
    .groupby(
        "AGE_GROUP",
        observed=True,
    )
    .agg(
        applicant_count=("SK_ID_CURR", "count"),
        default_count=("TARGET", "sum"),
        default_rate=("TARGET", "mean"),
    )
    .reset_index()
)

age_risk_summary["application_share_pct"] = (
    age_risk_summary["applicant_count"]
    / age_risk_summary["applicant_count"].sum()
    * 100
).round(2)

age_risk_summary["default_rate_pct"] = (
    age_risk_summary["default_rate"]
    * 100
).round(2)

age_risk_summary[
    [
        "AGE_GROUP",
        "applicant_count",
        "application_share_pct",
        "default_count",
        "default_rate_pct",
    ]
]

In [ ]:
age_risk_summary["standard_error"] = np.sqrt(
    age_risk_summary["default_rate"]
    * (1 - age_risk_summary["default_rate"])
    / age_risk_summary["applicant_count"]
)

age_risk_summary["ci_lower_pct"] = (
    age_risk_summary["default_rate"]
    - 1.96 * age_risk_summary["standard_error"]
) * 100

age_risk_summary["ci_upper_pct"] = (
    age_risk_summary["default_rate"]
    + 1.96 * age_risk_summary["standard_error"]
) * 100

x_positions = np.arange(len(age_risk_summary))

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    x_positions,
    age_risk_summary["default_rate_pct"],
    color="#EB5757",
    marker="o",
    linewidth=2.5,
    markersize=7,
)

ax.fill_between(
    x_positions,
    age_risk_summary["ci_lower_pct"],
    age_risk_summary["ci_upper_pct"],
    color="#EB5757",
    alpha=0.18,
    label="95% confidence interval",
)

for x_position, rate in zip(
    x_positions,
    age_risk_summary["default_rate_pct"],
):
    ax.text(
        x_position,
        rate + 0.3,
        f"{rate:.2f}%",
        ha="center",
    )

ax.set_xticks(x_positions)
ax.set_xticklabels(
    age_risk_summary["AGE_GROUP"]
)
ax.set_title("Default Rate by Applicant Age Group")
ax.set_xlabel("Age Group")
ax.set_ylabel("Default Rate (%)")
ax.set_ylim(0, 13)
ax.grid(axis="y", alpha=0.25)
ax.legend()

plt.tight_layout()
plt.show()

### Age Risk Finding

Default risk decreases monotonically across age groups, from 11.44% among applicants aged 20–29 to 4.92% among applicants aged 60–69. The youngest group has approximately 2.33 times the default rate of the oldest group. This is an association rather than a causal effect, and age must be evaluated carefully as a sensitive attribute.


In [ ]:
age_income_analysis = application_train[
    [
        "SK_ID_CURR",
        "TARGET",
        "DAYS_BIRTH",
        "AMT_INCOME_TOTAL",
    ]
].copy()

age_income_analysis["AGE_YEARS"] = (
    -age_income_analysis["DAYS_BIRTH"]
    / 365.25
)

age_income_analysis["AGE_GROUP"] = pd.cut(
    age_income_analysis["AGE_YEARS"],
    bins=[20, 30, 40, 50, 60, 70],
    labels=[
        "20–29",
        "30–39",
        "40–49",
        "50–59",
        "60–69",
    ],
    right=False,
)

income_quintile_code = pd.qcut(
    age_income_analysis["AMT_INCOME_TOTAL"],
    q=5,
    labels=False,
    duplicates="drop",
)

age_income_analysis["INCOME_QUINTILE"] = (
    income_quintile_code.map(
        {
            0: "Q1 – Lowest",
            1: "Q2",
            2: "Q3",
            3: "Q4",
            4: "Q5 – Highest",
        }
    )
)

age_income_summary = (
    age_income_analysis
    .groupby(
        ["AGE_GROUP", "INCOME_QUINTILE"],
        observed=True,
    )
    .agg(
        applicant_count=("SK_ID_CURR", "count"),
        default_rate=("TARGET", "mean"),
    )
    .reset_index()
)

age_income_summary["default_rate_pct"] = (
    age_income_summary["default_rate"]
    * 100
).round(2)

age_income_rate_matrix = (
    age_income_summary.pivot(
        index="AGE_GROUP",
        columns="INCOME_QUINTILE",
        values="default_rate_pct",
    )
)

print(
    "En küçük segmentteki başvuru sayısı:",
    age_income_summary["applicant_count"].min(),
)

age_income_rate_matrix

### Age and Income Interaction

The decline in default risk across age groups persists within every income quintile. The highest observed rate is 12.70% among applicants aged 20–29 in the lowest income quintile, while the lowest rate is 4.27% among applicants aged 60–69 in the highest income quintile. Income reduces risk most clearly in younger groups, suggesting a possible interaction between age and income.

In [ ]:
age_order = [
    "20–29",
    "30–39",
    "40–49",
    "50–59",
    "60–69",
]

income_order = [
    "Q1 – Lowest",
    "Q2",
    "Q3",
    "Q4",
    "Q5 – Highest",
]

heatmap_matrix = age_income_rate_matrix.loc[
    age_order,
    income_order,
]

heatmap_annotations = heatmap_matrix.map(
    lambda value: f"{value:.2f}%"
)

fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(
    heatmap_matrix,
    annot=heatmap_annotations,
    fmt="",
    cmap="RdYlGn_r",
    vmin=4,
    vmax=13,
    linewidths=0.7,
    linecolor="white",
    cbar_kws={
        "label": "Default Rate (%)"
    },
    ax=ax,
)

ax.set_title(
    "Default Rate by Age Group and Income Quintile"
)
ax.set_xlabel("Income Quintile")
ax.set_ylabel("Age Group")

plt.tight_layout()
plt.show()

In [ ]:
income_quantile_levels = [
    0.50,
    0.90,
    0.95,
    0.99,
    0.995,
    0.999,
    1.00,
]

income_quantile_summary = (
    application_train[
        "AMT_INCOME_TOTAL"
    ]
    .quantile(income_quantile_levels)
    .rename("income_amount")
    .reset_index()
    .rename(columns={"index": "quantile"})
)

income_quantile_summary[
    "quantile"
] = income_quantile_summary[
    "quantile"
].map(
    lambda value: f"{value:.1%}"
)

income_quantile_summary

In [ ]:
top_income_records = (
    application_train.nlargest(
        10,
        "AMT_INCOME_TOTAL",
    )[
        [
            "SK_ID_CURR",
            "TARGET",
            "AMT_INCOME_TOTAL",
            "AMT_CREDIT",
            "NAME_INCOME_TYPE",
            "OCCUPATION_TYPE",
            "ORGANIZATION_TYPE",
        ]
    ]
)

top_income_records

In [ ]:
income_q99 = application_train[
    "AMT_INCOME_TOTAL"
].quantile(0.99)

income_q999 = application_train[
    "AMT_INCOME_TOTAL"
].quantile(0.999)

income_tail_analysis = (
    application_train.assign(
        INCOME_TAIL_GROUP=pd.cut(
            application_train[
                "AMT_INCOME_TOTAL"
            ],
            bins=[
                -np.inf,
                income_q99,
                income_q999,
                np.inf,
            ],
            labels=[
                "Up to 99th percentile",
                "99th–99.9th percentile",
                "Above 99.9th percentile",
            ],
            include_lowest=True,
        )
    )
    .groupby(
        "INCOME_TAIL_GROUP",
        observed=True,
    )
    .agg(
        applicants=("SK_ID_CURR", "size"),
        defaults=("TARGET", "sum"),
        default_rate=("TARGET", "mean"),
        median_income=("AMT_INCOME_TOTAL", "median"),
        maximum_income=("AMT_INCOME_TOTAL", "max"),
        median_credit=("AMT_CREDIT", "median"),
    )
    .reset_index()
)

income_tail_analysis[
    "default_rate"
] *= 100

income_tail_analysis.round(2)

### Income Outlier Finding

Income is highly right-skewed, with a maximum value of 117 million compared with a 99.9th percentile of 900 thousand. Because the validity of this record cannot be confirmed, it will not be deleted. Log transformation and training-set-based capping will be evaluated during modeling. Lower default rates among high-income applicants represent an association, not a causal effect.

In [ ]:
financial_ratios = application_train[
    [
        "SK_ID_CURR",
        "TARGET",
        "AMT_INCOME_TOTAL",
        "AMT_CREDIT",
        "AMT_ANNUITY",
    ]
].copy()

financial_ratios[
    "CREDIT_INCOME_RATIO"
] = (
    financial_ratios["AMT_CREDIT"]
    / financial_ratios["AMT_INCOME_TOTAL"]
)

financial_ratios[
    "ANNUITY_INCOME_RATIO"
] = (
    financial_ratios["AMT_ANNUITY"]
    / financial_ratios["AMT_INCOME_TOTAL"]
)

financial_ratios.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True,
)

ratio_summary = (
    financial_ratios.groupby("TARGET")
    .agg(
        applicants=("SK_ID_CURR", "size"),
        median_credit_income_ratio=(
            "CREDIT_INCOME_RATIO",
            "median",
        ),
        p90_credit_income_ratio=(
            "CREDIT_INCOME_RATIO",
            lambda values: values.quantile(0.90),
        ),
        median_annuity_income_ratio=(
            "ANNUITY_INCOME_RATIO",
            "median",
        ),
        p90_annuity_income_ratio=(
            "ANNUITY_INCOME_RATIO",
            lambda values: values.quantile(0.90),
        ),
    )
    .reset_index()
)

ratio_summary.round(3)

In [ ]:
annuity_ratio_analysis = (
    financial_ratios.dropna(
        subset=["ANNUITY_INCOME_RATIO"]
    )
    .copy()
)

annuity_ratio_analysis[
    "ANNUITY_INCOME_DECILE"
] = pd.qcut(
    annuity_ratio_analysis[
        "ANNUITY_INCOME_RATIO"
    ],
    q=10,
    duplicates="drop",
)

annuity_decile_summary = (
    annuity_ratio_analysis.groupby(
        "ANNUITY_INCOME_DECILE",
        observed=True,
    )
    .agg(
        applicants=("SK_ID_CURR", "size"),
        defaults=("TARGET", "sum"),
        minimum_ratio=(
            "ANNUITY_INCOME_RATIO",
            "min",
        ),
        maximum_ratio=(
            "ANNUITY_INCOME_RATIO",
            "max",
        ),
        default_rate=("TARGET", "mean"),
    )
    .reset_index()
)

annuity_decile_summary.insert(
    0,
    "decile",
    [
        f"D{number}"
        for number in range(
            1,
            len(annuity_decile_summary) + 1,
        )
    ],
)

annuity_decile_summary[
    "default_rate"
] *= 100

annuity_decile_summary[
    [
        "decile",
        "applicants",
        "defaults",
        "minimum_ratio",
        "maximum_ratio",
        "default_rate",
    ]
].round(3)

In [ ]:
annuity_decile_summary[
    "default_rate_fraction"
] = (
    annuity_decile_summary["defaults"]
    / annuity_decile_summary["applicants"]
)

annuity_decile_summary[
    "standard_error"
] = np.sqrt(
    (
        annuity_decile_summary[
            "default_rate_fraction"
        ]
        * (
            1
            - annuity_decile_summary[
                "default_rate_fraction"
            ]
        )
    )
    / annuity_decile_summary["applicants"]
)

annuity_decile_summary["ci_lower"] = (
    annuity_decile_summary[
        "default_rate_fraction"
    ]
    - 1.96
    * annuity_decile_summary["standard_error"]
).clip(lower=0) * 100

annuity_decile_summary["ci_upper"] = (
    annuity_decile_summary[
        "default_rate_fraction"
    ]
    + 1.96
    * annuity_decile_summary["standard_error"]
).clip(upper=1) * 100

overall_default_rate = (
    application_train["TARGET"].mean() * 100
)

x_positions = np.arange(
    len(annuity_decile_summary)
)

fig, ax = plt.subplots(
    figsize=(11, 6)
)

ax.plot(
    x_positions,
    annuity_decile_summary["default_rate"],
    color="#2878B5",
    marker="o",
    linewidth=2.5,
)

ax.fill_between(
    x_positions,
    annuity_decile_summary["ci_lower"],
    annuity_decile_summary["ci_upper"],
    color="#2878B5",
    alpha=0.18,
    label="95% confidence interval",
)

ax.axhline(
    overall_default_rate,
    color="#555555",
    linestyle="--",
    linewidth=1.5,
    label=f"Overall default rate ({overall_default_rate:.2f}%)",
)

for x_value, risk_rate in zip(
    x_positions,
    annuity_decile_summary["default_rate"],
):
    ax.text(
        x_value,
        risk_rate + 0.20,
        f"{risk_rate:.2f}%",
        ha="center",
        fontsize=9,
    )

ax.set_xticks(x_positions)
ax.set_xticklabels(
    annuity_decile_summary["decile"]
)

ax.set_ylim(0, 10)
ax.set_xlabel("Annuity-to-Income Decile")
ax.set_ylabel("Default Rate (%)")

ax.set_title(
    "Default Rate by Annuity-to-Income Decile\n"
    "D1 represents the lowest ratio and D10 the highest"
)

ax.grid(
    axis="y",
    alpha=0.25,
)

ax.legend()
plt.tight_layout()
plt.show()

### Annuity-to-Income Risk Pattern

The default rate increases from 7.14% in the lowest annuity-to-income decile to 8.85% in D8, before leveling off and declining to 8.19% in D10. This pattern indicates a nonlinear association rather than a constant linear effect. The confidence interval for D10 overlaps the overall default rate, so the decline in the highest decile should not be overinterpreted. The ratio will be evaluated using nonlinear or tree-based modeling approaches. This analysis includes 307,499 applications; 12 records with missing annuity values were excluded.


In [ ]:
annuity_ratio_analysis[
    "DECILE_NUMBER"
] = (
    annuity_ratio_analysis[
        "ANNUITY_INCOME_DECILE"
    ].cat.codes
    + 1
)

upper_decile_profile = (
    annuity_ratio_analysis[
        annuity_ratio_analysis[
            "DECILE_NUMBER"
        ].isin([8, 9, 10])
    ]
    .copy()
)

upper_decile_profile[
    "AGE_YEARS"
] = (
    -application_train.loc[
        upper_decile_profile.index,
        "DAYS_BIRTH",
    ]
    / 365.25
)

upper_decile_profile[
    "NAME_INCOME_TYPE"
] = application_train.loc[
    upper_decile_profile.index,
    "NAME_INCOME_TYPE",
]

upper_decile_profile[
    "NAME_CONTRACT_TYPE"
] = application_train.loc[
    upper_decile_profile.index,
    "NAME_CONTRACT_TYPE",
]

upper_decile_comparison = (
    upper_decile_profile.groupby(
        "DECILE_NUMBER"
    )
    .agg(
        applicants=("SK_ID_CURR", "size"),
        default_rate=("TARGET", "mean"),
        median_age=("AGE_YEARS", "median"),
        median_income=(
            "AMT_INCOME_TOTAL",
            "median",
        ),
        median_credit=("AMT_CREDIT", "median"),
        pensioner_share=(
            "NAME_INCOME_TYPE",
            lambda values: (
                values.eq("Pensioner").mean()
                * 100
            ),
        ),
        cash_loan_share=(
            "NAME_CONTRACT_TYPE",
            lambda values: (
                values.eq("Cash loans").mean()
                * 100
            ),
        ),
    )
    .reset_index()
)

upper_decile_comparison[
    "default_rate"
] *= 100

upper_decile_comparison.round(2)

In [ ]:
upper_decile_profile[
    "AGE_GROUP"
] = pd.cut(
    upper_decile_profile["AGE_YEARS"],
    bins=[20, 30, 40, 50, 60, 70],
    labels=[
        "20–29",
        "30–39",
        "40–49",
        "50–59",
        "60–69",
    ],
    right=False,
    include_lowest=True,
)

age_stratified_decile_summary = (
    upper_decile_profile.groupby(
        [
            "AGE_GROUP",
            "DECILE_NUMBER",
        ],
        observed=True,
    )
    .agg(
        applicants=("SK_ID_CURR", "size"),
        defaults=("TARGET", "sum"),
        default_rate=("TARGET", "mean"),
    )
    .reset_index()
)

age_stratified_decile_summary[
    "default_rate"
] *= 100

age_stratified_decile_summary.round(2)

### Age-Stratified Financial Burden Analysis

The decline observed in the highest annuity-to-income decile is partly related to differences in applicant composition, but it is not fully explained by age. Within age groups, the D10 default rate is lower for applicants aged 30–39 and 50–59, nearly unchanged for those aged 20–29 and 40–49, and higher for applicants aged 60–69. This heterogeneous pattern suggests an interaction between age and relative financial burden. Therefore, the annuity-to-income ratio should not be interpreted as having a constant effect across all applicants.


In [ ]:
categorical_features = (
    application_train.select_dtypes(
        include="object"
    )
    .columns
    .tolist()
)

categorical_risk_tables = []

for feature in categorical_features:
    feature_summary = (
        application_train.groupby(
            feature,
            dropna=False,
        )
        .agg(
            applicants=("SK_ID_CURR", "size"),
            defaults=("TARGET", "sum"),
            default_rate=("TARGET", "mean"),
        )
        .reset_index()
        .rename(
            columns={feature: "category"}
        )
    )

    feature_summary["feature"] = feature

    feature_summary["category"] = (
        feature_summary["category"]
        .astype("string")
        .fillna("Missing")
    )

    categorical_risk_tables.append(
        feature_summary
    )

categorical_risk_summary = pd.concat(
    categorical_risk_tables,
    ignore_index=True,
)

overall_default_rate = (
    application_train["TARGET"].mean()
)

categorical_risk_summary[
    "default_rate"
] *= 100

categorical_risk_summary[
    "difference_from_overall"
] = (
    categorical_risk_summary["default_rate"]
    - overall_default_rate * 100
)

categorical_risk_summary[
    "absolute_difference"
] = categorical_risk_summary[
    "difference_from_overall"
].abs()

reliable_categorical_risks = (
    categorical_risk_summary[
        categorical_risk_summary[
            "applicants"
        ] >= 1000
    ]
    .sort_values(
        "absolute_difference",
        ascending=False,
    )
)

reliable_categorical_risks[
    [
        "feature",
        "category",
        "applicants",
        "defaults",
        "default_rate",
        "difference_from_overall",
    ]
].head(20).round(2)

### Categorical Risk Findings

Among categories containing at least 1,000 applications, low-skill laborers have the highest observed default rate at 17.15%, compared with the overall rate of 8.07%. Elevated rates are also observed among applicants working in Transport type 3, living in rented accommodation or with parents, and working as drivers or in construction. Lower rates are observed among accountants, higher-education applicants, pensioners, and several public-sector organization groups. These relationships are associative rather than causal. Occupation, education, housing, and organization variables may act as proxies for socioeconomic characteristics and will therefore require careful encoding and fairness assessment during modeling.


In [ ]:
import gc

temporary_objects = [
    "financial_ratios",
    "annuity_ratio_analysis",
    "upper_decile_profile",
    "categorical_risk_tables",
]

for object_name in temporary_objects:
    globals().pop(object_name, None)

gc.collect()

bureau = pd.read_csv(
    extract_path / "bureau.csv",
    low_memory=False,
)

bureau_memory_mb = (
    bureau.memory_usage(deep=True).sum()
    / 1024**2
)

bureau_overview = pd.DataFrame(
    {
        "metric": [
            "Total rows",
            "Total columns",
            "Unique bureau IDs",
            "Unique application IDs",
            "Duplicate bureau IDs",
            "Null bureau IDs",
            "Null application IDs",
            "Memory usage (MB)",
        ],
        "value": [
            len(bureau),
            bureau.shape[1],
            bureau["SK_ID_BUREAU"].nunique(),
            bureau["SK_ID_CURR"].nunique(),
            bureau["SK_ID_BUREAU"].duplicated().sum(),
            bureau["SK_ID_BUREAU"].isna().sum(),
            bureau["SK_ID_CURR"].isna().sum(),
            round(bureau_memory_mb, 2),
        ],
    }
)

bureau_overview

In [ ]:
bureau_records_per_customer = (
    bureau.groupby(
        "SK_ID_CURR",
        sort=False,
    )
    .size()
)

train_customer_ids = pd.Index(
    application_train["SK_ID_CURR"]
)

train_bureau_counts = (
    bureau_records_per_customer.reindex(
        train_customer_ids
    )
)

covered_train_counts = (
    train_bureau_counts.dropna()
)

customers_with_bureau = (
    train_bureau_counts.notna().sum()
)

customers_without_bureau = (
    train_bureau_counts.isna().sum()
)

bureau_coverage_summary = pd.DataFrame(
    {
        "metric": [
            "Total train customers",
            "Customers with bureau history",
            "Customers without bureau history",
            "Bureau coverage (%)",
            "Mean records per covered customer",
            "Median records per covered customer",
            "90th percentile records",
            "99th percentile records",
            "Maximum records",
        ],
        "value": [
            len(train_customer_ids),
            customers_with_bureau,
            customers_without_bureau,
            customers_with_bureau
            / len(train_customer_ids)
            * 100,
            covered_train_counts.mean(),
            covered_train_counts.median(),
            covered_train_counts.quantile(0.90),
            covered_train_counts.quantile(0.99),
            covered_train_counts.max(),
        ],
    }
)

bureau_coverage_summary.round(2)

In [ ]:
bureau_presence_analysis = application_train[
    [
        "SK_ID_CURR",
        "TARGET",
    ]
].copy()

bureau_presence_analysis[
    "BUREAU_RECORD_COUNT"
] = (
    bureau_presence_analysis[
        "SK_ID_CURR"
    ]
    .map(bureau_records_per_customer)
    .fillna(0)
    .astype("int16")
)

bureau_presence_analysis[
    "HAS_BUREAU_HISTORY"
] = (
    bureau_presence_analysis[
        "BUREAU_RECORD_COUNT"
    ] > 0
)

bureau_presence_summary = (
    bureau_presence_analysis.groupby(
        "HAS_BUREAU_HISTORY"
    )
    .agg(
        applicants=("SK_ID_CURR", "size"),
        defaults=("TARGET", "sum"),
        default_rate=("TARGET", "mean"),
        median_bureau_records=(
            "BUREAU_RECORD_COUNT",
            "median",
        ),
        mean_bureau_records=(
            "BUREAU_RECORD_COUNT",
            "mean",
        ),
    )
    .reset_index()
)

bureau_presence_summary[
    "applicant_share"
] = (
    bureau_presence_summary["applicants"]
    / len(bureau_presence_analysis)
    * 100
)

bureau_presence_summary[
    "default_rate"
] *= 100

bureau_presence_summary[
    [
        "HAS_BUREAU_HISTORY",
        "applicants",
        "applicant_share",
        "defaults",
        "default_rate",
        "median_bureau_records",
        "mean_bureau_records",
    ]
].round(2)

### Credit Bureau Coverage Finding

Applicants without an available credit bureau history have a default rate of 10.12%, compared with 7.73% among applicants with at least one bureau record. This represents a 2.39 percentage-point difference and approximately 31% higher relative default risk. The absence of bureau information is therefore an informative feature rather than an ordinary missing value. However, it may represent either limited credit history or incomplete bureau coverage and should not be interpreted causally.


In [ ]:
bureau_column_profile = pd.DataFrame(
    {
        "dtype": bureau.dtypes.astype(str),
        "missing_count": bureau.isna().sum(),
        "missing_percent": (
            bureau.isna().mean() * 100
        ),
        "unique_values": bureau.nunique(
            dropna=True
        ),
    }
)

bureau_column_profile = (
    bureau_column_profile.reset_index()
    .rename(
        columns={"index": "feature"}
    )
    .sort_values(
        "missing_percent",
        ascending=False,
    )
)

bureau_column_profile.round(2)

In [ ]:
quality_check_results = []

def add_quality_check(
    check_name,
    eligible_mask,
    issue_mask,
):
    eligible_count = int(
        eligible_mask.sum()
    )

    issue_count = int(
        (
            eligible_mask
            & issue_mask.fillna(False)
        ).sum()
    )

    issue_percent = (
        issue_count / eligible_count * 100
        if eligible_count > 0
        else np.nan
    )

    quality_check_results.append(
        {
            "check": check_name,
            "eligible_rows": eligible_count,
            "issue_rows": issue_count,
            "issue_percent": issue_percent,
        }
    )

add_quality_check(
    "DAYS_CREDIT is in the future",
    bureau["DAYS_CREDIT"].notna(),
    bureau["DAYS_CREDIT"] > 0,
)

add_quality_check(
    "DAYS_CREDIT_UPDATE is in the future",
    bureau["DAYS_CREDIT_UPDATE"].notna(),
    bureau["DAYS_CREDIT_UPDATE"] > 0,
)

add_quality_check(
    "Actual end date is in the future",
    bureau["DAYS_ENDDATE_FACT"].notna(),
    bureau["DAYS_ENDDATE_FACT"] > 0,
)

add_quality_check(
    "Negative overdue days",
    bureau["CREDIT_DAY_OVERDUE"].notna(),
    bureau["CREDIT_DAY_OVERDUE"] < 0,
)

add_quality_check(
    "Negative credit amount",
    bureau["AMT_CREDIT_SUM"].notna(),
    bureau["AMT_CREDIT_SUM"] < 0,
)

add_quality_check(
    "Negative debt amount",
    bureau["AMT_CREDIT_SUM_DEBT"].notna(),
    bureau["AMT_CREDIT_SUM_DEBT"] < 0,
)

add_quality_check(
    "Negative overdue amount",
    bureau[
        "AMT_CREDIT_SUM_OVERDUE"
    ].notna(),
    bureau[
        "AMT_CREDIT_SUM_OVERDUE"
    ] < 0,
)

add_quality_check(
    "Closed credit missing actual end date",
    bureau["CREDIT_ACTIVE"].eq("Closed"),
    bureau["DAYS_ENDDATE_FACT"].isna(),
)

add_quality_check(
    "Active credit has actual end date",
    bureau["CREDIT_ACTIVE"].eq("Active"),
    bureau["DAYS_ENDDATE_FACT"].notna(),
)

bureau_quality_checks = pd.DataFrame(
    quality_check_results
)

bureau_quality_checks.round(3)

In [ ]:
future_update_records = (
    bureau.loc[
        bureau["DAYS_CREDIT_UPDATE"] > 0,
        [
            "SK_ID_CURR",
            "SK_ID_BUREAU",
            "CREDIT_ACTIVE",
            "CREDIT_TYPE",
            "DAYS_CREDIT",
            "DAYS_CREDIT_UPDATE",
            "DAYS_CREDIT_ENDDATE",
            "DAYS_ENDDATE_FACT",
            "AMT_CREDIT_SUM",
            "AMT_CREDIT_SUM_DEBT",
        ],
    ]
    .sort_values(
        "DAYS_CREDIT_UPDATE",
        ascending=False,
    )
)

future_update_records

In [ ]:
excluded_future_update_count = len(
    future_update_records
)

bureau.drop(
    index=future_update_records.index,
    inplace=True,
)

print(
    "Excluded future-update records:",
    excluded_future_update_count,
)

print(
    "Remaining bureau records:",
    len(bureau),
)

negative_debt_records = bureau.loc[
    bureau["AMT_CREDIT_SUM_DEBT"] < 0
]

negative_debt_summary = (
    negative_debt_records.groupby(
        [
            "CREDIT_ACTIVE",
            "CREDIT_TYPE",
        ],
        dropna=False,
    )
    .agg(
        records=("SK_ID_BUREAU", "size"),
        customers=("SK_ID_CURR", "nunique"),
        median_negative_debt=(
            "AMT_CREDIT_SUM_DEBT",
            "median",
        ),
        minimum_debt=(
            "AMT_CREDIT_SUM_DEBT",
            "min",
        ),
        median_credit_amount=(
            "AMT_CREDIT_SUM",
            "median",
        ),
    )
    .reset_index()
)

negative_debt_summary[
    "share_of_negative_records"
] = (
    negative_debt_summary["records"]
    / len(negative_debt_records)
    * 100
)

negative_debt_summary = (
    negative_debt_summary.sort_values(
        "records",
        ascending=False,
    )
)

negative_debt_summary.head(15).round(2)

### Negative Bureau Debt Treatment

All negative debt balances are associated with credit card records. Approximately 73.91% belong to active credit cards and 26.09% to closed credit cards. The relatively small median negative balances suggest that these values may represent overpayments, positive card balances, refunds, or accounting adjustments rather than invalid debt. The records will therefore be retained. Negative balances will be separated from outstanding debt using a binary indicator and a surplus-balance feature.


In [ ]:
bureau_categorical_features = [
    "CREDIT_ACTIVE",
    "CREDIT_CURRENCY",
    "CREDIT_TYPE",
]

bureau_category_tables = []

for feature in bureau_categorical_features:
    category_summary = (
        bureau.groupby(
            feature,
            dropna=False,
        )
        .agg(
            records=("SK_ID_BUREAU", "size"),
            customers=("SK_ID_CURR", "nunique"),
        )
        .reset_index()
        .rename(
            columns={feature: "category"}
        )
    )

    category_summary["feature"] = feature

    category_summary[
        "record_share"
    ] = (
        category_summary["records"]
        / len(bureau)
        * 100
    )

    bureau_category_tables.append(
        category_summary
    )

bureau_category_distribution = pd.concat(
    bureau_category_tables,
    ignore_index=True,
)

bureau_category_distribution = (
    bureau_category_distribution.sort_values(
        ["feature", "records"],
        ascending=[True, False],
    )
)

bureau_category_distribution[
    [
        "feature",
        "category",
        "records",
        "customers",
        "record_share",
    ]
].round(3)

In [ ]:
memory_before_engineering = (
    bureau.memory_usage(deep=True).sum()
    / 1024**2
)

# Reduce memory usage for categorical columns
for feature in [
    "CREDIT_ACTIVE",
    "CREDIT_CURRENCY",
    "CREDIT_TYPE",
]:
    bureau[feature] = (
        bureau[feature].astype("category")
    )

main_credit_types = [
    "Consumer credit",
    "Credit card",
    "Car loan",
    "Mortgage",
    "Microloan",
]

bureau[
    "CREDIT_TYPE_GROUP"
] = pd.Categorical(
    np.where(
        bureau["CREDIT_TYPE"].isin(
            main_credit_types
        ),
        bureau["CREDIT_TYPE"],
        "Other",
    ),
    categories=main_credit_types + ["Other"],
)

# Negative debt treatment
bureau[
    "BUREAU_NEGATIVE_DEBT_FLAG"
] = (
    bureau["AMT_CREDIT_SUM_DEBT"] < 0
).astype("int8")

bureau[
    "BUREAU_DEBT_NONNEGATIVE"
] = bureau[
    "AMT_CREDIT_SUM_DEBT"
].clip(lower=0)

bureau[
    "BUREAU_BALANCE_SURPLUS"
] = (
    -bureau["AMT_CREDIT_SUM_DEBT"]
).clip(lower=0)

# Overdue status
bureau[
    "BUREAU_OVERDUE_FLAG"
] = (
    (
        bureau["CREDIT_DAY_OVERDUE"] > 0
    )
    | (
        bureau[
            "AMT_CREDIT_SUM_OVERDUE"
        ] > 0
    )
).astype("int8")

# Foreign currency indicator
bureau[
    "BUREAU_FOREIGN_CURRENCY_FLAG"
] = (
    bureau["CREDIT_CURRENCY"]
    != "currency 1"
).astype("int8")

# Credit recency
bureau[
    "BUREAU_CREDIT_RECENCY_DAYS"
] = (
    -bureau["DAYS_CREDIT"]
).astype("int16")

# Debt-to-credit ratio
bureau[
    "BUREAU_DEBT_TO_CREDIT_RATIO"
] = np.where(
    bureau["AMT_CREDIT_SUM"] > 0,
    (
        bureau[
            "BUREAU_DEBT_NONNEGATIVE"
        ]
        / bureau["AMT_CREDIT_SUM"]
    ),
    np.nan,
)

# Information availability indicators
bureau[
    "BUREAU_ANNUITY_AVAILABLE"
] = bureau[
    "AMT_ANNUITY"
].notna().astype("int8")

bureau[
    "BUREAU_MAX_OVERDUE_AVAILABLE"
] = bureau[
    "AMT_CREDIT_MAX_OVERDUE"
].notna().astype("int8")

bureau[
    "BUREAU_LIMIT_AVAILABLE"
] = bureau[
    "AMT_CREDIT_SUM_LIMIT"
].notna().astype("int8")

memory_after_engineering = (
    bureau.memory_usage(deep=True).sum()
    / 1024**2
)

bureau_engineering_summary = pd.DataFrame(
    {
        "metric": [
            "Bureau records",
            "Memory before engineering (MB)",
            "Memory after engineering (MB)",
            "Credit type groups",
            "Negative debt records",
            "Overdue records",
            "Foreign currency records",
            "Valid debt-to-credit ratios",
        ],
        "value": [
            len(bureau),
            memory_before_engineering,
            memory_after_engineering,
            bureau[
                "CREDIT_TYPE_GROUP"
            ].nunique(),
            bureau[
                "BUREAU_NEGATIVE_DEBT_FLAG"
            ].sum(),
            bureau[
                "BUREAU_OVERDUE_FLAG"
            ].sum(),
            bureau[
                "BUREAU_FOREIGN_CURRENCY_FLAG"
            ].sum(),
            bureau[
                "BUREAU_DEBT_TO_CREDIT_RATIO"
            ].notna().sum(),
        ],
    }
)

bureau_engineering_summary.round(2)

In [ ]:
# Additional availability and recency features
bureau[
    "BUREAU_UPDATE_RECENCY_DAYS"
] = (
    -bureau["DAYS_CREDIT_UPDATE"]
).astype("int32")

bureau[
    "BUREAU_DEBT_AVAILABLE"
] = bureau[
    "AMT_CREDIT_SUM_DEBT"
].notna().astype("int8")

# 1. Behaviour and data-availability aggregates
bureau_behavior_features = (
    bureau.groupby(
        "SK_ID_CURR",
        sort=False,
    )
    .agg(
        BUREAU_RECORD_COUNT=(
            "SK_ID_BUREAU",
            "size",
        ),
        BUREAU_CREDIT_RECENCY_MIN_DAYS=(
            "BUREAU_CREDIT_RECENCY_DAYS",
            "min",
        ),
        BUREAU_CREDIT_RECENCY_MEAN_DAYS=(
            "BUREAU_CREDIT_RECENCY_DAYS",
            "mean",
        ),
        BUREAU_CREDIT_RECENCY_MAX_DAYS=(
            "BUREAU_CREDIT_RECENCY_DAYS",
            "max",
        ),
        BUREAU_UPDATE_RECENCY_MIN_DAYS=(
            "BUREAU_UPDATE_RECENCY_DAYS",
            "min",
        ),
        BUREAU_UPDATE_RECENCY_MEAN_DAYS=(
            "BUREAU_UPDATE_RECENCY_DAYS",
            "mean",
        ),
        BUREAU_OVERDUE_RECORD_COUNT=(
            "BUREAU_OVERDUE_FLAG",
            "sum",
        ),
        BUREAU_OVERDUE_RECORD_SHARE=(
            "BUREAU_OVERDUE_FLAG",
            "mean",
        ),
        BUREAU_MAX_DAYS_OVERDUE=(
            "CREDIT_DAY_OVERDUE",
            "max",
        ),
        BUREAU_TOTAL_CREDIT_PROLONG=(
            "CNT_CREDIT_PROLONG",
            "sum",
        ),
        BUREAU_MAX_CREDIT_PROLONG=(
            "CNT_CREDIT_PROLONG",
            "max",
        ),
        BUREAU_NEGATIVE_DEBT_COUNT=(
            "BUREAU_NEGATIVE_DEBT_FLAG",
            "sum",
        ),
        BUREAU_NEGATIVE_DEBT_SHARE=(
            "BUREAU_NEGATIVE_DEBT_FLAG",
            "mean",
        ),
        BUREAU_TOTAL_BALANCE_SURPLUS=(
            "BUREAU_BALANCE_SURPLUS",
            "sum",
        ),
        BUREAU_MAX_BALANCE_SURPLUS=(
            "BUREAU_BALANCE_SURPLUS",
            "max",
        ),
        BUREAU_FOREIGN_CURRENCY_COUNT=(
            "BUREAU_FOREIGN_CURRENCY_FLAG",
            "sum",
        ),
        BUREAU_DEBT_COVERAGE=(
            "BUREAU_DEBT_AVAILABLE",
            "mean",
        ),
        BUREAU_ANNUITY_COVERAGE=(
            "BUREAU_ANNUITY_AVAILABLE",
            "mean",
        ),
        BUREAU_MAX_OVERDUE_COVERAGE=(
            "BUREAU_MAX_OVERDUE_AVAILABLE",
            "mean",
        ),
        BUREAU_LIMIT_COVERAGE=(
            "BUREAU_LIMIT_AVAILABLE",
            "mean",
        ),
    )
)

# 2. Monetary aggregates — currency 1 only
currency_1_mask = (
    bureau["CREDIT_CURRENCY"]
    == "currency 1"
)

bureau_amount_features = (
    bureau.loc[currency_1_mask]
    .groupby(
        "SK_ID_CURR",
        sort=False,
    )
    .agg(
        BUREAU_TOTAL_CREDIT_AMOUNT=(
            "AMT_CREDIT_SUM",
            "sum",
        ),
        BUREAU_MEAN_CREDIT_AMOUNT=(
            "AMT_CREDIT_SUM",
            "mean",
        ),
        BUREAU_MAX_CREDIT_AMOUNT=(
            "AMT_CREDIT_SUM",
            "max",
        ),
        BUREAU_TOTAL_DEBT_AMOUNT=(
            "BUREAU_DEBT_NONNEGATIVE",
            "sum",
        ),
        BUREAU_MEAN_DEBT_AMOUNT=(
            "BUREAU_DEBT_NONNEGATIVE",
            "mean",
        ),
        BUREAU_MAX_DEBT_AMOUNT=(
            "BUREAU_DEBT_NONNEGATIVE",
            "max",
        ),
        BUREAU_TOTAL_OVERDUE_AMOUNT=(
            "AMT_CREDIT_SUM_OVERDUE",
            "sum",
        ),
        BUREAU_MAX_OVERDUE_AMOUNT=(
            "AMT_CREDIT_SUM_OVERDUE",
            "max",
        ),
        BUREAU_MEAN_DEBT_CREDIT_RATIO=(
            "BUREAU_DEBT_TO_CREDIT_RATIO",
            "mean",
        ),
        BUREAU_MAX_DEBT_CREDIT_RATIO=(
            "BUREAU_DEBT_TO_CREDIT_RATIO",
            "max",
        ),
        BUREAU_TOTAL_ANNUITY=(
            "AMT_ANNUITY",
            "sum",
        ),
        BUREAU_MEAN_ANNUITY=(
            "AMT_ANNUITY",
            "mean",
        ),
        BUREAU_TOTAL_CREDIT_LIMIT=(
            "AMT_CREDIT_SUM_LIMIT",
            "sum",
        ),
        BUREAU_MAX_CREDIT_LIMIT=(
            "AMT_CREDIT_SUM_LIMIT",
            "max",
        ),
        BUREAU_MAX_RECORDED_OVERDUE=(
            "AMT_CREDIT_MAX_OVERDUE",
            "max",
        ),
    )
)

# 3. Join at customer grain
bureau_base_features = (
    bureau_behavior_features.join(
        bureau_amount_features,
        how="left",
    )
)

bureau_aggregation_check = pd.DataFrame(
    {
        "metric": [
            "Aggregated customers",
            "Aggregated features",
            "Unique customer index",
            "Duplicate customer index",
            "Source bureau records",
            "Reconciled record count",
            "Count reconciliation difference",
            "Aggregated memory (MB)",
        ],
        "value": [
            len(bureau_base_features),
            bureau_base_features.shape[1],
            bureau_base_features.index.nunique(),
            bureau_base_features.index.duplicated().sum(),
            len(bureau),
            bureau_base_features[
                "BUREAU_RECORD_COUNT"
            ].sum(),
            (
                bureau_base_features[
                    "BUREAU_RECORD_COUNT"
                ].sum()
                - len(bureau)
            ),
            (
                bureau_base_features
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

bureau_aggregation_check.round(2)

In [ ]:
def clean_category_name(value):
    return (
        str(value)
        .upper()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
    )


# CREDIT_ACTIVE counts
status_dummies = pd.get_dummies(
    bureau["CREDIT_ACTIVE"],
    dtype="int8",
)

status_dummies.columns = [
    f"BUREAU_STATUS_{clean_category_name(column)}_COUNT"
    for column in status_dummies.columns
]

status_dummies["SK_ID_CURR"] = (
    bureau["SK_ID_CURR"].to_numpy()
)

bureau_status_features = (
    status_dummies.groupby(
        "SK_ID_CURR",
        sort=False,
    )
    .sum()
    .astype("int16")
)


# CREDIT_TYPE_GROUP counts
type_dummies = pd.get_dummies(
    bureau["CREDIT_TYPE_GROUP"],
    dtype="int8",
)

type_dummies.columns = [
    f"BUREAU_TYPE_{clean_category_name(column)}_COUNT"
    for column in type_dummies.columns
]

type_dummies["SK_ID_CURR"] = (
    bureau["SK_ID_CURR"].to_numpy()
)

bureau_type_features = (
    type_dummies.groupby(
        "SK_ID_CURR",
        sort=False,
    )
    .sum()
    .astype("int16")
)


# Join categorical count features
bureau_base_features = (
    bureau_base_features
    .join(
        bureau_status_features,
        how="left",
    )
    .join(
        bureau_type_features,
        how="left",
    )
)

status_count_columns = list(
    bureau_status_features.columns
)

type_count_columns = list(
    bureau_type_features.columns
)

category_count_columns = (
    status_count_columns
    + type_count_columns
)


# Create category shares
for count_column in category_count_columns:
    share_column = count_column.replace(
        "_COUNT",
        "_SHARE",
    )

    bureau_base_features[
        share_column
    ] = (
        bureau_base_features[count_column]
        / bureau_base_features[
            "BUREAU_RECORD_COUNT"
        ]
    ).astype("float32")


# Reconciliation checks
status_count_difference = (
    bureau_base_features[
        status_count_columns
    ].sum(axis=1)
    - bureau_base_features[
        "BUREAU_RECORD_COUNT"
    ]
)

type_count_difference = (
    bureau_base_features[
        type_count_columns
    ].sum(axis=1)
    - bureau_base_features[
        "BUREAU_RECORD_COUNT"
    ]
)

original_bureau_customer_ids = pd.Index(
    bureau_records_per_customer.index
)

excluded_only_customer_ids = (
    original_bureau_customer_ids.difference(
        bureau_base_features.index
    )
)

excluded_only_train_count = int(
    excluded_only_customer_ids.isin(
        application_train["SK_ID_CURR"]
    ).sum()
)

bureau_category_check = pd.DataFrame(
    {
        "metric": [
            "Aggregated customers",
            "Total bureau features",
            "Duplicate customer index",
            "Status reconciliation errors",
            "Credit type reconciliation errors",
            "Customers removed entirely",
            "Removed train customers",
            "Aggregated memory (MB)",
        ],
        "value": [
            len(bureau_base_features),
            bureau_base_features.shape[1],
            bureau_base_features.index
            .duplicated()
            .sum(),
            status_count_difference
            .ne(0)
            .sum(),
            type_count_difference
            .ne(0)
            .sum(),
            len(excluded_only_customer_ids),
            excluded_only_train_count,
            (
                bureau_base_features
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

del status_dummies
del type_dummies
gc.collect()

bureau_category_check.round(2)

In [ ]:
bureau_features_train = (
    application_train[
        ["SK_ID_CURR"]
    ]
    .merge(
        bureau_base_features.reset_index(),
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one",
    )
)

# Determine history availability before filling counts
bureau_features_train[
    "BUREAU_HAS_HISTORY"
] = (
    bureau_features_train[
        "BUREAU_RECORD_COUNT"
    ]
    .notna()
    .astype("int8")
)

# Only genuine count features are filled with zero
zero_fill_count_columns = [
    "BUREAU_RECORD_COUNT",
    "BUREAU_OVERDUE_RECORD_COUNT",
    "BUREAU_TOTAL_CREDIT_PROLONG",
    "BUREAU_NEGATIVE_DEBT_COUNT",
    "BUREAU_FOREIGN_CURRENCY_COUNT",
] + category_count_columns

for feature in zero_fill_count_columns:
    bureau_features_train[feature] = (
        bureau_features_train[feature]
        .fillna(0)
        .astype("int16")
    )

# Logarithmic version of the skewed record count
bureau_features_train[
    "BUREAU_LOG_RECORD_COUNT"
] = np.log1p(
    bureau_features_train[
        "BUREAU_RECORD_COUNT"
    ]
).astype("float32")

# Keep the bureau ID map for bureau_balance.csv
bureau_id_map = bureau[
    [
        "SK_ID_BUREAU",
        "SK_ID_CURR",
    ]
].copy()

usable_history_count = int(
    bureau_features_train[
        "BUREAU_HAS_HISTORY"
    ].sum()
)

no_usable_history_count = (
    len(bureau_features_train)
    - usable_history_count
)

bureau_train_join_check = pd.DataFrame(
    {
        "metric": [
            "Application train rows",
            "Bureau feature rows",
            "Row count difference",
            "Unique application IDs",
            "Duplicate application IDs",
            "Bureau feature columns",
            "Customers with usable history",
            "Customers without usable history",
            "Time-filtered-only train customers",
            "Bureau ID map rows",
            "Duplicate bureau IDs in map",
            "Feature table memory (MB)",
        ],
        "value": [
            len(application_train),
            len(bureau_features_train),
            (
                len(bureau_features_train)
                - len(application_train)
            ),
            bureau_features_train[
                "SK_ID_CURR"
            ].nunique(),
            bureau_features_train[
                "SK_ID_CURR"
            ].duplicated().sum(),
            (
                bureau_features_train.shape[1]
                - 1
            ),
            usable_history_count,
            no_usable_history_count,
            excluded_only_train_count,
            len(bureau_id_map),
            bureau_id_map[
                "SK_ID_BUREAU"
            ].duplicated().sum(),
            (
                bureau_features_train
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

bureau_train_join_check.round(2)

In [ ]:
from importlib.util import find_spec

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

processed_data_path.mkdir(
    parents=True,
    exist_ok=True,
)

parquet_available = (
    find_spec("pyarrow") is not None
    or find_spec("fastparquet") is not None
)

if parquet_available:
    storage_format = "Parquet"

    bureau_feature_path = (
        processed_data_path
        / "bureau_features_train.parquet"
    )

    bureau_map_path = (
        processed_data_path
        / "bureau_id_map.parquet"
    )

    bureau_features_train.to_parquet(
        bureau_feature_path,
        index=False,
        compression="snappy",
    )

    bureau_id_map.to_parquet(
        bureau_map_path,
        index=False,
        compression="snappy",
    )

    feature_verification = pd.read_parquet(
        bureau_feature_path,
        columns=[
            "SK_ID_CURR",
            "BUREAU_RECORD_COUNT",
        ],
    )

    map_verification = pd.read_parquet(
        bureau_map_path,
        columns=[
            "SK_ID_BUREAU",
            "SK_ID_CURR",
        ],
    )

else:
    storage_format = "Pickle"

    bureau_feature_path = (
        processed_data_path
        / "bureau_features_train.pkl"
    )

    bureau_map_path = (
        processed_data_path
        / "bureau_id_map.pkl"
    )

    bureau_features_train.to_pickle(
        bureau_feature_path
    )

    bureau_id_map.to_pickle(
        bureau_map_path
    )

    feature_verification = (
        pd.read_pickle(
            bureau_feature_path
        )[
            [
                "SK_ID_CURR",
                "BUREAU_RECORD_COUNT",
            ]
        ]
    )

    map_verification = pd.read_pickle(
        bureau_map_path
    )


# Save data-quality audit evidence
future_update_records.to_csv(
    processed_data_path
    / "bureau_future_update_audit.csv",
    index=False,
)

bureau_quality_checks.to_csv(
    processed_data_path
    / "bureau_quality_checks.csv",
    index=False,
)


storage_validation = pd.DataFrame(
    {
        "metric": [
            "Storage format",
            "Feature file",
            "Feature file size (MB)",
            "Verified feature rows",
            "Duplicate feature IDs",
            "ID map file",
            "ID map file size (MB)",
            "Verified ID map rows",
            "Duplicate bureau IDs",
        ],
        "value": [
            storage_format,
            bureau_feature_path.name,
            round(
                bureau_feature_path.stat().st_size
                / 1024**2,
                2,
            ),
            len(feature_verification),
            feature_verification[
                "SK_ID_CURR"
            ].duplicated().sum(),
            bureau_map_path.name,
            round(
                bureau_map_path.stat().st_size
                / 1024**2,
                2,
            ),
            len(map_verification),
            map_verification[
                "SK_ID_BUREAU"
            ].duplicated().sum(),
        ],
    }
)


# Remove large raw and intermediate objects
objects_to_remove = [
    "bureau",
    "bureau_base_features",
    "bureau_behavior_features",
    "bureau_amount_features",
    "bureau_status_features",
    "bureau_type_features",
    "bureau_category_distribution",
    "bureau_category_tables",
    "negative_debt_records",
    "negative_debt_summary",
    "feature_verification",
    "map_verification",
]

for object_name in objects_to_remove:
    globals().pop(object_name, None)

gc.collect()

storage_validation

In [ ]:
bureau_balance = pd.read_csv(
    extract_path / "bureau_balance.csv",
    dtype={
        "SK_ID_BUREAU": "int64",
        "MONTHS_BALANCE": "int16",
        "STATUS": "category",
    },
    memory_map=True,
)

balance_bureau_ids = pd.Index(
    bureau_balance[
        "SK_ID_BUREAU"
    ].unique()
)

mapped_bureau_ids = pd.Index(
    bureau_id_map[
        "SK_ID_BUREAU"
    ]
)

matched_balance_ids = (
    balance_bureau_ids.intersection(
        mapped_bureau_ids
    )
)

unmapped_balance_ids = (
    balance_bureau_ids.difference(
        mapped_bureau_ids
    )
)

unmapped_balance_rows = int(
    bureau_balance[
        "SK_ID_BUREAU"
    ]
    .isin(unmapped_balance_ids)
    .sum()
)

bureau_balance_overview = pd.DataFrame(
    {
        "metric": [
            "Total balance rows",
            "Total columns",
            "Unique bureau IDs",
            "Duplicate bureau-month pairs",
            "Null bureau IDs",
            "Null month values",
            "Null status values",
            "Minimum month",
            "Maximum month",
            "Matched bureau IDs",
            "Unmapped bureau IDs",
            "Unmapped balance rows",
            "Bureau ID coverage (%)",
            "Status categories",
            "Memory usage (MB)",
        ],
        "value": [
            len(bureau_balance),
            bureau_balance.shape[1],
            len(balance_bureau_ids),
            bureau_balance.duplicated(
                subset=[
                    "SK_ID_BUREAU",
                    "MONTHS_BALANCE",
                ]
            ).sum(),
            bureau_balance[
                "SK_ID_BUREAU"
            ].isna().sum(),
            bureau_balance[
                "MONTHS_BALANCE"
            ].isna().sum(),
            bureau_balance[
                "STATUS"
            ].isna().sum(),
            bureau_balance[
                "MONTHS_BALANCE"
            ].min(),
            bureau_balance[
                "MONTHS_BALANCE"
            ].max(),
            len(matched_balance_ids),
            len(unmapped_balance_ids),
            unmapped_balance_rows,
            (
                len(matched_balance_ids)
                / len(mapped_bureau_ids)
                * 100
            ),
            bureau_balance[
                "STATUS"
            ].nunique(),
            (
                bureau_balance
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

bureau_balance_overview.round(2)

In [ ]:
import gc
import pandas as pd

# Filtering öncesindeki büyüklükler
total_balance_rows_before = len(bureau_balance)
total_balance_ids_before = bureau_balance["SK_ID_BUREAU"].nunique()

# Geçerli bureau ID evreni
valid_bureau_ids = pd.Index(
    bureau_id_map["SK_ID_BUREAU"].unique()
)

balance_bureau_ids = pd.Index(
    bureau_balance["SK_ID_BUREAU"].unique()
)

matched_balance_ids = balance_bureau_ids.intersection(valid_bureau_ids)
unmapped_balance_ids = balance_bureau_ids.difference(valid_bureau_ids)

# Satır seviyesinde eşleşme
mapped_balance_mask = bureau_balance["SK_ID_BUREAU"].isin(
    valid_bureau_ids
)

mapped_balance_rows = int(mapped_balance_mask.sum())
unmapped_balance_rows = int((~mapped_balance_mask).sum())

# Eşleşmeyen ID listesini sakla
unmapped_ids_audit = pd.DataFrame({
    "SK_ID_BUREAU": unmapped_balance_ids
})

unmapped_ids_audit.to_parquet(
    processed_data_path / "bureau_balance_unmapped_ids.parquet",
    index=False
)

# Eşleşmeyen kayıtların STATUS dağılımını sakla
unmapped_status_audit = (
    bureau_balance.loc[~mapped_balance_mask]
    .groupby("STATUS", observed=True)
    .agg(
        records=("SK_ID_BUREAU", "size"),
        bureau_ids=("SK_ID_BUREAU", "nunique")
    )
    .reset_index()
)

unmapped_status_audit.to_csv(
    processed_data_path / "bureau_balance_unmapped_status_audit.csv",
    index=False
)

# Eşleşme denetim özeti
mapping_reconciliation = pd.DataFrame({
    "Metric": [
        "Total balance rows before filtering",
        "Mapped balance rows",
        "Unmapped balance rows",
        "Total unique balance bureau IDs",
        "Matched balance bureau IDs",
        "Unmapped balance bureau IDs",
        "Balance ID mapping rate (%)",
        "Balance row mapping rate (%)",
        "Bureau credits with balance history (%)"
    ],
    "Value": [
        total_balance_rows_before,
        mapped_balance_rows,
        unmapped_balance_rows,
        total_balance_ids_before,
        len(matched_balance_ids),
        len(unmapped_balance_ids),
        len(matched_balance_ids) / total_balance_ids_before * 100,
        mapped_balance_rows / total_balance_rows_before * 100,
        len(matched_balance_ids) / len(valid_bureau_ids) * 100
    ]
})

# Sadece eşleşen kayıtlarla devam et
bureau_balance = (
    bureau_balance.loc[mapped_balance_mask]
    .copy()
)

# Kontroller
assert len(bureau_balance) == mapped_balance_rows
assert (
    bureau_balance["SK_ID_BUREAU"]
    .isin(valid_bureau_ids)
    .all()
)

del mapped_balance_mask
gc.collect()

display(mapping_reconciliation.round(2))

In [ ]:
status_meanings = {
    "0": "No delinquency",
    "1": "1–30 days past due",
    "2": "31–60 days past due",
    "3": "61–90 days past due",
    "4": "91–120 days past due",
    "5": "120+ days past due / severe status",
    "C": "Credit closed",
    "X": "Status unknown"
}

status_order = ["0", "1", "2", "3", "4", "5", "C", "X"]
status_order_map = {
    status: order
    for order, status in enumerate(status_order)
}

# Beklenmeyen kategori kontrolü
observed_statuses = {
    str(status)
    for status in bureau_balance["STATUS"].unique()
    if pd.notna(status)
}

unexpected_statuses = observed_statuses - set(status_order)

assert not unexpected_statuses, (
    f"Unexpected STATUS values: {unexpected_statuses}"
)

total_balance_records = len(bureau_balance)
total_balance_bureau_ids = (
    bureau_balance["SK_ID_BUREAU"].nunique()
)

status_summary = (
    bureau_balance
    .groupby("STATUS", observed=True)
    .agg(
        balance_records=("SK_ID_BUREAU", "size"),
        unique_bureau_ids=("SK_ID_BUREAU", "nunique")
    )
    .reset_index()
)

# Sadece sekiz satır olduğu için dönüşüm güvenli
status_summary["STATUS"] = (
    status_summary["STATUS"].astype("string")
)

status_summary["meaning"] = (
    status_summary["STATUS"].map(status_meanings)
)

status_summary["record_share_pct"] = (
    status_summary["balance_records"]
    / total_balance_records
    * 100
)

status_summary["bureau_id_coverage_pct"] = (
    status_summary["unique_bureau_ids"]
    / total_balance_bureau_ids
    * 100
)

status_summary["status_order"] = (
    status_summary["STATUS"].map(status_order_map)
)

status_summary = (
    status_summary
    .sort_values("status_order")
    .drop(columns="status_order")
    .reset_index(drop=True)
)

# Kayıt sayısı uzlaşma kontrolü
assert (
    status_summary["balance_records"].sum()
    == total_balance_records
)

status_summary.to_csv(
    processed_data_path / "bureau_balance_status_summary.csv",
    index=False
)

display(status_summary.round(3))

In [ ]:
import gc
import numpy as np
import pandas as pd

status_order = [
    "0", "1", "2", "3",
    "4", "5", "C", "X",
]

# 1. Credit timeline features
bureau_balance_timeline = (
    bureau_balance
    .groupby(
        "SK_ID_BUREAU",
        sort=False,
        observed=True,
    )
    .agg(
        BB_MONTH_COUNT=(
            "MONTHS_BALANCE",
            "size",
        ),
        BB_EARLIEST_MONTH=(
            "MONTHS_BALANCE",
            "min",
        ),
        BB_LATEST_MONTH=(
            "MONTHS_BALANCE",
            "max",
        ),
    )
)

# 2. Status counts per credit
bureau_balance_status_counts = (
    bureau_balance
    .groupby(
        [
            "SK_ID_BUREAU",
            "STATUS",
        ],
        sort=False,
        observed=True,
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        columns=status_order,
        fill_value=0,
    )
)

bureau_balance_status_counts.columns = [
    f"BB_STATUS_{status}_COUNT"
    for status in status_order
]

bureau_balance_status_counts = (
    bureau_balance_status_counts
    .astype("int16")
)

# 3. Join credit-level features
bureau_balance_credit_features = (
    bureau_balance_timeline.join(
        bureau_balance_status_counts,
        how="left",
        validate="one_to_one",
    )
)

# History span and reporting recency
bureau_balance_credit_features[
    "BB_HISTORY_SPAN_MONTHS"
] = (
    bureau_balance_credit_features[
        "BB_LATEST_MONTH"
    ]
    - bureau_balance_credit_features[
        "BB_EARLIEST_MONTH"
    ]
    + 1
).astype("int16")

bureau_balance_credit_features[
    "BB_MONTHS_SINCE_LATEST_REPORT"
] = (
    -bureau_balance_credit_features[
        "BB_LATEST_MONTH"
    ]
).astype("int16")

# Known risk observations: STATUS 0–5
known_status_columns = [
    f"BB_STATUS_{status}_COUNT"
    for status in ["0", "1", "2", "3", "4", "5"]
]

bureau_balance_credit_features[
    "BB_KNOWN_RISK_MONTH_COUNT"
] = (
    bureau_balance_credit_features[
        known_status_columns
    ]
    .sum(axis=1)
    .astype("int16")
)

# Delinquency counts
bureau_balance_credit_features[
    "BB_ANY_DPD_MONTH_COUNT"
] = (
    bureau_balance_credit_features[
        [
            "BB_STATUS_1_COUNT",
            "BB_STATUS_2_COUNT",
            "BB_STATUS_3_COUNT",
            "BB_STATUS_4_COUNT",
            "BB_STATUS_5_COUNT",
        ]
    ]
    .sum(axis=1)
    .astype("int16")
)

bureau_balance_credit_features[
    "BB_31_PLUS_DPD_MONTH_COUNT"
] = (
    bureau_balance_credit_features[
        [
            "BB_STATUS_2_COUNT",
            "BB_STATUS_3_COUNT",
            "BB_STATUS_4_COUNT",
            "BB_STATUS_5_COUNT",
        ]
    ]
    .sum(axis=1)
    .astype("int16")
)

bureau_balance_credit_features[
    "BB_91_PLUS_DPD_MONTH_COUNT"
] = (
    bureau_balance_credit_features[
        [
            "BB_STATUS_4_COUNT",
            "BB_STATUS_5_COUNT",
        ]
    ]
    .sum(axis=1)
    .astype("int16")
)

bureau_balance_credit_features[
    "BB_120_PLUS_DPD_MONTH_COUNT"
] = bureau_balance_credit_features[
    "BB_STATUS_5_COUNT"
].astype("int16")

# Maximum observed severity
bureau_balance_credit_features[
    "BB_MAX_DPD_SEVERITY"
] = np.select(
    [
        bureau_balance_credit_features[
            "BB_STATUS_5_COUNT"
        ] > 0,
        bureau_balance_credit_features[
            "BB_STATUS_4_COUNT"
        ] > 0,
        bureau_balance_credit_features[
            "BB_STATUS_3_COUNT"
        ] > 0,
        bureau_balance_credit_features[
            "BB_STATUS_2_COUNT"
        ] > 0,
        bureau_balance_credit_features[
            "BB_STATUS_1_COUNT"
        ] > 0,
    ],
    [5, 4, 3, 2, 1],
    default=0,
).astype("int8")

# Shares use only known risk months as denominator
known_risk_denominator = (
    bureau_balance_credit_features[
        "BB_KNOWN_RISK_MONTH_COUNT"
    ]
    .replace(0, np.nan)
)

bureau_balance_credit_features[
    "BB_ANY_DPD_MONTH_SHARE"
] = (
    bureau_balance_credit_features[
        "BB_ANY_DPD_MONTH_COUNT"
    ]
    / known_risk_denominator
).astype("float32")

bureau_balance_credit_features[
    "BB_31_PLUS_DPD_MONTH_SHARE"
] = (
    bureau_balance_credit_features[
        "BB_31_PLUS_DPD_MONTH_COUNT"
    ]
    / known_risk_denominator
).astype("float32")

bureau_balance_credit_features[
    "BB_91_PLUS_DPD_MONTH_SHARE"
] = (
    bureau_balance_credit_features[
        "BB_91_PLUS_DPD_MONTH_COUNT"
    ]
    / known_risk_denominator
).astype("float32")

bureau_balance_credit_features[
    "BB_UNKNOWN_MONTH_SHARE"
] = (
    bureau_balance_credit_features[
        "BB_STATUS_X_COUNT"
    ]
    / bureau_balance_credit_features[
        "BB_MONTH_COUNT"
    ]
).astype("float32")

bureau_balance_credit_features[
    "BB_CLOSED_MONTH_SHARE"
] = (
    bureau_balance_credit_features[
        "BB_STATUS_C_COUNT"
    ]
    / bureau_balance_credit_features[
        "BB_MONTH_COUNT"
    ]
).astype("float32")

# Reconciliation
status_count_columns = [
    f"BB_STATUS_{status}_COUNT"
    for status in status_order
]

status_reconciliation_difference = (
    bureau_balance_credit_features[
        status_count_columns
    ].sum(axis=1)
    - bureau_balance_credit_features[
        "BB_MONTH_COUNT"
    ]
)

bureau_balance_credit_check = pd.DataFrame(
    {
        "metric": [
            "Credit feature rows",
            "Unique credit IDs",
            "Duplicate credit IDs",
            "Total reconciled months",
            "Source balance rows",
            "Status reconciliation errors",
            "Credits with any delinquency",
            "Credits with 91+ day delinquency",
            "Credits with only C/X history",
            "Feature table memory (MB)",
        ],
        "value": [
            len(bureau_balance_credit_features),
            bureau_balance_credit_features
            .index.nunique(),
            bureau_balance_credit_features
            .index.duplicated().sum(),
            bureau_balance_credit_features[
                "BB_MONTH_COUNT"
            ].sum(),
            len(bureau_balance),
            status_reconciliation_difference
            .ne(0).sum(),
            (
                bureau_balance_credit_features[
                    "BB_ANY_DPD_MONTH_COUNT"
                ] > 0
            ).sum(),
            (
                bureau_balance_credit_features[
                    "BB_91_PLUS_DPD_MONTH_COUNT"
                ] > 0
            ).sum(),
            (
                bureau_balance_credit_features[
                    "BB_KNOWN_RISK_MONTH_COUNT"
                ] == 0
            ).sum(),
            (
                bureau_balance_credit_features
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

del bureau_balance_timeline
del bureau_balance_status_counts
gc.collect()

display(
    bureau_balance_credit_check.round(2)
)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import gc

# Restore the ID map when the kernel has restarted
if "processed_data_path" not in globals():
    processed_data_path = Path(
        str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "processed")
    )

if "bureau_id_map" not in globals():
    bureau_id_map = pd.read_parquet(
        processed_data_path
        / "bureau_id_map.parquet"
    )

# Credits in bureau_balance that cannot be mapped
unmapped_credit_feature_ids = (
    bureau_balance_credit_features
    .index
    .difference(
        pd.Index(
            bureau_id_map["SK_ID_BUREAU"]
        )
    )
)

# Connect each credit to its customer
credit_balance_customer_map = (
    bureau_id_map[
        [
            "SK_ID_BUREAU",
            "SK_ID_CURR",
        ]
    ]
    .merge(
        bureau_balance_credit_features
        .reset_index(),
        on="SK_ID_BUREAU",
        how="inner",
        validate="one_to_one",
    )
)

# Credit-level risk indicators
credit_balance_customer_map[
    "BB_CREDIT_ANY_DPD_FLAG"
] = (
    credit_balance_customer_map[
        "BB_ANY_DPD_MONTH_COUNT"
    ] > 0
).astype("int8")

credit_balance_customer_map[
    "BB_CREDIT_31_PLUS_DPD_FLAG"
] = (
    credit_balance_customer_map[
        "BB_31_PLUS_DPD_MONTH_COUNT"
    ] > 0
).astype("int8")

credit_balance_customer_map[
    "BB_CREDIT_91_PLUS_DPD_FLAG"
] = (
    credit_balance_customer_map[
        "BB_91_PLUS_DPD_MONTH_COUNT"
    ] > 0
).astype("int8")

credit_balance_customer_map[
    "BB_CREDIT_120_PLUS_DPD_FLAG"
] = (
    credit_balance_customer_map[
        "BB_120_PLUS_DPD_MONTH_COUNT"
    ] > 0
).astype("int8")

credit_balance_customer_map[
    "BB_CREDIT_ONLY_CX_FLAG"
] = (
    credit_balance_customer_map[
        "BB_KNOWN_RISK_MONTH_COUNT"
    ] == 0
).astype("int8")

# Aggregate credits to customer grain
bureau_balance_customer_features = (
    credit_balance_customer_map
    .groupby(
        "SK_ID_CURR",
        sort=False,
    )
    .agg(
        BB_CREDIT_COUNT=(
            "SK_ID_BUREAU",
            "size",
        ),
        BB_TOTAL_MONTH_COUNT=(
            "BB_MONTH_COUNT",
            "sum",
        ),
        BB_MEAN_MONTHS_PER_CREDIT=(
            "BB_MONTH_COUNT",
            "mean",
        ),
        BB_MAX_HISTORY_SPAN_MONTHS=(
            "BB_HISTORY_SPAN_MONTHS",
            "max",
        ),
        BB_MIN_MONTHS_SINCE_LATEST_REPORT=(
            "BB_MONTHS_SINCE_LATEST_REPORT",
            "min",
        ),
        BB_MAX_MONTHS_SINCE_LATEST_REPORT=(
            "BB_MONTHS_SINCE_LATEST_REPORT",
            "max",
        ),
        BB_MAX_DPD_SEVERITY=(
            "BB_MAX_DPD_SEVERITY",
            "max",
        ),
        BB_CREDITS_WITH_ANY_DPD=(
            "BB_CREDIT_ANY_DPD_FLAG",
            "sum",
        ),
        BB_CREDITS_WITH_31_PLUS_DPD=(
            "BB_CREDIT_31_PLUS_DPD_FLAG",
            "sum",
        ),
        BB_CREDITS_WITH_91_PLUS_DPD=(
            "BB_CREDIT_91_PLUS_DPD_FLAG",
            "sum",
        ),
        BB_CREDITS_WITH_120_PLUS_DPD=(
            "BB_CREDIT_120_PLUS_DPD_FLAG",
            "sum",
        ),
        BB_CREDITS_WITH_ONLY_CX=(
            "BB_CREDIT_ONLY_CX_FLAG",
            "sum",
        ),
        BB_TOTAL_KNOWN_RISK_MONTHS=(
            "BB_KNOWN_RISK_MONTH_COUNT",
            "sum",
        ),
        BB_TOTAL_ANY_DPD_MONTHS=(
            "BB_ANY_DPD_MONTH_COUNT",
            "sum",
        ),
        BB_TOTAL_31_PLUS_DPD_MONTHS=(
            "BB_31_PLUS_DPD_MONTH_COUNT",
            "sum",
        ),
        BB_TOTAL_91_PLUS_DPD_MONTHS=(
            "BB_91_PLUS_DPD_MONTH_COUNT",
            "sum",
        ),
        BB_TOTAL_120_PLUS_DPD_MONTHS=(
            "BB_120_PLUS_DPD_MONTH_COUNT",
            "sum",
        ),
        BB_TOTAL_CLOSED_MONTHS=(
            "BB_STATUS_C_COUNT",
            "sum",
        ),
        BB_TOTAL_UNKNOWN_MONTHS=(
            "BB_STATUS_X_COUNT",
            "sum",
        ),
    )
)

# Customer-level weighted ratios
credit_denominator = (
    bureau_balance_customer_features[
        "BB_CREDIT_COUNT"
    ].replace(0, np.nan)
)

known_month_denominator = (
    bureau_balance_customer_features[
        "BB_TOTAL_KNOWN_RISK_MONTHS"
    ].replace(0, np.nan)
)

total_month_denominator = (
    bureau_balance_customer_features[
        "BB_TOTAL_MONTH_COUNT"
    ].replace(0, np.nan)
)

bureau_balance_customer_features[
    "BB_CREDIT_ANY_DPD_SHARE"
] = (
    bureau_balance_customer_features[
        "BB_CREDITS_WITH_ANY_DPD"
    ]
    / credit_denominator
).astype("float32")

bureau_balance_customer_features[
    "BB_CREDIT_91_PLUS_DPD_SHARE"
] = (
    bureau_balance_customer_features[
        "BB_CREDITS_WITH_91_PLUS_DPD"
    ]
    / credit_denominator
).astype("float32")

bureau_balance_customer_features[
    "BB_ANY_DPD_MONTH_SHARE"
] = (
    bureau_balance_customer_features[
        "BB_TOTAL_ANY_DPD_MONTHS"
    ]
    / known_month_denominator
).astype("float32")

bureau_balance_customer_features[
    "BB_31_PLUS_DPD_MONTH_SHARE"
] = (
    bureau_balance_customer_features[
        "BB_TOTAL_31_PLUS_DPD_MONTHS"
    ]
    / known_month_denominator
).astype("float32")

bureau_balance_customer_features[
    "BB_91_PLUS_DPD_MONTH_SHARE"
] = (
    bureau_balance_customer_features[
        "BB_TOTAL_91_PLUS_DPD_MONTHS"
    ]
    / known_month_denominator
).astype("float32")

bureau_balance_customer_features[
    "BB_CLOSED_MONTH_SHARE"
] = (
    bureau_balance_customer_features[
        "BB_TOTAL_CLOSED_MONTHS"
    ]
    / total_month_denominator
).astype("float32")

bureau_balance_customer_features[
    "BB_UNKNOWN_MONTH_SHARE"
] = (
    bureau_balance_customer_features[
        "BB_TOTAL_UNKNOWN_MONTHS"
    ]
    / total_month_denominator
).astype("float32")

# Reduce memory
integer_columns = (
    bureau_balance_customer_features
    .select_dtypes(include=["int64"])
    .columns
)

for feature in integer_columns:
    bureau_balance_customer_features[
        feature
    ] = pd.to_numeric(
        bureau_balance_customer_features[
            feature
        ],
        downcast="integer",
    )

bureau_balance_customer_features[
    "BB_MEAN_MONTHS_PER_CREDIT"
] = (
    bureau_balance_customer_features[
        "BB_MEAN_MONTHS_PER_CREDIT"
    ].astype("float32")
)

# Reconciliation checks
customer_aggregation_check = pd.DataFrame(
    {
        "metric": [
            "Mapped credit feature rows",
            "Source credit feature rows",
            "Unmapped credit feature IDs",
            "Customer feature rows",
            "Unique customer index",
            "Duplicate customer index",
            "Reconciled credit count",
            "Reconciled month count",
            "Reconciled credits with any DPD",
            "Reconciled credits with 91+ DPD",
            "Customer feature columns",
            "Customer table memory (MB)",
        ],
        "value": [
            len(credit_balance_customer_map),
            len(bureau_balance_credit_features),
            len(unmapped_credit_feature_ids),
            len(bureau_balance_customer_features),
            bureau_balance_customer_features
            .index.nunique(),
            bureau_balance_customer_features
            .index.duplicated().sum(),
            bureau_balance_customer_features[
                "BB_CREDIT_COUNT"
            ].sum(),
            bureau_balance_customer_features[
                "BB_TOTAL_MONTH_COUNT"
            ].sum(),
            bureau_balance_customer_features[
                "BB_CREDITS_WITH_ANY_DPD"
            ].sum(),
            bureau_balance_customer_features[
                "BB_CREDITS_WITH_91_PLUS_DPD"
            ].sum(),
            bureau_balance_customer_features.shape[1],
            (
                bureau_balance_customer_features
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

gc.collect()

display(
    customer_aggregation_check.round(2)
)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import gc

if "processed_data_path" not in globals():
    processed_data_path = Path(
        str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "processed")
    )

# Always start from the clean saved feature table
bureau_features_train_base = (
    pd.read_parquet(
        processed_data_path
        / "bureau_features_train.parquet"
    )
)

# Preserve customer-level balance features separately
bureau_balance_customer_path = (
    processed_data_path
    / "bureau_balance_customer_features.parquet"
)

bureau_balance_customer_features.reset_index().to_parquet(
    bureau_balance_customer_path,
    index=False,
    compression="snappy",
)

# Safe one-to-one merge
bureau_features_train_enriched = (
    bureau_features_train_base.merge(
        bureau_balance_customer_features
        .reset_index(),
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one",
    )
)

# Determine history availability before filling nulls
bureau_features_train_enriched[
    "BB_HAS_HISTORY"
] = (
    bureau_features_train_enriched[
        "BB_CREDIT_COUNT"
    ]
    .notna()
    .astype("int8")
)

# Genuine counts can be zero when no balance history exists
bb_zero_fill_columns = [
    "BB_CREDIT_COUNT",
    "BB_TOTAL_MONTH_COUNT",
    "BB_CREDITS_WITH_ANY_DPD",
    "BB_CREDITS_WITH_31_PLUS_DPD",
    "BB_CREDITS_WITH_91_PLUS_DPD",
    "BB_CREDITS_WITH_120_PLUS_DPD",
    "BB_CREDITS_WITH_ONLY_CX",
    "BB_TOTAL_KNOWN_RISK_MONTHS",
    "BB_TOTAL_ANY_DPD_MONTHS",
    "BB_TOTAL_31_PLUS_DPD_MONTHS",
    "BB_TOTAL_91_PLUS_DPD_MONTHS",
    "BB_TOTAL_120_PLUS_DPD_MONTHS",
    "BB_TOTAL_CLOSED_MONTHS",
    "BB_TOTAL_UNKNOWN_MONTHS",
]

for feature in bb_zero_fill_columns:
    bureau_features_train_enriched[
        feature
    ] = (
        bureau_features_train_enriched[
            feature
        ]
        .fillna(0)
        .astype("int32")
    )

# Availability and ambiguity indicators
bureau_features_train_enriched[
    "BB_HAS_KNOWN_RISK_STATUS"
] = (
    bureau_features_train_enriched[
        "BB_TOTAL_KNOWN_RISK_MONTHS"
    ] > 0
).astype("int8")

bureau_features_train_enriched[
    "BB_ONLY_CX_HISTORY_FLAG"
] = (
    (
        bureau_features_train_enriched[
            "BB_HAS_HISTORY"
        ] == 1
    )
    & (
        bureau_features_train_enriched[
            "BB_TOTAL_KNOWN_RISK_MONTHS"
        ] == 0
    )
).astype("int8")

# Log versions for skewed volume variables
bureau_features_train_enriched[
    "BB_LOG_CREDIT_COUNT"
] = np.log1p(
    bureau_features_train_enriched[
        "BB_CREDIT_COUNT"
    ]
).astype("float32")

bureau_features_train_enriched[
    "BB_LOG_TOTAL_MONTH_COUNT"
] = np.log1p(
    bureau_features_train_enriched[
        "BB_TOTAL_MONTH_COUNT"
    ]
).astype("float32")

# Join and completeness checks
customers_with_balance_history = int(
    bureau_features_train_enriched[
        "BB_HAS_HISTORY"
    ].sum()
)

customers_without_balance_history = (
    len(bureau_features_train_enriched)
    - customers_with_balance_history
)

bureau_without_balance_history = int(
    (
        (
            bureau_features_train_enriched[
                "BUREAU_HAS_HISTORY"
            ] == 1
        )
        & (
            bureau_features_train_enriched[
                "BB_HAS_HISTORY"
            ] == 0
        )
    ).sum()
)

balance_without_bureau_history = int(
    (
        (
            bureau_features_train_enriched[
                "BB_HAS_HISTORY"
            ] == 1
        )
        & (
            bureau_features_train_enriched[
                "BUREAU_HAS_HISTORY"
            ] == 0
        )
    ).sum()
)

# Save enriched train features
enriched_bureau_path = (
    processed_data_path
    / "bureau_features_train_with_balance.parquet"
)

bureau_features_train_enriched.to_parquet(
    enriched_bureau_path,
    index=False,
    compression="snappy",
)

# Read-back verification
enriched_verification = pd.read_parquet(
    enriched_bureau_path,
    columns=[
        "SK_ID_CURR",
        "BUREAU_HAS_HISTORY",
        "BB_HAS_HISTORY",
    ],
)

bureau_balance_join_check = pd.DataFrame(
    {
        "metric": [
            "Base train rows",
            "Enriched train rows",
            "Row count difference",
            "Unique train IDs",
            "Duplicate train IDs",
            "Customers with balance history",
            "Customers without balance history",
            "Bureau history without balance history",
            "Balance history without bureau history",
            "Null values in zero-filled counts",
            "Total enriched feature columns",
            "Verified saved rows",
            "Saved file size (MB)",
            "Enriched memory (MB)",
        ],
        "value": [
            len(bureau_features_train_base),
            len(bureau_features_train_enriched),
            (
                len(bureau_features_train_enriched)
                - len(bureau_features_train_base)
            ),
            bureau_features_train_enriched[
                "SK_ID_CURR"
            ].nunique(),
            bureau_features_train_enriched[
                "SK_ID_CURR"
            ].duplicated().sum(),
            customers_with_balance_history,
            customers_without_balance_history,
            bureau_without_balance_history,
            balance_without_bureau_history,
            bureau_features_train_enriched[
                bb_zero_fill_columns
            ].isna().sum().sum(),
            (
                bureau_features_train_enriched.shape[1]
                - 1
            ),
            len(enriched_verification),
            (
                enriched_bureau_path.stat().st_size
                / 1024**2
            ),
            (
                bureau_features_train_enriched
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

display(
    bureau_balance_join_check.round(2)
)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import gc

if "project_path" not in globals():
    project_path = Path(
        str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
    )

if "extract_path" not in globals():
    extract_path = (
        project_path
        / "data"
        / "raw"
    )

if "processed_data_path" not in globals():
    processed_data_path = (
        project_path
        / "data"
        / "processed"
    )

# Preserve the latest bureau-balance audit
if "bureau_balance_join_check" in globals():
    bureau_balance_join_check.to_csv(
        processed_data_path
        / "bureau_balance_train_join_check.csv",
        index=False,
    )

# Remove only objects already saved or reproducible
objects_to_remove = [
    "bureau_balance",
    "bureau_balance_credit_features",
    "credit_balance_customer_map",
    "bureau_balance_customer_features",
    "bureau_features_train_base",
    "bureau_features_train_enriched",
    "enriched_verification",
    "bureau_id_map",
    "unmapped_credit_feature_ids",
    "status_reconciliation_difference",
]

for object_name in objects_to_remove:
    globals().pop(
        object_name,
        None,
    )

gc.collect()

In [ ]:
previous_application_path = (
    extract_path
    / "previous_application.csv"
)

previous_categorical_columns = [
    "NAME_CONTRACT_TYPE",
    "WEEKDAY_APPR_PROCESS_START",
    "FLAG_LAST_APPL_PER_CONTRACT",
    "NAME_CASH_LOAN_PURPOSE",
    "NAME_CONTRACT_STATUS",
    "NAME_PAYMENT_TYPE",
    "CODE_REJECT_REASON",
    "NAME_TYPE_SUITE",
    "NAME_CLIENT_TYPE",
    "NAME_GOODS_CATEGORY",
    "NAME_PORTFOLIO",
    "NAME_PRODUCT_TYPE",
    "CHANNEL_TYPE",
    "NAME_SELLER_INDUSTRY",
    "NAME_YIELD_GROUP",
    "PRODUCT_COMBINATION",
]

previous_float_columns = [
    "AMT_ANNUITY",
    "AMT_APPLICATION",
    "AMT_CREDIT",
    "AMT_DOWN_PAYMENT",
    "AMT_GOODS_PRICE",
    "RATE_DOWN_PAYMENT",
    "RATE_INTEREST_PRIMARY",
    "RATE_INTEREST_PRIVILEGED",
    "CNT_PAYMENT",
    "DAYS_FIRST_DRAWING",
    "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION",
    "DAYS_LAST_DUE",
    "DAYS_TERMINATION",
    "NFLAG_INSURED_ON_APPROVAL",
]

previous_dtype_map = {
    "SK_ID_PREV": "int32",
    "SK_ID_CURR": "int32",
    "HOUR_APPR_PROCESS_START": "int8",
    "NFLAG_LAST_APPL_IN_DAY": "int8",
    "DAYS_DECISION": "int16",
    "SELLERPLACE_AREA": "int32",
}

previous_dtype_map.update({
    feature: "category"
    for feature in previous_categorical_columns
})

previous_dtype_map.update({
    feature: "float32"
    for feature in previous_float_columns
})

previous_application = pd.read_csv(
    previous_application_path,
    dtype=previous_dtype_map,
    memory_map=True,
)

gc.collect()

In [ ]:
# Train IDs come from the verified feature file
verified_train_ids = pd.Index(
    pd.read_parquet(
        processed_data_path
        / "bureau_features_train_with_balance.parquet",
        columns=["SK_ID_CURR"],
    )["SK_ID_CURR"]
)

previous_customer_ids = pd.Index(
    previous_application[
        "SK_ID_CURR"
    ].unique()
)

matched_train_customer_ids = (
    previous_customer_ids.intersection(
        verified_train_ids
    )
)

train_previous_mask = (
    previous_application[
        "SK_ID_CURR"
    ].isin(verified_train_ids)
)

previous_missing_counts = (
    previous_application.isna().sum()
)

previous_application_overview = pd.DataFrame(
    {
        "metric": [
            "Previous application rows",
            "Total columns",
            "Unique previous application IDs",
            "Duplicate previous application IDs",
            "Unique customer IDs",
            "Null previous application IDs",
            "Null customer IDs",
            "Train-linked records",
            "Train customers with previous applications",
            "Train customer coverage (%)",
            "Columns with missing values",
            "Columns without missing values",
            "Minimum DAYS_DECISION",
            "Maximum DAYS_DECISION",
            "Future DAYS_DECISION records",
            "Categorical columns",
            "Memory usage (MB)",
        ],
        "value": [
            len(previous_application),
            previous_application.shape[1],
            previous_application[
                "SK_ID_PREV"
            ].nunique(),
            previous_application[
                "SK_ID_PREV"
            ].duplicated().sum(),
            previous_application[
                "SK_ID_CURR"
            ].nunique(),
            previous_application[
                "SK_ID_PREV"
            ].isna().sum(),
            previous_application[
                "SK_ID_CURR"
            ].isna().sum(),
            int(train_previous_mask.sum()),
            len(matched_train_customer_ids),
            (
                len(matched_train_customer_ids)
                / len(verified_train_ids)
                * 100
            ),
            int(
                previous_missing_counts
                .gt(0)
                .sum()
            ),
            int(
                previous_missing_counts
                .eq(0)
                .sum()
            ),
            previous_application[
                "DAYS_DECISION"
            ].min(),
            previous_application[
                "DAYS_DECISION"
            ].max(),
            (
                previous_application[
                    "DAYS_DECISION"
                ] > 0
            ).sum(),
            len(
                previous_application.select_dtypes(
                    include="category"
                ).columns
            ),
            (
                previous_application
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

del train_previous_mask
gc.collect()

display(
    previous_application_overview.round(2)
)

In [ ]:
# 1. Missing-value profile
previous_missing_summary = pd.DataFrame(
    {
        "feature": previous_application.columns,
        "dtype": [
            str(previous_application[column].dtype)
            for column in previous_application.columns
        ],
        "missing_count": [
            previous_application[column]
            .isna()
            .sum()
            for column in previous_application.columns
        ],
        "unique_non_null": [
            previous_application[column]
            .nunique(dropna=True)
            for column in previous_application.columns
        ],
    }
)

previous_missing_summary[
    "missing_pct"
] = (
    previous_missing_summary[
        "missing_count"
    ]
    / len(previous_application)
    * 100
)

previous_missing_summary = (
    previous_missing_summary[
        previous_missing_summary[
            "missing_count"
        ] > 0
    ]
    .sort_values(
        "missing_pct",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    previous_missing_summary.round(3)
)

In [ ]:
previous_date_columns = [
    "DAYS_FIRST_DRAWING",
    "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION",
    "DAYS_LAST_DUE",
    "DAYS_TERMINATION",
]

previous_date_quality_rows = []

for feature in previous_date_columns:
    missing_mask = (
        previous_application[feature]
        .isna()
    )

    sentinel_mask = (
        previous_application[feature]
        == 365243
    )

    valid_mask = (
        ~missing_mask
        & ~sentinel_mask
    )

    future_valid_mask = (
        valid_mask
        & (
            previous_application[feature]
            > 0
        )
    )

    valid_values = previous_application.loc[
        valid_mask,
        feature,
    ]

    previous_date_quality_rows.append(
        {
            "feature": feature,
            "missing_count": int(
                missing_mask.sum()
            ),
            "sentinel_365243_count": int(
                sentinel_mask.sum()
            ),
            "total_unavailable": int(
                (
                    missing_mask
                    | sentinel_mask
                ).sum()
            ),
            "unavailable_pct": (
                (
                    missing_mask
                    | sentinel_mask
                ).mean()
                * 100
            ),
            "valid_count": int(
                valid_mask.sum()
            ),
            "minimum_valid_day": (
                valid_values.min()
                if len(valid_values) > 0
                else np.nan
            ),
            "maximum_valid_day": (
                valid_values.max()
                if len(valid_values) > 0
                else np.nan
            ),
            "future_valid_records": int(
                future_valid_mask.sum()
            ),
        }
    )

previous_date_quality = pd.DataFrame(
    previous_date_quality_rows
)

display(
    previous_date_quality.round(3)
)

In [ ]:
previous_status_summary = (
    previous_application
    .groupby(
        "NAME_CONTRACT_STATUS",
        observed=True,
    )
    .agg(
        records=(
            "SK_ID_PREV",
            "size",
        ),
        customers=(
            "SK_ID_CURR",
            "nunique",
        ),
        median_days_since_decision=(
            "DAYS_DECISION",
            "median",
        ),
        mean_credit_amount=(
            "AMT_CREDIT",
            "mean",
        ),
    )
    .reset_index()
)

previous_status_summary[
    "record_share_pct"
] = (
    previous_status_summary[
        "records"
    ]
    / len(previous_application)
    * 100
)

previous_status_summary = (
    previous_status_summary
    .sort_values(
        "records",
        ascending=False,
    )
    .reset_index(drop=True)
)

# Reconciliation
assert (
    previous_status_summary[
        "records"
    ].sum()
    == len(previous_application)
)

# Save audit evidence
previous_missing_summary.to_csv(
    processed_data_path
    / "previous_application_missing_summary.csv",
    index=False,
)

previous_date_quality.to_csv(
    processed_data_path
    / "previous_application_date_quality.csv",
    index=False,
)

previous_status_summary.to_csv(
    processed_data_path
    / "previous_application_status_summary.csv",
    index=False,
)

display(
    previous_status_summary.round(2)
)

In [ ]:
previous_date_availability_names = {
    "DAYS_FIRST_DRAWING":
        "PREV_FIRST_DRAWING_AVAILABLE",
    "DAYS_FIRST_DUE":
        "PREV_FIRST_DUE_AVAILABLE",
    "DAYS_LAST_DUE_1ST_VERSION":
        "PREV_PLANNED_LAST_DUE_AVAILABLE",
    "DAYS_LAST_DUE":
        "PREV_ACTUAL_LAST_DUE_AVAILABLE",
    "DAYS_TERMINATION":
        "PREV_TERMINATION_AVAILABLE",
}

sentinel_value = 365243

sentinel_total_before = int(
    previous_application[
        previous_date_columns
    ]
    .eq(sentinel_value)
    .sum()
    .sum()
)

missing_total_before = int(
    previous_application[
        previous_date_columns
    ]
    .isna()
    .sum()
    .sum()
)

# Apply only when sentinel values still exist
if sentinel_total_before > 0:

    sentinel_matrix = (
        previous_application[
            previous_date_columns
        ].eq(sentinel_value)
    )

    missing_matrix = (
        previous_application[
            previous_date_columns
        ].isna()
    )

    valid_date_matrix = (
        ~missing_matrix
        & ~sentinel_matrix
    )

    # Row-level availability information
    previous_application[
        "PREV_DATE_SENTINEL_COUNT"
    ] = (
        sentinel_matrix
        .sum(axis=1)
        .astype("int8")
    )

    previous_application[
        "PREV_DATE_MISSING_COUNT"
    ] = (
        missing_matrix
        .sum(axis=1)
        .astype("int8")
    )

    previous_application[
        "PREV_DATE_AVAILABLE_COUNT"
    ] = (
        valid_date_matrix
        .sum(axis=1)
        .astype("int8")
    )

    # Availability by date field
    for (
        source_feature,
        availability_feature,
    ) in previous_date_availability_names.items():

        previous_application[
            availability_feature
        ] = (
            valid_date_matrix[
                source_feature
            ]
            .astype("int8")
        )

    # Planned final due date after current application
    previous_application[
        "PREV_PLANNED_END_AFTER_CURRENT_FLAG"
    ] = (
        (
            previous_application[
                "DAYS_LAST_DUE_1ST_VERSION"
            ] > 0
        )
        & (
            previous_application[
                "DAYS_LAST_DUE_1ST_VERSION"
            ] != sentinel_value
        )
    ).astype("int8")

    previous_application[
        "PREV_PLANNED_END_FUTURE_DAYS"
    ] = (
        previous_application[
            "DAYS_LAST_DUE_1ST_VERSION"
        ]
        .where(
            previous_application[
                "DAYS_LAST_DUE_1ST_VERSION"
            ] > 0
        )
        .astype("float32")
    )

    # Replace sentinel with genuine missing values
    for feature in previous_date_columns:
        previous_application[feature] = (
            previous_application[feature]
            .mask(
                previous_application[feature]
                == sentinel_value,
                np.nan,
            )
            .astype("float32")
        )

    del sentinel_matrix
    del missing_matrix
    del valid_date_matrix

else:
    required_engineered_features = [
        "PREV_DATE_SENTINEL_COUNT",
        "PREV_DATE_MISSING_COUNT",
        "PREV_DATE_AVAILABLE_COUNT",
        "PREV_PLANNED_END_AFTER_CURRENT_FLAG",
        "PREV_PLANNED_END_FUTURE_DAYS",
    ] + list(
        previous_date_availability_names.values()
    )

    missing_engineered_features = [
        feature
        for feature in required_engineered_features
        if feature
        not in previous_application.columns
    ]

    if missing_engineered_features:
        raise RuntimeError(
            "Sentinel values are already absent, but "
            "engineered availability columns are missing: "
            f"{missing_engineered_features}"
        )

    print(
        "Date sentinel transformation "
        "was already completed."
    )

In [ ]:
previous_application[
    "PREV_DECISION_RECENCY_DAYS"
] = (
    -previous_application[
        "DAYS_DECISION"
    ]
).astype("int16")

previous_application[
    "PREV_INSURANCE_INFO_AVAILABLE"
] = (
    previous_application[
        "NFLAG_INSURED_ON_APPROVAL"
    ]
    .notna()
    .astype("int8")
)

# Approved/refused/canceled indicators
previous_application[
    "PREV_APPROVED_FLAG"
] = (
    previous_application[
        "NAME_CONTRACT_STATUS"
    ] == "Approved"
).astype("int8")

previous_application[
    "PREV_REFUSED_FLAG"
] = (
    previous_application[
        "NAME_CONTRACT_STATUS"
    ] == "Refused"
).astype("int8")

previous_application[
    "PREV_CANCELED_FLAG"
] = (
    previous_application[
        "NAME_CONTRACT_STATUS"
    ] == "Canceled"
).astype("int8")

previous_application[
    "PREV_UNUSED_OFFER_FLAG"
] = (
    previous_application[
        "NAME_CONTRACT_STATUS"
    ] == "Unused offer"
).astype("int8")

In [ ]:
sentinel_total_after = int(
    previous_application[
        previous_date_columns
    ]
    .eq(sentinel_value)
    .sum()
    .sum()
)

missing_total_after = int(
    previous_application[
        previous_date_columns
    ]
    .isna()
    .sum()
    .sum()
)

positive_date_counts = {
    feature: int(
        (
            previous_application[feature]
            > 0
        ).sum()
    )
    for feature in previous_date_columns
}

previous_date_cleaning_check = pd.DataFrame(
    {
        "metric": [
            "Rows after transformation",
            "Sentinel values before",
            "Sentinel values after",
            "Missing values before",
            "Missing values after",
            "Expected missing values after",
            "Planned future end records",
            "Other positive date records",
            "Memory usage (MB)",
        ],
        "value": [
            len(previous_application),
            sentinel_total_before,
            sentinel_total_after,
            missing_total_before,
            missing_total_after,
            (
                missing_total_before
                + sentinel_total_before
            ),
            positive_date_counts[
                "DAYS_LAST_DUE_1ST_VERSION"
            ],
            sum(
                count
                for feature, count
                in positive_date_counts.items()
                if feature
                != "DAYS_LAST_DUE_1ST_VERSION"
            ),
            (
                previous_application
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

status_lifecycle_summary = (
    previous_application
    .groupby(
        "NAME_CONTRACT_STATUS",
        observed=True,
    )
    .agg(
        records=(
            "SK_ID_PREV",
            "size",
        ),
        mean_available_dates=(
            "PREV_DATE_AVAILABLE_COUNT",
            "mean",
        ),
        first_due_available_pct=(
            "PREV_FIRST_DUE_AVAILABLE",
            "mean",
        ),
        actual_last_due_available_pct=(
            "PREV_ACTUAL_LAST_DUE_AVAILABLE",
            "mean",
        ),
        termination_available_pct=(
            "PREV_TERMINATION_AVAILABLE",
            "mean",
        ),
        insurance_available_pct=(
            "PREV_INSURANCE_INFO_AVAILABLE",
            "mean",
        ),
        planned_end_after_current=(
            "PREV_PLANNED_END_AFTER_CURRENT_FLAG",
            "sum",
        ),
    )
    .reset_index()
)

percentage_columns = [
    "first_due_available_pct",
    "actual_last_due_available_pct",
    "termination_available_pct",
    "insurance_available_pct",
]

status_lifecycle_summary[
    percentage_columns
] = (
    status_lifecycle_summary[
        percentage_columns
    ] * 100
)

previous_date_cleaning_check.to_csv(
    processed_data_path
    / "previous_application_date_cleaning_check.csv",
    index=False,
)

status_lifecycle_summary.to_csv(
    processed_data_path
    / "previous_application_status_lifecycle_summary.csv",
    index=False,
)

display(
    previous_date_cleaning_check.round(2)
)

display(
    status_lifecycle_summary.round(2)
)

In [ ]:
previous_financial_columns = [
    "AMT_APPLICATION",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_DOWN_PAYMENT",
    "AMT_GOODS_PRICE",
    "CNT_PAYMENT",
    "RATE_DOWN_PAYMENT",
    "RATE_INTEREST_PRIMARY",
    "RATE_INTEREST_PRIVILEGED",
]

financial_quality_rows = []

for feature in previous_financial_columns:
    values = previous_application[feature]
    valid_values = values.dropna()

    quantiles = valid_values.quantile(
        [
            0.01,
            0.50,
            0.95,
            0.99,
        ]
    )

    financial_quality_rows.append(
        {
            "feature": feature,
            "missing_count": int(
                values.isna().sum()
            ),
            "missing_pct": (
                values.isna().mean()
                * 100
            ),
            "zero_count": int(
                values.eq(0).sum()
            ),
            "negative_count": int(
                values.lt(0).sum()
            ),
            "minimum": (
                valid_values.min()
                if len(valid_values) > 0
                else np.nan
            ),
            "p01": quantiles.get(
                0.01,
                np.nan,
            ),
            "median": quantiles.get(
                0.50,
                np.nan,
            ),
            "p95": quantiles.get(
                0.95,
                np.nan,
            ),
            "p99": quantiles.get(
                0.99,
                np.nan,
            ),
            "maximum": (
                valid_values.max()
                if len(valid_values) > 0
                else np.nan
            ),
        }
    )

previous_financial_quality = pd.DataFrame(
    financial_quality_rows
)

display(
    previous_financial_quality.round(4)
)

In [ ]:
application_amount = (
    previous_application[
        "AMT_APPLICATION"
    ]
)

credit_amount = (
    previous_application[
        "AMT_CREDIT"
    ]
)

annuity_amount = (
    previous_application[
        "AMT_ANNUITY"
    ]
)

down_payment_amount = (
    previous_application[
        "AMT_DOWN_PAYMENT"
    ]
)

goods_price_amount = (
    previous_application[
        "AMT_GOODS_PRICE"
    ]
)

application_credit_available = (
    application_amount.notna()
    & credit_amount.notna()
)

positive_application_credit = (
    application_amount.gt(0)
    & credit_amount.gt(0)
)

credit_application_ratio = (
    credit_amount
    / application_amount
).where(
    positive_application_credit
)

credit_application_difference = (
    credit_amount
    - application_amount
).where(
    application_credit_available
)

valid_down_payment = (
    down_payment_amount.notna()
    & application_amount.gt(0)
)

valid_annuity_credit = (
    annuity_amount.notna()
    & credit_amount.gt(0)
)

valid_goods_application = (
    goods_price_amount.notna()
    & application_amount.notna()
)

previous_financial_relationship_check = pd.DataFrame(
    {
        "metric": [
            "Application-credit available",
            "Positive application-credit pairs",
            "Zero application amounts",
            "Zero credit amounts",
            "Credit greater than application",
            "Credit lower than application",
            "Credit equal to application",
            "Credit/application ratio p01",
            "Credit/application ratio median",
            "Credit/application ratio p99",
            "Credit/application ratio maximum",
            "Down payment greater than application",
            "Annuity greater than credit",
            "Goods price different from application",
        ],
        "value": [
            int(
                application_credit_available.sum()
            ),
            int(
                positive_application_credit.sum()
            ),
            int(
                application_amount.eq(0).sum()
            ),
            int(
                credit_amount.eq(0).sum()
            ),
            int(
                (
                    credit_application_difference
                    > 0
                ).sum()
            ),
            int(
                (
                    credit_application_difference
                    < 0
                ).sum()
            ),
            int(
                (
                    credit_application_difference
                    == 0
                ).sum()
            ),
            credit_application_ratio.quantile(
                0.01
            ),
            credit_application_ratio.median(),
            credit_application_ratio.quantile(
                0.99
            ),
            credit_application_ratio.max(),
            int(
                (
                    valid_down_payment
                    & (
                        down_payment_amount
                        > application_amount
                    )
                ).sum()
            ),
            int(
                (
                    valid_annuity_credit
                    & (
                        annuity_amount
                        > credit_amount
                    )
                ).sum()
            ),
            int(
                (
                    valid_goods_application
                    & (
                        goods_price_amount
                        != application_amount
                    )
                ).sum()
            ),
        ],
    }
)

display(
    previous_financial_relationship_check
    .round(4)
)

In [ ]:
previous_status_amount_frame = pd.DataFrame(
    {
        "NAME_CONTRACT_STATUS":
            previous_application[
                "NAME_CONTRACT_STATUS"
            ],
        "AMT_APPLICATION":
            application_amount,
        "AMT_CREDIT":
            credit_amount,
        "AMT_ANNUITY":
            annuity_amount,
        "CREDIT_APPLICATION_RATIO":
            credit_application_ratio,
        "ZERO_APPLICATION_FLAG":
            application_amount
            .eq(0)
            .astype("int8"),
        "ZERO_CREDIT_FLAG":
            credit_amount
            .eq(0)
            .astype("int8"),
        "CREDIT_GREATER_FLAG":
            (
                credit_application_difference
                > 0
            ).astype("int8"),
    }
)

previous_status_amount_summary = (
    previous_status_amount_frame
    .groupby(
        "NAME_CONTRACT_STATUS",
        observed=True,
    )
    .agg(
        records=(
            "AMT_APPLICATION",
            "size",
        ),
        zero_application_pct=(
            "ZERO_APPLICATION_FLAG",
            "mean",
        ),
        zero_credit_pct=(
            "ZERO_CREDIT_FLAG",
            "mean",
        ),
        credit_greater_pct=(
            "CREDIT_GREATER_FLAG",
            "mean",
        ),
        median_application=(
            "AMT_APPLICATION",
            "median",
        ),
        median_credit=(
            "AMT_CREDIT",
            "median",
        ),
        median_annuity=(
            "AMT_ANNUITY",
            "median",
        ),
        median_credit_application_ratio=(
            "CREDIT_APPLICATION_RATIO",
            "median",
        ),
    )
    .reset_index()
)

percentage_features = [
    "zero_application_pct",
    "zero_credit_pct",
    "credit_greater_pct",
]

previous_status_amount_summary[
    percentage_features
] = (
    previous_status_amount_summary[
        percentage_features
    ] * 100
)

previous_financial_quality.to_csv(
    processed_data_path
    / "previous_application_financial_quality.csv",
    index=False,
)

previous_financial_relationship_check.to_csv(
    processed_data_path
    / "previous_application_financial_relationship_check.csv",
    index=False,
)

previous_status_amount_summary.to_csv(
    processed_data_path
    / "previous_application_status_amount_summary.csv",
    index=False,
)

del previous_status_amount_frame
gc.collect()

display(
    previous_status_amount_summary.round(3)
)

In [ ]:
# Recalculate safely in case the temporary series was removed
credit_application_ratio = (
    previous_application[
        "AMT_CREDIT"
    ]
    / previous_application[
        "AMT_APPLICATION"
    ]
).where(
    (
        previous_application[
            "AMT_CREDIT"
        ] > 0
    )
    & (
        previous_application[
            "AMT_APPLICATION"
        ] > 0
    )
)

ratio_thresholds = [
    1.5,
    2.0,
    5.0,
    10.0,
    20.0,
]

ratio_tail_rows = []

for threshold in ratio_thresholds:
    threshold_mask = (
        credit_application_ratio
        >= threshold
    )

    ratio_tail_rows.append(
        {
            "ratio_threshold": threshold,
            "records": int(
                threshold_mask.sum()
            ),
            "customers": int(
                previous_application.loc[
                    threshold_mask,
                    "SK_ID_CURR",
                ].nunique()
            ),
            "record_share_pct": (
                threshold_mask.mean()
                * 100
            ),
        }
    )

ratio_tail_summary = pd.DataFrame(
    ratio_tail_rows
)

display(
    ratio_tail_summary.round(5)
)

In [ ]:
ratio_tail_records = (
    previous_application.loc[
        credit_application_ratio >= 2,
        [
            "SK_ID_PREV",
            "SK_ID_CURR",
            "NAME_CONTRACT_STATUS",
            "NAME_CONTRACT_TYPE",
            "NAME_PRODUCT_TYPE",
            "AMT_APPLICATION",
            "AMT_CREDIT",
            "AMT_ANNUITY",
            "AMT_GOODS_PRICE",
            "DAYS_DECISION",
        ],
    ]
    .assign(
        CREDIT_APPLICATION_RATIO=(
            credit_application_ratio[
                credit_application_ratio >= 2
            ]
        )
    )
    .sort_values(
        "CREDIT_APPLICATION_RATIO",
        ascending=False,
    )
)

display(
    ratio_tail_records
    .head(15)
    .round(3)
)

In [ ]:
approved_zero_application_mask = (
    (
        previous_application[
            "NAME_CONTRACT_STATUS"
        ] == "Approved"
    )
    & (
        previous_application[
            "AMT_APPLICATION"
        ] == 0
    )
)

approved_zero_by_contract = (
    previous_application.loc[
        approved_zero_application_mask
    ]
    .groupby(
        "NAME_CONTRACT_TYPE",
        observed=True,
    )
    .agg(
        records=(
            "SK_ID_PREV",
            "size",
        ),
        customers=(
            "SK_ID_CURR",
            "nunique",
        ),
        zero_credit_records=(
            "AMT_CREDIT",
            lambda values: (
                values == 0
            ).sum(),
        ),
        median_credit=(
            "AMT_CREDIT",
            "median",
        ),
        maximum_credit=(
            "AMT_CREDIT",
            "max",
        ),
    )
    .reset_index()
    .sort_values(
        "records",
        ascending=False,
    )
)

approved_zero_by_contract[
    "record_share_pct"
] = (
    approved_zero_by_contract[
        "records"
    ]
    / approved_zero_by_contract[
        "records"
    ].sum()
    * 100
)

display(
    approved_zero_by_contract.round(3)
)

In [ ]:
negative_down_payment_mask = (
    (
        previous_application[
            "AMT_DOWN_PAYMENT"
        ] < 0
    )
    | (
        previous_application[
            "RATE_DOWN_PAYMENT"
        ] < 0
    )
)

negative_down_payment_records = (
    previous_application.loc[
        negative_down_payment_mask,
        [
            "SK_ID_PREV",
            "SK_ID_CURR",
            "NAME_CONTRACT_STATUS",
            "NAME_CONTRACT_TYPE",
            "AMT_APPLICATION",
            "AMT_CREDIT",
            "AMT_DOWN_PAYMENT",
            "RATE_DOWN_PAYMENT",
        ],
    ]
    .copy()
)

ratio_tail_summary.to_csv(
    processed_data_path
    / "previous_application_ratio_tail_summary.csv",
    index=False,
)

ratio_tail_records.head(100).to_csv(
    processed_data_path
    / "previous_application_ratio_tail_audit.csv",
    index=False,
)

approved_zero_by_contract.to_csv(
    processed_data_path
    / "previous_application_approved_zero_by_contract.csv",
    index=False,
)

negative_down_payment_records.to_csv(
    processed_data_path
    / "previous_application_negative_down_payment_audit.csv",
    index=False,
)

display(
    negative_down_payment_records.round(6)
)

In [ ]:
feature_count_before = (
    previous_application.shape[1]
)

# Preserve the original negative-value information
if (
    "PREV_NEGATIVE_DOWN_PAYMENT_FLAG"
    not in previous_application.columns
):
    previous_application[
        "PREV_NEGATIVE_DOWN_PAYMENT_FLAG"
    ] = (
        (
            previous_application[
                "AMT_DOWN_PAYMENT"
            ] < 0
        )
        | (
            previous_application[
                "RATE_DOWN_PAYMENT"
            ] < 0
        )
    ).astype("int8")

# Correct the two floating-point residuals
previous_application[
    "AMT_DOWN_PAYMENT"
] = (
    previous_application[
        "AMT_DOWN_PAYMENT"
    ]
    .clip(lower=0)
    .astype("float32")
)

previous_application[
    "RATE_DOWN_PAYMENT"
] = (
    previous_application[
        "RATE_DOWN_PAYMENT"
    ]
    .clip(lower=0)
    .astype("float32")
)

In [ ]:
financial_availability_map = {
    "AMT_CREDIT":
        "PREV_CREDIT_AVAILABLE",
    "AMT_ANNUITY":
        "PREV_ANNUITY_AVAILABLE",
    "AMT_DOWN_PAYMENT":
        "PREV_DOWN_PAYMENT_AVAILABLE",
    "AMT_GOODS_PRICE":
        "PREV_GOODS_PRICE_AVAILABLE",
    "CNT_PAYMENT":
        "PREV_PAYMENT_COUNT_AVAILABLE",
}

for (
    source_feature,
    availability_feature,
) in financial_availability_map.items():

    previous_application[
        availability_feature
    ] = (
        previous_application[
            source_feature
        ]
        .notna()
        .astype("int8")
    )

previous_application[
    "PREV_ZERO_APPLICATION_FLAG"
] = (
    previous_application[
        "AMT_APPLICATION"
    ] == 0
).astype("int8")

previous_application[
    "PREV_ZERO_CREDIT_FLAG"
] = (
    previous_application[
        "AMT_CREDIT"
    ] == 0
).astype("int8")

previous_application[
    "PREV_ZERO_ANNUITY_FLAG"
] = (
    previous_application[
        "AMT_ANNUITY"
    ] == 0
).astype("int8")

previous_application[
    "PREV_ZERO_PAYMENT_COUNT_FLAG"
] = (
    previous_application[
        "CNT_PAYMENT"
    ] == 0
).astype("int8")

In [ ]:
positive_application_mask = (
    previous_application[
        "AMT_APPLICATION"
    ] > 0
)

positive_credit_mask = (
    previous_application[
        "AMT_CREDIT"
    ] > 0
)

positive_goods_mask = (
    previous_application[
        "AMT_GOODS_PRICE"
    ] > 0
)

raw_credit_application_ratio = (
    previous_application[
        "AMT_CREDIT"
    ]
    / previous_application[
        "AMT_APPLICATION"
    ]
).where(
    positive_application_mask
    & positive_credit_mask
)

previous_application[
    "PREV_CREDIT_APPLICATION_RATIO_EXTREME_FLAG"
] = (
    raw_credit_application_ratio
    > 2.0
).astype("int8")

previous_application[
    "PREV_CREDIT_APPLICATION_RATIO"
] = (
    raw_credit_application_ratio
    .clip(upper=2.0)
    .astype("float32")
)

previous_application[
    "PREV_APPLICATION_CREDIT_DIFFERENCE"
] = (
    previous_application[
        "AMT_CREDIT"
    ]
    - previous_application[
        "AMT_APPLICATION"
    ]
).astype("float32")

previous_application[
    "PREV_ANNUITY_CREDIT_RATIO"
] = (
    previous_application[
        "AMT_ANNUITY"
    ]
    / previous_application[
        "AMT_CREDIT"
    ]
).where(
    positive_credit_mask
).astype("float32")

previous_application[
    "PREV_DOWN_PAYMENT_APPLICATION_RATIO"
] = (
    previous_application[
        "AMT_DOWN_PAYMENT"
    ]
    / previous_application[
        "AMT_APPLICATION"
    ]
).where(
    positive_application_mask
).astype("float32")

previous_application[
    "PREV_CREDIT_GOODS_RATIO"
] = (
    previous_application[
        "AMT_CREDIT"
    ]
    / previous_application[
        "AMT_GOODS_PRICE"
    ]
).where(
    positive_credit_mask
    & positive_goods_mask
).astype("float32")

previous_application[
    "PREV_TOTAL_SCHEDULED_PAYMENT"
] = (
    previous_application[
        "AMT_ANNUITY"
    ]
    * previous_application[
        "CNT_PAYMENT"
    ]
).astype("float32")

previous_application[
    "PREV_SCHEDULED_PAYMENT_CREDIT_RATIO"
] = (
    previous_application[
        "PREV_TOTAL_SCHEDULED_PAYMENT"
    ]
    / previous_application[
        "AMT_CREDIT"
    ]
).where(
    positive_credit_mask
).astype("float32")

In [ ]:
previous_application[
    "PREV_PLANNED_TERM_DAYS"
] = (
    previous_application[
        "DAYS_LAST_DUE_1ST_VERSION"
    ]
    - previous_application[
        "DAYS_FIRST_DUE"
    ]
).astype("float32")

previous_application[
    "PREV_ACTUAL_TERM_DAYS"
] = (
    previous_application[
        "DAYS_LAST_DUE"
    ]
    - previous_application[
        "DAYS_FIRST_DUE"
    ]
).astype("float32")

previous_application[
    "PREV_EARLY_COMPLETION_DAYS"
] = (
    previous_application[
        "DAYS_LAST_DUE_1ST_VERSION"
    ]
    - previous_application[
        "DAYS_TERMINATION"
    ]
).astype("float32")

In [ ]:
dropped_sparse_interest_columns = [
    "RATE_INTEREST_PRIMARY",
    "RATE_INTEREST_PRIVILEGED",
]

previous_application.drop(
    columns=dropped_sparse_interest_columns,
    errors="ignore",
    inplace=True,
)

In [ ]:
previous_row_feature_path = (
    processed_data_path
    / "previous_application_row_features.parquet"
)

previous_application.to_parquet(
    previous_row_feature_path,
    index=False,
    compression="snappy",
)

previous_row_verification = pd.read_parquet(
    previous_row_feature_path,
    columns=[
        "SK_ID_PREV",
        "SK_ID_CURR",
        "PREV_CREDIT_APPLICATION_RATIO",
    ],
)

previous_row_feature_check = pd.DataFrame(
    {
        "metric": [
            "Rows after feature engineering",
            "Unique previous application IDs",
            "Duplicate previous application IDs",
            "Negative down-payment values after",
            "Negative down-payment-rate values after",
            "Valid credit/application ratios",
            "Extreme ratio flags",
            "Maximum capped ratio",
            "Zero application flags",
            "Negative planned term records",
            "Negative actual term records",
            "Features added",
            "Current total columns",
            "Verified saved rows",
            "Saved file size (MB)",
            "Current memory usage (MB)",
        ],
        "value": [
            len(previous_application),
            previous_application[
                "SK_ID_PREV"
            ].nunique(),
            previous_application[
                "SK_ID_PREV"
            ].duplicated().sum(),
            (
                previous_application[
                    "AMT_DOWN_PAYMENT"
                ] < 0
            ).sum(),
            (
                previous_application[
                    "RATE_DOWN_PAYMENT"
                ] < 0
            ).sum(),
            previous_application[
                "PREV_CREDIT_APPLICATION_RATIO"
            ].notna().sum(),
            previous_application[
                "PREV_CREDIT_APPLICATION_RATIO_EXTREME_FLAG"
            ].sum(),
            previous_application[
                "PREV_CREDIT_APPLICATION_RATIO"
            ].max(),
            previous_application[
                "PREV_ZERO_APPLICATION_FLAG"
            ].sum(),
            (
                previous_application[
                    "PREV_PLANNED_TERM_DAYS"
                ] < 0
            ).sum(),
            (
                previous_application[
                    "PREV_ACTUAL_TERM_DAYS"
                ] < 0
            ).sum(),
            (
                previous_application.shape[1]
                - feature_count_before
                + len(
                    dropped_sparse_interest_columns
                )
            ),
            previous_application.shape[1],
            len(previous_row_verification),
            (
                previous_row_feature_path
                .stat()
                .st_size
                / 1024**2
            ),
            (
                previous_application
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

display(
    previous_row_feature_check.round(2)
)

In [ ]:
# 1. All previous applications
previous_customer_base_features = (
    previous_application
    .groupby(
        "SK_ID_CURR",
        sort=False,
    )
    .agg(
        PREV_APPLICATION_COUNT=(
            "SK_ID_PREV",
            "size",
        ),
        PREV_APPROVED_COUNT=(
            "PREV_APPROVED_FLAG",
            "sum",
        ),
        PREV_REFUSED_COUNT=(
            "PREV_REFUSED_FLAG",
            "sum",
        ),
        PREV_CANCELED_COUNT=(
            "PREV_CANCELED_FLAG",
            "sum",
        ),
        PREV_UNUSED_OFFER_COUNT=(
            "PREV_UNUSED_OFFER_FLAG",
            "sum",
        ),
        PREV_DECISION_RECENCY_MIN_DAYS=(
            "PREV_DECISION_RECENCY_DAYS",
            "min",
        ),
        PREV_DECISION_RECENCY_MEAN_DAYS=(
            "PREV_DECISION_RECENCY_DAYS",
            "mean",
        ),
        PREV_DECISION_RECENCY_MAX_DAYS=(
            "PREV_DECISION_RECENCY_DAYS",
            "max",
        ),
        PREV_TOTAL_APPLICATION_AMOUNT=(
            "AMT_APPLICATION",
            "sum",
        ),
        PREV_MEAN_APPLICATION_AMOUNT=(
            "AMT_APPLICATION",
            "mean",
        ),
        PREV_MAX_APPLICATION_AMOUNT=(
            "AMT_APPLICATION",
            "max",
        ),
        PREV_TOTAL_CREDIT_AMOUNT=(
            "AMT_CREDIT",
            "sum",
        ),
        PREV_MEAN_CREDIT_AMOUNT=(
            "AMT_CREDIT",
            "mean",
        ),
        PREV_MAX_CREDIT_AMOUNT=(
            "AMT_CREDIT",
            "max",
        ),
        PREV_MEAN_ANNUITY=(
            "AMT_ANNUITY",
            "mean",
        ),
        PREV_MAX_ANNUITY=(
            "AMT_ANNUITY",
            "max",
        ),
        PREV_MEAN_PAYMENT_COUNT=(
            "CNT_PAYMENT",
            "mean",
        ),
        PREV_MAX_PAYMENT_COUNT=(
            "CNT_PAYMENT",
            "max",
        ),
        PREV_MEAN_CREDIT_APPLICATION_RATIO=(
            "PREV_CREDIT_APPLICATION_RATIO",
            "mean",
        ),
        PREV_MAX_CREDIT_APPLICATION_RATIO=(
            "PREV_CREDIT_APPLICATION_RATIO",
            "max",
        ),
        PREV_MEAN_ANNUITY_CREDIT_RATIO=(
            "PREV_ANNUITY_CREDIT_RATIO",
            "mean",
        ),
        PREV_MEAN_SCHEDULED_PAYMENT_CREDIT_RATIO=(
            "PREV_SCHEDULED_PAYMENT_CREDIT_RATIO",
            "mean",
        ),
        PREV_ZERO_APPLICATION_COUNT=(
            "PREV_ZERO_APPLICATION_FLAG",
            "sum",
        ),
        PREV_ZERO_CREDIT_COUNT=(
            "PREV_ZERO_CREDIT_FLAG",
            "sum",
        ),
        PREV_EXTREME_RATIO_COUNT=(
            "PREV_CREDIT_APPLICATION_RATIO_EXTREME_FLAG",
            "sum",
        ),
        PREV_NEGATIVE_DOWN_PAYMENT_AUDIT_COUNT=(
            "PREV_NEGATIVE_DOWN_PAYMENT_FLAG",
            "sum",
        ),
        PREV_MEAN_DATE_AVAILABLE_COUNT=(
            "PREV_DATE_AVAILABLE_COUNT",
            "mean",
        ),
        PREV_FIRST_DUE_AVAILABLE_SHARE=(
            "PREV_FIRST_DUE_AVAILABLE",
            "mean",
        ),
        PREV_TERMINATION_AVAILABLE_SHARE=(
            "PREV_TERMINATION_AVAILABLE",
            "mean",
        ),
        PREV_INSURANCE_INFO_AVAILABLE_SHARE=(
            "PREV_INSURANCE_INFO_AVAILABLE",
            "mean",
        ),
        PREV_PLANNED_END_AFTER_CURRENT_COUNT=(
            "PREV_PLANNED_END_AFTER_CURRENT_FLAG",
            "sum",
        ),
        PREV_MAX_PLANNED_END_FUTURE_DAYS=(
            "PREV_PLANNED_END_FUTURE_DAYS",
            "max",
        ),
    )
)

In [ ]:
status_count_columns = [
    "PREV_APPROVED_COUNT",
    "PREV_REFUSED_COUNT",
    "PREV_CANCELED_COUNT",
    "PREV_UNUSED_OFFER_COUNT",
]

for count_feature in status_count_columns:
    share_feature = count_feature.replace(
        "_COUNT",
        "_SHARE",
    )

    previous_customer_base_features[
        share_feature
    ] = (
        previous_customer_base_features[
            count_feature
        ]
        / previous_customer_base_features[
            "PREV_APPLICATION_COUNT"
        ]
    ).astype("float32")

In [ ]:
def clean_previous_category_name(value):
    return (
        str(value)
        .upper()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
    )

previous_contract_dummies = pd.get_dummies(
    previous_application[
        "NAME_CONTRACT_TYPE"
    ],
    dtype="int8",
)

previous_contract_dummies.columns = [
    (
        "PREV_CONTRACT_"
        f"{clean_previous_category_name(column)}"
        "_COUNT"
    )
    for column
    in previous_contract_dummies.columns
]

previous_contract_dummies[
    "SK_ID_CURR"
] = (
    previous_application[
        "SK_ID_CURR"
    ].to_numpy()
)

previous_contract_features = (
    previous_contract_dummies
    .groupby(
        "SK_ID_CURR",
        sort=False,
    )
    .sum()
    .astype("int16")
)

previous_customer_base_features = (
    previous_customer_base_features.join(
        previous_contract_features,
        how="left",
        validate="one_to_one",
    )
)

contract_count_columns = list(
    previous_contract_features.columns
)

for count_feature in contract_count_columns:
    share_feature = count_feature.replace(
        "_COUNT",
        "_SHARE",
    )

    previous_customer_base_features[
        share_feature
    ] = (
        previous_customer_base_features[
            count_feature
        ]
        / previous_customer_base_features[
            "PREV_APPLICATION_COUNT"
        ]
    ).astype("float32")

In [ ]:
approved_previous_applications = (
    previous_application.loc[
        previous_application[
            "PREV_APPROVED_FLAG"
        ] == 1
    ]
)

approved_previous_features = (
    approved_previous_applications
    .groupby(
        "SK_ID_CURR",
        sort=False,
    )
    .agg(
        APPROVED_PREV_TOTAL_CREDIT_AMOUNT=(
            "AMT_CREDIT",
            "sum",
        ),
        APPROVED_PREV_MEAN_CREDIT_AMOUNT=(
            "AMT_CREDIT",
            "mean",
        ),
        APPROVED_PREV_MAX_CREDIT_AMOUNT=(
            "AMT_CREDIT",
            "max",
        ),
        APPROVED_PREV_MEAN_APPLICATION_AMOUNT=(
            "AMT_APPLICATION",
            "mean",
        ),
        APPROVED_PREV_MEAN_ANNUITY=(
            "AMT_ANNUITY",
            "mean",
        ),
        APPROVED_PREV_MAX_ANNUITY=(
            "AMT_ANNUITY",
            "max",
        ),
        APPROVED_PREV_MEAN_DOWN_PAYMENT=(
            "AMT_DOWN_PAYMENT",
            "mean",
        ),
        APPROVED_PREV_MEAN_GOODS_PRICE=(
            "AMT_GOODS_PRICE",
            "mean",
        ),
        APPROVED_PREV_MEAN_PAYMENT_COUNT=(
            "CNT_PAYMENT",
            "mean",
        ),
        APPROVED_PREV_MAX_PAYMENT_COUNT=(
            "CNT_PAYMENT",
            "max",
        ),
        APPROVED_PREV_MEAN_CREDIT_APPLICATION_RATIO=(
            "PREV_CREDIT_APPLICATION_RATIO",
            "mean",
        ),
        APPROVED_PREV_MEAN_ANNUITY_CREDIT_RATIO=(
            "PREV_ANNUITY_CREDIT_RATIO",
            "mean",
        ),
        APPROVED_PREV_MEAN_SCHEDULED_PAYMENT_CREDIT_RATIO=(
            "PREV_SCHEDULED_PAYMENT_CREDIT_RATIO",
            "mean",
        ),
        APPROVED_PREV_DECISION_RECENCY_MIN_DAYS=(
            "PREV_DECISION_RECENCY_DAYS",
            "min",
        ),
        APPROVED_PREV_DECISION_RECENCY_MEAN_DAYS=(
            "PREV_DECISION_RECENCY_DAYS",
            "mean",
        ),
        APPROVED_PREV_MEAN_PLANNED_TERM_DAYS=(
            "PREV_PLANNED_TERM_DAYS",
            "mean",
        ),
        APPROVED_PREV_MAX_PLANNED_TERM_DAYS=(
            "PREV_PLANNED_TERM_DAYS",
            "max",
        ),
        APPROVED_PREV_MEAN_ACTUAL_TERM_DAYS=(
            "PREV_ACTUAL_TERM_DAYS",
            "mean",
        ),
        APPROVED_PREV_MAX_ACTUAL_TERM_DAYS=(
            "PREV_ACTUAL_TERM_DAYS",
            "max",
        ),
        APPROVED_PREV_MEAN_EARLY_COMPLETION_DAYS=(
            "PREV_EARLY_COMPLETION_DAYS",
            "mean",
        ),
        APPROVED_PREV_INSURED_SHARE=(
            "NFLAG_INSURED_ON_APPROVAL",
            "mean",
        ),
        APPROVED_PREV_ZERO_APPLICATION_COUNT=(
            "PREV_ZERO_APPLICATION_FLAG",
            "sum",
        ),
        APPROVED_PREV_EXTREME_RATIO_COUNT=(
            "PREV_CREDIT_APPLICATION_RATIO_EXTREME_FLAG",
            "sum",
        ),
        APPROVED_PREV_PLANNED_END_AFTER_CURRENT_COUNT=(
            "PREV_PLANNED_END_AFTER_CURRENT_FLAG",
            "sum",
        ),
    )
)

previous_customer_features = (
    previous_customer_base_features.join(
        approved_previous_features,
        how="left",
        validate="one_to_one",
    )
)

In [ ]:
integer_64_columns = (
    previous_customer_features
    .select_dtypes(include=["int64"])
    .columns
)

for feature in integer_64_columns:
    previous_customer_features[
        feature
    ] = pd.to_numeric(
        previous_customer_features[
            feature
        ],
        downcast="integer",
    )

float_64_columns = (
    previous_customer_features
    .select_dtypes(include=["float64"])
    .columns
)

for feature in float_64_columns:
    previous_customer_features[
        feature
    ] = (
        previous_customer_features[
            feature
        ].astype("float32")
    )

In [ ]:
status_reconciliation_difference = (
    previous_customer_features[
        status_count_columns
    ].sum(axis=1)
    - previous_customer_features[
        "PREV_APPLICATION_COUNT"
    ]
)

contract_reconciliation_difference = (
    previous_customer_features[
        contract_count_columns
    ].sum(axis=1)
    - previous_customer_features[
        "PREV_APPLICATION_COUNT"
    ]
)

previous_customer_feature_path = (
    processed_data_path
    / "previous_application_customer_features.parquet"
)

previous_customer_features.reset_index().to_parquet(
    previous_customer_feature_path,
    index=False,
    compression="snappy",
)

previous_customer_verification = pd.read_parquet(
    previous_customer_feature_path,
    columns=[
        "SK_ID_CURR",
        "PREV_APPLICATION_COUNT",
        "PREV_APPROVED_COUNT",
    ],
)

previous_customer_check = pd.DataFrame(
    {
        "metric": [
            "Customer feature rows",
            "Unique customer index",
            "Duplicate customer index",
            "Reconciled application count",
            "Source application rows",
            "Reconciled approved count",
            "Source approved rows",
            "Status reconciliation errors",
            "Contract reconciliation errors",
            "Customers without approved history",
            "Customer feature columns",
            "Verified saved rows",
            "Saved file size (MB)",
            "Customer table memory (MB)",
        ],
        "value": [
            len(previous_customer_features),
            previous_customer_features
            .index.nunique(),
            previous_customer_features
            .index.duplicated().sum(),
            previous_customer_features[
                "PREV_APPLICATION_COUNT"
            ].sum(),
            len(previous_application),
            previous_customer_features[
                "PREV_APPROVED_COUNT"
            ].sum(),
            len(
                approved_previous_applications
            ),
            status_reconciliation_difference
            .ne(0).sum(),
            contract_reconciliation_difference
            .ne(0).sum(),
            (
                previous_customer_features[
                    "PREV_APPROVED_COUNT"
                ] == 0
            ).sum(),
            previous_customer_features.shape[1],
            len(previous_customer_verification),
            (
                previous_customer_feature_path
                .stat()
                .st_size
                / 1024**2
            ),
            (
                previous_customer_features
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

del previous_contract_dummies
del previous_contract_features
del approved_previous_applications
del approved_previous_features
gc.collect()

display(
    previous_customer_check.round(2)
)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import gc

if "processed_data_path" not in globals():
    processed_data_path = Path(
        str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "processed")
    )

train_base_path = (
    processed_data_path
    / "bureau_features_train_with_balance.parquet"
)

previous_customer_feature_path = (
    processed_data_path
    / "previous_application_customer_features.parquet"
)

# Always load clean saved versions
train_features_base = pd.read_parquet(
    train_base_path
)

previous_train_features = pd.read_parquet(
    previous_customer_feature_path
)

# No unexpected overlapping features should exist
overlapping_feature_columns = (
    train_features_base.columns
    .intersection(
        previous_train_features.columns
    )
    .difference(
        ["SK_ID_CURR"]
    )
)

if len(overlapping_feature_columns) > 0:
    raise ValueError(
        "Unexpected overlapping columns: "
        f"{list(overlapping_feature_columns)}"
    )

# Safe one-to-one left join
train_features_bureau_previous = (
    train_features_base.merge(
        previous_train_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one",
    )
)

# Determine history before filling counts
train_features_bureau_previous[
    "PREV_HAS_HISTORY"
] = (
    train_features_bureau_previous[
        "PREV_APPLICATION_COUNT"
    ]
    .notna()
    .astype("int8")
)

In [ ]:
previous_contract_count_columns = [
    feature
    for feature
    in previous_train_features.columns
    if (
        feature.startswith(
            "PREV_CONTRACT_"
        )
        and feature.endswith(
            "_COUNT"
        )
    )
]

previous_zero_fill_count_columns = [
    "PREV_APPLICATION_COUNT",
    "PREV_APPROVED_COUNT",
    "PREV_REFUSED_COUNT",
    "PREV_CANCELED_COUNT",
    "PREV_UNUSED_OFFER_COUNT",
    "PREV_ZERO_APPLICATION_COUNT",
    "PREV_ZERO_CREDIT_COUNT",
    "PREV_EXTREME_RATIO_COUNT",
    "PREV_NEGATIVE_DOWN_PAYMENT_AUDIT_COUNT",
    "PREV_PLANNED_END_AFTER_CURRENT_COUNT",
    "APPROVED_PREV_ZERO_APPLICATION_COUNT",
    "APPROVED_PREV_EXTREME_RATIO_COUNT",
    "APPROVED_PREV_PLANNED_END_AFTER_CURRENT_COUNT",
] + previous_contract_count_columns

missing_count_features = [
    feature
    for feature
    in previous_zero_fill_count_columns
    if feature
    not in train_features_bureau_previous.columns
]

if missing_count_features:
    raise RuntimeError(
        "Missing expected count features: "
        f"{missing_count_features}"
    )

for feature in previous_zero_fill_count_columns:
    train_features_bureau_previous[
        feature
    ] = (
        train_features_bureau_previous[
            feature
        ]
        .fillna(0)
        .astype("int32")
    )

In [ ]:
train_features_bureau_previous[
    "PREV_HAS_APPROVED_HISTORY"
] = (
    train_features_bureau_previous[
        "PREV_APPROVED_COUNT"
    ] > 0
).astype("int8")

train_features_bureau_previous[
    "PREV_HAS_REFUSED_HISTORY"
] = (
    train_features_bureau_previous[
        "PREV_REFUSED_COUNT"
    ] > 0
).astype("int8")

train_features_bureau_previous[
    "PREV_LOG_APPLICATION_COUNT"
] = np.log1p(
    train_features_bureau_previous[
        "PREV_APPLICATION_COUNT"
    ]
).astype("float32")

train_features_bureau_previous[
    "PREV_LOG_APPROVED_COUNT"
] = np.log1p(
    train_features_bureau_previous[
        "PREV_APPROVED_COUNT"
    ]
).astype("float32")

In [ ]:
train_bureau_previous_path = (
    processed_data_path
    / "application_train_features_bureau_previous.parquet"
)

train_features_bureau_previous.to_parquet(
    train_bureau_previous_path,
    index=False,
    compression="snappy",
)

train_previous_verification = pd.read_parquet(
    train_bureau_previous_path,
    columns=[
        "SK_ID_CURR",
        "BUREAU_HAS_HISTORY",
        "BB_HAS_HISTORY",
        "PREV_HAS_HISTORY",
    ],
)

customers_with_previous_history = int(
    train_features_bureau_previous[
        "PREV_HAS_HISTORY"
    ].sum()
)

customers_without_previous_history = (
    len(train_features_bureau_previous)
    - customers_with_previous_history
)

customers_with_approved_history = int(
    train_features_bureau_previous[
        "PREV_HAS_APPROVED_HISTORY"
    ].sum()
)

previous_history_without_approval = int(
    (
        (
            train_features_bureau_previous[
                "PREV_HAS_HISTORY"
            ] == 1
        )
        & (
            train_features_bureau_previous[
                "PREV_HAS_APPROVED_HISTORY"
            ] == 0
        )
    ).sum()
)

previous_train_join_check = pd.DataFrame(
    {
        "metric": [
            "Base train rows",
            "Final train rows",
            "Row count difference",
            "Unique train IDs",
            "Duplicate train IDs",
            "Unexpected overlapping features",
            "Customers with previous history",
            "Customers without previous history",
            "Customers with approved history",
            "Previous history without approval",
            "Nulls in zero-filled counts",
            "Final feature columns",
            "Verified saved rows",
            "Saved file size (MB)",
            "Final table memory (MB)",
        ],
        "value": [
            len(train_features_base),
            len(train_features_bureau_previous),
            (
                len(train_features_bureau_previous)
                - len(train_features_base)
            ),
            train_features_bureau_previous[
                "SK_ID_CURR"
            ].nunique(),
            train_features_bureau_previous[
                "SK_ID_CURR"
            ].duplicated().sum(),
            len(overlapping_feature_columns),
            customers_with_previous_history,
            customers_without_previous_history,
            customers_with_approved_history,
            previous_history_without_approval,
            train_features_bureau_previous[
                previous_zero_fill_count_columns
            ].isna().sum().sum(),
            (
                train_features_bureau_previous.shape[1]
                - 1
            ),
            len(train_previous_verification),
            (
                train_bureau_previous_path
                .stat()
                .st_size
                / 1024**2
            ),
            (
                train_features_bureau_previous
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

previous_train_join_check.to_csv(
    processed_data_path
    / "previous_application_train_join_check.csv",
    index=False,
)

display(
    previous_train_join_check.round(2)
)

In [ ]:
import gc
from pathlib import Path
import pandas as pd
import numpy as np

objects_to_remove = [
    "previous_application",
    "previous_customer_features",
    "previous_train_features",
    "train_features_base",
    "train_features_bureau_previous",
    "train_previous_verification",
    "previous_customer_verification",
    "credit_application_ratio",
    "raw_credit_application_ratio",
]

for object_name in objects_to_remove:
    globals().pop(
        object_name,
        None,
    )

gc.collect()

In [ ]:
if "project_path" not in globals():
    project_path = Path(
        str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
    )

if "extract_path" not in globals():
    extract_path = (
        project_path
        / "data"
        / "raw"
    )

if "processed_data_path" not in globals():
    processed_data_path = (
        project_path
        / "data"
        / "processed"
    )

pos_cash_balance = pd.read_csv(
    extract_path
    / "POS_CASH_balance.csv",
    dtype={
        "SK_ID_PREV": "int32",
        "SK_ID_CURR": "int32",
        "MONTHS_BALANCE": "int16",
        "CNT_INSTALMENT": "float32",
        "CNT_INSTALMENT_FUTURE": "float32",
        "NAME_CONTRACT_STATUS": "category",
        "SK_DPD": "int32",
        "SK_DPD_DEF": "int32",
    },
    memory_map=True,
)

In [ ]:
previous_id_map = pd.read_parquet(
    processed_data_path
    / "previous_application_row_features.parquet",
    columns=[
        "SK_ID_PREV",
        "SK_ID_CURR",
    ],
)

previous_id_map_check = (
    previous_id_map
    .groupby(
        "SK_ID_PREV",
        sort=False,
    )["SK_ID_CURR"]
    .nunique()
)

conflicting_previous_customer_ids = int(
    (
        previous_id_map_check > 1
    ).sum()
)

valid_previous_ids = pd.Index(
    previous_id_map[
        "SK_ID_PREV"
    ].unique()
)

pos_previous_ids = pd.Index(
    pos_cash_balance[
        "SK_ID_PREV"
    ].unique()
)

matched_pos_previous_ids = (
    pos_previous_ids.intersection(
        valid_previous_ids
    )
)

unmapped_pos_previous_ids = (
    pos_previous_ids.difference(
        valid_previous_ids
    )
)

mapped_pos_mask = (
    pos_cash_balance[
        "SK_ID_PREV"
    ].isin(valid_previous_ids)
)

mapped_pos_rows = int(
    mapped_pos_mask.sum()
)

unmapped_pos_rows = int(
    (~mapped_pos_mask).sum()
)

duplicate_credit_month_pairs = int(
    pos_cash_balance.duplicated(
        subset=[
            "SK_ID_PREV",
            "MONTHS_BALANCE",
        ]
    ).sum()
)

pos_cash_overview = pd.DataFrame(
    {
        "metric": [
            "Total POS CASH rows",
            "Total columns",
            "Unique previous application IDs",
            "Unique customer IDs",
            "Duplicate credit-month pairs",
            "Null previous application IDs",
            "Null customer IDs",
            "Null month values",
            "Minimum month",
            "Maximum month",
            "Future month records",
            "Minimum DPD",
            "Negative DPD records",
            "Minimum DPD DEF",
            "Negative DPD DEF records",
            "Matched previous application IDs",
            "Unmapped previous application IDs",
            "Mapped POS CASH rows",
            "Unmapped POS CASH rows",
            "POS ID mapping rate (%)",
            "POS row mapping rate (%)",
            "Previous applications with POS history (%)",
            "Conflicting previous-customer mappings",
            "Contract status categories",
            "Memory usage (MB)",
        ],
        "value": [
            len(pos_cash_balance),
            pos_cash_balance.shape[1],
            pos_cash_balance[
                "SK_ID_PREV"
            ].nunique(),
            pos_cash_balance[
                "SK_ID_CURR"
            ].nunique(),
            duplicate_credit_month_pairs,
            pos_cash_balance[
                "SK_ID_PREV"
            ].isna().sum(),
            pos_cash_balance[
                "SK_ID_CURR"
            ].isna().sum(),
            pos_cash_balance[
                "MONTHS_BALANCE"
            ].isna().sum(),
            pos_cash_balance[
                "MONTHS_BALANCE"
            ].min(),
            pos_cash_balance[
                "MONTHS_BALANCE"
            ].max(),
            (
                pos_cash_balance[
                    "MONTHS_BALANCE"
                ] > 0
            ).sum(),
            pos_cash_balance[
                "SK_DPD"
            ].min(),
            (
                pos_cash_balance[
                    "SK_DPD"
                ] < 0
            ).sum(),
            pos_cash_balance[
                "SK_DPD_DEF"
            ].min(),
            (
                pos_cash_balance[
                    "SK_DPD_DEF"
                ] < 0
            ).sum(),
            len(matched_pos_previous_ids),
            len(unmapped_pos_previous_ids),
            mapped_pos_rows,
            unmapped_pos_rows,
            (
                len(matched_pos_previous_ids)
                / len(pos_previous_ids)
                * 100
            ),
            (
                mapped_pos_rows
                / len(pos_cash_balance)
                * 100
            ),
            (
                len(matched_pos_previous_ids)
                / len(valid_previous_ids)
                * 100
            ),
            conflicting_previous_customer_ids,
            pos_cash_balance[
                "NAME_CONTRACT_STATUS"
            ].nunique(),
            (
                pos_cash_balance
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

display(
    pos_cash_overview.round(2)
)

In [ ]:
# Unique SK_ID_PREV -> SK_ID_CURR mapping inside POS CASH
pos_id_map = (
    pos_cash_balance[
        [
            "SK_ID_PREV",
            "SK_ID_CURR",
        ]
    ]
    .drop_duplicates()
)

pos_conflicting_previous_ids = int(
    (
        pos_id_map.groupby(
            "SK_ID_PREV",
            sort=False,
        )["SK_ID_CURR"]
        .nunique()
        > 1
    ).sum()
)

if pos_conflicting_previous_ids > 0:
    raise RuntimeError(
        "Some POS SK_ID_PREV values map "
        "to multiple customers."
    )

pos_id_map = (
    pos_id_map
    .drop_duplicates(
        subset="SK_ID_PREV"
    )
)

In [ ]:
previous_id_map_unique = (
    previous_id_map[
        [
            "SK_ID_PREV",
            "SK_ID_CURR",
        ]
    ]
    .drop_duplicates(
        subset="SK_ID_PREV"
    )
)

pos_previous_mapping_check = (
    pos_id_map.merge(
        previous_id_map_unique,
        on="SK_ID_PREV",
        how="left",
        suffixes=(
            "_POS",
            "_PREVIOUS",
        ),
        validate="one_to_one",
    )
)

pos_previous_mapping_check[
    "PREVIOUS_ID_FOUND"
] = (
    pos_previous_mapping_check[
        "SK_ID_CURR_PREVIOUS"
    ].notna()
)

pos_previous_mapping_check[
    "CUSTOMER_MAPPING_CONSISTENT"
] = (
    ~pos_previous_mapping_check[
        "PREVIOUS_ID_FOUND"
    ]
    | (
        pos_previous_mapping_check[
            "SK_ID_CURR_POS"
        ]
        == pos_previous_mapping_check[
            "SK_ID_CURR_PREVIOUS"
        ]
    )
)

inconsistent_customer_mappings = int(
    (
        pos_previous_mapping_check[
            "PREVIOUS_ID_FOUND"
        ]
        & ~pos_previous_mapping_check[
            "CUSTOMER_MAPPING_CONSISTENT"
        ]
    ).sum()
)

In [ ]:
train_application_ids = pd.Index(
    pd.read_parquet(
        processed_data_path
        / "application_train_features_bureau_previous.parquet",
        columns=[
            "SK_ID_CURR",
        ],
    )["SK_ID_CURR"]
)

test_application_ids = pd.Index(
    pd.read_csv(
        extract_path
        / "application_test.csv",
        usecols=[
            "SK_ID_CURR",
        ],
        dtype={
            "SK_ID_CURR": "int32",
        },
    )["SK_ID_CURR"]
)

all_application_ids = (
    train_application_ids.union(
        test_application_ids
    )
)

pos_customer_ids = pd.Index(
    pos_cash_balance[
        "SK_ID_CURR"
    ].unique()
)

orphan_pos_customer_ids = (
    pos_customer_ids.difference(
        all_application_ids
    )
)

orphan_pos_row_count = int(
    pos_cash_balance[
        "SK_ID_CURR"
    ]
    .isin(orphan_pos_customer_ids)
    .sum()
)

In [ ]:
unmapped_previous_mask = (
    ~pos_cash_balance[
        "SK_ID_PREV"
    ].isin(valid_previous_ids)
)

pos_unmapped_previous_audit = (
    pos_cash_balance.loc[
        unmapped_previous_mask
    ]
    .groupby(
        [
            "SK_ID_PREV",
            "SK_ID_CURR",
        ],
        sort=False,
        observed=True,
    )
    .agg(
        records=(
            "MONTHS_BALANCE",
            "size",
        ),
        earliest_month=(
            "MONTHS_BALANCE",
            "min",
        ),
        latest_month=(
            "MONTHS_BALANCE",
            "max",
        ),
        maximum_dpd=(
            "SK_DPD",
            "max",
        ),
    )
    .reset_index()
)

pos_unmapped_previous_audit[
    "CUSTOMER_IN_APPLICATION_UNIVERSE"
] = (
    pos_unmapped_previous_audit[
        "SK_ID_CURR"
    ].isin(all_application_ids)
)

pos_unmapped_previous_audit.to_parquet(
    processed_data_path
    / "pos_cash_unmapped_previous_audit.parquet",
    index=False,
    compression="snappy",
)

In [ ]:
valid_unmapped_previous_ids = int(
    (
        ~pos_previous_mapping_check[
            "PREVIOUS_ID_FOUND"
        ]
        & pos_previous_mapping_check[
            "SK_ID_CURR_POS"
        ].isin(all_application_ids)
    ).sum()
)

invalid_unmapped_previous_ids = int(
    (
        ~pos_previous_mapping_check[
            "PREVIOUS_ID_FOUND"
        ]
        & ~pos_previous_mapping_check[
            "SK_ID_CURR_POS"
        ].isin(all_application_ids)
    ).sum()
)

pos_mapping_reconciliation = pd.DataFrame(
    {
        "metric": [
            "Unique POS previous IDs",
            "Matched previous IDs",
            "Unmapped previous IDs",
            "Consistent matched mappings",
            "Inconsistent matched mappings",
            "Unmapped IDs with valid customer",
            "Unmapped IDs with invalid customer",
            "Unique POS customers",
            "Application universe customers",
            "Orphan POS customers",
            "Orphan POS rows",
            "Rows safe to retain",
            "Rows requiring exclusion",
        ],
        "value": [
            len(pos_previous_mapping_check),
            int(
                pos_previous_mapping_check[
                    "PREVIOUS_ID_FOUND"
                ].sum()
            ),
            int(
                (
                    ~pos_previous_mapping_check[
                        "PREVIOUS_ID_FOUND"
                    ]
                ).sum()
            ),
            int(
                (
                    pos_previous_mapping_check[
                        "PREVIOUS_ID_FOUND"
                    ]
                    & pos_previous_mapping_check[
                        "CUSTOMER_MAPPING_CONSISTENT"
                    ]
                ).sum()
            ),
            inconsistent_customer_mappings,
            valid_unmapped_previous_ids,
            invalid_unmapped_previous_ids,
            len(pos_customer_ids),
            len(all_application_ids),
            len(orphan_pos_customer_ids),
            orphan_pos_row_count,
            (
                len(pos_cash_balance)
                - orphan_pos_row_count
            ),
            orphan_pos_row_count,
        ],
    }
)

pos_mapping_reconciliation.to_csv(
    processed_data_path
    / "pos_cash_mapping_reconciliation.csv",
    index=False,
)

display(
    pos_mapping_reconciliation
)

In [ ]:
pos_cash_balance[
    "POS_DPD_FLAG"
] = (
    pos_cash_balance[
        "SK_DPD"
    ] > 0
).astype("int8")

pos_cash_balance[
    "POS_DPD_DEF_FLAG"
] = (
    pos_cash_balance[
        "SK_DPD_DEF"
    ] > 0
).astype("int8")

pos_cash_balance[
    "POS_DPD_30_PLUS_FLAG"
] = (
    pos_cash_balance[
        "SK_DPD"
    ] >= 30
).astype("int8")

pos_cash_balance[
    "POS_DPD_90_PLUS_FLAG"
] = (
    pos_cash_balance[
        "SK_DPD"
    ] >= 90
).astype("int8")

In [ ]:
pos_status_summary = (
    pos_cash_balance
    .groupby(
        "NAME_CONTRACT_STATUS",
        observed=True,
    )
    .agg(
        records=(
            "SK_ID_PREV",
            "size",
        ),
        previous_application_ids=(
            "SK_ID_PREV",
            "nunique",
        ),
        customers=(
            "SK_ID_CURR",
            "nunique",
        ),
        dpd_records=(
            "POS_DPD_FLAG",
            "sum",
        ),
        dpd_def_records=(
            "POS_DPD_DEF_FLAG",
            "sum",
        ),
        maximum_dpd=(
            "SK_DPD",
            "max",
        ),
        mean_instalment_count=(
            "CNT_INSTALMENT",
            "mean",
        ),
        mean_future_instalment_count=(
            "CNT_INSTALMENT_FUTURE",
            "mean",
        ),
    )
    .reset_index()
)

pos_status_summary[
    "record_share_pct"
] = (
    pos_status_summary[
        "records"
    ]
    / len(pos_cash_balance)
    * 100
)

pos_status_summary[
    "dpd_record_share_pct"
] = (
    pos_status_summary[
        "dpd_records"
    ]
    / pos_status_summary[
        "records"
    ]
    * 100
)

pos_status_summary[
    "dpd_def_record_share_pct"
] = (
    pos_status_summary[
        "dpd_def_records"
    ]
    / pos_status_summary[
        "records"
    ]
    * 100
)

pos_status_summary = (
    pos_status_summary
    .sort_values(
        "records",
        ascending=False,
    )
    .reset_index(drop=True)
)

assert (
    pos_status_summary[
        "records"
    ].sum()
    == len(pos_cash_balance)
)

display(
    pos_status_summary.round(3)
)

In [ ]:
pos_dpd_tail_rows = []

for feature in [
    "SK_DPD",
    "SK_DPD_DEF",
]:
    positive_values = (
        pos_cash_balance.loc[
            pos_cash_balance[
                feature
            ] > 0,
            feature,
        ]
    )

    pos_dpd_tail_rows.append(
        {
            "feature": feature,
            "positive_records": len(
                positive_values
            ),
            "positive_record_share_pct": (
                len(positive_values)
                / len(pos_cash_balance)
                * 100
            ),
            "previous_application_ids": (
                pos_cash_balance.loc[
                    pos_cash_balance[
                        feature
                    ] > 0,
                    "SK_ID_PREV",
                ].nunique()
            ),
            "customers": (
                pos_cash_balance.loc[
                    pos_cash_balance[
                        feature
                    ] > 0,
                    "SK_ID_CURR",
                ].nunique()
            ),
            "positive_median": (
                positive_values.median()
                if len(positive_values) > 0
                else np.nan
            ),
            "positive_p90": (
                positive_values.quantile(
                    0.90
                )
                if len(positive_values) > 0
                else np.nan
            ),
            "positive_p99": (
                positive_values.quantile(
                    0.99
                )
                if len(positive_values) > 0
                else np.nan
            ),
            "maximum": (
                positive_values.max()
                if len(positive_values) > 0
                else np.nan
            ),
        }
    )

pos_dpd_tail_summary = pd.DataFrame(
    pos_dpd_tail_rows
)

display(
    pos_dpd_tail_summary.round(3)
)

In [ ]:
instalment_available_mask = (
    pos_cash_balance[
        "CNT_INSTALMENT"
    ].notna()
    & pos_cash_balance[
        "CNT_INSTALMENT_FUTURE"
    ].notna()
)

completed_status_mask = (
    pos_cash_balance[
        "NAME_CONTRACT_STATUS"
    ] == "Completed"
)

active_status_mask = (
    pos_cash_balance[
        "NAME_CONTRACT_STATUS"
    ] == "Active"
)

pos_cash_quality_check = pd.DataFrame(
    {
        "metric": [
            "Total rows",
            "Missing instalment count",
            "Missing future instalment count",
            "Negative instalment count",
            "Negative future instalment count",
            "Future instalments greater than total",
            "Zero total instalments",
            "Zero future instalments",
            "Positive DPD records",
            "30+ DPD records",
            "90+ DPD records",
            "Positive DPD DEF records",
            "DPD DEF greater than DPD",
            "DPD DEF positive while DPD zero",
            "Completed rows with future instalments",
            "Active rows with zero future instalments",
        ],
        "value": [
            len(pos_cash_balance),
            pos_cash_balance[
                "CNT_INSTALMENT"
            ].isna().sum(),
            pos_cash_balance[
                "CNT_INSTALMENT_FUTURE"
            ].isna().sum(),
            (
                pos_cash_balance[
                    "CNT_INSTALMENT"
                ] < 0
            ).sum(),
            (
                pos_cash_balance[
                    "CNT_INSTALMENT_FUTURE"
                ] < 0
            ).sum(),
            (
                instalment_available_mask
                & (
                    pos_cash_balance[
                        "CNT_INSTALMENT_FUTURE"
                    ]
                    > pos_cash_balance[
                        "CNT_INSTALMENT"
                    ]
                )
            ).sum(),
            (
                pos_cash_balance[
                    "CNT_INSTALMENT"
                ] == 0
            ).sum(),
            (
                pos_cash_balance[
                    "CNT_INSTALMENT_FUTURE"
                ] == 0
            ).sum(),
            pos_cash_balance[
                "POS_DPD_FLAG"
            ].sum(),
            pos_cash_balance[
                "POS_DPD_30_PLUS_FLAG"
            ].sum(),
            pos_cash_balance[
                "POS_DPD_90_PLUS_FLAG"
            ].sum(),
            pos_cash_balance[
                "POS_DPD_DEF_FLAG"
            ].sum(),
            (
                pos_cash_balance[
                    "SK_DPD_DEF"
                ]
                > pos_cash_balance[
                    "SK_DPD"
                ]
            ).sum(),
            (
                (
                    pos_cash_balance[
                        "SK_DPD_DEF"
                    ] > 0
                )
                & (
                    pos_cash_balance[
                        "SK_DPD"
                    ] == 0
                )
            ).sum(),
            (
                completed_status_mask
                & (
                    pos_cash_balance[
                        "CNT_INSTALMENT_FUTURE"
                    ] > 0
                )
            ).sum(),
            (
                active_status_mask
                & (
                    pos_cash_balance[
                        "CNT_INSTALMENT_FUTURE"
                    ] == 0
                )
            ).sum(),
        ],
    }
)

pos_status_summary.to_csv(
    processed_data_path
    / "pos_cash_status_summary.csv",
    index=False,
)

pos_dpd_tail_summary.to_csv(
    processed_data_path
    / "pos_cash_dpd_tail_summary.csv",
    index=False,
)

pos_cash_quality_check.to_csv(
    processed_data_path
    / "pos_cash_quality_check.csv",
    index=False,
)

display(
    pos_cash_quality_check
)

In [ ]:
# Recency: positive number of months before current application
pos_cash_balance[
    "POS_MONTH_RECENCY"
] = (
    -pos_cash_balance[
        "MONTHS_BALANCE"
    ]
).astype("int16")

# Availability indicators
pos_cash_balance[
    "POS_INSTALMENT_AVAILABLE"
] = (
    pos_cash_balance[
        "CNT_INSTALMENT"
    ].notna()
).astype("int8")

pos_cash_balance[
    "POS_FUTURE_INSTALMENT_AVAILABLE"
] = (
    pos_cash_balance[
        "CNT_INSTALMENT_FUTURE"
    ].notna()
).astype("int8")

# Schedule inconsistency indicator
pos_cash_balance[
    "POS_FUTURE_GT_TOTAL_FLAG"
] = (
    (
        pos_cash_balance[
            "CNT_INSTALMENT"
        ].notna()
    )
    & (
        pos_cash_balance[
            "CNT_INSTALMENT_FUTURE"
        ].notna()
    )
    & (
        pos_cash_balance[
            "CNT_INSTALMENT_FUTURE"
        ]
        > pos_cash_balance[
            "CNT_INSTALMENT"
        ]
    )
).astype("int8")

In [ ]:
pos_cash_balance[
    "POS_DPD_EXTREME_FLAG"
] = (
    pos_cash_balance[
        "SK_DPD"
    ] > 2801
).astype("int8")

pos_cash_balance[
    "POS_DPD_DEF_EXTREME_FLAG"
] = (
    pos_cash_balance[
        "SK_DPD_DEF"
    ] > 1940
).astype("int8")

pos_cash_balance[
    "POS_DPD_CAPPED"
] = (
    pos_cash_balance[
        "SK_DPD"
    ]
    .clip(upper=2801)
    .astype("int16")
)

pos_cash_balance[
    "POS_DPD_DEF_CAPPED"
] = (
    pos_cash_balance[
        "SK_DPD_DEF"
    ]
    .clip(upper=1940)
    .astype("int16")
)

In [ ]:
valid_instalment_ratio_mask = (
    pos_cash_balance[
        "CNT_INSTALMENT"
    ].gt(0)
    & pos_cash_balance[
        "CNT_INSTALMENT_FUTURE"
    ].notna()
)

pos_cash_balance[
    "POS_REMAINING_INSTALMENT_RATIO"
] = (
    (
        pos_cash_balance[
            "CNT_INSTALMENT_FUTURE"
        ]
        / pos_cash_balance[
            "CNT_INSTALMENT"
        ]
    )
    .where(
        valid_instalment_ratio_mask
    )
    .clip(
        lower=0,
        upper=1,
    )
    .astype("float32")
)

In [ ]:
pos_credit_behavior_features = (
    pos_cash_balance
    .groupby(
        "SK_ID_PREV",
        sort=False,
        observed=True,
    )
    .agg(
        POS_SK_ID_CURR=(
            "SK_ID_CURR",
            "first",
        ),
        POS_CUSTOMER_NUNIQUE=(
            "SK_ID_CURR",
            "nunique",
        ),
        POS_MONTH_RECORD_COUNT=(
            "MONTHS_BALANCE",
            "size",
        ),
        POS_RECENCY_MIN_MONTHS=(
            "POS_MONTH_RECENCY",
            "min",
        ),
        POS_RECENCY_MEAN_MONTHS=(
            "POS_MONTH_RECENCY",
            "mean",
        ),
        POS_RECENCY_MAX_MONTHS=(
            "POS_MONTH_RECENCY",
            "max",
        ),
        POS_MAX_DPD_RAW=(
            "SK_DPD",
            "max",
        ),
        POS_MAX_DPD_CAPPED=(
            "POS_DPD_CAPPED",
            "max",
        ),
        POS_MEAN_DPD_CAPPED=(
            "POS_DPD_CAPPED",
            "mean",
        ),
        POS_MAX_DPD_DEF_RAW=(
            "SK_DPD_DEF",
            "max",
        ),
        POS_MAX_DPD_DEF_CAPPED=(
            "POS_DPD_DEF_CAPPED",
            "max",
        ),
        POS_MEAN_DPD_DEF_CAPPED=(
            "POS_DPD_DEF_CAPPED",
            "mean",
        ),
        POS_DPD_RECORD_COUNT=(
            "POS_DPD_FLAG",
            "sum",
        ),
        POS_DPD_DEF_RECORD_COUNT=(
            "POS_DPD_DEF_FLAG",
            "sum",
        ),
        POS_DPD_30_PLUS_RECORD_COUNT=(
            "POS_DPD_30_PLUS_FLAG",
            "sum",
        ),
        POS_DPD_90_PLUS_RECORD_COUNT=(
            "POS_DPD_90_PLUS_FLAG",
            "sum",
        ),
        POS_DPD_EXTREME_RECORD_COUNT=(
            "POS_DPD_EXTREME_FLAG",
            "sum",
        ),
        POS_DPD_DEF_EXTREME_RECORD_COUNT=(
            "POS_DPD_DEF_EXTREME_FLAG",
            "sum",
        ),
        POS_MEAN_INSTALMENT_COUNT=(
            "CNT_INSTALMENT",
            "mean",
        ),
        POS_MAX_INSTALMENT_COUNT=(
            "CNT_INSTALMENT",
            "max",
        ),
        POS_MEAN_FUTURE_INSTALMENT_COUNT=(
            "CNT_INSTALMENT_FUTURE",
            "mean",
        ),
        POS_MIN_FUTURE_INSTALMENT_COUNT=(
            "CNT_INSTALMENT_FUTURE",
            "min",
        ),
        POS_MEAN_REMAINING_INSTALMENT_RATIO=(
            "POS_REMAINING_INSTALMENT_RATIO",
            "mean",
        ),
        POS_MIN_REMAINING_INSTALMENT_RATIO=(
            "POS_REMAINING_INSTALMENT_RATIO",
            "min",
        ),
        POS_INSTALMENT_COVERAGE=(
            "POS_INSTALMENT_AVAILABLE",
            "mean",
        ),
        POS_FUTURE_INSTALMENT_COVERAGE=(
            "POS_FUTURE_INSTALMENT_AVAILABLE",
            "mean",
        ),
        POS_FUTURE_GT_TOTAL_COUNT=(
            "POS_FUTURE_GT_TOTAL_FLAG",
            "sum",
        ),
    )
)

In [ ]:
pos_credit_behavior_features[
    "POS_HISTORY_SPAN_MONTHS"
] = (
    pos_credit_behavior_features[
        "POS_RECENCY_MAX_MONTHS"
    ]
    - pos_credit_behavior_features[
        "POS_RECENCY_MIN_MONTHS"
    ]
    + 1
).astype("int16")

pos_credit_behavior_features[
    "POS_LOG_MAX_DPD"
] = np.log1p(
    pos_credit_behavior_features[
        "POS_MAX_DPD_RAW"
    ]
).astype("float32")

pos_credit_behavior_features[
    "POS_LOG_MAX_DPD_DEF"
] = np.log1p(
    pos_credit_behavior_features[
        "POS_MAX_DPD_DEF_RAW"
    ]
).astype("float32")

In [ ]:
def clean_pos_category_name(value):
    return (
        str(value)
        .upper()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
    )

pos_status_categories = [
    str(status)
    for status in pos_cash_balance[
        "NAME_CONTRACT_STATUS"
    ].cat.categories
]

pos_credit_status_counts = (
    pos_cash_balance
    .groupby(
        [
            "SK_ID_PREV",
            "NAME_CONTRACT_STATUS",
        ],
        sort=False,
        observed=True,
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        columns=pos_status_categories,
        fill_value=0,
    )
)

pos_credit_status_counts.columns = [
    (
        "POS_STATUS_"
        f"{clean_pos_category_name(status)}"
        "_COUNT"
    )
    for status
    in pos_status_categories
]

pos_credit_status_counts = (
    pos_credit_status_counts
    .astype("int16")
)

pos_credit_features = (
    pos_credit_behavior_features.join(
        pos_credit_status_counts,
        how="left",
        validate="one_to_one",
    )
)

pos_status_count_columns = list(
    pos_credit_status_counts.columns
)

for count_feature in pos_status_count_columns:
    share_feature = count_feature.replace(
        "_COUNT",
        "_SHARE",
    )

    pos_credit_features[
        share_feature
    ] = (
        pos_credit_features[
            count_feature
        ]
        / pos_credit_features[
            "POS_MONTH_RECORD_COUNT"
        ]
    ).astype("float32")

In [ ]:
pos_dpd_count_columns = [
    "POS_DPD_RECORD_COUNT",
    "POS_DPD_DEF_RECORD_COUNT",
    "POS_DPD_30_PLUS_RECORD_COUNT",
    "POS_DPD_90_PLUS_RECORD_COUNT",
]

for count_feature in pos_dpd_count_columns:
    share_feature = count_feature.replace(
        "_COUNT",
        "_SHARE",
    )

    pos_credit_features[
        share_feature
    ] = (
        pos_credit_features[
            count_feature
        ]
        / pos_credit_features[
            "POS_MONTH_RECORD_COUNT"
        ]
    ).astype("float32")

In [ ]:
pos_status_reconciliation_difference = (
    pos_credit_features[
        pos_status_count_columns
    ].sum(axis=1)
    - pos_credit_features[
        "POS_MONTH_RECORD_COUNT"
    ]
)

pos_credit_feature_path = (
    processed_data_path
    / "pos_cash_credit_features.parquet"
)

pos_credit_features.reset_index().to_parquet(
    pos_credit_feature_path,
    index=False,
    compression="snappy",
)

pos_credit_verification = pd.read_parquet(
    pos_credit_feature_path,
    columns=[
        "SK_ID_PREV",
        "POS_SK_ID_CURR",
        "POS_MONTH_RECORD_COUNT",
    ],
)

pos_credit_feature_check = pd.DataFrame(
    {
        "metric": [
            "Credit feature rows",
            "Unique credit index",
            "Duplicate credit index",
            "Customer mapping conflicts",
            "Reconciled POS rows",
            "Source POS rows",
            "Status reconciliation errors",
            "Reconciled DPD records",
            "Reconciled 30+ DPD records",
            "Reconciled 90+ DPD records",
            "Reconciled DPD DEF records",
            "Reconciled future-greater records",
            "Credit feature columns",
            "Verified saved rows",
            "Saved file size (MB)",
            "Credit table memory (MB)",
        ],
        "value": [
            len(pos_credit_features),
            pos_credit_features
            .index.nunique(),
            pos_credit_features
            .index.duplicated().sum(),
            (
                pos_credit_features[
                    "POS_CUSTOMER_NUNIQUE"
                ] > 1
            ).sum(),
            pos_credit_features[
                "POS_MONTH_RECORD_COUNT"
            ].sum(),
            len(pos_cash_balance),
            pos_status_reconciliation_difference
            .ne(0).sum(),
            pos_credit_features[
                "POS_DPD_RECORD_COUNT"
            ].sum(),
            pos_credit_features[
                "POS_DPD_30_PLUS_RECORD_COUNT"
            ].sum(),
            pos_credit_features[
                "POS_DPD_90_PLUS_RECORD_COUNT"
            ].sum(),
            pos_credit_features[
                "POS_DPD_DEF_RECORD_COUNT"
            ].sum(),
            pos_credit_features[
                "POS_FUTURE_GT_TOTAL_COUNT"
            ].sum(),
            pos_credit_features.shape[1],
            len(pos_credit_verification),
            (
                pos_credit_feature_path
                .stat()
                .st_size
                / 1024**2
            ),
            (
                pos_credit_features
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

display(
    pos_credit_feature_check.round(2)
)

In [ ]:
# Recover the status column list without hidden cell dependency
pos_status_count_columns = [
    feature
    for feature
    in pos_credit_features.columns
    if (
        feature.startswith(
            "POS_STATUS_"
        )
        and feature.endswith(
            "_COUNT"
        )
    )
]

required_pos_status_columns = [
    "POS_STATUS_ACTIVE_COUNT",
    "POS_STATUS_COMPLETED_COUNT",
    "POS_STATUS_DEMAND_COUNT",
    "POS_STATUS_AMORTIZED_DEBT_COUNT",
]

missing_pos_status_columns = [
    feature
    for feature
    in required_pos_status_columns
    if feature
    not in pos_credit_features.columns
]

if missing_pos_status_columns:
    raise RuntimeError(
        "Missing expected POS status columns: "
        f"{missing_pos_status_columns}"
    )

In [ ]:
pos_credit_features[
    "POS_CREDIT_EVER_DPD_FLAG"
] = (
    pos_credit_features[
        "POS_DPD_RECORD_COUNT"
    ] > 0
).astype("int8")

pos_credit_features[
    "POS_CREDIT_EVER_DPD_DEF_FLAG"
] = (
    pos_credit_features[
        "POS_DPD_DEF_RECORD_COUNT"
    ] > 0
).astype("int8")

pos_credit_features[
    "POS_CREDIT_EVER_30_PLUS_DPD_FLAG"
] = (
    pos_credit_features[
        "POS_DPD_30_PLUS_RECORD_COUNT"
    ] > 0
).astype("int8")

pos_credit_features[
    "POS_CREDIT_EVER_90_PLUS_DPD_FLAG"
] = (
    pos_credit_features[
        "POS_DPD_90_PLUS_RECORD_COUNT"
    ] > 0
).astype("int8")

pos_credit_features[
    "POS_CREDIT_EVER_DEMAND_FLAG"
] = (
    pos_credit_features[
        "POS_STATUS_DEMAND_COUNT"
    ] > 0
).astype("int8")

pos_credit_features[
    "POS_CREDIT_EVER_AMORTIZED_DEBT_FLAG"
] = (
    pos_credit_features[
        "POS_STATUS_AMORTIZED_DEBT_COUNT"
    ] > 0
).astype("int8")

pos_credit_features[
    "POS_CREDIT_EVER_COMPLETED_FLAG"
] = (
    pos_credit_features[
        "POS_STATUS_COMPLETED_COUNT"
    ] > 0
).astype("int8")

pos_credit_features[
    "POS_CREDIT_FUTURE_GT_TOTAL_FLAG"
] = (
    pos_credit_features[
        "POS_FUTURE_GT_TOTAL_COUNT"
    ] > 0
).astype("int8")

In [ ]:
pos_customer_base_features = (
    pos_credit_features
    .groupby(
        "POS_SK_ID_CURR",
        sort=False,
    )
    .agg(
        POS_CREDIT_COUNT=(
            "POS_MONTH_RECORD_COUNT",
            "size",
        ),
        POS_TOTAL_MONTH_RECORD_COUNT=(
            "POS_MONTH_RECORD_COUNT",
            "sum",
        ),
        POS_RECENCY_MIN_MONTHS=(
            "POS_RECENCY_MIN_MONTHS",
            "min",
        ),
        POS_RECENCY_MEAN_MONTHS=(
            "POS_RECENCY_MEAN_MONTHS",
            "mean",
        ),
        POS_RECENCY_MAX_MONTHS=(
            "POS_RECENCY_MAX_MONTHS",
            "max",
        ),
        POS_MEAN_HISTORY_SPAN_MONTHS=(
            "POS_HISTORY_SPAN_MONTHS",
            "mean",
        ),
        POS_MAX_HISTORY_SPAN_MONTHS=(
            "POS_HISTORY_SPAN_MONTHS",
            "max",
        ),
        POS_MAX_DPD_RAW=(
            "POS_MAX_DPD_RAW",
            "max",
        ),
        POS_MAX_DPD_CAPPED=(
            "POS_MAX_DPD_CAPPED",
            "max",
        ),
        POS_MEAN_CREDIT_MAX_DPD_CAPPED=(
            "POS_MAX_DPD_CAPPED",
            "mean",
        ),
        POS_MAX_DPD_DEF_RAW=(
            "POS_MAX_DPD_DEF_RAW",
            "max",
        ),
        POS_MAX_DPD_DEF_CAPPED=(
            "POS_MAX_DPD_DEF_CAPPED",
            "max",
        ),
        POS_MAX_LOG_DPD=(
            "POS_LOG_MAX_DPD",
            "max",
        ),
        POS_MAX_LOG_DPD_DEF=(
            "POS_LOG_MAX_DPD_DEF",
            "max",
        ),
        POS_TOTAL_DPD_RECORD_COUNT=(
            "POS_DPD_RECORD_COUNT",
            "sum",
        ),
        POS_TOTAL_DPD_DEF_RECORD_COUNT=(
            "POS_DPD_DEF_RECORD_COUNT",
            "sum",
        ),
        POS_TOTAL_30_PLUS_DPD_RECORD_COUNT=(
            "POS_DPD_30_PLUS_RECORD_COUNT",
            "sum",
        ),
        POS_TOTAL_90_PLUS_DPD_RECORD_COUNT=(
            "POS_DPD_90_PLUS_RECORD_COUNT",
            "sum",
        ),
        POS_TOTAL_DPD_EXTREME_RECORD_COUNT=(
            "POS_DPD_EXTREME_RECORD_COUNT",
            "sum",
        ),
        POS_TOTAL_DPD_DEF_EXTREME_RECORD_COUNT=(
            "POS_DPD_DEF_EXTREME_RECORD_COUNT",
            "sum",
        ),
        POS_CREDITS_WITH_DPD=(
            "POS_CREDIT_EVER_DPD_FLAG",
            "sum",
        ),
        POS_CREDITS_WITH_DPD_DEF=(
            "POS_CREDIT_EVER_DPD_DEF_FLAG",
            "sum",
        ),
        POS_CREDITS_WITH_30_PLUS_DPD=(
            "POS_CREDIT_EVER_30_PLUS_DPD_FLAG",
            "sum",
        ),
        POS_CREDITS_WITH_90_PLUS_DPD=(
            "POS_CREDIT_EVER_90_PLUS_DPD_FLAG",
            "sum",
        ),
        POS_CREDITS_WITH_DEMAND=(
            "POS_CREDIT_EVER_DEMAND_FLAG",
            "sum",
        ),
        POS_CREDITS_WITH_AMORTIZED_DEBT=(
            "POS_CREDIT_EVER_AMORTIZED_DEBT_FLAG",
            "sum",
        ),
        POS_CREDITS_WITH_COMPLETED_STATUS=(
            "POS_CREDIT_EVER_COMPLETED_FLAG",
            "sum",
        ),
        POS_MEAN_INSTALMENT_COUNT=(
            "POS_MEAN_INSTALMENT_COUNT",
            "mean",
        ),
        POS_MAX_INSTALMENT_COUNT=(
            "POS_MAX_INSTALMENT_COUNT",
            "max",
        ),
        POS_MEAN_FUTURE_INSTALMENT_COUNT=(
            "POS_MEAN_FUTURE_INSTALMENT_COUNT",
            "mean",
        ),
        POS_MIN_FUTURE_INSTALMENT_COUNT=(
            "POS_MIN_FUTURE_INSTALMENT_COUNT",
            "min",
        ),
        POS_MEAN_REMAINING_INSTALMENT_RATIO=(
            "POS_MEAN_REMAINING_INSTALMENT_RATIO",
            "mean",
        ),
        POS_MIN_REMAINING_INSTALMENT_RATIO=(
            "POS_MIN_REMAINING_INSTALMENT_RATIO",
            "min",
        ),
        POS_MEAN_INSTALMENT_COVERAGE=(
            "POS_INSTALMENT_COVERAGE",
            "mean",
        ),
        POS_MEAN_FUTURE_INSTALMENT_COVERAGE=(
            "POS_FUTURE_INSTALMENT_COVERAGE",
            "mean",
        ),
        POS_TOTAL_FUTURE_GT_TOTAL_COUNT=(
            "POS_FUTURE_GT_TOTAL_COUNT",
            "sum",
        ),
        POS_CREDITS_WITH_FUTURE_GT_TOTAL=(
            "POS_CREDIT_FUTURE_GT_TOTAL_FLAG",
            "sum",
        ),
    )
)

In [ ]:
pos_customer_status_features = (
    pos_credit_features
    .groupby(
        "POS_SK_ID_CURR",
        sort=False,
    )[
        pos_status_count_columns
    ]
    .sum()
)

pos_customer_features = (
    pos_customer_base_features.join(
        pos_customer_status_features,
        how="left",
        validate="one_to_one",
    )
)

In [ ]:
pos_customer_dpd_count_columns = [
    "POS_TOTAL_DPD_RECORD_COUNT",
    "POS_TOTAL_DPD_DEF_RECORD_COUNT",
    "POS_TOTAL_30_PLUS_DPD_RECORD_COUNT",
    "POS_TOTAL_90_PLUS_DPD_RECORD_COUNT",
]

for count_feature in pos_customer_dpd_count_columns:
    share_feature = count_feature.replace(
        "_COUNT",
        "_SHARE",
    )

    pos_customer_features[
        share_feature
    ] = (
        pos_customer_features[
            count_feature
        ]
        / pos_customer_features[
            "POS_TOTAL_MONTH_RECORD_COUNT"
        ]
    ).astype("float32")

for count_feature in pos_status_count_columns:
    share_feature = count_feature.replace(
        "_COUNT",
        "_SHARE",
    )

    pos_customer_features[
        share_feature
    ] = (
        pos_customer_features[
            count_feature
        ]
        / pos_customer_features[
            "POS_TOTAL_MONTH_RECORD_COUNT"
        ]
    ).astype("float32")

In [ ]:
credit_denominator = (
    pos_customer_features[
        "POS_CREDIT_COUNT"
    ].replace(0, np.nan)
)

pos_customer_features[
    "POS_CREDIT_DPD_SHARE"
] = (
    pos_customer_features[
        "POS_CREDITS_WITH_DPD"
    ]
    / credit_denominator
).astype("float32")

pos_customer_features[
    "POS_CREDIT_30_PLUS_DPD_SHARE"
] = (
    pos_customer_features[
        "POS_CREDITS_WITH_30_PLUS_DPD"
    ]
    / credit_denominator
).astype("float32")

pos_customer_features[
    "POS_CREDIT_90_PLUS_DPD_SHARE"
] = (
    pos_customer_features[
        "POS_CREDITS_WITH_90_PLUS_DPD"
    ]
    / credit_denominator
).astype("float32")

pos_customer_features[
    "POS_LOG_CREDIT_COUNT"
] = np.log1p(
    pos_customer_features[
        "POS_CREDIT_COUNT"
    ]
).astype("float32")

pos_customer_features[
    "POS_LOG_TOTAL_MONTH_RECORD_COUNT"
] = np.log1p(
    pos_customer_features[
        "POS_TOTAL_MONTH_RECORD_COUNT"
    ]
).astype("float32")

In [ ]:
for feature in (
    pos_customer_features
    .select_dtypes(include=["int64"])
    .columns
):
    pos_customer_features[
        feature
    ] = pd.to_numeric(
        pos_customer_features[
            feature
        ],
        downcast="integer",
    )

for feature in (
    pos_customer_features
    .select_dtypes(include=["float64"])
    .columns
):
    pos_customer_features[
        feature
    ] = (
        pos_customer_features[
            feature
        ].astype("float32")
    )

In [ ]:
pos_customer_status_difference = (
    pos_customer_features[
        pos_status_count_columns
    ].sum(axis=1)
    - pos_customer_features[
        "POS_TOTAL_MONTH_RECORD_COUNT"
    ]
)

pos_customer_feature_path = (
    processed_data_path
    / "pos_cash_customer_features.parquet"
)

pos_customer_features.reset_index(
    names="SK_ID_CURR"
).to_parquet(
    pos_customer_feature_path,
    index=False,
    compression="snappy",
)

pos_customer_verification = pd.read_parquet(
    pos_customer_feature_path,
    columns=[
        "SK_ID_CURR",
        "POS_CREDIT_COUNT",
        "POS_TOTAL_MONTH_RECORD_COUNT",
    ],
)

pos_customer_feature_check = pd.DataFrame(
    {
        "metric": [
            "Customer feature rows",
            "Unique customer index",
            "Duplicate customer index",
            "Reconciled credit count",
            "Source credit rows",
            "Reconciled POS month records",
            "Source POS rows",
            "Status reconciliation errors",
            "Reconciled DPD records",
            "Reconciled 30+ DPD records",
            "Reconciled 90+ DPD records",
            "Reconciled future-greater records",
            "Customers with any DPD",
            "Customers with 90+ DPD",
            "Customer feature columns",
            "Verified saved rows",
            "Saved file size (MB)",
            "Customer table memory (MB)",
        ],
        "value": [
            len(pos_customer_features),
            pos_customer_features
            .index.nunique(),
            pos_customer_features
            .index.duplicated().sum(),
            pos_customer_features[
                "POS_CREDIT_COUNT"
            ].sum(),
            len(pos_credit_features),
            pos_customer_features[
                "POS_TOTAL_MONTH_RECORD_COUNT"
            ].sum(),
            len(pos_cash_balance),
            pos_customer_status_difference
            .ne(0).sum(),
            pos_customer_features[
                "POS_TOTAL_DPD_RECORD_COUNT"
            ].sum(),
            pos_customer_features[
                "POS_TOTAL_30_PLUS_DPD_RECORD_COUNT"
            ].sum(),
            pos_customer_features[
                "POS_TOTAL_90_PLUS_DPD_RECORD_COUNT"
            ].sum(),
            pos_customer_features[
                "POS_TOTAL_FUTURE_GT_TOTAL_COUNT"
            ].sum(),
            (
                pos_customer_features[
                    "POS_CREDITS_WITH_DPD"
                ] > 0
            ).sum(),
            (
                pos_customer_features[
                    "POS_CREDITS_WITH_90_PLUS_DPD"
                ] > 0
            ).sum(),
            pos_customer_features.shape[1],
            len(pos_customer_verification),
            (
                pos_customer_feature_path
                .stat()
                .st_size
                / 1024**2
            ),
            (
                pos_customer_features
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

display(
    pos_customer_feature_check.round(2)
)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import gc

if "processed_data_path" not in globals():
    processed_data_path = Path(
        str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "processed")
    )

train_bureau_previous_path = (
    processed_data_path
    / "application_train_features_bureau_previous.parquet"
)

pos_customer_feature_path = (
    processed_data_path
    / "pos_cash_customer_features.parquet"
)

# Always start from clean saved files
train_features_base = pd.read_parquet(
    train_bureau_previous_path
)

pos_train_features = pd.read_parquet(
    pos_customer_feature_path
)

# Check unexpected overlap
pos_overlapping_columns = (
    train_features_base.columns
    .intersection(
        pos_train_features.columns
    )
    .difference(
        ["SK_ID_CURR"]
    )
)

if len(pos_overlapping_columns) > 0:
    raise ValueError(
        "Unexpected overlapping POS columns: "
        f"{list(pos_overlapping_columns)}"
    )

# Safe one-to-one merge
train_features_with_pos = (
    train_features_base.merge(
        pos_train_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one",
    )
)

# Determine history before filling counts
train_features_with_pos[
    "POS_HAS_HISTORY"
] = (
    train_features_with_pos[
        "POS_CREDIT_COUNT"
    ]
    .notna()
    .astype("int8")
)

In [ ]:
pos_status_count_columns = [
    feature
    for feature
    in pos_train_features.columns
    if (
        feature.startswith(
            "POS_STATUS_"
        )
        and feature.endswith(
            "_COUNT"
        )
    )
]

pos_zero_fill_count_columns = [
    "POS_CREDIT_COUNT",
    "POS_TOTAL_MONTH_RECORD_COUNT",
    "POS_TOTAL_DPD_RECORD_COUNT",
    "POS_TOTAL_DPD_DEF_RECORD_COUNT",
    "POS_TOTAL_30_PLUS_DPD_RECORD_COUNT",
    "POS_TOTAL_90_PLUS_DPD_RECORD_COUNT",
    "POS_TOTAL_DPD_EXTREME_RECORD_COUNT",
    "POS_TOTAL_DPD_DEF_EXTREME_RECORD_COUNT",
    "POS_CREDITS_WITH_DPD",
    "POS_CREDITS_WITH_DPD_DEF",
    "POS_CREDITS_WITH_30_PLUS_DPD",
    "POS_CREDITS_WITH_90_PLUS_DPD",
    "POS_CREDITS_WITH_DEMAND",
    "POS_CREDITS_WITH_AMORTIZED_DEBT",
    "POS_CREDITS_WITH_COMPLETED_STATUS",
    "POS_TOTAL_FUTURE_GT_TOTAL_COUNT",
    "POS_CREDITS_WITH_FUTURE_GT_TOTAL",
] + pos_status_count_columns

missing_pos_count_columns = [
    feature
    for feature
    in pos_zero_fill_count_columns
    if feature
    not in train_features_with_pos.columns
]

if missing_pos_count_columns:
    raise RuntimeError(
        "Missing expected POS count columns: "
        f"{missing_pos_count_columns}"
    )

for feature in pos_zero_fill_count_columns:
    train_features_with_pos[
        feature
    ] = (
        train_features_with_pos[
            feature
        ]
        .fillna(0)
        .astype("int32")
    )

In [ ]:
train_features_with_pos[
    "POS_HAS_DPD_HISTORY"
] = (
    train_features_with_pos[
        "POS_CREDITS_WITH_DPD"
    ] > 0
).astype("int8")

train_features_with_pos[
    "POS_HAS_90_PLUS_DPD_HISTORY"
] = (
    train_features_with_pos[
        "POS_CREDITS_WITH_90_PLUS_DPD"
    ] > 0
).astype("int8")

# Recalculate count logs so no-history customers safely receive log(1)=0
train_features_with_pos[
    "POS_LOG_CREDIT_COUNT"
] = np.log1p(
    train_features_with_pos[
        "POS_CREDIT_COUNT"
    ]
).astype("float32")

train_features_with_pos[
    "POS_LOG_TOTAL_MONTH_RECORD_COUNT"
] = np.log1p(
    train_features_with_pos[
        "POS_TOTAL_MONTH_RECORD_COUNT"
    ]
).astype("float32")

In [ ]:
train_with_pos_path = (
    processed_data_path
    / "application_train_features_bureau_previous_pos.parquet"
)

train_features_with_pos.to_parquet(
    train_with_pos_path,
    index=False,
    compression="snappy",
)

train_pos_verification = pd.read_parquet(
    train_with_pos_path,
    columns=[
        "SK_ID_CURR",
        "PREV_HAS_HISTORY",
        "POS_HAS_HISTORY",
        "POS_HAS_DPD_HISTORY",
    ],
)

customers_with_pos_history = int(
    train_features_with_pos[
        "POS_HAS_HISTORY"
    ].sum()
)

customers_without_pos_history = (
    len(train_features_with_pos)
    - customers_with_pos_history
)

previous_history_without_pos = int(
    (
        (
            train_features_with_pos[
                "PREV_HAS_HISTORY"
            ] == 1
        )
        & (
            train_features_with_pos[
                "POS_HAS_HISTORY"
            ] == 0
        )
    ).sum()
)

pos_without_previous_history = int(
    (
        (
            train_features_with_pos[
                "POS_HAS_HISTORY"
            ] == 1
        )
        & (
            train_features_with_pos[
                "PREV_HAS_HISTORY"
            ] == 0
        )
    ).sum()
)

train_pos_join_check = pd.DataFrame(
    {
        "metric": [
            "Base train rows",
            "Final train rows",
            "Row count difference",
            "Unique train IDs",
            "Duplicate train IDs",
            "Unexpected overlapping features",
            "Customers with POS history",
            "Customers without POS history",
            "Previous history without POS",
            "POS history without previous history",
            "Customers with any POS DPD",
            "Customers with 90+ POS DPD",
            "Nulls in zero-filled counts",
            "Final feature columns",
            "Verified saved rows",
            "Saved file size (MB)",
            "Final table memory (MB)",
        ],
        "value": [
            len(train_features_base),
            len(train_features_with_pos),
            (
                len(train_features_with_pos)
                - len(train_features_base)
            ),
            train_features_with_pos[
                "SK_ID_CURR"
            ].nunique(),
            train_features_with_pos[
                "SK_ID_CURR"
            ].duplicated().sum(),
            len(pos_overlapping_columns),
            customers_with_pos_history,
            customers_without_pos_history,
            previous_history_without_pos,
            pos_without_previous_history,
            train_features_with_pos[
                "POS_HAS_DPD_HISTORY"
            ].sum(),
            train_features_with_pos[
                "POS_HAS_90_PLUS_DPD_HISTORY"
            ].sum(),
            train_features_with_pos[
                pos_zero_fill_count_columns
            ].isna().sum().sum(),
            (
                train_features_with_pos.shape[1]
                - 1
            ),
            len(train_pos_verification),
            (
                train_with_pos_path
                .stat()
                .st_size
                / 1024**2
            ),
            (
                train_features_with_pos
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

train_pos_join_check.to_csv(
    processed_data_path
    / "pos_cash_train_join_check.csv",
    index=False,
)

display(
    train_pos_join_check.round(2)
)

In [ ]:
import gc
from pathlib import Path
import pandas as pd
import numpy as np

objects_to_remove = [
    "pos_cash_balance",
    "pos_credit_features",
    "pos_customer_features",
    "pos_customer_base_features",
    "pos_customer_status_features",
    "pos_credit_behavior_features",
    "pos_credit_status_counts",
    "pos_train_features",
    "train_features_base",
    "train_features_with_pos",
    "train_pos_verification",
    "pos_credit_verification",
    "pos_customer_verification",
    "pos_id_map",
    "pos_previous_mapping_check",
    "pos_unmapped_previous_audit",
]

for object_name in objects_to_remove:
    globals().pop(
        object_name,
        None,
    )

gc.collect()

In [ ]:
if "project_path" not in globals():
    project_path = Path(
        str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
    )

if "extract_path" not in globals():
    extract_path = (
        project_path
        / "data"
        / "raw"
    )

if "processed_data_path" not in globals():
    processed_data_path = (
        project_path
        / "data"
        / "processed"
    )

installments_payments = pd.read_csv(
    extract_path
    / "installments_payments.csv",
    dtype={
        "SK_ID_PREV": "int32",
        "SK_ID_CURR": "int32",
        "NUM_INSTALMENT_VERSION": "float32",
        "NUM_INSTALMENT_NUMBER": "int16",
        "DAYS_INSTALMENT": "int16",
        "DAYS_ENTRY_PAYMENT": "float32",
        "AMT_INSTALMENT": "float32",
        "AMT_PAYMENT": "float32",
    },
    memory_map=True,
)

In [6]:
scheduled_key_columns = [
    "SK_ID_PREV",
    "NUM_INSTALMENT_VERSION",
    "NUM_INSTALMENT_NUMBER",
]

duplicate_scheduled_key_mask = (
    installments_payments.duplicated(
        subset=scheduled_key_columns,
        keep=False,
    )
)

duplicate_scheduled_key_rows = int(
    duplicate_scheduled_key_mask.sum()
)

duplicate_scheduled_keys = int(
    installments_payments.loc[
        duplicate_scheduled_key_mask,
        scheduled_key_columns,
    ]
    .drop_duplicates()
    .shape[0]
)

entry_payment_missing_mismatch = int(
    (
        installments_payments[
            "DAYS_ENTRY_PAYMENT"
        ].isna()
        != installments_payments[
            "AMT_PAYMENT"
        ].isna()
    ).sum()
)

installments_overview = pd.DataFrame(
    {
        "metric": [
            "Total installment payment rows",
            "Total columns",
            "Unique previous application IDs",
            "Unique customer IDs",
            "Exact duplicate rows",
            "Duplicate scheduled-key rows",
            "Duplicate scheduled keys",
            "Null previous application IDs",
            "Null customer IDs",
            "Null scheduled days",
            "Null payment-entry days",
            "Null installment amounts",
            "Null payment amounts",
            "Entry/payment missing mismatch",
            "Minimum scheduled day",
            "Maximum scheduled day",
            "Future scheduled-day records",
            "Minimum payment-entry day",
            "Maximum payment-entry day",
            "Future payment-entry records",
            "Negative installment amounts",
            "Negative payment amounts",
            "Zero installment amounts",
            "Zero payment amounts",
            "Memory usage (MB)",
        ],
        "value": [
            len(installments_payments),
            installments_payments.shape[1],
            installments_payments[
                "SK_ID_PREV"
            ].nunique(),
            installments_payments[
                "SK_ID_CURR"
            ].nunique(),
            installments_payments
            .duplicated()
            .sum(),
            duplicate_scheduled_key_rows,
            duplicate_scheduled_keys,
            installments_payments[
                "SK_ID_PREV"
            ].isna().sum(),
            installments_payments[
                "SK_ID_CURR"
            ].isna().sum(),
            installments_payments[
                "DAYS_INSTALMENT"
            ].isna().sum(),
            installments_payments[
                "DAYS_ENTRY_PAYMENT"
            ].isna().sum(),
            installments_payments[
                "AMT_INSTALMENT"
            ].isna().sum(),
            installments_payments[
                "AMT_PAYMENT"
            ].isna().sum(),
            entry_payment_missing_mismatch,
            installments_payments[
                "DAYS_INSTALMENT"
            ].min(),
            installments_payments[
                "DAYS_INSTALMENT"
            ].max(),
            (
                installments_payments[
                    "DAYS_INSTALMENT"
                ] > 0
            ).sum(),
            installments_payments[
                "DAYS_ENTRY_PAYMENT"
            ].min(),
            installments_payments[
                "DAYS_ENTRY_PAYMENT"
            ].max(),
            (
                installments_payments[
                    "DAYS_ENTRY_PAYMENT"
                ] > 0
            ).sum(),
            (
                installments_payments[
                    "AMT_INSTALMENT"
                ] < 0
            ).sum(),
            (
                installments_payments[
                    "AMT_PAYMENT"
                ] < 0
            ).sum(),
            (
                installments_payments[
                    "AMT_INSTALMENT"
                ] == 0
            ).sum(),
            (
                installments_payments[
                    "AMT_PAYMENT"
                ] == 0
            ).sum(),
            (
                installments_payments
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

display(
    installments_overview.round(2)
)

,metric,value
0,Total installment payment rows,13605401.0
1,Total columns,8.0
2,Unique previous application IDs,997752.0
3,Unique customer IDs,339587.0
4,Exact duplicate rows,0.0
5,Duplicate scheduled-key rows,1294388.0
6,Duplicate scheduled keys,640905.0
7,Null previous application IDs,0.0
8,Null customer IDs,0.0
9,Null scheduled days,0.0


In [7]:
# Reload clean previous ID map if necessary
if "previous_id_map" not in globals():
    previous_id_map = pd.read_parquet(
        processed_data_path
        / "previous_application_row_features.parquet",
        columns=[
            "SK_ID_PREV",
            "SK_ID_CURR",
        ],
    )

previous_id_map_unique = (
    previous_id_map
    .drop_duplicates(
        subset="SK_ID_PREV"
    )
)

installment_id_map = (
    installments_payments[
        [
            "SK_ID_PREV",
            "SK_ID_CURR",
        ]
    ]
    .drop_duplicates()
)

installment_internal_conflicts = int(
    (
        installment_id_map.groupby(
            "SK_ID_PREV",
            sort=False,
        )["SK_ID_CURR"]
        .nunique()
        > 1
    ).sum()
)

if installment_internal_conflicts == 0:
    installment_id_map = (
        installment_id_map
        .drop_duplicates(
            subset="SK_ID_PREV"
        )
    )

installment_previous_mapping = (
    installment_id_map.merge(
        previous_id_map_unique,
        on="SK_ID_PREV",
        how="left",
        suffixes=(
            "_INSTALLMENT",
            "_PREVIOUS",
        ),
        validate="one_to_one",
    )
)

installment_previous_mapping[
    "PREVIOUS_ID_FOUND"
] = (
    installment_previous_mapping[
        "SK_ID_CURR_PREVIOUS"
    ].notna()
)

installment_previous_mapping[
    "CUSTOMER_MAPPING_CONSISTENT"
] = (
    ~installment_previous_mapping[
        "PREVIOUS_ID_FOUND"
    ]
    | (
        installment_previous_mapping[
            "SK_ID_CURR_INSTALLMENT"
        ]
        == installment_previous_mapping[
            "SK_ID_CURR_PREVIOUS"
        ]
    )
)

installment_mapping_conflicts = int(
    (
        installment_previous_mapping[
            "PREVIOUS_ID_FOUND"
        ]
        & ~installment_previous_mapping[
            "CUSTOMER_MAPPING_CONSISTENT"
        ]
    ).sum()
)

In [8]:
train_application_ids = pd.Index(
    pd.read_parquet(
        processed_data_path
        / "application_train_features_bureau_previous_pos.parquet",
        columns=[
            "SK_ID_CURR",
        ],
    )["SK_ID_CURR"]
)

test_application_ids = pd.Index(
    pd.read_csv(
        extract_path
        / "application_test.csv",
        usecols=[
            "SK_ID_CURR",
        ],
        dtype={
            "SK_ID_CURR": "int32",
        },
    )["SK_ID_CURR"]
)

all_application_ids = (
    train_application_ids.union(
        test_application_ids
    )
)

installment_customer_ids = pd.Index(
    installments_payments[
        "SK_ID_CURR"
    ].unique()
)

orphan_installment_customer_ids = (
    installment_customer_ids.difference(
        all_application_ids
    )
)

orphan_installment_rows = int(
    installments_payments[
        "SK_ID_CURR"
    ]
    .isin(
        orphan_installment_customer_ids
    )
    .sum()
)

unmapped_previous_mask = (
    ~installments_payments[
        "SK_ID_PREV"
    ].isin(
        previous_id_map_unique[
            "SK_ID_PREV"
        ]
    )
)

installment_mapping_check = pd.DataFrame(
    {
        "metric": [
            "Unique installment previous IDs",
            "Matched previous IDs",
            "Unmapped previous IDs",
            "Internal previous-customer conflicts",
            "Matched customer mapping conflicts",
            "Unique installment customers",
            "Application universe customers",
            "Orphan installment customers",
            "Orphan installment rows",
            "Unmapped previous rows",
            "Rows safe to retain",
        ],
        "value": [
            len(installment_previous_mapping),
            installment_previous_mapping[
                "PREVIOUS_ID_FOUND"
            ].sum(),
            (
                ~installment_previous_mapping[
                    "PREVIOUS_ID_FOUND"
                ]
            ).sum(),
            installment_internal_conflicts,
            installment_mapping_conflicts,
            len(installment_customer_ids),
            len(all_application_ids),
            len(orphan_installment_customer_ids),
            orphan_installment_rows,
            unmapped_previous_mask.sum(),
            (
                len(installments_payments)
                - orphan_installment_rows
            ),
        ],
    }
)

display(
    installment_mapping_check
)

,metric,value
0,Unique installment previous IDs,997752
1,Matched previous IDs,958905
2,Unmapped previous IDs,38847
3,Internal previous-customer conflicts,0
4,Matched customer mapping conflicts,0
5,Unique installment customers,339587
6,Application universe customers,356255
7,Orphan installment customers,0
8,Orphan installment rows,0
9,Unmapped previous rows,1250826


In [9]:
display(
    installments_overview.round(2)
)

,metric,value
0,Total installment payment rows,13605401.0
1,Total columns,8.0
2,Unique previous application IDs,997752.0
3,Unique customer IDs,339587.0
4,Exact duplicate rows,0.0
5,Duplicate scheduled-key rows,1294388.0
6,Duplicate scheduled keys,640905.0
7,Null previous application IDs,0.0
8,Null customer IDs,0.0
9,Null scheduled days,0.0


In [10]:
# Preserve audit evidence before releasing the raw table
if "installments_overview" in globals():
    installments_overview.to_csv(
        processed_data_path
        / "installments_overview.csv",
        index=False,
    )

if "installment_mapping_check" in globals():
    installment_mapping_check.to_csv(
        processed_data_path
        / "installment_mapping_check.csv",
        index=False,
    )

installment_source_row_count = len(
    installments_payments
)

installment_source_previous_ids = (
    installments_payments[
        "SK_ID_PREV"
    ].nunique()
)

installment_source_customers = (
    installments_payments[
        "SK_ID_CURR"
    ].nunique()
)

scheduled_key_columns = [
    "SK_ID_PREV",
    "NUM_INSTALMENT_VERSION",
    "NUM_INSTALMENT_NUMBER",
]

split_payment_mask = (
    installments_payments.duplicated(
        subset=scheduled_key_columns,
        keep=False,
    )
)

In [11]:
single_payment_schedules = (
    installments_payments.loc[
        ~split_payment_mask,
        [
            "SK_ID_PREV",
            "NUM_INSTALMENT_VERSION",
            "NUM_INSTALMENT_NUMBER",
            "SK_ID_CURR",
            "DAYS_INSTALMENT",
            "DAYS_ENTRY_PAYMENT",
            "AMT_INSTALMENT",
            "AMT_PAYMENT",
        ],
    ]
    .copy()
    .rename(
        columns={
            "DAYS_INSTALMENT":
                "INST_SCHEDULED_DAY",
            "DAYS_ENTRY_PAYMENT":
                "INST_FIRST_PAYMENT_DAY",
            "AMT_INSTALMENT":
                "INST_SCHEDULED_AMOUNT",
            "AMT_PAYMENT":
                "INST_TOTAL_PAYMENT",
        }
    )
)

single_payment_schedules[
    "INST_LAST_PAYMENT_DAY"
] = single_payment_schedules[
    "INST_FIRST_PAYMENT_DAY"
]

single_payment_schedules[
    "INST_PAYMENT_PART_COUNT"
] = np.int16(1)

single_payment_schedules[
    "INST_OBSERVED_PAYMENT_PART_COUNT"
] = (
    single_payment_schedules[
        "INST_TOTAL_PAYMENT"
    ]
    .notna()
    .astype("int16")
)

single_payment_schedules[
    "INST_CUSTOMER_NUNIQUE"
] = np.int16(1)

single_payment_schedules[
    "INST_SCHEDULED_DAY_NUNIQUE"
] = np.int16(1)

single_payment_schedules[
    "INST_SCHEDULED_AMOUNT_NUNIQUE"
] = np.int16(1)

In [12]:
split_payment_rows = (
    installments_payments.loc[
        split_payment_mask
    ]
)

split_payment_group = (
    split_payment_rows.groupby(
        scheduled_key_columns,
        sort=False,
        observed=True,
    )
)

split_payment_schedules = (
    split_payment_group
    .agg(
        SK_ID_CURR=(
            "SK_ID_CURR",
            "first",
        ),
        INST_CUSTOMER_NUNIQUE=(
            "SK_ID_CURR",
            "nunique",
        ),
        INST_SCHEDULED_DAY=(
            "DAYS_INSTALMENT",
            "first",
        ),
        INST_SCHEDULED_DAY_NUNIQUE=(
            "DAYS_INSTALMENT",
            "nunique",
        ),
        INST_SCHEDULED_AMOUNT=(
            "AMT_INSTALMENT",
            "first",
        ),
        INST_SCHEDULED_AMOUNT_NUNIQUE=(
            "AMT_INSTALMENT",
            "nunique",
        ),
        INST_FIRST_PAYMENT_DAY=(
            "DAYS_ENTRY_PAYMENT",
            "min",
        ),
        INST_LAST_PAYMENT_DAY=(
            "DAYS_ENTRY_PAYMENT",
            "max",
        ),
        INST_PAYMENT_PART_COUNT=(
            "SK_ID_PREV",
            "size",
        ),
        INST_OBSERVED_PAYMENT_PART_COUNT=(
            "AMT_PAYMENT",
            "count",
        ),
    )
)

# min_count=1 prevents all-missing payments becoming zero
split_payment_schedules[
    "INST_TOTAL_PAYMENT"
] = (
    split_payment_group[
        "AMT_PAYMENT"
    ].sum(min_count=1)
)

split_payment_schedules = (
    split_payment_schedules
    .reset_index()
)

In [13]:
for feature in [
    "INST_CUSTOMER_NUNIQUE",
    "INST_SCHEDULED_DAY_NUNIQUE",
    "INST_SCHEDULED_AMOUNT_NUNIQUE",
    "INST_PAYMENT_PART_COUNT",
    "INST_OBSERVED_PAYMENT_PART_COUNT",
]:
    split_payment_schedules[
        feature
    ] = (
        split_payment_schedules[
            feature
        ].astype("int16")
    )

for feature in [
    "INST_FIRST_PAYMENT_DAY",
    "INST_LAST_PAYMENT_DAY",
    "INST_SCHEDULED_AMOUNT",
    "INST_TOTAL_PAYMENT",
]:
    split_payment_schedules[
        feature
    ] = (
        split_payment_schedules[
            feature
        ].astype("float32")
    )

In [14]:
del split_payment_rows
del split_payment_group
del installments_payments
del split_payment_mask
gc.collect()

installment_schedule = pd.concat(
    [
        single_payment_schedules,
        split_payment_schedules,
    ],
    ignore_index=True,
    copy=False,
)

del single_payment_schedules
del split_payment_schedules
gc.collect()

NameError: name 'gc' is not defined

In [ ]:
schedule_duplicate_count = int(
    installment_schedule.duplicated(
        subset=scheduled_key_columns
    ).sum()
)

missing_payment_part_count = int(
    (
        installment_schedule[
            "INST_PAYMENT_PART_COUNT"
        ]
        - installment_schedule[
            "INST_OBSERVED_PAYMENT_PART_COUNT"
        ]
    ).sum()
)

installment_schedule_path = (
    processed_data_path
    / "installment_schedule_level.parquet"
)

installment_schedule.to_parquet(
    installment_schedule_path,
    index=False,
    compression="snappy",
)

schedule_verification = pd.read_parquet(
    installment_schedule_path,
    columns=[
        "SK_ID_PREV",
        "SK_ID_CURR",
        "NUM_INSTALMENT_VERSION",
        "NUM_INSTALMENT_NUMBER",
    ],
)

installment_schedule_check = pd.DataFrame(
    {
        "metric": [
            "Schedule-level rows",
            "Unique scheduled keys",
            "Duplicate scheduled keys after merge",
            "Reconciled payment-part rows",
            "Source payment rows",
            "Split-payment schedules",
            "Single-payment schedules",
            "Missing payment parts",
            "Schedules without any payment",
            "Customer mapping conflicts",
            "Scheduled-day conflicts",
            "Scheduled-amount conflicts",
            "Unique previous application IDs",
            "Source previous application IDs",
            "Unique customer IDs",
            "Source customer IDs",
            "Verified saved rows",
            "Saved file size (MB)",
            "Schedule table memory (MB)",
        ],
        "value": [
            len(installment_schedule),
            installment_schedule[
                scheduled_key_columns
            ].drop_duplicates().shape[0],
            schedule_duplicate_count,
            installment_schedule[
                "INST_PAYMENT_PART_COUNT"
            ].sum(),
            installment_source_row_count,
            (
                installment_schedule[
                    "INST_PAYMENT_PART_COUNT"
                ] > 1
            ).sum(),
            (
                installment_schedule[
                    "INST_PAYMENT_PART_COUNT"
                ] == 1
            ).sum(),
            missing_payment_part_count,
            installment_schedule[
                "INST_TOTAL_PAYMENT"
            ].isna().sum(),
            (
                installment_schedule[
                    "INST_CUSTOMER_NUNIQUE"
                ] > 1
            ).sum(),
            (
                installment_schedule[
                    "INST_SCHEDULED_DAY_NUNIQUE"
                ] > 1
            ).sum(),
            (
                installment_schedule[
                    "INST_SCHEDULED_AMOUNT_NUNIQUE"
                ] > 1
            ).sum(),
            installment_schedule[
                "SK_ID_PREV"
            ].nunique(),
            installment_source_previous_ids,
            installment_schedule[
                "SK_ID_CURR"
            ].nunique(),
            installment_source_customers,
            len(schedule_verification),
            (
                installment_schedule_path
                .stat()
                .st_size
                / 1024**2
            ),
            (
                installment_schedule
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

display(
    installment_schedule_check.round(2)
)

In [ ]:
payment_available_mask = (
    installment_schedule[
        "INST_TOTAL_PAYMENT"
    ].notna()
    & installment_schedule[
        "INST_LAST_PAYMENT_DAY"
    ].notna()
)

payment_delay_days = (
    installment_schedule[
        "INST_LAST_PAYMENT_DAY"
    ]
    - installment_schedule[
        "INST_SCHEDULED_DAY"
    ]
).where(
    payment_available_mask
).astype("float32")

days_late = (
    payment_delay_days
    .clip(lower=0)
)

days_early = (
    -payment_delay_days
    .clip(upper=0)
)

positive_scheduled_amount_mask = (
    installment_schedule[
        "INST_SCHEDULED_AMOUNT"
    ] > 0
)

payment_ratio_raw = (
    installment_schedule[
        "INST_TOTAL_PAYMENT"
    ]
    / installment_schedule[
        "INST_SCHEDULED_AMOUNT"
    ]
).where(
    payment_available_mask
    & positive_scheduled_amount_mask
)

payment_difference = (
    installment_schedule[
        "INST_TOTAL_PAYMENT"
    ]
    - installment_schedule[
        "INST_SCHEDULED_AMOUNT"
    ]
).where(
    payment_available_mask
)

# A tolerance of 1 currency unit avoids floating-point noise
underpayment_mask = (
    payment_difference < -1
)

overpayment_mask = (
    payment_difference > 1
)

approximately_equal_payment_mask = (
    payment_available_mask
    & payment_difference.abs().le(1)
)

In [ ]:
installment_behavior_check = pd.DataFrame(
    {
        "metric": [
            "Schedule rows",
            "Schedules with payment",
            "Schedules without payment",
            "Split-payment schedules",
            "Early-paid schedules",
            "On-time schedules",
            "Late-paid schedules",
            "30+ days late schedules",
            "90+ days late schedules",
            "Underpaid schedules",
            "Overpaid schedules",
            "Approximately equal payments",
            "Zero scheduled amounts",
            "Zero payment amounts",
            "Valid payment ratios",
            "Payment ratio greater than 2",
            "Payment ratio greater than 5",
            "Payment ratio greater than 10",
        ],
        "value": [
            len(installment_schedule),
            int(
                payment_available_mask.sum()
            ),
            int(
                (~payment_available_mask).sum()
            ),
            int(
                (
                    installment_schedule[
                        "INST_PAYMENT_PART_COUNT"
                    ] > 1
                ).sum()
            ),
            int(
                (
                    payment_delay_days < 0
                ).sum()
            ),
            int(
                (
                    payment_delay_days == 0
                ).sum()
            ),
            int(
                (
                    payment_delay_days > 0
                ).sum()
            ),
            int(
                (
                    payment_delay_days >= 30
                ).sum()
            ),
            int(
                (
                    payment_delay_days >= 90
                ).sum()
            ),
            int(
                underpayment_mask.sum()
            ),
            int(
                overpayment_mask.sum()
            ),
            int(
                approximately_equal_payment_mask
                .sum()
            ),
            int(
                (
                    installment_schedule[
                        "INST_SCHEDULED_AMOUNT"
                    ] == 0
                ).sum()
            ),
            int(
                (
                    installment_schedule[
                        "INST_TOTAL_PAYMENT"
                    ] == 0
                ).sum()
            ),
            int(
                payment_ratio_raw.notna().sum()
            ),
            int(
                (
                    payment_ratio_raw > 2
                ).sum()
            ),
            int(
                (
                    payment_ratio_raw > 5
                ).sum()
            ),
            int(
                (
                    payment_ratio_raw > 10
                ).sum()
            ),
        ],
    }
)

display(
    installment_behavior_check
)

In [ ]:
def positive_tail_summary(
    values,
    feature_name,
):
    positive_values = values[
        values > 0
    ].dropna()

    return {
        "feature": feature_name,
        "positive_records": len(
            positive_values
        ),
        "positive_share_pct": (
            len(positive_values)
            / len(installment_schedule)
            * 100
        ),
        "median": (
            positive_values.median()
            if len(positive_values) > 0
            else np.nan
        ),
        "p90": (
            positive_values.quantile(
                0.90
            )
            if len(positive_values) > 0
            else np.nan
        ),
        "p95": (
            positive_values.quantile(
                0.95
            )
            if len(positive_values) > 0
            else np.nan
        ),
        "p99": (
            positive_values.quantile(
                0.99
            )
            if len(positive_values) > 0
            else np.nan
        ),
        "maximum": (
            positive_values.max()
            if len(positive_values) > 0
            else np.nan
        ),
    }

installment_tail_summary = pd.DataFrame(
    [
        positive_tail_summary(
            days_late,
            "DAYS_LATE",
        ),
        positive_tail_summary(
            days_early,
            "DAYS_EARLY",
        ),
        positive_tail_summary(
            payment_ratio_raw,
            "PAYMENT_RATIO",
        ),
    ]
)

installment_behavior_check.to_csv(
    processed_data_path
    / "installment_behavior_check.csv",
    index=False,
)

installment_tail_summary.to_csv(
    processed_data_path
    / "installment_tail_summary.csv",
    index=False,
)

display(
    installment_tail_summary.round(3)
)

In [ ]:
display(
    installment_tail_summary.round(3)
)

In [ ]:
import gc
import numpy as np
import pandas as pd

# Reload after a kernel restart if necessary
if "installment_schedule" not in globals():
    installment_schedule = pd.read_parquet(
        processed_data_path
        / "installment_schedule_level.parquet"
    )

required_columns = {
    "SK_ID_PREV",
    "SK_ID_CURR",
    "INST_SCHEDULED_DAY",
    "INST_LAST_PAYMENT_DAY",
    "INST_SCHEDULED_AMOUNT",
    "INST_TOTAL_PAYMENT",
}

missing_columns = (
    required_columns
    - set(installment_schedule.columns)
)

assert not missing_columns, (
    f"Missing schedule columns: {missing_columns}"
)

# Locate the payment-part count column safely
payment_part_candidates = [
    "INST_PAYMENT_PART_COUNT",
    "INST_PAYMENT_PARTS",
    "INST_SOURCE_ROW_COUNT",
]

payment_part_count_column = next(
    (
        column
        for column in payment_part_candidates
        if column in installment_schedule.columns
    ),
    None,
)

assert payment_part_count_column is not None, (
    "Payment-part count column could not be found."
)

columns_before_engineering = set(
    installment_schedule.columns
)

# Availability
payment_available_mask = (
    installment_schedule[
        "INST_TOTAL_PAYMENT"
    ].notna()
    & installment_schedule[
        "INST_LAST_PAYMENT_DAY"
    ].notna()
)

installment_schedule[
    "INST_PAYMENT_AVAILABLE"
] = payment_available_mask.astype("int8")

installment_schedule[
    "INST_SPLIT_PAYMENT_FLAG"
] = (
    installment_schedule[
        payment_part_count_column
    ] > 1
).astype("int8")


# Payment timing
payment_delay_days = (
    installment_schedule[
        "INST_LAST_PAYMENT_DAY"
    ]
    - installment_schedule[
        "INST_SCHEDULED_DAY"
    ]
).where(payment_available_mask)

installment_schedule[
    "INST_EARLY_PAYMENT_FLAG"
] = (
    payment_available_mask
    & payment_delay_days.lt(0)
).astype("int8")

installment_schedule[
    "INST_ON_TIME_PAYMENT_FLAG"
] = (
    payment_available_mask
    & payment_delay_days.eq(0)
).astype("int8")

installment_schedule[
    "INST_LATE_PAYMENT_FLAG"
] = (
    payment_available_mask
    & payment_delay_days.gt(0)
).astype("int8")

installment_schedule[
    "INST_30PLUS_LATE_FLAG"
] = (
    payment_available_mask
    & payment_delay_days.ge(30)
).astype("int8")

installment_schedule[
    "INST_90PLUS_LATE_FLAG"
] = (
    payment_available_mask
    & payment_delay_days.ge(90)
).astype("int8")

installment_schedule[
    "INST_DAYS_LATE_RAW"
] = (
    payment_delay_days
    .clip(lower=0)
    .astype("float32")
)

installment_schedule[
    "INST_DAYS_LATE_CAP_120"
] = (
    installment_schedule[
        "INST_DAYS_LATE_RAW"
    ]
    .clip(upper=120)
    .astype("float32")
)

installment_schedule[
    "INST_LOG1P_DAYS_LATE"
] = np.log1p(
    installment_schedule[
        "INST_DAYS_LATE_CAP_120"
    ]
).astype("float32")

installment_schedule[
    "INST_DAYS_EARLY_CAP_90"
] = (
    -payment_delay_days.clip(upper=0)
).clip(upper=90).astype("float32")


# Payment amount behaviour
payment_difference = (
    installment_schedule[
        "INST_TOTAL_PAYMENT"
    ]
    - installment_schedule[
        "INST_SCHEDULED_AMOUNT"
    ]
).where(payment_available_mask)

installment_schedule[
    "INST_UNDERPAYMENT_FLAG"
] = (
    payment_available_mask
    & payment_difference.lt(-1)
).astype("int8")

installment_schedule[
    "INST_OVERPAYMENT_FLAG"
] = (
    payment_available_mask
    & payment_difference.gt(1)
).astype("int8")

installment_schedule[
    "INST_EQUAL_PAYMENT_FLAG"
] = (
    payment_available_mask
    & payment_difference.abs().le(1)
).astype("int8")

installment_schedule[
    "INST_UNDERPAYMENT_AMOUNT"
] = (
    (-payment_difference)
    .where(payment_difference.lt(-1), 0)
    .where(payment_available_mask)
    .astype("float32")
)

installment_schedule[
    "INST_OVERPAYMENT_AMOUNT"
] = (
    payment_difference
    .where(payment_difference.gt(1), 0)
    .where(payment_available_mask)
    .astype("float32")
)


# Robust payment ratio
positive_scheduled_amount_mask = (
    installment_schedule[
        "INST_SCHEDULED_AMOUNT"
    ] > 0
)

payment_ratio_available_mask = (
    payment_available_mask
    & positive_scheduled_amount_mask
)

payment_ratio_raw = (
    installment_schedule[
        "INST_TOTAL_PAYMENT"
    ]
    / installment_schedule[
        "INST_SCHEDULED_AMOUNT"
    ]
).where(payment_ratio_available_mask)

installment_schedule[
    "INST_PAYMENT_RATIO_AVAILABLE"
] = (
    payment_ratio_available_mask
    .astype("int8")
)

installment_schedule[
    "INST_PAYMENT_RATIO_CAP_2"
] = (
    payment_ratio_raw
    .clip(lower=0, upper=2)
    .astype("float32")
)

installment_schedule[
    "INST_PAYMENT_RATIO_EXTREME_FLAG"
] = (
    payment_ratio_available_mask
    & payment_ratio_raw.gt(2)
).astype("int8")

installment_schedule[
    "INST_ZERO_SCHEDULED_AMOUNT_FLAG"
] = (
    installment_schedule[
        "INST_SCHEDULED_AMOUNT"
    ].eq(0)
).astype("int8")


# Reconciliation
timing_reconciliation_difference = (
    installment_schedule[
        [
            "INST_EARLY_PAYMENT_FLAG",
            "INST_ON_TIME_PAYMENT_FLAG",
            "INST_LATE_PAYMENT_FLAG",
        ]
    ].sum(axis=1)
    - installment_schedule[
        "INST_PAYMENT_AVAILABLE"
    ]
)

amount_reconciliation_difference = (
    installment_schedule[
        [
            "INST_UNDERPAYMENT_FLAG",
            "INST_OVERPAYMENT_FLAG",
            "INST_EQUAL_PAYMENT_FLAG",
        ]
    ].sum(axis=1)
    - installment_schedule[
        "INST_PAYMENT_AVAILABLE"
    ]
)

installment_feature_check = pd.DataFrame(
    {
        "metric": [
            "Schedule rows",
            "Payment-availability reconciliation errors",
            "Timing reconciliation errors",
            "Amount reconciliation errors",
            "Late-payment schedules",
            "30+ late schedules",
            "90+ late schedules",
            "Extreme payment-ratio schedules",
            "Maximum raw days late",
            "Maximum capped days late",
            "Maximum capped days early",
            "Maximum capped payment ratio",
            "Features added",
            "Current memory usage (MB)",
        ],
        "value": [
            len(installment_schedule),
            (
                installment_schedule[
                    "INST_PAYMENT_AVAILABLE"
                ].sum()
                - int(payment_available_mask.sum())
            ),
            timing_reconciliation_difference.ne(0).sum(),
            amount_reconciliation_difference.ne(0).sum(),
            installment_schedule[
                "INST_LATE_PAYMENT_FLAG"
            ].sum(),
            installment_schedule[
                "INST_30PLUS_LATE_FLAG"
            ].sum(),
            installment_schedule[
                "INST_90PLUS_LATE_FLAG"
            ].sum(),
            installment_schedule[
                "INST_PAYMENT_RATIO_EXTREME_FLAG"
            ].sum(),
            installment_schedule[
                "INST_DAYS_LATE_RAW"
            ].max(),
            installment_schedule[
                "INST_DAYS_LATE_CAP_120"
            ].max(),
            installment_schedule[
                "INST_DAYS_EARLY_CAP_90"
            ].max(),
            installment_schedule[
                "INST_PAYMENT_RATIO_CAP_2"
            ].max(),
            len(
                set(installment_schedule.columns)
                - columns_before_engineering
            ),
            (
                installment_schedule
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

assert timing_reconciliation_difference.ne(0).sum() == 0
assert amount_reconciliation_difference.ne(0).sum() == 0

del payment_delay_days
del payment_difference
del payment_ratio_raw
del timing_reconciliation_difference
del amount_reconciliation_difference
gc.collect()

display(installment_feature_check.round(2))

In [ ]:
display(
    installment_feature_check.round(2)
)

In [ ]:
import gc
import numpy as np
import pandas as pd

payment_part_candidates = [
    "INST_PAYMENT_PART_COUNT",
    "INST_PAYMENT_PARTS",
    "INST_SOURCE_ROW_COUNT",
]

payment_part_count_column = next(
    (
        column
        for column in payment_part_candidates
        if column in installment_schedule.columns
    ),
    None,
)

assert payment_part_count_column is not None

# Source values for later reconciliation
source_schedule_rows = len(
    installment_schedule
)

source_payment_part_rows = int(
    installment_schedule[
        payment_part_count_column
    ].sum()
)

source_late_count = int(
    installment_schedule[
        "INST_LATE_PAYMENT_FLAG"
    ].sum()
)

source_30plus_count = int(
    installment_schedule[
        "INST_30PLUS_LATE_FLAG"
    ].sum()
)

source_90plus_count = int(
    installment_schedule[
        "INST_90PLUS_LATE_FLAG"
    ].sum()
)

source_extreme_ratio_count = int(
    installment_schedule[
        "INST_PAYMENT_RATIO_EXTREME_FLAG"
    ].sum()
)


# One row per previous credit
installment_credit_grouped = (
    installment_schedule.groupby(
        "SK_ID_PREV",
        sort=False,
    )
)

installment_credit_features = (
    installment_credit_grouped.agg(
        SK_ID_CURR=(
            "SK_ID_CURR",
            "first",
        ),
        INST_CREDIT_CUSTOMER_NUNIQUE=(
            "SK_ID_CURR",
            "nunique",
        ),
        INST_CREDIT_SCHEDULE_COUNT=(
            "SK_ID_CURR",
            "size",
        ),
        INST_CREDIT_PAYMENT_PART_ROW_COUNT=(
            payment_part_count_column,
            "sum",
        ),
        INST_CREDIT_PAYMENT_AVAILABLE_COUNT=(
            "INST_PAYMENT_AVAILABLE",
            "sum",
        ),
        INST_CREDIT_SPLIT_PAYMENT_COUNT=(
            "INST_SPLIT_PAYMENT_FLAG",
            "sum",
        ),
        INST_CREDIT_EARLY_PAYMENT_COUNT=(
            "INST_EARLY_PAYMENT_FLAG",
            "sum",
        ),
        INST_CREDIT_ON_TIME_PAYMENT_COUNT=(
            "INST_ON_TIME_PAYMENT_FLAG",
            "sum",
        ),
        INST_CREDIT_LATE_PAYMENT_COUNT=(
            "INST_LATE_PAYMENT_FLAG",
            "sum",
        ),
        INST_CREDIT_30PLUS_LATE_COUNT=(
            "INST_30PLUS_LATE_FLAG",
            "sum",
        ),
        INST_CREDIT_90PLUS_LATE_COUNT=(
            "INST_90PLUS_LATE_FLAG",
            "sum",
        ),
        INST_CREDIT_UNDERPAYMENT_COUNT=(
            "INST_UNDERPAYMENT_FLAG",
            "sum",
        ),
        INST_CREDIT_OVERPAYMENT_COUNT=(
            "INST_OVERPAYMENT_FLAG",
            "sum",
        ),
        INST_CREDIT_EQUAL_PAYMENT_COUNT=(
            "INST_EQUAL_PAYMENT_FLAG",
            "sum",
        ),
        INST_CREDIT_RATIO_AVAILABLE_COUNT=(
            "INST_PAYMENT_RATIO_AVAILABLE",
            "sum",
        ),
        INST_CREDIT_EXTREME_RATIO_COUNT=(
            "INST_PAYMENT_RATIO_EXTREME_FLAG",
            "sum",
        ),
        INST_CREDIT_ZERO_SCHEDULED_AMOUNT_COUNT=(
            "INST_ZERO_SCHEDULED_AMOUNT_FLAG",
            "sum",
        ),
        INST_CREDIT_MIN_SCHEDULED_DAY=(
            "INST_SCHEDULED_DAY",
            "min",
        ),
        INST_CREDIT_MAX_SCHEDULED_DAY=(
            "INST_SCHEDULED_DAY",
            "max",
        ),
        INST_CREDIT_LAST_PAYMENT_DAY=(
            "INST_LAST_PAYMENT_DAY",
            "max",
        ),
        INST_CREDIT_TOTAL_SCHEDULED_AMOUNT=(
            "INST_SCHEDULED_AMOUNT",
            "sum",
        ),
        INST_CREDIT_MEAN_SCHEDULED_AMOUNT=(
            "INST_SCHEDULED_AMOUNT",
            "mean",
        ),
        INST_CREDIT_MAX_SCHEDULED_AMOUNT=(
            "INST_SCHEDULED_AMOUNT",
            "max",
        ),
        INST_CREDIT_TOTAL_PAYMENT_AMOUNT=(
            "INST_TOTAL_PAYMENT",
            "sum",
        ),
        INST_CREDIT_MEAN_PAYMENT_AMOUNT=(
            "INST_TOTAL_PAYMENT",
            "mean",
        ),
        INST_CREDIT_MAX_PAYMENT_AMOUNT=(
            "INST_TOTAL_PAYMENT",
            "max",
        ),
        INST_CREDIT_TOTAL_UNDERPAYMENT_AMOUNT=(
            "INST_UNDERPAYMENT_AMOUNT",
            "sum",
        ),
        INST_CREDIT_TOTAL_OVERPAYMENT_AMOUNT=(
            "INST_OVERPAYMENT_AMOUNT",
            "sum",
        ),
        INST_CREDIT_MEAN_DAYS_LATE_RAW=(
            "INST_DAYS_LATE_RAW",
            "mean",
        ),
        INST_CREDIT_MAX_DAYS_LATE_RAW=(
            "INST_DAYS_LATE_RAW",
            "max",
        ),
        INST_CREDIT_MEAN_DAYS_LATE_CAP_120=(
            "INST_DAYS_LATE_CAP_120",
            "mean",
        ),
        INST_CREDIT_MAX_DAYS_LATE_CAP_120=(
            "INST_DAYS_LATE_CAP_120",
            "max",
        ),
        INST_CREDIT_MEAN_LOG1P_DAYS_LATE=(
            "INST_LOG1P_DAYS_LATE",
            "mean",
        ),
        INST_CREDIT_MAX_LOG1P_DAYS_LATE=(
            "INST_LOG1P_DAYS_LATE",
            "max",
        ),
        INST_CREDIT_MEAN_DAYS_EARLY_CAP_90=(
            "INST_DAYS_EARLY_CAP_90",
            "mean",
        ),
        INST_CREDIT_MAX_DAYS_EARLY_CAP_90=(
            "INST_DAYS_EARLY_CAP_90",
            "max",
        ),
        INST_CREDIT_MEAN_PAYMENT_RATIO_CAP_2=(
            "INST_PAYMENT_RATIO_CAP_2",
            "mean",
        ),
        INST_CREDIT_MAX_PAYMENT_RATIO_CAP_2=(
            "INST_PAYMENT_RATIO_CAP_2",
            "max",
        ),
    )
)

# Credits with no recorded payment must remain missing
no_payment_credit_mask = (
    installment_credit_features[
        "INST_CREDIT_PAYMENT_AVAILABLE_COUNT"
    ].eq(0)
)

payment_amount_columns = [
    "INST_CREDIT_TOTAL_PAYMENT_AMOUNT",
    "INST_CREDIT_MEAN_PAYMENT_AMOUNT",
    "INST_CREDIT_MAX_PAYMENT_AMOUNT",
    "INST_CREDIT_TOTAL_UNDERPAYMENT_AMOUNT",
    "INST_CREDIT_TOTAL_OVERPAYMENT_AMOUNT",
]

installment_credit_features.loc[
    no_payment_credit_mask,
    payment_amount_columns,
] = np.nan


# Positive recency and history-span features
installment_credit_features[
    "INST_CREDIT_SCHEDULE_RECENCY_DAYS"
] = (
    -installment_credit_features[
        "INST_CREDIT_MAX_SCHEDULED_DAY"
    ]
).astype("float32")

installment_credit_features[
    "INST_CREDIT_PAYMENT_RECENCY_DAYS"
] = (
    -installment_credit_features[
        "INST_CREDIT_LAST_PAYMENT_DAY"
    ]
).astype("float32")

installment_credit_features[
    "INST_CREDIT_SCHEDULE_HISTORY_SPAN_DAYS"
] = (
    installment_credit_features[
        "INST_CREDIT_MAX_SCHEDULED_DAY"
    ]
    - installment_credit_features[
        "INST_CREDIT_MIN_SCHEDULED_DAY"
    ]
).astype("float32")


# Count columns
credit_count_columns = [
    "INST_CREDIT_SCHEDULE_COUNT",
    "INST_CREDIT_PAYMENT_PART_ROW_COUNT",
    "INST_CREDIT_PAYMENT_AVAILABLE_COUNT",
    "INST_CREDIT_SPLIT_PAYMENT_COUNT",
    "INST_CREDIT_EARLY_PAYMENT_COUNT",
    "INST_CREDIT_ON_TIME_PAYMENT_COUNT",
    "INST_CREDIT_LATE_PAYMENT_COUNT",
    "INST_CREDIT_30PLUS_LATE_COUNT",
    "INST_CREDIT_90PLUS_LATE_COUNT",
    "INST_CREDIT_UNDERPAYMENT_COUNT",
    "INST_CREDIT_OVERPAYMENT_COUNT",
    "INST_CREDIT_EQUAL_PAYMENT_COUNT",
    "INST_CREDIT_RATIO_AVAILABLE_COUNT",
    "INST_CREDIT_EXTREME_RATIO_COUNT",
    "INST_CREDIT_ZERO_SCHEDULED_AMOUNT_COUNT",
]

installment_credit_features[
    credit_count_columns
] = installment_credit_features[
    credit_count_columns
].astype("int32")


# Shares with the appropriate denominator
schedule_share_columns = {
    "INST_CREDIT_SPLIT_PAYMENT_COUNT":
        "INST_CREDIT_SPLIT_PAYMENT_SHARE",
    "INST_CREDIT_ZERO_SCHEDULED_AMOUNT_COUNT":
        "INST_CREDIT_ZERO_SCHEDULED_AMOUNT_SHARE",
}

for count_column, share_column in (
    schedule_share_columns.items()
):
    installment_credit_features[
        share_column
    ] = (
        installment_credit_features[
            count_column
        ]
        / installment_credit_features[
            "INST_CREDIT_SCHEDULE_COUNT"
        ]
    ).astype("float32")


payment_share_columns = {
    "INST_CREDIT_EARLY_PAYMENT_COUNT":
        "INST_CREDIT_EARLY_PAYMENT_SHARE",
    "INST_CREDIT_ON_TIME_PAYMENT_COUNT":
        "INST_CREDIT_ON_TIME_PAYMENT_SHARE",
    "INST_CREDIT_LATE_PAYMENT_COUNT":
        "INST_CREDIT_LATE_PAYMENT_SHARE",
    "INST_CREDIT_30PLUS_LATE_COUNT":
        "INST_CREDIT_30PLUS_LATE_SHARE",
    "INST_CREDIT_90PLUS_LATE_COUNT":
        "INST_CREDIT_90PLUS_LATE_SHARE",
    "INST_CREDIT_UNDERPAYMENT_COUNT":
        "INST_CREDIT_UNDERPAYMENT_SHARE",
    "INST_CREDIT_OVERPAYMENT_COUNT":
        "INST_CREDIT_OVERPAYMENT_SHARE",
    "INST_CREDIT_EQUAL_PAYMENT_COUNT":
        "INST_CREDIT_EQUAL_PAYMENT_SHARE",
}

payment_denominator = (
    installment_credit_features[
        "INST_CREDIT_PAYMENT_AVAILABLE_COUNT"
    ].replace(0, np.nan)
)

for count_column, share_column in (
    payment_share_columns.items()
):
    installment_credit_features[
        share_column
    ] = (
        installment_credit_features[
            count_column
        ]
        / payment_denominator
    ).astype("float32")

installment_credit_features[
    "INST_CREDIT_PAYMENT_COVERAGE"
] = (
    installment_credit_features[
        "INST_CREDIT_PAYMENT_AVAILABLE_COUNT"
    ]
    / installment_credit_features[
        "INST_CREDIT_SCHEDULE_COUNT"
    ]
).astype("float32")

ratio_denominator = (
    installment_credit_features[
        "INST_CREDIT_RATIO_AVAILABLE_COUNT"
    ].replace(0, np.nan)
)

installment_credit_features[
    "INST_CREDIT_EXTREME_RATIO_SHARE"
] = (
    installment_credit_features[
        "INST_CREDIT_EXTREME_RATIO_COUNT"
    ]
    / ratio_denominator
).astype("float32")


# Total paid / scheduled ratio
credit_total_ratio_raw = (
    installment_credit_features[
        "INST_CREDIT_TOTAL_PAYMENT_AMOUNT"
    ]
    / installment_credit_features[
        "INST_CREDIT_TOTAL_SCHEDULED_AMOUNT"
    ]
).where(
    installment_credit_features[
        "INST_CREDIT_TOTAL_SCHEDULED_AMOUNT"
    ].gt(0)
    & ~no_payment_credit_mask
)

installment_credit_features[
    "INST_CREDIT_TOTAL_PAYMENT_RATIO_CAP_2"
] = (
    credit_total_ratio_raw
    .clip(lower=0, upper=2)
    .astype("float32")
)

installment_credit_features[
    "INST_CREDIT_TOTAL_RATIO_EXTREME_FLAG"
] = (
    credit_total_ratio_raw.gt(2)
    .fillna(False)
    .astype("int8")
)


# Reconciliation
credit_timing_difference = (
    installment_credit_features[
        [
            "INST_CREDIT_EARLY_PAYMENT_COUNT",
            "INST_CREDIT_ON_TIME_PAYMENT_COUNT",
            "INST_CREDIT_LATE_PAYMENT_COUNT",
        ]
    ].sum(axis=1)
    - installment_credit_features[
        "INST_CREDIT_PAYMENT_AVAILABLE_COUNT"
    ]
)

credit_amount_difference = (
    installment_credit_features[
        [
            "INST_CREDIT_UNDERPAYMENT_COUNT",
            "INST_CREDIT_OVERPAYMENT_COUNT",
            "INST_CREDIT_EQUAL_PAYMENT_COUNT",
        ]
    ].sum(axis=1)
    - installment_credit_features[
        "INST_CREDIT_PAYMENT_AVAILABLE_COUNT"
    ]
)

customer_mapping_conflicts = int(
    installment_credit_features[
        "INST_CREDIT_CUSTOMER_NUNIQUE"
    ].ne(1).sum()
)

assert installment_credit_features.index.is_unique
assert customer_mapping_conflicts == 0
assert credit_timing_difference.ne(0).sum() == 0
assert credit_amount_difference.ne(0).sum() == 0
assert (
    installment_credit_features[
        "INST_CREDIT_SCHEDULE_COUNT"
    ].sum()
    == source_schedule_rows
)
assert (
    installment_credit_features[
        "INST_CREDIT_PAYMENT_PART_ROW_COUNT"
    ].sum()
    == source_payment_part_rows
)

installment_credit_features.drop(
    columns=[
        "INST_CREDIT_CUSTOMER_NUNIQUE",
        "INST_CREDIT_MIN_SCHEDULED_DAY",
        "INST_CREDIT_MAX_SCHEDULED_DAY",
        "INST_CREDIT_LAST_PAYMENT_DAY",
    ],
    inplace=True,
)


# Save and verify
installment_credit_path = (
    processed_data_path
    / "installment_credit_features.parquet"
)

installment_credit_output = (
    installment_credit_features
    .reset_index()
)

installment_credit_output.to_parquet(
    installment_credit_path,
    index=False,
    compression="snappy",
)

credit_verification = pd.read_parquet(
    installment_credit_path,
    columns=[
        "SK_ID_PREV",
        "SK_ID_CURR",
        "INST_CREDIT_SCHEDULE_COUNT",
    ],
)

installment_credit_check = pd.DataFrame(
    {
        "metric": [
            "Credit feature rows",
            "Unique credit IDs",
            "Duplicate credit IDs",
            "Customer mapping conflicts",
            "Reconciled schedule rows",
            "Source schedule rows",
            "Reconciled payment-part rows",
            "Source payment rows",
            "Timing reconciliation errors",
            "Amount reconciliation errors",
            "Reconciled late schedules",
            "Reconciled 30+ late schedules",
            "Reconciled 90+ late schedules",
            "Reconciled extreme-ratio schedules",
            "Credit feature columns",
            "Verified saved rows",
            "Saved file size (MB)",
            "Credit table memory (MB)",
        ],
        "value": [
            len(installment_credit_features),
            installment_credit_features
            .index.nunique(),
            installment_credit_features
            .index.duplicated().sum(),
            customer_mapping_conflicts,
            installment_credit_features[
                "INST_CREDIT_SCHEDULE_COUNT"
            ].sum(),
            source_schedule_rows,
            installment_credit_features[
                "INST_CREDIT_PAYMENT_PART_ROW_COUNT"
            ].sum(),
            source_payment_part_rows,
            credit_timing_difference.ne(0).sum(),
            credit_amount_difference.ne(0).sum(),
            installment_credit_features[
                "INST_CREDIT_LATE_PAYMENT_COUNT"
            ].sum(),
            installment_credit_features[
                "INST_CREDIT_30PLUS_LATE_COUNT"
            ].sum(),
            installment_credit_features[
                "INST_CREDIT_90PLUS_LATE_COUNT"
            ].sum(),
            installment_credit_features[
                "INST_CREDIT_EXTREME_RATIO_COUNT"
            ].sum(),
            installment_credit_features.shape[1],
            len(credit_verification),
            (
                installment_credit_path.stat().st_size
                / 1024**2
            ),
            (
                installment_credit_features
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

# Release the 1 GB schedule-level table
del installment_credit_grouped
del installment_credit_output
del credit_verification
del credit_timing_difference
del credit_amount_difference
del credit_total_ratio_raw
del payment_denominator
del ratio_denominator
del no_payment_credit_mask
del installment_schedule
gc.collect()

display(installment_credit_check.round(2))

In [ ]:
import gc
import numpy as np
import pandas as pd

# Reload after a kernel restart if necessary
if "installment_credit_features" not in globals():
    installment_credit_features = (
        pd.read_parquet(
            processed_data_path
            / "installment_credit_features.parquet"
        )
        .set_index("SK_ID_PREV")
    )

assert installment_credit_features.index.is_unique

# Source values for reconciliation
source_credit_rows = len(
    installment_credit_features
)

source_schedule_rows = int(
    installment_credit_features[
        "INST_CREDIT_SCHEDULE_COUNT"
    ].sum()
)

source_payment_part_rows = int(
    installment_credit_features[
        "INST_CREDIT_PAYMENT_PART_ROW_COUNT"
    ].sum()
)

source_late_count = int(
    installment_credit_features[
        "INST_CREDIT_LATE_PAYMENT_COUNT"
    ].sum()
)

source_30plus_count = int(
    installment_credit_features[
        "INST_CREDIT_30PLUS_LATE_COUNT"
    ].sum()
)

source_90plus_count = int(
    installment_credit_features[
        "INST_CREDIT_90PLUS_LATE_COUNT"
    ].sum()
)


# Credit-level indicator columns
credit_flag_definitions = {
    "_INST_CREDIT_HAS_PAYMENT_FLAG":
        installment_credit_features[
            "INST_CREDIT_PAYMENT_AVAILABLE_COUNT"
        ].gt(0),

    "_INST_CREDIT_HAS_SPLIT_PAYMENT_FLAG":
        installment_credit_features[
            "INST_CREDIT_SPLIT_PAYMENT_COUNT"
        ].gt(0),

    "_INST_CREDIT_HAS_LATE_PAYMENT_FLAG":
        installment_credit_features[
            "INST_CREDIT_LATE_PAYMENT_COUNT"
        ].gt(0),

    "_INST_CREDIT_HAS_30PLUS_LATE_FLAG":
        installment_credit_features[
            "INST_CREDIT_30PLUS_LATE_COUNT"
        ].gt(0),

    "_INST_CREDIT_HAS_90PLUS_LATE_FLAG":
        installment_credit_features[
            "INST_CREDIT_90PLUS_LATE_COUNT"
        ].gt(0),

    "_INST_CREDIT_HAS_UNDERPAYMENT_FLAG":
        installment_credit_features[
            "INST_CREDIT_UNDERPAYMENT_COUNT"
        ].gt(0),

    "_INST_CREDIT_HAS_OVERPAYMENT_FLAG":
        installment_credit_features[
            "INST_CREDIT_OVERPAYMENT_COUNT"
        ].gt(0),

    "_INST_CREDIT_HAS_EXTREME_RATIO_FLAG":
        installment_credit_features[
            "INST_CREDIT_EXTREME_RATIO_COUNT"
        ].gt(0),

    "_INST_CREDIT_WITHOUT_PAYMENT_FLAG":
        installment_credit_features[
            "INST_CREDIT_PAYMENT_AVAILABLE_COUNT"
        ].eq(0),

    "_INST_CREDIT_INCOMPLETE_PAYMENT_FLAG": (
        installment_credit_features[
            "INST_CREDIT_PAYMENT_AVAILABLE_COUNT"
        ]
        < installment_credit_features[
            "INST_CREDIT_SCHEDULE_COUNT"
        ]
    ),

    "_INST_CREDIT_TOTAL_RATIO_EXTREME_FLAG":
        installment_credit_features[
            "INST_CREDIT_TOTAL_RATIO_EXTREME_FLAG"
        ].eq(1),
}

for column, values in credit_flag_definitions.items():
    installment_credit_features[column] = (
        values.astype("int8")
    )


# Weighted numerators for correct customer-level means
weighted_feature_definitions = {
    "_INST_WEIGHTED_DAYS_LATE_RAW": (
        "INST_CREDIT_MEAN_DAYS_LATE_RAW",
        "INST_CREDIT_PAYMENT_AVAILABLE_COUNT",
    ),
    "_INST_WEIGHTED_DAYS_LATE_CAP_120": (
        "INST_CREDIT_MEAN_DAYS_LATE_CAP_120",
        "INST_CREDIT_PAYMENT_AVAILABLE_COUNT",
    ),
    "_INST_WEIGHTED_LOG1P_DAYS_LATE": (
        "INST_CREDIT_MEAN_LOG1P_DAYS_LATE",
        "INST_CREDIT_PAYMENT_AVAILABLE_COUNT",
    ),
    "_INST_WEIGHTED_DAYS_EARLY_CAP_90": (
        "INST_CREDIT_MEAN_DAYS_EARLY_CAP_90",
        "INST_CREDIT_PAYMENT_AVAILABLE_COUNT",
    ),
    "_INST_WEIGHTED_PAYMENT_RATIO_CAP_2": (
        "INST_CREDIT_MEAN_PAYMENT_RATIO_CAP_2",
        "INST_CREDIT_RATIO_AVAILABLE_COUNT",
    ),
}

for output_column, (
    value_column,
    weight_column,
) in weighted_feature_definitions.items():
    installment_credit_features[
        output_column
    ] = (
        installment_credit_features[value_column]
        * installment_credit_features[weight_column]
    ).astype("float32")


# Columns that are additive across credits
customer_sum_source_columns = [
    "INST_CREDIT_SCHEDULE_COUNT",
    "INST_CREDIT_PAYMENT_PART_ROW_COUNT",
    "INST_CREDIT_PAYMENT_AVAILABLE_COUNT",
    "INST_CREDIT_SPLIT_PAYMENT_COUNT",
    "INST_CREDIT_EARLY_PAYMENT_COUNT",
    "INST_CREDIT_ON_TIME_PAYMENT_COUNT",
    "INST_CREDIT_LATE_PAYMENT_COUNT",
    "INST_CREDIT_30PLUS_LATE_COUNT",
    "INST_CREDIT_90PLUS_LATE_COUNT",
    "INST_CREDIT_UNDERPAYMENT_COUNT",
    "INST_CREDIT_OVERPAYMENT_COUNT",
    "INST_CREDIT_EQUAL_PAYMENT_COUNT",
    "INST_CREDIT_RATIO_AVAILABLE_COUNT",
    "INST_CREDIT_EXTREME_RATIO_COUNT",
    "INST_CREDIT_ZERO_SCHEDULED_AMOUNT_COUNT",
    "INST_CREDIT_TOTAL_SCHEDULED_AMOUNT",
    "INST_CREDIT_TOTAL_PAYMENT_AMOUNT",
    "INST_CREDIT_TOTAL_UNDERPAYMENT_AMOUNT",
    "INST_CREDIT_TOTAL_OVERPAYMENT_AMOUNT",
]

aggregation_specification = {
    "INST_CUSTOMER_CREDIT_COUNT": (
        "INST_CREDIT_SCHEDULE_COUNT",
        "size",
    )
}

for source_column in customer_sum_source_columns:
    output_column = source_column.replace(
        "INST_CREDIT_",
        "INST_CUSTOMER_",
    )

    aggregation_specification[
        output_column
    ] = (
        source_column,
        "sum",
    )


# Count credits exhibiting each behaviour
credit_flag_output_map = {
    "INST_CUSTOMER_CREDITS_WITH_PAYMENT_COUNT":
        "_INST_CREDIT_HAS_PAYMENT_FLAG",
    "INST_CUSTOMER_CREDITS_WITH_SPLIT_PAYMENT_COUNT":
        "_INST_CREDIT_HAS_SPLIT_PAYMENT_FLAG",
    "INST_CUSTOMER_CREDITS_WITH_LATE_PAYMENT_COUNT":
        "_INST_CREDIT_HAS_LATE_PAYMENT_FLAG",
    "INST_CUSTOMER_CREDITS_WITH_30PLUS_LATE_COUNT":
        "_INST_CREDIT_HAS_30PLUS_LATE_FLAG",
    "INST_CUSTOMER_CREDITS_WITH_90PLUS_LATE_COUNT":
        "_INST_CREDIT_HAS_90PLUS_LATE_FLAG",
    "INST_CUSTOMER_CREDITS_WITH_UNDERPAYMENT_COUNT":
        "_INST_CREDIT_HAS_UNDERPAYMENT_FLAG",
    "INST_CUSTOMER_CREDITS_WITH_OVERPAYMENT_COUNT":
        "_INST_CREDIT_HAS_OVERPAYMENT_FLAG",
    "INST_CUSTOMER_CREDITS_WITH_EXTREME_RATIO_COUNT":
        "_INST_CREDIT_HAS_EXTREME_RATIO_FLAG",
    "INST_CUSTOMER_CREDITS_WITHOUT_PAYMENT_COUNT":
        "_INST_CREDIT_WITHOUT_PAYMENT_FLAG",
    "INST_CUSTOMER_CREDITS_WITH_INCOMPLETE_PAYMENT_COUNT":
        "_INST_CREDIT_INCOMPLETE_PAYMENT_FLAG",
    "INST_CUSTOMER_CREDITS_WITH_TOTAL_RATIO_EXTREME_COUNT":
        "_INST_CREDIT_TOTAL_RATIO_EXTREME_FLAG",
}

for (
    output_column,
    source_column,
) in credit_flag_output_map.items():
    aggregation_specification[
        output_column
    ] = (
        source_column,
        "sum",
    )


# Maxima, recency and history characteristics
aggregation_specification.update(
    {
        "INST_CUSTOMER_MAX_SCHEDULED_AMOUNT": (
            "INST_CREDIT_MAX_SCHEDULED_AMOUNT",
            "max",
        ),
        "INST_CUSTOMER_MAX_PAYMENT_AMOUNT": (
            "INST_CREDIT_MAX_PAYMENT_AMOUNT",
            "max",
        ),
        "INST_CUSTOMER_MAX_DAYS_LATE_RAW": (
            "INST_CREDIT_MAX_DAYS_LATE_RAW",
            "max",
        ),
        "INST_CUSTOMER_MAX_DAYS_LATE_CAP_120": (
            "INST_CREDIT_MAX_DAYS_LATE_CAP_120",
            "max",
        ),
        "INST_CUSTOMER_MAX_LOG1P_DAYS_LATE": (
            "INST_CREDIT_MAX_LOG1P_DAYS_LATE",
            "max",
        ),
        "INST_CUSTOMER_MAX_DAYS_EARLY_CAP_90": (
            "INST_CREDIT_MAX_DAYS_EARLY_CAP_90",
            "max",
        ),
        "INST_CUSTOMER_MAX_PAYMENT_RATIO_CAP_2": (
            "INST_CREDIT_MAX_PAYMENT_RATIO_CAP_2",
            "max",
        ),
        "INST_CUSTOMER_MIN_SCHEDULE_RECENCY_DAYS": (
            "INST_CREDIT_SCHEDULE_RECENCY_DAYS",
            "min",
        ),
        "INST_CUSTOMER_MEAN_SCHEDULE_RECENCY_DAYS": (
            "INST_CREDIT_SCHEDULE_RECENCY_DAYS",
            "mean",
        ),
        "INST_CUSTOMER_MAX_SCHEDULE_RECENCY_DAYS": (
            "INST_CREDIT_SCHEDULE_RECENCY_DAYS",
            "max",
        ),
        "INST_CUSTOMER_MIN_PAYMENT_RECENCY_DAYS": (
            "INST_CREDIT_PAYMENT_RECENCY_DAYS",
            "min",
        ),
        "INST_CUSTOMER_MEAN_PAYMENT_RECENCY_DAYS": (
            "INST_CREDIT_PAYMENT_RECENCY_DAYS",
            "mean",
        ),
        "INST_CUSTOMER_MAX_PAYMENT_RECENCY_DAYS": (
            "INST_CREDIT_PAYMENT_RECENCY_DAYS",
            "max",
        ),
        "INST_CUSTOMER_MEAN_SCHEDULE_HISTORY_SPAN_DAYS": (
            "INST_CREDIT_SCHEDULE_HISTORY_SPAN_DAYS",
            "mean",
        ),
        "INST_CUSTOMER_MAX_SCHEDULE_HISTORY_SPAN_DAYS": (
            "INST_CREDIT_SCHEDULE_HISTORY_SPAN_DAYS",
            "max",
        ),
        "_INST_CUSTOMER_WEIGHTED_DAYS_LATE_RAW": (
            "_INST_WEIGHTED_DAYS_LATE_RAW",
            "sum",
        ),
        "_INST_CUSTOMER_WEIGHTED_DAYS_LATE_CAP_120": (
            "_INST_WEIGHTED_DAYS_LATE_CAP_120",
            "sum",
        ),
        "_INST_CUSTOMER_WEIGHTED_LOG1P_DAYS_LATE": (
            "_INST_WEIGHTED_LOG1P_DAYS_LATE",
            "sum",
        ),
        "_INST_CUSTOMER_WEIGHTED_DAYS_EARLY_CAP_90": (
            "_INST_WEIGHTED_DAYS_EARLY_CAP_90",
            "sum",
        ),
        "_INST_CUSTOMER_WEIGHTED_PAYMENT_RATIO_CAP_2": (
            "_INST_WEIGHTED_PAYMENT_RATIO_CAP_2",
            "sum",
        ),
    }
)


# Aggregate to one row per customer
installment_customer_features = (
    installment_credit_features
    .groupby(
        "SK_ID_CURR",
        sort=False,
    )
    .agg(**aggregation_specification)
)


# Keep totals missing when no payment exists
no_customer_payment_mask = (
    installment_customer_features[
        "INST_CUSTOMER_PAYMENT_AVAILABLE_COUNT"
    ].eq(0)
)

customer_payment_amount_columns = [
    "INST_CUSTOMER_TOTAL_PAYMENT_AMOUNT",
    "INST_CUSTOMER_TOTAL_UNDERPAYMENT_AMOUNT",
    "INST_CUSTOMER_TOTAL_OVERPAYMENT_AMOUNT",
]

installment_customer_features.loc[
    no_customer_payment_mask,
    customer_payment_amount_columns,
] = np.nan


# Correct weighted customer-level averages
payment_denominator = (
    installment_customer_features[
        "INST_CUSTOMER_PAYMENT_AVAILABLE_COUNT"
    ].replace(0, np.nan)
)

ratio_denominator = (
    installment_customer_features[
        "INST_CUSTOMER_RATIO_AVAILABLE_COUNT"
    ].replace(0, np.nan)
)

installment_customer_features[
    "INST_CUSTOMER_MEAN_DAYS_LATE_RAW"
] = (
    installment_customer_features[
        "_INST_CUSTOMER_WEIGHTED_DAYS_LATE_RAW"
    ]
    / payment_denominator
).astype("float32")

installment_customer_features[
    "INST_CUSTOMER_MEAN_DAYS_LATE_CAP_120"
] = (
    installment_customer_features[
        "_INST_CUSTOMER_WEIGHTED_DAYS_LATE_CAP_120"
    ]
    / payment_denominator
).astype("float32")

installment_customer_features[
    "INST_CUSTOMER_MEAN_LOG1P_DAYS_LATE"
] = (
    installment_customer_features[
        "_INST_CUSTOMER_WEIGHTED_LOG1P_DAYS_LATE"
    ]
    / payment_denominator
).astype("float32")

installment_customer_features[
    "INST_CUSTOMER_MEAN_DAYS_EARLY_CAP_90"
] = (
    installment_customer_features[
        "_INST_CUSTOMER_WEIGHTED_DAYS_EARLY_CAP_90"
    ]
    / payment_denominator
).astype("float32")

installment_customer_features[
    "INST_CUSTOMER_MEAN_PAYMENT_RATIO_CAP_2"
] = (
    installment_customer_features[
        "_INST_CUSTOMER_WEIGHTED_PAYMENT_RATIO_CAP_2"
    ]
    / ratio_denominator
).astype("float32")


# Overall monetary means
installment_customer_features[
    "INST_CUSTOMER_MEAN_SCHEDULED_AMOUNT"
] = (
    installment_customer_features[
        "INST_CUSTOMER_TOTAL_SCHEDULED_AMOUNT"
    ]
    / installment_customer_features[
        "INST_CUSTOMER_SCHEDULE_COUNT"
    ]
).astype("float32")

installment_customer_features[
    "INST_CUSTOMER_MEAN_PAYMENT_AMOUNT"
] = (
    installment_customer_features[
        "INST_CUSTOMER_TOTAL_PAYMENT_AMOUNT"
    ]
    / payment_denominator
).astype("float32")


# Schedule-level shares
schedule_share_map = {
    "INST_CUSTOMER_SPLIT_PAYMENT_COUNT":
        "INST_CUSTOMER_SPLIT_PAYMENT_SHARE",
    "INST_CUSTOMER_ZERO_SCHEDULED_AMOUNT_COUNT":
        "INST_CUSTOMER_ZERO_SCHEDULED_AMOUNT_SHARE",
}

for count_column, share_column in (
    schedule_share_map.items()
):
    installment_customer_features[
        share_column
    ] = (
        installment_customer_features[count_column]
        / installment_customer_features[
            "INST_CUSTOMER_SCHEDULE_COUNT"
        ]
    ).astype("float32")


# Payment-level shares
payment_share_map = {
    "INST_CUSTOMER_EARLY_PAYMENT_COUNT":
        "INST_CUSTOMER_EARLY_PAYMENT_SHARE",
    "INST_CUSTOMER_ON_TIME_PAYMENT_COUNT":
        "INST_CUSTOMER_ON_TIME_PAYMENT_SHARE",
    "INST_CUSTOMER_LATE_PAYMENT_COUNT":
        "INST_CUSTOMER_LATE_PAYMENT_SHARE",
    "INST_CUSTOMER_30PLUS_LATE_COUNT":
        "INST_CUSTOMER_30PLUS_LATE_SHARE",
    "INST_CUSTOMER_90PLUS_LATE_COUNT":
        "INST_CUSTOMER_90PLUS_LATE_SHARE",
    "INST_CUSTOMER_UNDERPAYMENT_COUNT":
        "INST_CUSTOMER_UNDERPAYMENT_SHARE",
    "INST_CUSTOMER_OVERPAYMENT_COUNT":
        "INST_CUSTOMER_OVERPAYMENT_SHARE",
    "INST_CUSTOMER_EQUAL_PAYMENT_COUNT":
        "INST_CUSTOMER_EQUAL_PAYMENT_SHARE",
}

for count_column, share_column in (
    payment_share_map.items()
):
    installment_customer_features[
        share_column
    ] = (
        installment_customer_features[count_column]
        / payment_denominator
    ).astype("float32")

installment_customer_features[
    "INST_CUSTOMER_PAYMENT_COVERAGE"
] = (
    installment_customer_features[
        "INST_CUSTOMER_PAYMENT_AVAILABLE_COUNT"
    ]
    / installment_customer_features[
        "INST_CUSTOMER_SCHEDULE_COUNT"
    ]
).astype("float32")

installment_customer_features[
    "INST_CUSTOMER_EXTREME_RATIO_SHARE"
] = (
    installment_customer_features[
        "INST_CUSTOMER_EXTREME_RATIO_COUNT"
    ]
    / ratio_denominator
).astype("float32")


# Credit-level behaviour shares
customer_credit_denominator = (
    installment_customer_features[
        "INST_CUSTOMER_CREDIT_COUNT"
    ]
)

credit_count_columns = list(
    credit_flag_output_map.keys()
)

for count_column in credit_count_columns:
    share_column = count_column.replace(
        "_COUNT",
        "_SHARE",
    )

    installment_customer_features[
        share_column
    ] = (
        installment_customer_features[count_column]
        / customer_credit_denominator
    ).astype("float32")


# Total payment / scheduled amount
customer_total_ratio_raw = (
    installment_customer_features[
        "INST_CUSTOMER_TOTAL_PAYMENT_AMOUNT"
    ]
    / installment_customer_features[
        "INST_CUSTOMER_TOTAL_SCHEDULED_AMOUNT"
    ]
).where(
    installment_customer_features[
        "INST_CUSTOMER_TOTAL_SCHEDULED_AMOUNT"
    ].gt(0)
    & ~no_customer_payment_mask
)

installment_customer_features[
    "INST_CUSTOMER_TOTAL_PAYMENT_RATIO_CAP_2"
] = (
    customer_total_ratio_raw
    .clip(lower=0, upper=2)
    .astype("float32")
)

installment_customer_features[
    "INST_CUSTOMER_TOTAL_RATIO_EXTREME_FLAG"
] = (
    customer_total_ratio_raw.gt(2)
    .fillna(False)
    .astype("int8")
)

installment_customer_features[
    "INST_CUSTOMER_LOG_CREDIT_COUNT"
] = np.log1p(
    installment_customer_features[
        "INST_CUSTOMER_CREDIT_COUNT"
    ]
).astype("float32")

installment_customer_features[
    "INST_CUSTOMER_LOG_SCHEDULE_COUNT"
] = np.log1p(
    installment_customer_features[
        "INST_CUSTOMER_SCHEDULE_COUNT"
    ]
).astype("float32")


# Remove weighted intermediate columns
weighted_customer_columns = [
    column
    for column in installment_customer_features.columns
    if column.startswith(
        "_INST_CUSTOMER_WEIGHTED_"
    )
]

installment_customer_features.drop(
    columns=weighted_customer_columns,
    inplace=True,
)


# Optimize genuine count columns
customer_count_columns = [
    "INST_CUSTOMER_CREDIT_COUNT",
] + [
    column.replace(
        "INST_CREDIT_",
        "INST_CUSTOMER_",
    )
    for column in customer_sum_source_columns
    if column.endswith("_COUNT")
] + credit_count_columns

customer_count_columns = list(
    dict.fromkeys(customer_count_columns)
)

installment_customer_features[
    customer_count_columns
] = installment_customer_features[
    customer_count_columns
].astype("int32")


# Reconciliation
customer_timing_difference = (
    installment_customer_features[
        [
            "INST_CUSTOMER_EARLY_PAYMENT_COUNT",
            "INST_CUSTOMER_ON_TIME_PAYMENT_COUNT",
            "INST_CUSTOMER_LATE_PAYMENT_COUNT",
        ]
    ].sum(axis=1)
    - installment_customer_features[
        "INST_CUSTOMER_PAYMENT_AVAILABLE_COUNT"
    ]
)

customer_amount_difference = (
    installment_customer_features[
        [
            "INST_CUSTOMER_UNDERPAYMENT_COUNT",
            "INST_CUSTOMER_OVERPAYMENT_COUNT",
            "INST_CUSTOMER_EQUAL_PAYMENT_COUNT",
        ]
    ].sum(axis=1)
    - installment_customer_features[
        "INST_CUSTOMER_PAYMENT_AVAILABLE_COUNT"
    ]
)

assert installment_customer_features.index.is_unique
assert customer_timing_difference.ne(0).sum() == 0
assert customer_amount_difference.ne(0).sum() == 0
assert (
    installment_customer_features[
        "INST_CUSTOMER_CREDIT_COUNT"
    ].sum()
    == source_credit_rows
)
assert (
    installment_customer_features[
        "INST_CUSTOMER_SCHEDULE_COUNT"
    ].sum()
    == source_schedule_rows
)


# Save and verify
installment_customer_path = (
    processed_data_path
    / "installment_customer_features.parquet"
)

installment_customer_output = (
    installment_customer_features
    .reset_index()
)

installment_customer_output.to_parquet(
    installment_customer_path,
    index=False,
    compression="snappy",
)

customer_verification = pd.read_parquet(
    installment_customer_path,
    columns=[
        "SK_ID_CURR",
        "INST_CUSTOMER_CREDIT_COUNT",
        "INST_CUSTOMER_SCHEDULE_COUNT",
    ],
)

installment_customer_check = pd.DataFrame(
    {
        "metric": [
            "Customer feature rows",
            "Unique customer index",
            "Duplicate customer index",
            "Reconciled credit count",
            "Source credit rows",
            "Reconciled schedule count",
            "Source schedule rows",
            "Reconciled payment-part count",
            "Source payment rows",
            "Timing reconciliation errors",
            "Amount reconciliation errors",
            "Reconciled late schedules",
            "Reconciled 30+ late schedules",
            "Reconciled 90+ late schedules",
            "Customers with any late payment",
            "Customers with 30+ late payment",
            "Customers with 90+ late payment",
            "Customers without recorded payment",
            "Customer feature columns",
            "Verified saved rows",
            "Saved file size (MB)",
            "Customer table memory (MB)",
        ],
        "value": [
            len(installment_customer_features),
            installment_customer_features
            .index.nunique(),
            installment_customer_features
            .index.duplicated().sum(),
            installment_customer_features[
                "INST_CUSTOMER_CREDIT_COUNT"
            ].sum(),
            source_credit_rows,
            installment_customer_features[
                "INST_CUSTOMER_SCHEDULE_COUNT"
            ].sum(),
            source_schedule_rows,
            installment_customer_features[
                "INST_CUSTOMER_PAYMENT_PART_ROW_COUNT"
            ].sum(),
            source_payment_part_rows,
            customer_timing_difference.ne(0).sum(),
            customer_amount_difference.ne(0).sum(),
            installment_customer_features[
                "INST_CUSTOMER_LATE_PAYMENT_COUNT"
            ].sum(),
            installment_customer_features[
                "INST_CUSTOMER_30PLUS_LATE_COUNT"
            ].sum(),
            installment_customer_features[
                "INST_CUSTOMER_90PLUS_LATE_COUNT"
            ].sum(),
            installment_customer_features[
                "INST_CUSTOMER_LATE_PAYMENT_COUNT"
            ].gt(0).sum(),
            installment_customer_features[
                "INST_CUSTOMER_30PLUS_LATE_COUNT"
            ].gt(0).sum(),
            installment_customer_features[
                "INST_CUSTOMER_90PLUS_LATE_COUNT"
            ].gt(0).sum(),
            no_customer_payment_mask.sum(),
            installment_customer_features.shape[1],
            len(customer_verification),
            (
                installment_customer_path.stat().st_size
                / 1024**2
            ),
            (
                installment_customer_features
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

installment_customer_check.to_csv(
    processed_data_path
    / "installment_customer_feature_check.csv",
    index=False,
)

# Credit table is safely persisted and no longer required
del installment_credit_features
del installment_customer_output
del customer_verification
del customer_timing_difference
del customer_amount_difference
del customer_total_ratio_raw
del payment_denominator
del ratio_denominator
del customer_credit_denominator
del no_customer_payment_mask
gc.collect()

display(installment_customer_check.round(2))

In [ ]:
import gc
import numpy as np
import pandas as pd

base_train_feature_path = (
    processed_data_path
    / "application_train_features_bureau_previous_pos.parquet"
)

installment_customer_path = (
    processed_data_path
    / "installment_customer_features.parquet"
)

previous_customer_path = (
    processed_data_path
    / "previous_application_customer_features.parquet"
)


# Always load clean persisted tables
base_train_features = pd.read_parquet(
    base_train_feature_path
)

installment_customer_data = pd.read_parquet(
    installment_customer_path
)

previous_customer_ids = pd.Index(
    pd.read_parquet(
        previous_customer_path,
        columns=["SK_ID_CURR"],
    )["SK_ID_CURR"]
)


# Key validations
assert base_train_features[
    "SK_ID_CURR"
].is_unique

assert installment_customer_data[
    "SK_ID_CURR"
].is_unique

base_train_rows = len(
    base_train_features
)

base_feature_count = (
    base_train_features.shape[1] - 1
)

installment_feature_columns = [
    column
    for column in installment_customer_data.columns
    if column != "SK_ID_CURR"
]

unexpected_overlap = sorted(
    set(installment_feature_columns)
    .intersection(base_train_features.columns)
)

assert not unexpected_overlap, (
    "Unexpected overlapping installment features: "
    f"{unexpected_overlap}"
)


# Preserve original dtypes before merging
installment_dtype_map = {
    column: installment_customer_data[column].dtype
    for column in installment_feature_columns
}


# Safe one-to-one left merge
application_train_features_final = (
    base_train_features.merge(
        installment_customer_data,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one",
    )
)


# Availability indicator before filling counts
application_train_features_final[
    "INST_HAS_HISTORY"
] = (
    application_train_features_final[
        "INST_CUSTOMER_CREDIT_COUNT"
    ]
    .notna()
    .astype("int8")
)


# Only genuine counts, flags and log-counts become zero
installment_zero_fill_columns = [
    column
    for column in installment_feature_columns
    if (
        column.endswith("_COUNT")
        or column.endswith("_FLAG")
    )
]

installment_zero_fill_columns += [
    "INST_CUSTOMER_LOG_CREDIT_COUNT",
    "INST_CUSTOMER_LOG_SCHEDULE_COUNT",
]

installment_zero_fill_columns = list(
    dict.fromkeys(
        installment_zero_fill_columns
    )
)

for column in installment_zero_fill_columns:
    application_train_features_final[
        column
    ] = (
        application_train_features_final[column]
        .fillna(0)
        .astype(installment_dtype_map[column])
    )


# Cross-source history checks
train_customer_ids = (
    application_train_features_final[
        "SK_ID_CURR"
    ]
)

previous_history_mask = (
    train_customer_ids.isin(
        previous_customer_ids
    )
)

installment_history_mask = (
    application_train_features_final[
        "INST_HAS_HISTORY"
    ].eq(1)
)

previous_without_installment_count = int(
    (
        previous_history_mask
        & ~installment_history_mask
    ).sum()
)

installment_without_previous_count = int(
    (
        installment_history_mask
        & ~previous_history_mask
    ).sum()
)


# Final table validations
assert len(
    application_train_features_final
) == base_train_rows

assert application_train_features_final[
    "SK_ID_CURR"
].is_unique

nulls_in_zero_filled_columns = int(
    application_train_features_final[
        installment_zero_fill_columns
    ].isna().sum().sum()
)

assert nulls_in_zero_filled_columns == 0


# Save final feature table
final_train_feature_path = (
    processed_data_path
    / (
        "application_train_features_"
        "bureau_previous_pos_installments.parquet"
    )
)

application_train_features_final.to_parquet(
    final_train_feature_path,
    index=False,
    compression="snappy",
)


# Verify saved result
final_verification = pd.read_parquet(
    final_train_feature_path,
    columns=[
        "SK_ID_CURR",
        "INST_HAS_HISTORY",
        "INST_CUSTOMER_CREDIT_COUNT",
        "INST_CUSTOMER_LATE_PAYMENT_COUNT",
        "INST_CUSTOMER_90PLUS_LATE_COUNT",
    ],
)


installment_train_join_check = pd.DataFrame(
    {
        "metric": [
            "Base train rows",
            "Final train rows",
            "Row count difference",
            "Unique train IDs",
            "Duplicate train IDs",
            "Unexpected overlapping features",
            "Customers with installment history",
            "Customers without installment history",
            "Previous history without installments",
            "Installment history without previous history",
            "Customers with any late payment",
            "Customers with 30+ late payment",
            "Customers with 90+ late payment",
            "History customers without recorded payment",
            "Nulls in zero-filled columns",
            "Base feature columns",
            "Installment features added",
            "Final feature columns",
            "Verified saved rows",
            "Saved file size (MB)",
            "Final table memory (MB)",
        ],
        "value": [
            base_train_rows,
            len(application_train_features_final),
            (
                len(application_train_features_final)
                - base_train_rows
            ),
            application_train_features_final[
                "SK_ID_CURR"
            ].nunique(),
            application_train_features_final[
                "SK_ID_CURR"
            ].duplicated().sum(),
            len(unexpected_overlap),
            installment_history_mask.sum(),
            (~installment_history_mask).sum(),
            previous_without_installment_count,
            installment_without_previous_count,
            application_train_features_final[
                "INST_CUSTOMER_LATE_PAYMENT_COUNT"
            ].gt(0).sum(),
            application_train_features_final[
                "INST_CUSTOMER_30PLUS_LATE_COUNT"
            ].gt(0).sum(),
            application_train_features_final[
                "INST_CUSTOMER_90PLUS_LATE_COUNT"
            ].gt(0).sum(),
            (
                installment_history_mask
                & application_train_features_final[
                    "INST_CUSTOMER_PAYMENT_AVAILABLE_COUNT"
                ].eq(0)
            ).sum(),
            nulls_in_zero_filled_columns,
            base_feature_count,
            len(installment_feature_columns) + 1,
            (
                application_train_features_final
                .shape[1] - 1
            ),
            len(final_verification),
            (
                final_train_feature_path
                .stat().st_size
                / 1024**2
            ),
            (
                application_train_features_final
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

installment_train_join_check.to_csv(
    processed_data_path
    / "installment_train_join_check.csv",
    index=False,
)


# Clear large intermediate tables
del base_train_features
del installment_customer_data
del installment_customer_features
del application_train_features_final
del final_verification
del previous_customer_ids
gc.collect()

display(
    installment_train_join_check.round(2)
)

In [ ]:
import gc
import numpy as np
import pandas as pd

credit_card_dtype = {
    "SK_ID_PREV": "int32",
    "SK_ID_CURR": "int32",
    "MONTHS_BALANCE": "int16",

    "AMT_BALANCE": "float32",
    "AMT_CREDIT_LIMIT_ACTUAL": "float32",
    "AMT_DRAWINGS_ATM_CURRENT": "float32",
    "AMT_DRAWINGS_CURRENT": "float32",
    "AMT_DRAWINGS_OTHER_CURRENT": "float32",
    "AMT_DRAWINGS_POS_CURRENT": "float32",
    "AMT_INST_MIN_REGULARITY": "float32",
    "AMT_PAYMENT_CURRENT": "float32",
    "AMT_PAYMENT_TOTAL_CURRENT": "float32",
    "AMT_RECEIVABLE_PRINCIPAL": "float32",
    "AMT_RECIVABLE": "float32",
    "AMT_TOTAL_RECEIVABLE": "float32",

    "CNT_DRAWINGS_ATM_CURRENT": "float32",
    "CNT_DRAWINGS_CURRENT": "float32",
    "CNT_DRAWINGS_OTHER_CURRENT": "float32",
    "CNT_DRAWINGS_POS_CURRENT": "float32",
    "CNT_INSTALMENT_MATURE_CUM": "float32",

    "NAME_CONTRACT_STATUS": "category",
    "SK_DPD": "int16",
    "SK_DPD_DEF": "int16",
}

credit_card_balance = pd.read_csv(
    extract_path / "credit_card_balance.csv",
    dtype=credit_card_dtype,
    memory_map=True,
)

# Previous-application ID map
previous_id_map = pd.read_parquet(
    processed_data_path
    / "previous_application_row_features.parquet",
    columns=[
        "SK_ID_PREV",
        "SK_ID_CURR",
    ],
)

assert previous_id_map[
    "SK_ID_PREV"
].is_unique


# Application train + test customer universe
application_train_ids = pd.read_csv(
    extract_path / "application_train.csv",
    usecols=["SK_ID_CURR"],
    dtype={"SK_ID_CURR": "int32"},
)

application_test_ids = pd.read_csv(
    extract_path / "application_test.csv",
    usecols=["SK_ID_CURR"],
    dtype={"SK_ID_CURR": "int32"},
)

application_universe_ids = pd.Index(
    pd.concat(
        [
            application_train_ids[
                "SK_ID_CURR"
            ],
            application_test_ids[
                "SK_ID_CURR"
            ],
        ],
        ignore_index=True,
    ).unique()
)


# Credit-card ID/customer relationship
credit_card_id_customer_counts = (
    credit_card_balance.groupby(
        "SK_ID_PREV",
        sort=False,
    )["SK_ID_CURR"]
    .nunique()
)

internal_mapping_conflicts = int(
    credit_card_id_customer_counts
    .gt(1)
    .sum()
)

assert internal_mapping_conflicts == 0

credit_card_id_map = (
    credit_card_balance[
        [
            "SK_ID_PREV",
            "SK_ID_CURR",
        ]
    ]
    .drop_duplicates(
        subset="SK_ID_PREV"
    )
)

credit_card_previous_ids = pd.Index(
    credit_card_id_map[
        "SK_ID_PREV"
    ]
)

previous_application_ids = pd.Index(
    previous_id_map[
        "SK_ID_PREV"
    ]
)

matched_previous_ids = (
    credit_card_previous_ids
    .intersection(previous_application_ids)
)

unmapped_previous_ids = (
    credit_card_previous_ids
    .difference(previous_application_ids)
)

previous_mapped_row_mask = (
    credit_card_balance[
        "SK_ID_PREV"
    ].isin(previous_application_ids)
)

mapped_credit_card_rows = int(
    previous_mapped_row_mask.sum()
)

unmapped_credit_card_rows = int(
    (~previous_mapped_row_mask).sum()
)


# Verify customer IDs for matched previous applications
matched_mapping_check = (
    credit_card_id_map[
        credit_card_id_map[
            "SK_ID_PREV"
        ].isin(matched_previous_ids)
    ]
    .merge(
        previous_id_map,
        on="SK_ID_PREV",
        how="inner",
        suffixes=(
            "_CREDIT_CARD",
            "_PREVIOUS",
        ),
        validate="one_to_one",
    )
)

matched_customer_mapping_conflicts = int(
    (
        matched_mapping_check[
            "SK_ID_CURR_CREDIT_CARD"
        ]
        != matched_mapping_check[
            "SK_ID_CURR_PREVIOUS"
        ]
    ).sum()
)

assert matched_customer_mapping_conflicts == 0


# Direct SK_ID_CURR determines whether a row is safe
orphan_customer_mask = (
    ~credit_card_balance[
        "SK_ID_CURR"
    ].isin(application_universe_ids)
)

orphan_credit_card_rows = int(
    orphan_customer_mask.sum()
)

orphan_credit_card_customers = int(
    credit_card_balance.loc[
        orphan_customer_mask,
        "SK_ID_CURR",
    ].nunique()
)

rows_safe_to_retain = (
    len(credit_card_balance)
    - orphan_credit_card_rows
)


# Save unmatched previous IDs for audit
unmapped_credit_card_audit = (
    credit_card_id_map[
        credit_card_id_map[
            "SK_ID_PREV"
        ].isin(unmapped_previous_ids)
    ]
    .sort_values("SK_ID_PREV")
)

unmapped_credit_card_audit.to_parquet(
    processed_data_path
    / "credit_card_unmapped_previous_ids.parquet",
    index=False,
    compression="snappy",
)


# Raw-table overview
credit_card_overview = pd.DataFrame(
    {
        "metric": [
            "Total credit-card rows",
            "Total columns",
            "Unique previous application IDs",
            "Unique customer IDs",
            "Duplicate credit-month pairs",
            "Null previous application IDs",
            "Null customer IDs",
            "Null month values",
            "Minimum month",
            "Maximum month",
            "Future month records",
            "Minimum DPD",
            "Negative DPD records",
            "Minimum DPD DEF",
            "Negative DPD DEF records",
            "DPD DEF greater than DPD",
            "Matched previous application IDs",
            "Unmapped previous application IDs",
            "Mapped credit-card rows",
            "Unmapped credit-card rows",
            "Previous ID mapping rate (%)",
            "Previous applications with card history (%)",
            "Internal previous-customer conflicts",
            "Matched customer mapping conflicts",
            "Orphan credit-card customers",
            "Orphan credit-card rows",
            "Rows safe to retain",
            "Contract status categories",
            "Memory usage (MB)",
        ],
        "value": [
            len(credit_card_balance),
            credit_card_balance.shape[1],
            credit_card_balance[
                "SK_ID_PREV"
            ].nunique(),
            credit_card_balance[
                "SK_ID_CURR"
            ].nunique(),
            credit_card_balance.duplicated(
                subset=[
                    "SK_ID_PREV",
                    "MONTHS_BALANCE",
                ]
            ).sum(),
            credit_card_balance[
                "SK_ID_PREV"
            ].isna().sum(),
            credit_card_balance[
                "SK_ID_CURR"
            ].isna().sum(),
            credit_card_balance[
                "MONTHS_BALANCE"
            ].isna().sum(),
            credit_card_balance[
                "MONTHS_BALANCE"
            ].min(),
            credit_card_balance[
                "MONTHS_BALANCE"
            ].max(),
            credit_card_balance[
                "MONTHS_BALANCE"
            ].gt(0).sum(),
            credit_card_balance[
                "SK_DPD"
            ].min(),
            credit_card_balance[
                "SK_DPD"
            ].lt(0).sum(),
            credit_card_balance[
                "SK_DPD_DEF"
            ].min(),
            credit_card_balance[
                "SK_DPD_DEF"
            ].lt(0).sum(),
            (
                credit_card_balance[
                    "SK_DPD_DEF"
                ]
                > credit_card_balance[
                    "SK_DPD"
                ]
            ).sum(),
            len(matched_previous_ids),
            len(unmapped_previous_ids),
            mapped_credit_card_rows,
            unmapped_credit_card_rows,
            (
                len(matched_previous_ids)
                / len(credit_card_previous_ids)
                * 100
            ),
            (
                len(matched_previous_ids)
                / len(previous_application_ids)
                * 100
            ),
            internal_mapping_conflicts,
            matched_customer_mapping_conflicts,
            orphan_credit_card_customers,
            orphan_credit_card_rows,
            rows_safe_to_retain,
            credit_card_balance[
                "NAME_CONTRACT_STATUS"
            ].nunique(),
            (
                credit_card_balance
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

credit_card_overview.to_csv(
    processed_data_path
    / "credit_card_raw_overview.csv",
    index=False,
)

del previous_id_map
del matched_mapping_check
del credit_card_id_customer_counts
del application_train_ids
del application_test_ids
del previous_mapped_row_mask
del orphan_customer_mask
gc.collect()

display(credit_card_overview.round(2))

In [ ]:
import numpy as np
import pandas as pd

# Reusable row-level delinquency features
credit_card_balance[
    "CC_MONTH_RECENCY_DAYS"
] = (
    -credit_card_balance[
        "MONTHS_BALANCE"
    ]
).astype("int16")

credit_card_balance[
    "CC_DPD_FLAG"
] = (
    credit_card_balance["SK_DPD"] > 0
).astype("int8")

credit_card_balance[
    "CC_DPD_30PLUS_FLAG"
] = (
    credit_card_balance["SK_DPD"] >= 30
).astype("int8")

credit_card_balance[
    "CC_DPD_90PLUS_FLAG"
] = (
    credit_card_balance["SK_DPD"] >= 90
).astype("int8")

credit_card_balance[
    "CC_DPD_DEF_FLAG"
] = (
    credit_card_balance[
        "SK_DPD_DEF"
    ] > 0
).astype("int8")


# 1. Contract-status distribution
credit_card_status_summary = (
    credit_card_balance
    .groupby(
        "NAME_CONTRACT_STATUS",
        observed=True,
        sort=False,
    )
    .agg(
        records=(
            "SK_ID_PREV",
            "size",
        ),
        credit_accounts=(
            "SK_ID_PREV",
            "nunique",
        ),
        customers=(
            "SK_ID_CURR",
            "nunique",
        ),
        positive_dpd_records=(
            "CC_DPD_FLAG",
            "sum",
        ),
        dpd_30plus_records=(
            "CC_DPD_30PLUS_FLAG",
            "sum",
        ),
        dpd_90plus_records=(
            "CC_DPD_90PLUS_FLAG",
            "sum",
        ),
        positive_dpd_def_records=(
            "CC_DPD_DEF_FLAG",
            "sum",
        ),
        mean_month_recency=(
            "CC_MONTH_RECENCY_DAYS",
            "mean",
        ),
        mean_balance=(
            "AMT_BALANCE",
            "mean",
        ),
        mean_credit_limit=(
            "AMT_CREDIT_LIMIT_ACTUAL",
            "mean",
        ),
        maximum_dpd=(
            "SK_DPD",
            "max",
        ),
    )
    .reset_index()
)

credit_card_status_summary[
    "record_share_pct"
] = (
    credit_card_status_summary["records"]
    / len(credit_card_balance)
    * 100
)

credit_card_status_summary[
    "positive_dpd_share_pct"
] = (
    credit_card_status_summary[
        "positive_dpd_records"
    ]
    / credit_card_status_summary["records"]
    * 100
)

credit_card_status_summary[
    "dpd_90plus_share_pct"
] = (
    credit_card_status_summary[
        "dpd_90plus_records"
    ]
    / credit_card_status_summary["records"]
    * 100
)

credit_card_status_summary = (
    credit_card_status_summary
    .sort_values(
        "records",
        ascending=False,
    )
    .reset_index(drop=True)
)


# 2. Missing-value summary
credit_card_missing_summary = pd.DataFrame(
    {
        "feature": (
            credit_card_balance.columns
        ),
        "dtype": [
            str(dtype)
            for dtype in credit_card_balance.dtypes
        ],
        "missing_records": (
            credit_card_balance
            .isna()
            .sum()
            .to_numpy()
        ),
        "unique_non_null_values": [
            credit_card_balance[
                column
            ].nunique(dropna=True)
            for column in credit_card_balance.columns
        ],
    }
)

credit_card_missing_summary[
    "missing_share_pct"
] = (
    credit_card_missing_summary[
        "missing_records"
    ]
    / len(credit_card_balance)
    * 100
)

credit_card_missing_summary = (
    credit_card_missing_summary[
        credit_card_missing_summary[
            "missing_records"
        ].gt(0)
    ]
    .sort_values(
        [
            "missing_records",
            "feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


# 3. Overall delinquency audit
credit_card_dpd_summary = pd.DataFrame(
    {
        "metric": [
            "Total credit-card records",
            "Positive DPD records",
            "30+ DPD records",
            "90+ DPD records",
            "Positive DPD DEF records",
            "Credits with any DPD",
            "Credits with 30+ DPD",
            "Credits with 90+ DPD",
            "Customers with any DPD",
            "Customers with 30+ DPD",
            "Customers with 90+ DPD",
            "Maximum DPD",
            "Maximum DPD DEF",
        ],
        "value": [
            len(credit_card_balance),
            credit_card_balance[
                "CC_DPD_FLAG"
            ].sum(),
            credit_card_balance[
                "CC_DPD_30PLUS_FLAG"
            ].sum(),
            credit_card_balance[
                "CC_DPD_90PLUS_FLAG"
            ].sum(),
            credit_card_balance[
                "CC_DPD_DEF_FLAG"
            ].sum(),
            credit_card_balance.loc[
                credit_card_balance[
                    "CC_DPD_FLAG"
                ].eq(1),
                "SK_ID_PREV",
            ].nunique(),
            credit_card_balance.loc[
                credit_card_balance[
                    "CC_DPD_30PLUS_FLAG"
                ].eq(1),
                "SK_ID_PREV",
            ].nunique(),
            credit_card_balance.loc[
                credit_card_balance[
                    "CC_DPD_90PLUS_FLAG"
                ].eq(1),
                "SK_ID_PREV",
            ].nunique(),
            credit_card_balance.loc[
                credit_card_balance[
                    "CC_DPD_FLAG"
                ].eq(1),
                "SK_ID_CURR",
            ].nunique(),
            credit_card_balance.loc[
                credit_card_balance[
                    "CC_DPD_30PLUS_FLAG"
                ].eq(1),
                "SK_ID_CURR",
            ].nunique(),
            credit_card_balance.loc[
                credit_card_balance[
                    "CC_DPD_90PLUS_FLAG"
                ].eq(1),
                "SK_ID_CURR",
            ].nunique(),
            credit_card_balance[
                "SK_DPD"
            ].max(),
            credit_card_balance[
                "SK_DPD_DEF"
            ].max(),
        ],
    }
)


# Save audit evidence
credit_card_status_summary.to_csv(
    processed_data_path
    / "credit_card_status_summary.csv",
    index=False,
)

credit_card_missing_summary.to_csv(
    processed_data_path
    / "credit_card_missing_summary.csv",
    index=False,
)

credit_card_dpd_summary.to_csv(
    processed_data_path
    / "credit_card_dpd_summary.csv",
    index=False,
)

display(
    credit_card_status_summary.round(3)
)

display(
    credit_card_missing_summary.round(3)
)

display(
    credit_card_dpd_summary.round(2)
)

In [ ]:
import numpy as np
import pandas as pd

drawing_component_columns = [
    "AMT_DRAWINGS_ATM_CURRENT",
    "AMT_DRAWINGS_OTHER_CURRENT",
    "AMT_DRAWINGS_POS_CURRENT",
    "CNT_DRAWINGS_ATM_CURRENT",
    "CNT_DRAWINGS_OTHER_CURRENT",
    "CNT_DRAWINGS_POS_CURRENT",
]

# Availability indicators
drawing_components_available_mask = (
    credit_card_balance[
        drawing_component_columns
    ].notna().all(axis=1)
)

credit_card_balance[
    "CC_DRAWING_COMPONENTS_AVAILABLE"
] = (
    drawing_components_available_mask
    .astype("int8")
)

credit_card_balance[
    "CC_PAYMENT_CURRENT_AVAILABLE"
] = (
    credit_card_balance[
        "AMT_PAYMENT_CURRENT"
    ].notna().astype("int8")
)

credit_card_balance[
    "CC_MIN_INSTALMENT_AVAILABLE"
] = (
    credit_card_balance[
        "AMT_INST_MIN_REGULARITY"
    ].notna().astype("int8")
)


# Availability by contract status
credit_card_availability_by_status = (
    credit_card_balance
    .groupby(
        "NAME_CONTRACT_STATUS",
        observed=True,
        sort=False,
    )
    .agg(
        records=(
            "SK_ID_PREV",
            "size",
        ),
        drawing_component_coverage=(
            "CC_DRAWING_COMPONENTS_AVAILABLE",
            "mean",
        ),
        payment_current_coverage=(
            "CC_PAYMENT_CURRENT_AVAILABLE",
            "mean",
        ),
        minimum_instalment_coverage=(
            "CC_MIN_INSTALMENT_AVAILABLE",
            "mean",
        ),
    )
    .reset_index()
)

coverage_columns = [
    "drawing_component_coverage",
    "payment_current_coverage",
    "minimum_instalment_coverage",
]

credit_card_availability_by_status[
    coverage_columns
] = (
    credit_card_availability_by_status[
        coverage_columns
    ] * 100
)

credit_card_availability_by_status = (
    credit_card_availability_by_status
    .sort_values(
        "records",
        ascending=False,
    )
    .reset_index(drop=True)
)


# Numeric amount quality
financial_columns = [
    "AMT_BALANCE",
    "AMT_CREDIT_LIMIT_ACTUAL",
    "AMT_DRAWINGS_CURRENT",
    "AMT_PAYMENT_CURRENT",
    "AMT_PAYMENT_TOTAL_CURRENT",
    "AMT_INST_MIN_REGULARITY",
    "AMT_RECEIVABLE_PRINCIPAL",
    "AMT_RECIVABLE",
    "AMT_TOTAL_RECEIVABLE",
]

financial_quality_rows = []

for column in financial_columns:
    values = credit_card_balance[column]

    quantiles = values.quantile(
        [0.50, 0.95, 0.99]
    )

    financial_quality_rows.append(
        {
            "feature": column,
            "missing_records": (
                values.isna().sum()
            ),
            "negative_records": (
                values.lt(0).sum()
            ),
            "zero_records": (
                values.eq(0).sum()
            ),
            "median": quantiles.loc[0.50],
            "p95": quantiles.loc[0.95],
            "p99": quantiles.loc[0.99],
            "maximum": values.max(),
        }
    )

credit_card_financial_quality = pd.DataFrame(
    financial_quality_rows
)


# Drawing-component reconciliation
drawing_amount_component_sum = (
    credit_card_balance[
        [
            "AMT_DRAWINGS_ATM_CURRENT",
            "AMT_DRAWINGS_OTHER_CURRENT",
            "AMT_DRAWINGS_POS_CURRENT",
        ]
    ].sum(axis=1)
)

drawing_count_component_sum = (
    credit_card_balance[
        [
            "CNT_DRAWINGS_ATM_CURRENT",
            "CNT_DRAWINGS_OTHER_CURRENT",
            "CNT_DRAWINGS_POS_CURRENT",
        ]
    ].sum(axis=1)
)

drawing_amount_difference = (
    credit_card_balance[
        "AMT_DRAWINGS_CURRENT"
    ]
    - drawing_amount_component_sum
)

drawing_count_difference = (
    credit_card_balance[
        "CNT_DRAWINGS_CURRENT"
    ]
    - drawing_count_component_sum
)

component_missing_reference = (
    credit_card_balance[
        drawing_component_columns[0]
    ].isna()
)

component_missing_alignment_errors = 0

for column in drawing_component_columns[1:]:
    component_missing_alignment_errors += int(
        (
            credit_card_balance[
                column
            ].isna()
            != component_missing_reference
        ).sum()
    )

minimum_maturity_missing_mismatch = int(
    (
        credit_card_balance[
            "AMT_INST_MIN_REGULARITY"
        ].isna()
        != credit_card_balance[
            "CNT_INSTALMENT_MATURE_CUM"
        ].isna()
    ).sum()
)


# Ratios for tail inspection
positive_limit_mask = (
    credit_card_balance[
        "AMT_CREDIT_LIMIT_ACTUAL"
    ] > 0
)

positive_minimum_payment_mask = (
    credit_card_balance[
        "AMT_INST_MIN_REGULARITY"
    ] > 0
)

balance_limit_ratio = (
    credit_card_balance[
        "AMT_BALANCE"
    ]
    / credit_card_balance[
        "AMT_CREDIT_LIMIT_ACTUAL"
    ]
).where(positive_limit_mask)

drawing_limit_ratio = (
    credit_card_balance[
        "AMT_DRAWINGS_CURRENT"
    ]
    / credit_card_balance[
        "AMT_CREDIT_LIMIT_ACTUAL"
    ]
).where(positive_limit_mask)

payment_minimum_ratio = (
    credit_card_balance[
        "AMT_PAYMENT_TOTAL_CURRENT"
    ]
    / credit_card_balance[
        "AMT_INST_MIN_REGULARITY"
    ]
).where(positive_minimum_payment_mask)

ratio_series = {
    "BALANCE_LIMIT_RATIO":
        balance_limit_ratio,
    "DRAWING_LIMIT_RATIO":
        drawing_limit_ratio,
    "PAYMENT_MINIMUM_RATIO":
        payment_minimum_ratio,
}

ratio_tail_rows = []

for ratio_name, values in ratio_series.items():
    quantiles = values.quantile(
        [0.50, 0.90, 0.95, 0.99]
    )

    ratio_tail_rows.append(
        {
            "ratio": ratio_name,
            "valid_records": (
                values.notna().sum()
            ),
            "negative_records": (
                values.lt(0).sum()
            ),
            "median": quantiles.loc[0.50],
            "p90": quantiles.loc[0.90],
            "p95": quantiles.loc[0.95],
            "p99": quantiles.loc[0.99],
            "maximum": values.max(),
        }
    )

credit_card_ratio_tails = pd.DataFrame(
    ratio_tail_rows
)


# Consistency summary
credit_card_consistency_check = pd.DataFrame(
    {
        "metric": [
            "Total rows",
            "Rows with positive credit limit",
            "Rows with zero credit limit",
            "Rows with negative credit limit",
            "Negative balance records",
            "Balance greater than credit limit",
            "Drawing components available",
            "Component missing alignment errors",
            "Drawing amount reconciliation errors",
            "Drawing count reconciliation errors",
            "Minimum/maturity missing mismatch",
            "Payment current missing",
            "Payment current missing with zero total payment",
            "Payment current missing with positive total payment",
            "Minimum instalment missing",
            "Positive total-payment records",
        ],
        "value": [
            len(credit_card_balance),
            positive_limit_mask.sum(),
            credit_card_balance[
                "AMT_CREDIT_LIMIT_ACTUAL"
            ].eq(0).sum(),
            credit_card_balance[
                "AMT_CREDIT_LIMIT_ACTUAL"
            ].lt(0).sum(),
            credit_card_balance[
                "AMT_BALANCE"
            ].lt(0).sum(),
            (
                credit_card_balance[
                    "AMT_BALANCE"
                ]
                > credit_card_balance[
                    "AMT_CREDIT_LIMIT_ACTUAL"
                ]
            ).sum(),
            drawing_components_available_mask.sum(),
            component_missing_alignment_errors,
            (
                drawing_components_available_mask
                & drawing_amount_difference
                .abs().gt(1)
            ).sum(),
            (
                drawing_components_available_mask
                & drawing_count_difference
                .abs().gt(0.001)
            ).sum(),
            minimum_maturity_missing_mismatch,
            credit_card_balance[
                "AMT_PAYMENT_CURRENT"
            ].isna().sum(),
            (
                credit_card_balance[
                    "AMT_PAYMENT_CURRENT"
                ].isna()
                & credit_card_balance[
                    "AMT_PAYMENT_TOTAL_CURRENT"
                ].eq(0)
            ).sum(),
            (
                credit_card_balance[
                    "AMT_PAYMENT_CURRENT"
                ].isna()
                & credit_card_balance[
                    "AMT_PAYMENT_TOTAL_CURRENT"
                ].gt(0)
            ).sum(),
            credit_card_balance[
                "AMT_INST_MIN_REGULARITY"
            ].isna().sum(),
            credit_card_balance[
                "AMT_PAYMENT_TOTAL_CURRENT"
            ].gt(0).sum(),
        ],
    }
)


# Save audits
credit_card_availability_by_status.to_csv(
    processed_data_path
    / "credit_card_availability_by_status.csv",
    index=False,
)

credit_card_financial_quality.to_csv(
    processed_data_path
    / "credit_card_financial_quality.csv",
    index=False,
)

credit_card_ratio_tails.to_csv(
    processed_data_path
    / "credit_card_ratio_tails.csv",
    index=False,
)

credit_card_consistency_check.to_csv(
    processed_data_path
    / "credit_card_consistency_check.csv",
    index=False,
)

display(
    credit_card_availability_by_status.round(3)
)

display(
    credit_card_financial_quality.round(3)
)

display(
    credit_card_ratio_tails.round(3)
)

display(
    credit_card_consistency_check.round(2)
)

In [ ]:
import numpy as np
import pandas as pd

columns_before_financial_engineering = set(
    credit_card_balance.columns
)

positive_limit_mask = (
    credit_card_balance[
        "AMT_CREDIT_LIMIT_ACTUAL"
    ] > 0
)

positive_minimum_mask = (
    credit_card_balance[
        "AMT_INST_MIN_REGULARITY"
    ] > 0
)

drawing_components_available_mask = (
    credit_card_balance[
        [
            "AMT_DRAWINGS_ATM_CURRENT",
            "AMT_DRAWINGS_OTHER_CURRENT",
            "AMT_DRAWINGS_POS_CURRENT",
            "CNT_DRAWINGS_ATM_CURRENT",
            "CNT_DRAWINGS_OTHER_CURRENT",
            "CNT_DRAWINGS_POS_CURRENT",
        ]
    ].notna().all(axis=1)
)


# Negative balance treatment
credit_card_balance[
    "CC_NEGATIVE_BALANCE_FLAG"
] = (
    credit_card_balance[
        "AMT_BALANCE"
    ] < 0
).astype("int8")

credit_card_balance[
    "CC_BALANCE_NONNEGATIVE"
] = (
    credit_card_balance[
        "AMT_BALANCE"
    ]
    .clip(lower=0)
    .astype("float32")
)

credit_card_balance[
    "CC_BALANCE_SURPLUS"
] = (
    -credit_card_balance[
        "AMT_BALANCE"
    ]
).clip(lower=0).astype("float32")


# Credit-limit availability
credit_card_balance[
    "CC_POSITIVE_LIMIT_FLAG"
] = (
    positive_limit_mask.astype("int8")
)

credit_card_balance[
    "CC_ZERO_LIMIT_FLAG"
] = (
    credit_card_balance[
        "AMT_CREDIT_LIMIT_ACTUAL"
    ].eq(0)
).astype("int8")


# Balance / credit-limit utilization
balance_limit_ratio_raw = (
    credit_card_balance[
        "CC_BALANCE_NONNEGATIVE"
    ]
    / credit_card_balance[
        "AMT_CREDIT_LIMIT_ACTUAL"
    ]
).where(positive_limit_mask)

credit_card_balance[
    "CC_UTILIZATION_AVAILABLE"
] = (
    balance_limit_ratio_raw
    .notna()
    .astype("int8")
)

credit_card_balance[
    "CC_BALANCE_LIMIT_RATIO_CAP_2"
] = (
    balance_limit_ratio_raw
    .clip(lower=0, upper=2)
    .astype("float32")
)

credit_card_balance[
    "CC_OVER_LIMIT_FLAG"
] = (
    positive_limit_mask
    & balance_limit_ratio_raw.gt(1)
).astype("int8")

credit_card_balance[
    "CC_UTILIZATION_EXTREME_FLAG"
] = (
    positive_limit_mask
    & balance_limit_ratio_raw.gt(2)
).astype("int8")


# Drawing / credit-limit ratio
drawing_limit_ratio_raw = (
    credit_card_balance[
        "AMT_DRAWINGS_CURRENT"
    ]
    / credit_card_balance[
        "AMT_CREDIT_LIMIT_ACTUAL"
    ]
).where(positive_limit_mask)

credit_card_balance[
    "CC_DRAWING_LIMIT_RATIO_CAP_2"
] = (
    drawing_limit_ratio_raw
    .clip(lower=0, upper=2)
    .astype("float32")
)

credit_card_balance[
    "CC_DRAWING_OVER_LIMIT_FLAG"
] = (
    positive_limit_mask
    & drawing_limit_ratio_raw.gt(1)
).astype("int8")

credit_card_balance[
    "CC_DRAWING_RATIO_EXTREME_FLAG"
] = (
    positive_limit_mask
    & drawing_limit_ratio_raw.gt(2)
).astype("int8")

credit_card_balance[
    "CC_NEGATIVE_DRAWING_FLAG"
] = (
    credit_card_balance[
        "AMT_DRAWINGS_CURRENT"
    ].lt(0)
).astype("int8")


# Payment / minimum-instalment ratio
payment_minimum_ratio_raw = (
    credit_card_balance[
        "AMT_PAYMENT_TOTAL_CURRENT"
    ]
    / credit_card_balance[
        "AMT_INST_MIN_REGULARITY"
    ]
).where(positive_minimum_mask)

credit_card_balance[
    "CC_PAYMENT_MIN_RATIO_AVAILABLE"
] = (
    payment_minimum_ratio_raw
    .notna()
    .astype("int8")
)

credit_card_balance[
    "CC_PAYMENT_MIN_RATIO_CAP_50"
] = (
    payment_minimum_ratio_raw
    .clip(lower=0, upper=50)
    .astype("float32")
)

credit_card_balance[
    "CC_LOG1P_PAYMENT_MIN_RATIO"
] = np.log1p(
    credit_card_balance[
        "CC_PAYMENT_MIN_RATIO_CAP_50"
    ]
).astype("float32")

credit_card_balance[
    "CC_PAYMENT_MIN_RATIO_EXTREME_FLAG"
] = (
    positive_minimum_mask
    & payment_minimum_ratio_raw.gt(50)
).astype("int8")

credit_card_balance[
    "CC_PAYMENT_BELOW_MIN_FLAG"
] = (
    positive_minimum_mask
    & (
        credit_card_balance[
            "AMT_PAYMENT_TOTAL_CURRENT"
        ]
        < (
            credit_card_balance[
                "AMT_INST_MIN_REGULARITY"
            ] - 1
        )
    )
).astype("int8")

credit_card_balance[
    "CC_PAYMENT_SHORTFALL_AMOUNT"
] = (
    credit_card_balance[
        "AMT_INST_MIN_REGULARITY"
    ]
    - credit_card_balance[
        "AMT_PAYMENT_TOTAL_CURRENT"
    ]
).clip(lower=0).where(
    positive_minimum_mask
).astype("float32")


# Robust DPD features
credit_card_balance[
    "CC_DPD_CAP_120"
] = (
    credit_card_balance[
        "SK_DPD"
    ]
    .clip(lower=0, upper=120)
    .astype("int16")
)

credit_card_balance[
    "CC_LOG1P_DPD"
] = np.log1p(
    credit_card_balance[
        "CC_DPD_CAP_120"
    ]
).astype("float32")

credit_card_balance[
    "CC_DPD_DEF_CAP_120"
] = (
    credit_card_balance[
        "SK_DPD_DEF"
    ]
    .clip(lower=0, upper=120)
    .astype("int16")
)

credit_card_balance[
    "CC_LOG1P_DPD_DEF"
] = np.log1p(
    credit_card_balance[
        "CC_DPD_DEF_CAP_120"
    ]
).astype("float32")


# Drawing reconciliation flag
drawing_amount_component_sum = (
    credit_card_balance[
        [
            "AMT_DRAWINGS_ATM_CURRENT",
            "AMT_DRAWINGS_OTHER_CURRENT",
            "AMT_DRAWINGS_POS_CURRENT",
        ]
    ].sum(axis=1)
)

drawing_amount_difference = (
    credit_card_balance[
        "AMT_DRAWINGS_CURRENT"
    ]
    - drawing_amount_component_sum
)

credit_card_balance[
    "CC_DRAWING_AMOUNT_RECONCILIATION_FLAG"
] = (
    drawing_components_available_mask
    & drawing_amount_difference.abs().gt(1)
).astype("int8")


# Validation summary
credit_card_feature_check = pd.DataFrame(
    {
        "metric": [
            "Rows after feature engineering",
            "Negative balance flags",
            "Zero credit-limit flags",
            "Valid utilization ratios",
            "Over-limit records",
            "Extreme utilization records",
            "Maximum capped utilization",
            "Drawing over-limit records",
            "Extreme drawing-ratio records",
            "Maximum capped drawing ratio",
            "Valid payment/minimum ratios",
            "Payment-below-minimum records",
            "Extreme payment/minimum ratios",
            "Maximum capped payment/minimum ratio",
            "Positive DPD records",
            "30+ DPD records",
            "90+ DPD records",
            "Drawing reconciliation flags",
            "Features added",
            "Current total columns",
            "Current memory usage (MB)",
        ],
        "value": [
            len(credit_card_balance),
            credit_card_balance[
                "CC_NEGATIVE_BALANCE_FLAG"
            ].sum(),
            credit_card_balance[
                "CC_ZERO_LIMIT_FLAG"
            ].sum(),
            credit_card_balance[
                "CC_UTILIZATION_AVAILABLE"
            ].sum(),
            credit_card_balance[
                "CC_OVER_LIMIT_FLAG"
            ].sum(),
            credit_card_balance[
                "CC_UTILIZATION_EXTREME_FLAG"
            ].sum(),
            credit_card_balance[
                "CC_BALANCE_LIMIT_RATIO_CAP_2"
            ].max(),
            credit_card_balance[
                "CC_DRAWING_OVER_LIMIT_FLAG"
            ].sum(),
            credit_card_balance[
                "CC_DRAWING_RATIO_EXTREME_FLAG"
            ].sum(),
            credit_card_balance[
                "CC_DRAWING_LIMIT_RATIO_CAP_2"
            ].max(),
            credit_card_balance[
                "CC_PAYMENT_MIN_RATIO_AVAILABLE"
            ].sum(),
            credit_card_balance[
                "CC_PAYMENT_BELOW_MIN_FLAG"
            ].sum(),
            credit_card_balance[
                "CC_PAYMENT_MIN_RATIO_EXTREME_FLAG"
            ].sum(),
            credit_card_balance[
                "CC_PAYMENT_MIN_RATIO_CAP_50"
            ].max(),
            credit_card_balance[
                "CC_DPD_FLAG"
            ].sum(),
            credit_card_balance[
                "CC_DPD_30PLUS_FLAG"
            ].sum(),
            credit_card_balance[
                "CC_DPD_90PLUS_FLAG"
            ].sum(),
            credit_card_balance[
                "CC_DRAWING_AMOUNT_RECONCILIATION_FLAG"
            ].sum(),
            len(
                set(credit_card_balance.columns)
                - columns_before_financial_engineering
            ),
            credit_card_balance.shape[1],
            (
                credit_card_balance
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

assert (
    credit_card_balance[
        "CC_BALANCE_LIMIT_RATIO_CAP_2"
    ].max()
    <= 2
)

assert (
    credit_card_balance[
        "CC_DRAWING_LIMIT_RATIO_CAP_2"
    ].max()
    <= 2
)

assert (
    credit_card_balance[
        "CC_PAYMENT_MIN_RATIO_CAP_50"
    ].max()
    <= 50
)

display(
    credit_card_feature_check.round(2)
)

In [ ]:
import gc
import numpy as np
import pandas as pd

def clean_cc_category_name(value):
    return (
        str(value)
        .upper()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
    )


# Source totals for reconciliation
source_credit_card_rows = len(
    credit_card_balance
)

source_credit_card_ids = (
    credit_card_balance[
        "SK_ID_PREV"
    ].nunique()
)

source_dpd_count = int(
    credit_card_balance[
        "CC_DPD_FLAG"
    ].sum()
)

source_30plus_dpd_count = int(
    credit_card_balance[
        "CC_DPD_30PLUS_FLAG"
    ].sum()
)

source_90plus_dpd_count = int(
    credit_card_balance[
        "CC_DPD_90PLUS_FLAG"
    ].sum()
)


# Row flags to aggregate
account_flag_source_map = {
    "CC_ACCOUNT_DPD_COUNT":
        "CC_DPD_FLAG",
    "CC_ACCOUNT_30PLUS_DPD_COUNT":
        "CC_DPD_30PLUS_FLAG",
    "CC_ACCOUNT_90PLUS_DPD_COUNT":
        "CC_DPD_90PLUS_FLAG",
    "CC_ACCOUNT_DPD_DEF_COUNT":
        "CC_DPD_DEF_FLAG",
    "CC_ACCOUNT_DRAWING_COMPONENT_AVAILABLE_COUNT":
        "CC_DRAWING_COMPONENTS_AVAILABLE",
    "CC_ACCOUNT_PAYMENT_CURRENT_AVAILABLE_COUNT":
        "CC_PAYMENT_CURRENT_AVAILABLE",
    "CC_ACCOUNT_MIN_INSTALMENT_AVAILABLE_COUNT":
        "CC_MIN_INSTALMENT_AVAILABLE",
    "CC_ACCOUNT_NEGATIVE_BALANCE_COUNT":
        "CC_NEGATIVE_BALANCE_FLAG",
    "CC_ACCOUNT_POSITIVE_LIMIT_COUNT":
        "CC_POSITIVE_LIMIT_FLAG",
    "CC_ACCOUNT_ZERO_LIMIT_COUNT":
        "CC_ZERO_LIMIT_FLAG",
    "CC_ACCOUNT_OVER_LIMIT_COUNT":
        "CC_OVER_LIMIT_FLAG",
    "CC_ACCOUNT_EXTREME_UTILIZATION_COUNT":
        "CC_UTILIZATION_EXTREME_FLAG",
    "CC_ACCOUNT_DRAWING_OVER_LIMIT_COUNT":
        "CC_DRAWING_OVER_LIMIT_FLAG",
    "CC_ACCOUNT_EXTREME_DRAWING_RATIO_COUNT":
        "CC_DRAWING_RATIO_EXTREME_FLAG",
    "CC_ACCOUNT_NEGATIVE_DRAWING_COUNT":
        "CC_NEGATIVE_DRAWING_FLAG",
    "CC_ACCOUNT_PAYMENT_RATIO_AVAILABLE_COUNT":
        "CC_PAYMENT_MIN_RATIO_AVAILABLE",
    "CC_ACCOUNT_PAYMENT_BELOW_MIN_COUNT":
        "CC_PAYMENT_BELOW_MIN_FLAG",
    "CC_ACCOUNT_EXTREME_PAYMENT_RATIO_COUNT":
        "CC_PAYMENT_MIN_RATIO_EXTREME_FLAG",
    "CC_ACCOUNT_DRAWING_RECONCILIATION_COUNT":
        "CC_DRAWING_AMOUNT_RECONCILIATION_FLAG",
}


# Main aggregation specification
account_aggregation = {
    "SK_ID_CURR": (
        "SK_ID_CURR",
        "first",
    ),
    "CC_ACCOUNT_CUSTOMER_NUNIQUE": (
        "SK_ID_CURR",
        "nunique",
    ),
    "CC_ACCOUNT_MONTH_COUNT": (
        "MONTHS_BALANCE",
        "size",
    ),
    "CC_ACCOUNT_MIN_MONTH_RECENCY_DAYS": (
        "CC_MONTH_RECENCY_DAYS",
        "min",
    ),
    "CC_ACCOUNT_MEAN_MONTH_RECENCY_DAYS": (
        "CC_MONTH_RECENCY_DAYS",
        "mean",
    ),
    "CC_ACCOUNT_MAX_MONTH_RECENCY_DAYS": (
        "CC_MONTH_RECENCY_DAYS",
        "max",
    ),

    # Balance and limit
    "CC_ACCOUNT_MEAN_BALANCE": (
        "AMT_BALANCE",
        "mean",
    ),
    "CC_ACCOUNT_MAX_BALANCE": (
        "AMT_BALANCE",
        "max",
    ),
    "CC_ACCOUNT_MEAN_BALANCE_NONNEGATIVE": (
        "CC_BALANCE_NONNEGATIVE",
        "mean",
    ),
    "CC_ACCOUNT_MAX_BALANCE_NONNEGATIVE": (
        "CC_BALANCE_NONNEGATIVE",
        "max",
    ),
    "CC_ACCOUNT_TOTAL_BALANCE_SURPLUS": (
        "CC_BALANCE_SURPLUS",
        "sum",
    ),
    "CC_ACCOUNT_MAX_BALANCE_SURPLUS": (
        "CC_BALANCE_SURPLUS",
        "max",
    ),
    "CC_ACCOUNT_MEAN_CREDIT_LIMIT": (
        "AMT_CREDIT_LIMIT_ACTUAL",
        "mean",
    ),
    "CC_ACCOUNT_MAX_CREDIT_LIMIT": (
        "AMT_CREDIT_LIMIT_ACTUAL",
        "max",
    ),

    # Drawings
    "CC_ACCOUNT_TOTAL_DRAWINGS": (
        "AMT_DRAWINGS_CURRENT",
        "sum",
    ),
    "CC_ACCOUNT_MEAN_DRAWINGS": (
        "AMT_DRAWINGS_CURRENT",
        "mean",
    ),
    "CC_ACCOUNT_MAX_DRAWINGS": (
        "AMT_DRAWINGS_CURRENT",
        "max",
    ),
    "CC_ACCOUNT_TOTAL_ATM_DRAWINGS": (
        "AMT_DRAWINGS_ATM_CURRENT",
        "sum",
    ),
    "CC_ACCOUNT_TOTAL_POS_DRAWINGS": (
        "AMT_DRAWINGS_POS_CURRENT",
        "sum",
    ),
    "CC_ACCOUNT_TOTAL_OTHER_DRAWINGS": (
        "AMT_DRAWINGS_OTHER_CURRENT",
        "sum",
    ),
    "CC_ACCOUNT_TOTAL_DRAWING_COUNT": (
        "CNT_DRAWINGS_CURRENT",
        "sum",
    ),

    # Payments
    "CC_ACCOUNT_TOTAL_PAYMENT": (
        "AMT_PAYMENT_TOTAL_CURRENT",
        "sum",
    ),
    "CC_ACCOUNT_MEAN_PAYMENT": (
        "AMT_PAYMENT_TOTAL_CURRENT",
        "mean",
    ),
    "CC_ACCOUNT_MAX_PAYMENT": (
        "AMT_PAYMENT_TOTAL_CURRENT",
        "max",
    ),
    "CC_ACCOUNT_MEAN_MIN_INSTALMENT": (
        "AMT_INST_MIN_REGULARITY",
        "mean",
    ),
    "CC_ACCOUNT_MAX_MIN_INSTALMENT": (
        "AMT_INST_MIN_REGULARITY",
        "max",
    ),
    "CC_ACCOUNT_TOTAL_PAYMENT_SHORTFALL": (
        "CC_PAYMENT_SHORTFALL_AMOUNT",
        "sum",
    ),
    "CC_ACCOUNT_MEAN_PAYMENT_SHORTFALL": (
        "CC_PAYMENT_SHORTFALL_AMOUNT",
        "mean",
    ),
    "CC_ACCOUNT_MAX_PAYMENT_SHORTFALL": (
        "CC_PAYMENT_SHORTFALL_AMOUNT",
        "max",
    ),

    # Receivables
    "CC_ACCOUNT_MEAN_RECEIVABLE_PRINCIPAL": (
        "AMT_RECEIVABLE_PRINCIPAL",
        "mean",
    ),
    "CC_ACCOUNT_MAX_RECEIVABLE_PRINCIPAL": (
        "AMT_RECEIVABLE_PRINCIPAL",
        "max",
    ),
    "CC_ACCOUNT_MEAN_TOTAL_RECEIVABLE": (
        "AMT_TOTAL_RECEIVABLE",
        "mean",
    ),
    "CC_ACCOUNT_MIN_TOTAL_RECEIVABLE": (
        "AMT_TOTAL_RECEIVABLE",
        "min",
    ),
    "CC_ACCOUNT_MAX_TOTAL_RECEIVABLE": (
        "AMT_TOTAL_RECEIVABLE",
        "max",
    ),

    # Ratios
    "CC_ACCOUNT_MEAN_UTILIZATION_CAP_2": (
        "CC_BALANCE_LIMIT_RATIO_CAP_2",
        "mean",
    ),
    "CC_ACCOUNT_MAX_UTILIZATION_CAP_2": (
        "CC_BALANCE_LIMIT_RATIO_CAP_2",
        "max",
    ),
    "CC_ACCOUNT_MEAN_DRAWING_RATIO_CAP_2": (
        "CC_DRAWING_LIMIT_RATIO_CAP_2",
        "mean",
    ),
    "CC_ACCOUNT_MAX_DRAWING_RATIO_CAP_2": (
        "CC_DRAWING_LIMIT_RATIO_CAP_2",
        "max",
    ),
    "CC_ACCOUNT_MEAN_PAYMENT_RATIO_CAP_50": (
        "CC_PAYMENT_MIN_RATIO_CAP_50",
        "mean",
    ),
    "CC_ACCOUNT_MAX_PAYMENT_RATIO_CAP_50": (
        "CC_PAYMENT_MIN_RATIO_CAP_50",
        "max",
    ),
    "CC_ACCOUNT_MEAN_LOG1P_PAYMENT_RATIO": (
        "CC_LOG1P_PAYMENT_MIN_RATIO",
        "mean",
    ),
    "CC_ACCOUNT_MAX_LOG1P_PAYMENT_RATIO": (
        "CC_LOG1P_PAYMENT_MIN_RATIO",
        "max",
    ),

    # Delinquency
    "CC_ACCOUNT_MEAN_DPD_RAW": (
        "SK_DPD",
        "mean",
    ),
    "CC_ACCOUNT_MAX_DPD_RAW": (
        "SK_DPD",
        "max",
    ),
    "CC_ACCOUNT_MEAN_DPD_CAP_120": (
        "CC_DPD_CAP_120",
        "mean",
    ),
    "CC_ACCOUNT_MAX_DPD_CAP_120": (
        "CC_DPD_CAP_120",
        "max",
    ),
    "CC_ACCOUNT_MEAN_LOG1P_DPD": (
        "CC_LOG1P_DPD",
        "mean",
    ),
    "CC_ACCOUNT_MAX_LOG1P_DPD": (
        "CC_LOG1P_DPD",
        "max",
    ),
    "CC_ACCOUNT_MEAN_DPD_DEF_RAW": (
        "SK_DPD_DEF",
        "mean",
    ),
    "CC_ACCOUNT_MAX_DPD_DEF_RAW": (
        "SK_DPD_DEF",
        "max",
    ),
    "CC_ACCOUNT_MEAN_DPD_DEF_CAP_120": (
        "CC_DPD_DEF_CAP_120",
        "mean",
    ),
    "CC_ACCOUNT_MAX_DPD_DEF_CAP_120": (
        "CC_DPD_DEF_CAP_120",
        "max",
    ),
    "CC_ACCOUNT_MEAN_LOG1P_DPD_DEF": (
        "CC_LOG1P_DPD_DEF",
        "mean",
    ),
    "CC_ACCOUNT_MAX_LOG1P_DPD_DEF": (
        "CC_LOG1P_DPD_DEF",
        "max",
    ),
}

for output_column, source_column in (
    account_flag_source_map.items()
):
    account_aggregation[output_column] = (
        source_column,
        "sum",
    )


# Aggregate monthly records to account level
credit_card_account_features = (
    credit_card_balance
    .groupby(
        "SK_ID_PREV",
        sort=False,
    )
    .agg(**account_aggregation)
)


# Contract-status counts
credit_card_status_dummies = pd.get_dummies(
    credit_card_balance[
        "NAME_CONTRACT_STATUS"
    ],
    dtype="int8",
)

credit_card_status_dummies.columns = [
    (
        "CC_ACCOUNT_STATUS_"
        f"{clean_cc_category_name(column)}"
        "_COUNT"
    )
    for column in credit_card_status_dummies.columns
]

credit_card_status_dummies[
    "SK_ID_PREV"
] = credit_card_balance[
    "SK_ID_PREV"
].to_numpy()

credit_card_account_status_features = (
    credit_card_status_dummies
    .groupby(
        "SK_ID_PREV",
        sort=False,
    )
    .sum()
    .astype("int16")
)

status_count_columns = list(
    credit_card_account_status_features.columns
)

credit_card_account_features = (
    credit_card_account_features.join(
        credit_card_account_status_features,
        how="left",
    )
)


# Latest monthly snapshot
latest_month_index = (
    credit_card_balance
    .groupby(
        "SK_ID_PREV",
        sort=False,
    )["MONTHS_BALANCE"]
    .idxmax()
)

latest_snapshot_columns = {
    "AMT_BALANCE":
        "CC_ACCOUNT_LATEST_BALANCE",
    "AMT_CREDIT_LIMIT_ACTUAL":
        "CC_ACCOUNT_LATEST_CREDIT_LIMIT",
    "AMT_DRAWINGS_CURRENT":
        "CC_ACCOUNT_LATEST_DRAWINGS",
    "AMT_PAYMENT_TOTAL_CURRENT":
        "CC_ACCOUNT_LATEST_PAYMENT",
    "AMT_TOTAL_RECEIVABLE":
        "CC_ACCOUNT_LATEST_TOTAL_RECEIVABLE",
    "AMT_INST_MIN_REGULARITY":
        "CC_ACCOUNT_LATEST_MIN_INSTALMENT",
    "CC_BALANCE_LIMIT_RATIO_CAP_2":
        "CC_ACCOUNT_LATEST_UTILIZATION_CAP_2",
    "CC_DRAWING_LIMIT_RATIO_CAP_2":
        "CC_ACCOUNT_LATEST_DRAWING_RATIO_CAP_2",
    "CC_PAYMENT_MIN_RATIO_CAP_50":
        "CC_ACCOUNT_LATEST_PAYMENT_RATIO_CAP_50",
    "SK_DPD":
        "CC_ACCOUNT_LATEST_DPD",
    "SK_DPD_DEF":
        "CC_ACCOUNT_LATEST_DPD_DEF",
}

credit_card_latest_snapshot = (
    credit_card_balance.loc[
        latest_month_index,
        [
            "SK_ID_PREV",
            "NAME_CONTRACT_STATUS",
        ] + list(
            latest_snapshot_columns.keys()
        ),
    ]
    .set_index("SK_ID_PREV")
)

credit_card_latest_numeric = (
    credit_card_latest_snapshot[
        list(latest_snapshot_columns.keys())
    ]
    .rename(
        columns=latest_snapshot_columns
    )
)

latest_status_dummies = pd.get_dummies(
    credit_card_latest_snapshot[
        "NAME_CONTRACT_STATUS"
    ],
    dtype="int8",
)

latest_status_dummies.columns = [
    (
        "CC_ACCOUNT_LATEST_STATUS_"
        f"{clean_cc_category_name(column)}"
        "_FLAG"
    )
    for column in latest_status_dummies.columns
]

credit_card_account_features = (
    credit_card_account_features
    .join(
        credit_card_latest_numeric,
        how="left",
    )
    .join(
        latest_status_dummies,
        how="left",
    )
)


# History span
credit_card_account_features[
    "CC_ACCOUNT_HISTORY_SPAN_MONTHS"
] = (
    credit_card_account_features[
        "CC_ACCOUNT_MAX_MONTH_RECENCY_DAYS"
    ]
    - credit_card_account_features[
        "CC_ACCOUNT_MIN_MONTH_RECENCY_DAYS"
    ]
).astype("int16")


# Correct missing total shortfall
no_minimum_information_mask = (
    credit_card_account_features[
        "CC_ACCOUNT_PAYMENT_RATIO_AVAILABLE_COUNT"
    ].eq(0)
)

credit_card_account_features.loc[
    no_minimum_information_mask,
    "CC_ACCOUNT_TOTAL_PAYMENT_SHORTFALL",
] = np.nan


# Count dtypes
account_count_columns = [
    "CC_ACCOUNT_MONTH_COUNT",
] + list(
    account_flag_source_map.keys()
) + status_count_columns

credit_card_account_features[
    account_count_columns
] = credit_card_account_features[
    account_count_columns
].astype("int16")


# Shares using appropriate denominators
month_denominator = (
    credit_card_account_features[
        "CC_ACCOUNT_MONTH_COUNT"
    ]
)

month_share_count_columns = [
    "CC_ACCOUNT_DPD_COUNT",
    "CC_ACCOUNT_30PLUS_DPD_COUNT",
    "CC_ACCOUNT_90PLUS_DPD_COUNT",
    "CC_ACCOUNT_DPD_DEF_COUNT",
    "CC_ACCOUNT_DRAWING_COMPONENT_AVAILABLE_COUNT",
    "CC_ACCOUNT_PAYMENT_CURRENT_AVAILABLE_COUNT",
    "CC_ACCOUNT_MIN_INSTALMENT_AVAILABLE_COUNT",
    "CC_ACCOUNT_NEGATIVE_BALANCE_COUNT",
    "CC_ACCOUNT_ZERO_LIMIT_COUNT",
    "CC_ACCOUNT_NEGATIVE_DRAWING_COUNT",
    "CC_ACCOUNT_DRAWING_RECONCILIATION_COUNT",
]

for count_column in month_share_count_columns:
    share_column = count_column.replace(
        "_COUNT",
        "_SHARE",
    )

    credit_card_account_features[
        share_column
    ] = (
        credit_card_account_features[count_column]
        / month_denominator
    ).astype("float32")


positive_limit_denominator = (
    credit_card_account_features[
        "CC_ACCOUNT_POSITIVE_LIMIT_COUNT"
    ].replace(0, np.nan)
)

for count_column in [
    "CC_ACCOUNT_OVER_LIMIT_COUNT",
    "CC_ACCOUNT_EXTREME_UTILIZATION_COUNT",
    "CC_ACCOUNT_DRAWING_OVER_LIMIT_COUNT",
    "CC_ACCOUNT_EXTREME_DRAWING_RATIO_COUNT",
]:
    share_column = count_column.replace(
        "_COUNT",
        "_SHARE",
    )

    credit_card_account_features[
        share_column
    ] = (
        credit_card_account_features[count_column]
        / positive_limit_denominator
    ).astype("float32")


payment_ratio_denominator = (
    credit_card_account_features[
        "CC_ACCOUNT_PAYMENT_RATIO_AVAILABLE_COUNT"
    ].replace(0, np.nan)
)

for count_column in [
    "CC_ACCOUNT_PAYMENT_BELOW_MIN_COUNT",
    "CC_ACCOUNT_EXTREME_PAYMENT_RATIO_COUNT",
]:
    share_column = count_column.replace(
        "_COUNT",
        "_SHARE",
    )

    credit_card_account_features[
        share_column
    ] = (
        credit_card_account_features[count_column]
        / payment_ratio_denominator
    ).astype("float32")


# Status shares
for count_column in status_count_columns:
    share_column = count_column.replace(
        "_COUNT",
        "_SHARE",
    )

    credit_card_account_features[
        share_column
    ] = (
        credit_card_account_features[count_column]
        / month_denominator
    ).astype("float32")


# Log count
credit_card_account_features[
    "CC_ACCOUNT_LOG_MONTH_COUNT"
] = np.log1p(
    credit_card_account_features[
        "CC_ACCOUNT_MONTH_COUNT"
    ]
).astype("float32")


# Reconciliation
customer_mapping_conflicts = int(
    credit_card_account_features[
        "CC_ACCOUNT_CUSTOMER_NUNIQUE"
    ].ne(1).sum()
)

status_reconciliation_difference = (
    credit_card_account_features[
        status_count_columns
    ].sum(axis=1)
    - credit_card_account_features[
        "CC_ACCOUNT_MONTH_COUNT"
    ]
)

assert credit_card_account_features.index.is_unique
assert customer_mapping_conflicts == 0
assert status_reconciliation_difference.ne(0).sum() == 0
assert (
    credit_card_account_features[
        "CC_ACCOUNT_MONTH_COUNT"
    ].sum()
    == source_credit_card_rows
)

credit_card_account_features.drop(
    columns="CC_ACCOUNT_CUSTOMER_NUNIQUE",
    inplace=True,
)


# Save and verify
credit_card_account_path = (
    processed_data_path
    / "credit_card_account_features.parquet"
)

credit_card_account_output = (
    credit_card_account_features
    .reset_index()
)

credit_card_account_output.to_parquet(
    credit_card_account_path,
    index=False,
    compression="snappy",
)

account_verification = pd.read_parquet(
    credit_card_account_path,
    columns=[
        "SK_ID_PREV",
        "SK_ID_CURR",
        "CC_ACCOUNT_MONTH_COUNT",
    ],
)

credit_card_account_check = pd.DataFrame(
    {
        "metric": [
            "Account feature rows",
            "Unique account IDs",
            "Duplicate account IDs",
            "Customer mapping conflicts",
            "Reconciled monthly rows",
            "Source credit-card rows",
            "Status reconciliation errors",
            "Reconciled DPD records",
            "Reconciled 30+ DPD records",
            "Reconciled 90+ DPD records",
            "Accounts with any DPD",
            "Accounts with 30+ DPD",
            "Accounts with 90+ DPD",
            "Account feature columns",
            "Verified saved rows",
            "Saved file size (MB)",
            "Account table memory (MB)",
        ],
        "value": [
            len(credit_card_account_features),
            credit_card_account_features
            .index.nunique(),
            credit_card_account_features
            .index.duplicated().sum(),
            customer_mapping_conflicts,
            credit_card_account_features[
                "CC_ACCOUNT_MONTH_COUNT"
            ].sum(),
            source_credit_card_rows,
            status_reconciliation_difference
            .ne(0).sum(),
            credit_card_account_features[
                "CC_ACCOUNT_DPD_COUNT"
            ].sum(),
            credit_card_account_features[
                "CC_ACCOUNT_30PLUS_DPD_COUNT"
            ].sum(),
            credit_card_account_features[
                "CC_ACCOUNT_90PLUS_DPD_COUNT"
            ].sum(),
            credit_card_account_features[
                "CC_ACCOUNT_DPD_COUNT"
            ].gt(0).sum(),
            credit_card_account_features[
                "CC_ACCOUNT_30PLUS_DPD_COUNT"
            ].gt(0).sum(),
            credit_card_account_features[
                "CC_ACCOUNT_90PLUS_DPD_COUNT"
            ].gt(0).sum(),
            credit_card_account_features.shape[1],
            len(account_verification),
            (
                credit_card_account_path
                .stat().st_size
                / 1024**2
            ),
            (
                credit_card_account_features
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

credit_card_account_check.to_csv(
    processed_data_path
    / "credit_card_account_feature_check.csv",
    index=False,
)

# Remove the large monthly table after successful persistence
objects_to_remove = [
    "credit_card_balance",
    "credit_card_status_dummies",
    "credit_card_account_status_features",
    "credit_card_latest_snapshot",
    "credit_card_latest_numeric",
    "latest_status_dummies",
    "latest_month_index",
    "account_verification",
    "balance_limit_ratio_raw",
    "drawing_limit_ratio_raw",
    "payment_minimum_ratio_raw",
    "drawing_amount_component_sum",
    "drawing_amount_difference",
    "drawing_components_available_mask",
    "positive_limit_mask",
    "positive_minimum_mask",
]

for object_name in objects_to_remove:
    globals().pop(object_name, None)

gc.collect()

display(
    credit_card_account_check.round(2)
)

In [ ]:
import gc
import numpy as np
import pandas as pd

# Reload after kernel restart
if "credit_card_account_features" not in globals():
    credit_card_account_features = (
        pd.read_parquet(
            processed_data_path
            / "credit_card_account_features.parquet"
        )
        .set_index("SK_ID_PREV")
    )

assert credit_card_account_features.index.is_unique


# Correct structural all-missing component totals
component_total_columns = [
    "CC_ACCOUNT_TOTAL_ATM_DRAWINGS",
    "CC_ACCOUNT_TOTAL_POS_DRAWINGS",
    "CC_ACCOUNT_TOTAL_OTHER_DRAWINGS",
]

no_component_information_mask = (
    credit_card_account_features[
        "CC_ACCOUNT_DRAWING_COMPONENT_AVAILABLE_COUNT"
    ].eq(0)
)

accounts_with_structural_missing_components = int(
    no_component_information_mask.sum()
)

credit_card_account_features.loc[
    no_component_information_mask,
    component_total_columns,
] = np.nan

assert (
    credit_card_account_features.loc[
        no_component_information_mask,
        component_total_columns,
    ].isna().all().all()
)


# Update corrected account-level checkpoint
credit_card_account_path = (
    processed_data_path
    / "credit_card_account_features.parquet"
)

credit_card_account_features.reset_index().to_parquet(
    credit_card_account_path,
    index=False,
    compression="snappy",
)


# Source totals
source_account_rows = len(
    credit_card_account_features
)

source_month_rows = int(
    credit_card_account_features[
        "CC_ACCOUNT_MONTH_COUNT"
    ].sum()
)

source_dpd_count = int(
    credit_card_account_features[
        "CC_ACCOUNT_DPD_COUNT"
    ].sum()
)

source_30plus_dpd_count = int(
    credit_card_account_features[
        "CC_ACCOUNT_30PLUS_DPD_COUNT"
    ].sum()
)

source_90plus_dpd_count = int(
    credit_card_account_features[
        "CC_ACCOUNT_90PLUS_DPD_COUNT"
    ].sum()
)


# Account behaviour indicators
account_behaviour_map = {
    "CC_CUSTOMER_ACCOUNTS_WITH_DPD_COUNT":
        "CC_ACCOUNT_DPD_COUNT",
    "CC_CUSTOMER_ACCOUNTS_WITH_30PLUS_DPD_COUNT":
        "CC_ACCOUNT_30PLUS_DPD_COUNT",
    "CC_CUSTOMER_ACCOUNTS_WITH_90PLUS_DPD_COUNT":
        "CC_ACCOUNT_90PLUS_DPD_COUNT",
    "CC_CUSTOMER_ACCOUNTS_WITH_NEGATIVE_BALANCE_COUNT":
        "CC_ACCOUNT_NEGATIVE_BALANCE_COUNT",
    "CC_CUSTOMER_ACCOUNTS_WITH_OVER_LIMIT_COUNT":
        "CC_ACCOUNT_OVER_LIMIT_COUNT",
    "CC_CUSTOMER_ACCOUNTS_WITH_EXTREME_UTILIZATION_COUNT":
        "CC_ACCOUNT_EXTREME_UTILIZATION_COUNT",
    "CC_CUSTOMER_ACCOUNTS_WITH_PAYMENT_BELOW_MIN_COUNT":
        "CC_ACCOUNT_PAYMENT_BELOW_MIN_COUNT",
    "CC_CUSTOMER_ACCOUNTS_WITH_EXTREME_PAYMENT_RATIO_COUNT":
        "CC_ACCOUNT_EXTREME_PAYMENT_RATIO_COUNT",
    "CC_CUSTOMER_ACCOUNTS_WITH_DRAWING_RECONCILIATION_COUNT":
        "CC_ACCOUNT_DRAWING_RECONCILIATION_COUNT",
    "CC_CUSTOMER_ACCOUNTS_WITH_DEMAND_STATUS_COUNT":
        "CC_ACCOUNT_STATUS_DEMAND_COUNT",
}

account_behaviour_temp_columns = {}

for output_column, source_column in (
    account_behaviour_map.items()
):
    temp_column = (
        f"_TEMP_{output_column}"
    )

    credit_card_account_features[
        temp_column
    ] = (
        credit_card_account_features[
            source_column
        ].gt(0).astype("int8")
    )

    account_behaviour_temp_columns[
        output_column
    ] = temp_column


# Weighted monthly means
weighted_mean_configuration = {
    "CC_CUSTOMER_MEAN_MONTH_RECENCY_DAYS": (
        "CC_ACCOUNT_MEAN_MONTH_RECENCY_DAYS",
        "CC_ACCOUNT_MONTH_COUNT",
    ),
    "CC_CUSTOMER_MEAN_BALANCE": (
        "CC_ACCOUNT_MEAN_BALANCE",
        "CC_ACCOUNT_MONTH_COUNT",
    ),
    "CC_CUSTOMER_MEAN_BALANCE_NONNEGATIVE": (
        "CC_ACCOUNT_MEAN_BALANCE_NONNEGATIVE",
        "CC_ACCOUNT_MONTH_COUNT",
    ),
    "CC_CUSTOMER_MEAN_CREDIT_LIMIT": (
        "CC_ACCOUNT_MEAN_CREDIT_LIMIT",
        "CC_ACCOUNT_MONTH_COUNT",
    ),
    "CC_CUSTOMER_MEAN_DRAWINGS": (
        "CC_ACCOUNT_MEAN_DRAWINGS",
        "CC_ACCOUNT_MONTH_COUNT",
    ),
    "CC_CUSTOMER_MEAN_PAYMENT": (
        "CC_ACCOUNT_MEAN_PAYMENT",
        "CC_ACCOUNT_MONTH_COUNT",
    ),
    "CC_CUSTOMER_MEAN_RECEIVABLE_PRINCIPAL": (
        "CC_ACCOUNT_MEAN_RECEIVABLE_PRINCIPAL",
        "CC_ACCOUNT_MONTH_COUNT",
    ),
    "CC_CUSTOMER_MEAN_TOTAL_RECEIVABLE": (
        "CC_ACCOUNT_MEAN_TOTAL_RECEIVABLE",
        "CC_ACCOUNT_MONTH_COUNT",
    ),
    "CC_CUSTOMER_MEAN_DPD_RAW": (
        "CC_ACCOUNT_MEAN_DPD_RAW",
        "CC_ACCOUNT_MONTH_COUNT",
    ),
    "CC_CUSTOMER_MEAN_DPD_CAP_120": (
        "CC_ACCOUNT_MEAN_DPD_CAP_120",
        "CC_ACCOUNT_MONTH_COUNT",
    ),
    "CC_CUSTOMER_MEAN_LOG1P_DPD": (
        "CC_ACCOUNT_MEAN_LOG1P_DPD",
        "CC_ACCOUNT_MONTH_COUNT",
    ),
    "CC_CUSTOMER_MEAN_DPD_DEF_RAW": (
        "CC_ACCOUNT_MEAN_DPD_DEF_RAW",
        "CC_ACCOUNT_MONTH_COUNT",
    ),
    "CC_CUSTOMER_MEAN_DPD_DEF_CAP_120": (
        "CC_ACCOUNT_MEAN_DPD_DEF_CAP_120",
        "CC_ACCOUNT_MONTH_COUNT",
    ),
    "CC_CUSTOMER_MEAN_LOG1P_DPD_DEF": (
        "CC_ACCOUNT_MEAN_LOG1P_DPD_DEF",
        "CC_ACCOUNT_MONTH_COUNT",
    ),
    "CC_CUSTOMER_MEAN_MIN_INSTALMENT": (
        "CC_ACCOUNT_MEAN_MIN_INSTALMENT",
        "CC_ACCOUNT_MIN_INSTALMENT_AVAILABLE_COUNT",
    ),
    "CC_CUSTOMER_MEAN_PAYMENT_SHORTFALL": (
        "CC_ACCOUNT_MEAN_PAYMENT_SHORTFALL",
        "CC_ACCOUNT_PAYMENT_RATIO_AVAILABLE_COUNT",
    ),
    "CC_CUSTOMER_MEAN_UTILIZATION_CAP_2": (
        "CC_ACCOUNT_MEAN_UTILIZATION_CAP_2",
        "CC_ACCOUNT_POSITIVE_LIMIT_COUNT",
    ),
    "CC_CUSTOMER_MEAN_DRAWING_RATIO_CAP_2": (
        "CC_ACCOUNT_MEAN_DRAWING_RATIO_CAP_2",
        "CC_ACCOUNT_POSITIVE_LIMIT_COUNT",
    ),
    "CC_CUSTOMER_MEAN_PAYMENT_RATIO_CAP_50": (
        "CC_ACCOUNT_MEAN_PAYMENT_RATIO_CAP_50",
        "CC_ACCOUNT_PAYMENT_RATIO_AVAILABLE_COUNT",
    ),
    "CC_CUSTOMER_MEAN_LOG1P_PAYMENT_RATIO": (
        "CC_ACCOUNT_MEAN_LOG1P_PAYMENT_RATIO",
        "CC_ACCOUNT_PAYMENT_RATIO_AVAILABLE_COUNT",
    ),
}

weighted_temp_columns = {}

for output_column, (
    value_column,
    weight_column,
) in weighted_mean_configuration.items():
    temp_column = (
        f"_TEMP_WEIGHTED_{output_column}"
    )

    credit_card_account_features[
        temp_column
    ] = (
        credit_card_account_features[
            value_column
        ]
        * credit_card_account_features[
            weight_column
        ]
    ).astype("float32")

    weighted_temp_columns[
        output_column
    ] = (
        temp_column,
        weight_column,
    )


# Additive account-level count columns
account_count_source_columns = [
    column
    for column in credit_card_account_features.columns
    if (
        column.startswith("CC_ACCOUNT_")
        and column.endswith("_COUNT")
    )
]

customer_aggregation = {
    "CC_CUSTOMER_ACCOUNT_COUNT": (
        "SK_ID_CURR",
        "size",
    )
}

for source_column in account_count_source_columns:
    output_column = source_column.replace(
        "CC_ACCOUNT_",
        "CC_CUSTOMER_",
    )

    customer_aggregation[
        output_column
    ] = (
        source_column,
        "sum",
    )


# Additive financial totals
financial_total_columns = [
    "CC_ACCOUNT_TOTAL_BALANCE_SURPLUS",
    "CC_ACCOUNT_TOTAL_DRAWINGS",
    "CC_ACCOUNT_TOTAL_ATM_DRAWINGS",
    "CC_ACCOUNT_TOTAL_POS_DRAWINGS",
    "CC_ACCOUNT_TOTAL_OTHER_DRAWINGS",
    "CC_ACCOUNT_TOTAL_PAYMENT",
    "CC_ACCOUNT_TOTAL_PAYMENT_SHORTFALL",
]

for source_column in financial_total_columns:
    output_column = source_column.replace(
        "CC_ACCOUNT_",
        "CC_CUSTOMER_",
    )

    customer_aggregation[
        output_column
    ] = (
        source_column,
        "sum",
    )


# Minimum and maximum features
minimum_source_columns = [
    column
    for column in credit_card_account_features.columns
    if (
        column.startswith("CC_ACCOUNT_MIN_")
        and not column.endswith(
            ("_COUNT", "_SHARE", "_FLAG")
        )
    )
]

maximum_source_columns = [
    column
    for column in credit_card_account_features.columns
    if (
        column.startswith("CC_ACCOUNT_MAX_")
        and not column.endswith(
            ("_COUNT", "_SHARE", "_FLAG")
        )
    )
]

for source_column in minimum_source_columns:
    output_column = source_column.replace(
        "CC_ACCOUNT_",
        "CC_CUSTOMER_",
    )

    customer_aggregation[
        output_column
    ] = (
        source_column,
        "min",
    )

for source_column in maximum_source_columns:
    output_column = source_column.replace(
        "CC_ACCOUNT_",
        "CC_CUSTOMER_",
    )

    customer_aggregation[
        output_column
    ] = (
        source_column,
        "max",
    )


# Account history spans
customer_aggregation.update(
    {
        "CC_CUSTOMER_MEAN_ACCOUNT_HISTORY_SPAN_MONTHS": (
            "CC_ACCOUNT_HISTORY_SPAN_MONTHS",
            "mean",
        ),
        "CC_CUSTOMER_MAX_ACCOUNT_HISTORY_SPAN_MONTHS": (
            "CC_ACCOUNT_HISTORY_SPAN_MONTHS",
            "max",
        ),
    }
)


# Latest monetary snapshots are additive across accounts
latest_sum_columns = {
    "CC_ACCOUNT_LATEST_BALANCE":
        "CC_CUSTOMER_LATEST_TOTAL_BALANCE",
    "CC_ACCOUNT_LATEST_CREDIT_LIMIT":
        "CC_CUSTOMER_LATEST_TOTAL_CREDIT_LIMIT",
    "CC_ACCOUNT_LATEST_DRAWINGS":
        "CC_CUSTOMER_LATEST_TOTAL_DRAWINGS",
    "CC_ACCOUNT_LATEST_PAYMENT":
        "CC_CUSTOMER_LATEST_TOTAL_PAYMENT",
    "CC_ACCOUNT_LATEST_TOTAL_RECEIVABLE":
        "CC_CUSTOMER_LATEST_TOTAL_RECEIVABLE",
    "CC_ACCOUNT_LATEST_MIN_INSTALMENT":
        "CC_CUSTOMER_LATEST_TOTAL_MIN_INSTALMENT",
}

for (
    source_column,
    output_column,
) in latest_sum_columns.items():
    customer_aggregation[
        output_column
    ] = (
        source_column,
        "sum",
    )


# Latest ratios and DPD use maximum severity
latest_max_columns = {
    "CC_ACCOUNT_LATEST_UTILIZATION_CAP_2":
        "CC_CUSTOMER_LATEST_MAX_UTILIZATION_CAP_2",
    "CC_ACCOUNT_LATEST_DRAWING_RATIO_CAP_2":
        "CC_CUSTOMER_LATEST_MAX_DRAWING_RATIO_CAP_2",
    "CC_ACCOUNT_LATEST_PAYMENT_RATIO_CAP_50":
        "CC_CUSTOMER_LATEST_MAX_PAYMENT_RATIO_CAP_50",
    "CC_ACCOUNT_LATEST_DPD":
        "CC_CUSTOMER_LATEST_MAX_DPD",
    "CC_ACCOUNT_LATEST_DPD_DEF":
        "CC_CUSTOMER_LATEST_MAX_DPD_DEF",
}

for (
    source_column,
    output_column,
) in latest_max_columns.items():
    customer_aggregation[
        output_column
    ] = (
        source_column,
        "max",
    )


# Latest account-status counts
latest_status_flag_columns = [
    column
    for column in credit_card_account_features.columns
    if (
        column.startswith(
            "CC_ACCOUNT_LATEST_STATUS_"
        )
        and column.endswith("_FLAG")
    )
]

latest_status_customer_columns = []

for source_column in latest_status_flag_columns:
    output_column = (
        source_column
        .replace(
            "CC_ACCOUNT_LATEST_STATUS_",
            "CC_CUSTOMER_LATEST_STATUS_",
        )
        .replace(
            "_FLAG",
            "_ACCOUNT_COUNT",
        )
    )

    latest_status_customer_columns.append(
        output_column
    )

    customer_aggregation[
        output_column
    ] = (
        source_column,
        "sum",
    )


# Behaviour-account counts
for (
    output_column,
    temp_column,
) in account_behaviour_temp_columns.items():
    customer_aggregation[
        output_column
    ] = (
        temp_column,
        "sum",
    )


# Weighted numerators
weighted_numerator_columns = {}

for output_column, (
    temp_column,
    weight_column,
) in weighted_temp_columns.items():
    numerator_column = (
        f"_TEMP_NUMERATOR_{output_column}"
    )

    weighted_numerator_columns[
        output_column
    ] = (
        numerator_column,
        weight_column,
    )

    customer_aggregation[
        numerator_column
    ] = (
        temp_column,
        "sum",
    )


# Aggregate to customer level
credit_card_customer_features = (
    credit_card_account_features
    .groupby(
        "SK_ID_CURR",
        sort=False,
    )
    .agg(**customer_aggregation)
)


# Restore correctly weighted means
for output_column, (
    numerator_column,
    account_weight_column,
) in weighted_numerator_columns.items():
    customer_weight_column = (
        account_weight_column.replace(
            "CC_ACCOUNT_",
            "CC_CUSTOMER_",
        )
    )

    denominator = (
        credit_card_customer_features[
            customer_weight_column
        ].replace(0, np.nan)
    )

    credit_card_customer_features[
        output_column
    ] = (
        credit_card_customer_features[
            numerator_column
        ]
        / denominator
    ).astype("float32")


credit_card_customer_features.drop(
    columns=[
        numerator_column
        for numerator_column, _ in (
            weighted_numerator_columns.values()
        )
    ],
    inplace=True,
)


# Preserve structural missing component totals
no_customer_component_information = (
    credit_card_customer_features[
        "CC_CUSTOMER_DRAWING_COMPONENT_AVAILABLE_COUNT"
    ].eq(0)
)

customer_component_total_columns = [
    "CC_CUSTOMER_TOTAL_ATM_DRAWINGS",
    "CC_CUSTOMER_TOTAL_POS_DRAWINGS",
    "CC_CUSTOMER_TOTAL_OTHER_DRAWINGS",
]

credit_card_customer_features.loc[
    no_customer_component_information,
    customer_component_total_columns,
] = np.nan


# Preserve missing payment-shortfall information
no_customer_minimum_information = (
    credit_card_customer_features[
        "CC_CUSTOMER_PAYMENT_RATIO_AVAILABLE_COUNT"
    ].eq(0)
)

credit_card_customer_features.loc[
    no_customer_minimum_information,
    "CC_CUSTOMER_TOTAL_PAYMENT_SHORTFALL",
] = np.nan


# Customer history span
credit_card_customer_features[
    "CC_CUSTOMER_HISTORY_SPAN_MONTHS"
] = (
    credit_card_customer_features[
        "CC_CUSTOMER_MAX_MONTH_RECENCY_DAYS"
    ]
    - credit_card_customer_features[
        "CC_CUSTOMER_MIN_MONTH_RECENCY_DAYS"
    ]
).astype("int16")


# Shares based on monthly records
month_denominator = (
    credit_card_customer_features[
        "CC_CUSTOMER_MONTH_COUNT"
    ]
)

monthly_share_count_columns = [
    "CC_CUSTOMER_DPD_COUNT",
    "CC_CUSTOMER_30PLUS_DPD_COUNT",
    "CC_CUSTOMER_90PLUS_DPD_COUNT",
    "CC_CUSTOMER_DPD_DEF_COUNT",
    "CC_CUSTOMER_DRAWING_COMPONENT_AVAILABLE_COUNT",
    "CC_CUSTOMER_PAYMENT_CURRENT_AVAILABLE_COUNT",
    "CC_CUSTOMER_MIN_INSTALMENT_AVAILABLE_COUNT",
    "CC_CUSTOMER_NEGATIVE_BALANCE_COUNT",
    "CC_CUSTOMER_ZERO_LIMIT_COUNT",
    "CC_CUSTOMER_NEGATIVE_DRAWING_COUNT",
    "CC_CUSTOMER_DRAWING_RECONCILIATION_COUNT",
]

for count_column in monthly_share_count_columns:
    share_column = count_column.replace(
        "_COUNT",
        "_SHARE",
    )

    credit_card_customer_features[
        share_column
    ] = (
        credit_card_customer_features[count_column]
        / month_denominator
    ).astype("float32")


# Limit-based shares
positive_limit_denominator = (
    credit_card_customer_features[
        "CC_CUSTOMER_POSITIVE_LIMIT_COUNT"
    ].replace(0, np.nan)
)

for count_column in [
    "CC_CUSTOMER_OVER_LIMIT_COUNT",
    "CC_CUSTOMER_EXTREME_UTILIZATION_COUNT",
    "CC_CUSTOMER_DRAWING_OVER_LIMIT_COUNT",
    "CC_CUSTOMER_EXTREME_DRAWING_RATIO_COUNT",
]:
    share_column = count_column.replace(
        "_COUNT",
        "_SHARE",
    )

    credit_card_customer_features[
        share_column
    ] = (
        credit_card_customer_features[count_column]
        / positive_limit_denominator
    ).astype("float32")


# Payment-minimum shares
payment_ratio_denominator = (
    credit_card_customer_features[
        "CC_CUSTOMER_PAYMENT_RATIO_AVAILABLE_COUNT"
    ].replace(0, np.nan)
)

for count_column in [
    "CC_CUSTOMER_PAYMENT_BELOW_MIN_COUNT",
    "CC_CUSTOMER_EXTREME_PAYMENT_RATIO_COUNT",
]:
    share_column = count_column.replace(
        "_COUNT",
        "_SHARE",
    )

    credit_card_customer_features[
        share_column
    ] = (
        credit_card_customer_features[count_column]
        / payment_ratio_denominator
    ).astype("float32")


# Status shares
customer_status_count_columns = [
    column.replace(
        "CC_ACCOUNT_",
        "CC_CUSTOMER_",
    )
    for column in account_count_source_columns
    if column.startswith(
        "CC_ACCOUNT_STATUS_"
    )
]

for count_column in customer_status_count_columns:
    share_column = count_column.replace(
        "_COUNT",
        "_SHARE",
    )

    credit_card_customer_features[
        share_column
    ] = (
        credit_card_customer_features[count_column]
        / month_denominator
    ).astype("float32")


# Account behaviour and latest-status shares
account_denominator = (
    credit_card_customer_features[
        "CC_CUSTOMER_ACCOUNT_COUNT"
    ]
)

for count_column in list(
    account_behaviour_map.keys()
) + latest_status_customer_columns:
    share_column = count_column.replace(
        "_COUNT",
        "_SHARE",
    )

    credit_card_customer_features[
        share_column
    ] = (
        credit_card_customer_features[count_column]
        / account_denominator
    ).astype("float32")


# Average drawing transactions per month
credit_card_customer_features[
    "CC_CUSTOMER_MEAN_DRAWING_COUNT_PER_MONTH"
] = (
    credit_card_customer_features[
        "CC_CUSTOMER_TOTAL_DRAWING_COUNT"
    ]
    / month_denominator
).astype("float32")


# Latest combined utilization
latest_total_utilization_raw = (
    credit_card_customer_features[
        "CC_CUSTOMER_LATEST_TOTAL_BALANCE"
    ].clip(lower=0)
    / credit_card_customer_features[
        "CC_CUSTOMER_LATEST_TOTAL_CREDIT_LIMIT"
    ]
).where(
    credit_card_customer_features[
        "CC_CUSTOMER_LATEST_TOTAL_CREDIT_LIMIT"
    ].gt(0)
)

credit_card_customer_features[
    "CC_CUSTOMER_LATEST_TOTAL_UTILIZATION_CAP_2"
] = (
    latest_total_utilization_raw
    .clip(lower=0, upper=2)
    .astype("float32")
)

credit_card_customer_features[
    "CC_CUSTOMER_LATEST_OVER_LIMIT_FLAG"
] = (
    latest_total_utilization_raw.gt(1)
    .fillna(False)
    .astype("int8")
)


# Log counts
credit_card_customer_features[
    "CC_CUSTOMER_LOG_ACCOUNT_COUNT"
] = np.log1p(
    credit_card_customer_features[
        "CC_CUSTOMER_ACCOUNT_COUNT"
    ]
).astype("float32")

credit_card_customer_features[
    "CC_CUSTOMER_LOG_MONTH_COUNT"
] = np.log1p(
    credit_card_customer_features[
        "CC_CUSTOMER_MONTH_COUNT"
    ]
).astype("float32")


# Optimize count columns
customer_count_columns = [
    "CC_CUSTOMER_ACCOUNT_COUNT",
] + [
    column.replace(
        "CC_ACCOUNT_",
        "CC_CUSTOMER_",
    )
    for column in account_count_source_columns
] + list(
    account_behaviour_map.keys()
) + latest_status_customer_columns

customer_count_columns = list(
    dict.fromkeys(customer_count_columns)
)

credit_card_customer_features[
    customer_count_columns
] = credit_card_customer_features[
    customer_count_columns
].astype("int32")


# Reconciliation
status_reconciliation_difference = (
    credit_card_customer_features[
        customer_status_count_columns
    ].sum(axis=1)
    - credit_card_customer_features[
        "CC_CUSTOMER_MONTH_COUNT"
    ]
)

latest_status_reconciliation_difference = (
    credit_card_customer_features[
        latest_status_customer_columns
    ].sum(axis=1)
    - credit_card_customer_features[
        "CC_CUSTOMER_ACCOUNT_COUNT"
    ]
)

assert credit_card_customer_features.index.is_unique
assert status_reconciliation_difference.ne(0).sum() == 0
assert (
    latest_status_reconciliation_difference
    .ne(0).sum()
    == 0
)
assert (
    credit_card_customer_features[
        "CC_CUSTOMER_ACCOUNT_COUNT"
    ].sum()
    == source_account_rows
)
assert (
    credit_card_customer_features[
        "CC_CUSTOMER_MONTH_COUNT"
    ].sum()
    == source_month_rows
)


# Save and verify
credit_card_customer_path = (
    processed_data_path
    / "credit_card_customer_features.parquet"
)

credit_card_customer_output = (
    credit_card_customer_features
    .reset_index()
)

credit_card_customer_output.to_parquet(
    credit_card_customer_path,
    index=False,
    compression="snappy",
)

customer_verification = pd.read_parquet(
    credit_card_customer_path,
    columns=[
        "SK_ID_CURR",
        "CC_CUSTOMER_ACCOUNT_COUNT",
        "CC_CUSTOMER_MONTH_COUNT",
    ],
)

credit_card_customer_check = pd.DataFrame(
    {
        "metric": [
            "Customer feature rows",
            "Unique customer index",
            "Duplicate customer index",
            "Reconciled account count",
            "Source account rows",
            "Reconciled monthly rows",
            "Source monthly rows",
            "Status reconciliation errors",
            "Latest-status reconciliation errors",
            "Reconciled DPD records",
            "Reconciled 30+ DPD records",
            "Reconciled 90+ DPD records",
            "Customers with any DPD",
            "Customers with 30+ DPD",
            "Customers with 90+ DPD",
            "Customers with multiple card accounts",
            "Accounts corrected from structural zero",
            "Customer feature columns",
            "Verified saved rows",
            "Saved file size (MB)",
            "Customer table memory (MB)",
        ],
        "value": [
            len(credit_card_customer_features),
            credit_card_customer_features
            .index.nunique(),
            credit_card_customer_features
            .index.duplicated().sum(),
            credit_card_customer_features[
                "CC_CUSTOMER_ACCOUNT_COUNT"
            ].sum(),
            source_account_rows,
            credit_card_customer_features[
                "CC_CUSTOMER_MONTH_COUNT"
            ].sum(),
            source_month_rows,
            status_reconciliation_difference
            .ne(0).sum(),
            latest_status_reconciliation_difference
            .ne(0).sum(),
            credit_card_customer_features[
                "CC_CUSTOMER_DPD_COUNT"
            ].sum(),
            credit_card_customer_features[
                "CC_CUSTOMER_30PLUS_DPD_COUNT"
            ].sum(),
            credit_card_customer_features[
                "CC_CUSTOMER_90PLUS_DPD_COUNT"
            ].sum(),
            credit_card_customer_features[
                "CC_CUSTOMER_DPD_COUNT"
            ].gt(0).sum(),
            credit_card_customer_features[
                "CC_CUSTOMER_30PLUS_DPD_COUNT"
            ].gt(0).sum(),
            credit_card_customer_features[
                "CC_CUSTOMER_90PLUS_DPD_COUNT"
            ].gt(0).sum(),
            credit_card_customer_features[
                "CC_CUSTOMER_ACCOUNT_COUNT"
            ].gt(1).sum(),
            accounts_with_structural_missing_components,
            credit_card_customer_features.shape[1],
            len(customer_verification),
            (
                credit_card_customer_path
                .stat().st_size
                / 1024**2
            ),
            (
                credit_card_customer_features
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

credit_card_customer_check.to_csv(
    processed_data_path
    / "credit_card_customer_feature_check.csv",
    index=False,
)


# Clear account-level intermediates
del credit_card_account_features
del credit_card_customer_output
del customer_verification
del status_reconciliation_difference
del latest_status_reconciliation_difference
del latest_total_utilization_raw
del no_component_information_mask
del no_customer_component_information
del no_customer_minimum_information
gc.collect()

display(
    credit_card_customer_check.round(2)
)

In [ ]:
import warnings
import pandas as pd

warnings.filterwarnings(
    "ignore",
    category=pd.errors.PerformanceWarning,
)

check_path = (
    processed_data_path
    / "credit_card_customer_feature_check.csv"
)

if check_path.exists():
    credit_card_customer_check = pd.read_csv(
        check_path
    )

# Remove DataFrame fragmentation
if "credit_card_customer_features" in globals():
    credit_card_customer_features = (
        credit_card_customer_features.copy()
    )

display(
    credit_card_customer_check.round(2)
)

In [ ]:
check_path = (
    processed_data_path
    / "credit_card_customer_feature_check.csv"
)

print("Kontrol dosyası mevcut:", check_path.exists())

if check_path.exists():
    check_result = pd.read_csv(check_path)

    print(
        check_result.to_string(
            index=False
        )
    )

In [ ]:
import gc
import numpy as np
import pandas as pd

base_feature_path = (
    processed_data_path
    / (
        "application_train_features_"
        "bureau_previous_pos_installments.parquet"
    )
)

credit_card_customer_path = (
    processed_data_path
    / "credit_card_customer_features.parquet"
)

previous_customer_path = (
    processed_data_path
    / "previous_application_customer_features.parquet"
)


# Always start from clean persisted tables
base_train_features = pd.read_parquet(
    base_feature_path
)

credit_card_customer_data = pd.read_parquet(
    credit_card_customer_path
)

previous_customer_ids = pd.Index(
    pd.read_parquet(
        previous_customer_path,
        columns=["SK_ID_CURR"],
    )["SK_ID_CURR"]
)


# Key checks
assert base_train_features[
    "SK_ID_CURR"
].is_unique

assert credit_card_customer_data[
    "SK_ID_CURR"
].is_unique

base_train_rows = len(
    base_train_features
)

base_feature_count = (
    base_train_features.shape[1] - 1
)

credit_card_feature_columns = [
    column
    for column in credit_card_customer_data.columns
    if column != "SK_ID_CURR"
]

credit_card_dtype_map = {
    column: credit_card_customer_data[column].dtype
    for column in credit_card_feature_columns
}

unexpected_overlap = sorted(
    set(credit_card_feature_columns)
    .intersection(base_train_features.columns)
)

assert not unexpected_overlap, (
    "Unexpected credit-card overlaps: "
    f"{unexpected_overlap}"
)


# Safe one-to-one left merge
application_train_history_features = (
    base_train_features.merge(
        credit_card_customer_data,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one",
    )
)


# History indicator before filling counts
application_train_history_features[
    "CC_HAS_HISTORY"
] = (
    application_train_history_features[
        "CC_CUSTOMER_ACCOUNT_COUNT"
    ]
    .notna()
    .astype("int8")
)


# Only genuine counts, flags and log-counts become zero
credit_card_zero_fill_columns = [
    column
    for column in credit_card_feature_columns
    if (
        column.endswith("_COUNT")
        or column.endswith("_FLAG")
    )
]

credit_card_zero_fill_columns += [
    "CC_CUSTOMER_LOG_ACCOUNT_COUNT",
    "CC_CUSTOMER_LOG_MONTH_COUNT",
]

credit_card_zero_fill_columns = list(
    dict.fromkeys(
        credit_card_zero_fill_columns
    )
)

application_train_history_features[
    credit_card_zero_fill_columns
] = (
    application_train_history_features[
        credit_card_zero_fill_columns
    ].fillna(0)
)


# Restore original dtypes in small batches
dtype_groups = {}

for column in credit_card_zero_fill_columns:
    dtype_name = str(
        credit_card_dtype_map[column]
    )

    dtype_groups.setdefault(
        dtype_name,
        [],
    ).append(column)

for dtype_name, columns in dtype_groups.items():
    application_train_history_features[
        columns
    ] = (
        application_train_history_features[
            columns
        ].astype(dtype_name)
    )


# Cross-source history checks
train_customer_ids = (
    application_train_history_features[
        "SK_ID_CURR"
    ]
)

credit_card_history_mask = (
    application_train_history_features[
        "CC_HAS_HISTORY"
    ].eq(1)
)

previous_history_mask = (
    train_customer_ids.isin(
        previous_customer_ids
    )
)

previous_without_card_count = int(
    (
        previous_history_mask
        & ~credit_card_history_mask
    ).sum()
)

card_without_previous_count = int(
    (
        credit_card_history_mask
        & ~previous_history_mask
    ).sum()
)


# Final validations
assert (
    len(application_train_history_features)
    == base_train_rows
)

assert application_train_history_features[
    "SK_ID_CURR"
].is_unique

nulls_in_zero_filled_columns = int(
    application_train_history_features[
        credit_card_zero_fill_columns
    ].isna().sum().sum()
)

assert nulls_in_zero_filled_columns == 0


# Defragment once before saving
application_train_history_features = (
    application_train_history_features.copy()
)


# Save canonical relational-history feature table
all_sources_feature_path = (
    processed_data_path
    / "application_train_features_all_sources.parquet"
)

application_train_history_features.to_parquet(
    all_sources_feature_path,
    index=False,
    compression="snappy",
)


# Verify saved table
final_verification = pd.read_parquet(
    all_sources_feature_path,
    columns=[
        "SK_ID_CURR",
        "CC_HAS_HISTORY",
        "CC_CUSTOMER_ACCOUNT_COUNT",
        "CC_CUSTOMER_DPD_COUNT",
        "CC_CUSTOMER_90PLUS_DPD_COUNT",
    ],
)

assert len(final_verification) == base_train_rows
assert final_verification[
    "SK_ID_CURR"
].is_unique


# Feature-source manifest
final_feature_count = (
    application_train_history_features
    .shape[1] - 1
)

feature_source_manifest = pd.DataFrame(
    {
        "feature_source": [
            "Bureau and bureau balance",
            "Previous applications",
            "POS CASH",
            "Installment payments",
            "Credit-card balance",
            "Total relational-history features",
        ],
        "feature_count": [
            88,
            161 - 88,
            228 - 161,
            309 - 228,
            len(credit_card_feature_columns) + 1,
            final_feature_count,
        ],
    }
)

assert (
    feature_source_manifest.iloc[
        :-1
    ]["feature_count"].sum()
    == final_feature_count
)

feature_source_manifest.to_csv(
    processed_data_path
    / "relational_feature_source_manifest.csv",
    index=False,
)


# Final reconciliation summary
credit_card_train_join_check = pd.DataFrame(
    {
        "metric": [
            "Base train rows",
            "Final train rows",
            "Row count difference",
            "Unique train IDs",
            "Duplicate train IDs",
            "Unexpected overlapping features",
            "Customers with credit-card history",
            "Customers without credit-card history",
            "Previous history without credit card",
            "Credit-card history without previous history",
            "Customers with any card DPD",
            "Customers with 30+ card DPD",
            "Customers with 90+ card DPD",
            "Customers with multiple card accounts",
            "Customers without component information",
            "Nulls in zero-filled columns",
            "Base feature columns",
            "Credit-card features added",
            "Final relational feature columns",
            "Verified saved rows",
            "Saved file size (MB)",
            "Final table memory (MB)",
        ],
        "value": [
            base_train_rows,
            len(application_train_history_features),
            (
                len(application_train_history_features)
                - base_train_rows
            ),
            application_train_history_features[
                "SK_ID_CURR"
            ].nunique(),
            application_train_history_features[
                "SK_ID_CURR"
            ].duplicated().sum(),
            len(unexpected_overlap),
            credit_card_history_mask.sum(),
            (~credit_card_history_mask).sum(),
            previous_without_card_count,
            card_without_previous_count,
            application_train_history_features[
                "CC_CUSTOMER_DPD_COUNT"
            ].gt(0).sum(),
            application_train_history_features[
                "CC_CUSTOMER_30PLUS_DPD_COUNT"
            ].gt(0).sum(),
            application_train_history_features[
                "CC_CUSTOMER_90PLUS_DPD_COUNT"
            ].gt(0).sum(),
            application_train_history_features[
                "CC_CUSTOMER_ACCOUNT_COUNT"
            ].gt(1).sum(),
            (
                credit_card_history_mask
                & application_train_history_features[
                    "CC_CUSTOMER_DRAWING_COMPONENT_AVAILABLE_COUNT"
                ].eq(0)
            ).sum(),
            nulls_in_zero_filled_columns,
            base_feature_count,
            len(credit_card_feature_columns) + 1,
            final_feature_count,
            len(final_verification),
            (
                all_sources_feature_path
                .stat().st_size
                / 1024**2
            ),
            (
                application_train_history_features
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

credit_card_train_join_check.to_csv(
    processed_data_path
    / "credit_card_train_join_check.csv",
    index=False,
)


# Release large intermediate objects
del base_train_features
del credit_card_customer_data
del credit_card_customer_features
del application_train_history_features
del final_verification
del previous_customer_ids
gc.collect()

display(
    credit_card_train_join_check.round(2)
)

display(
    feature_source_manifest
)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import gc

project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

extract_path = (
    project_path
    / "data"
    / "raw"
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

required_files = [
    (
        "Base feature table",
        processed_data_path
        / (
            "application_train_features_"
            "bureau_previous_pos_installments.parquet"
        ),
    ),
    (
        "Credit-card customer features",
        processed_data_path
        / "credit_card_customer_features.parquet",
    ),
    (
        "Previous customer features",
        processed_data_path
        / "previous_application_customer_features.parquet",
    ),
]

path_setup_check = pd.DataFrame(
    {
        "file": [
            name
            for name, _ in required_files
        ],
        "exists": [
            path.exists()
            for _, path in required_files
        ],
        "size_mb": [
            (
                path.stat().st_size / 1024**2
                if path.exists()
                else np.nan
            )
            for _, path in required_files
        ],
    }
)

assert path_setup_check[
    "exists"
].all(), "One or more required files are missing."

display(
    path_setup_check.round(2)
)

In [ ]:
objects_to_remove = [
    "base_train_features",
    "credit_card_customer_data",
    "credit_card_customer_features",
    "application_train_history_features",
    "final_verification",
    "previous_customer_ids",
]

for object_name in objects_to_remove:
    globals().pop(
        object_name,
        None,
    )

gc.collect()

display(
    credit_card_train_join_check.round(2)
)

display(
    feature_source_manifest
)

In [ ]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd

# =========================================================
# 1. Paths
# =========================================================
project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

base_feature_path = (
    processed_data_path
    / (
        "application_train_features_"
        "bureau_previous_pos_installments.parquet"
    )
)

credit_card_customer_path = (
    processed_data_path
    / "credit_card_customer_features.parquet"
)

previous_customer_path = (
    processed_data_path
    / "previous_application_customer_features.parquet"
)

all_sources_feature_path = (
    processed_data_path
    / "application_train_features_all_sources.parquet"
)

required_paths = [
    base_feature_path,
    credit_card_customer_path,
    previous_customer_path,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

assert not missing_paths, (
    f"Missing required files: {missing_paths}"
)


# =========================================================
# 2. Load clean persisted tables
# =========================================================
base_train_features = pd.read_parquet(
    base_feature_path
)

credit_card_customer_data = pd.read_parquet(
    credit_card_customer_path
)

previous_customer_ids = pd.Index(
    pd.read_parquet(
        previous_customer_path,
        columns=["SK_ID_CURR"],
    )["SK_ID_CURR"]
)

assert base_train_features[
    "SK_ID_CURR"
].is_unique

assert credit_card_customer_data[
    "SK_ID_CURR"
].is_unique

base_train_rows = len(
    base_train_features
)

base_feature_count = (
    base_train_features.shape[1] - 1
)


# =========================================================
# 3. Validate feature overlap
# =========================================================
credit_card_feature_columns = [
    column
    for column in credit_card_customer_data.columns
    if column != "SK_ID_CURR"
]

credit_card_dtype_map = {
    column: credit_card_customer_data[column].dtype
    for column in credit_card_feature_columns
}

unexpected_overlap = sorted(
    set(credit_card_feature_columns)
    .intersection(base_train_features.columns)
)

assert not unexpected_overlap, (
    f"Unexpected overlaps: {unexpected_overlap}"
)


# =========================================================
# 4. Safe one-to-one merge
# =========================================================
application_train_history_features = (
    base_train_features.merge(
        credit_card_customer_data,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one",
    )
)

application_train_history_features[
    "CC_HAS_HISTORY"
] = (
    application_train_history_features[
        "CC_CUSTOMER_ACCOUNT_COUNT"
    ]
    .notna()
    .astype("int8")
)


# =========================================================
# 5. Fill genuine counts and flags only
# =========================================================
credit_card_zero_fill_columns = [
    column
    for column in credit_card_feature_columns
    if (
        column.endswith("_COUNT")
        or column.endswith("_FLAG")
    )
]

credit_card_zero_fill_columns += [
    "CC_CUSTOMER_LOG_ACCOUNT_COUNT",
    "CC_CUSTOMER_LOG_MONTH_COUNT",
]

credit_card_zero_fill_columns = list(
    dict.fromkeys(
        credit_card_zero_fill_columns
    )
)

application_train_history_features[
    credit_card_zero_fill_columns
] = (
    application_train_history_features[
        credit_card_zero_fill_columns
    ].fillna(0)
)

# Restore dtypes in batches
dtype_groups = {}

for column in credit_card_zero_fill_columns:
    dtype_name = str(
        credit_card_dtype_map[column]
    )

    dtype_groups.setdefault(
        dtype_name,
        [],
    ).append(column)

for dtype_name, columns in dtype_groups.items():
    application_train_history_features[
        columns
    ] = (
        application_train_history_features[
            columns
        ].astype(dtype_name)
    )


# =========================================================
# 6. Cross-source checks
# =========================================================
train_customer_ids = (
    application_train_history_features[
        "SK_ID_CURR"
    ]
)

credit_card_history_mask = (
    application_train_history_features[
        "CC_HAS_HISTORY"
    ].eq(1)
)

previous_history_mask = (
    train_customer_ids.isin(
        previous_customer_ids
    )
)

previous_without_card_count = int(
    (
        previous_history_mask
        & ~credit_card_history_mask
    ).sum()
)

card_without_previous_count = int(
    (
        credit_card_history_mask
        & ~previous_history_mask
    ).sum()
)

nulls_in_zero_filled_columns = int(
    application_train_history_features[
        credit_card_zero_fill_columns
    ].isna().sum().sum()
)

assert (
    len(application_train_history_features)
    == base_train_rows
)

assert application_train_history_features[
    "SK_ID_CURR"
].is_unique

assert nulls_in_zero_filled_columns == 0


# =========================================================
# 7. Defragment, save and verify
# =========================================================
application_train_history_features = (
    application_train_history_features.copy()
)

application_train_history_features.to_parquet(
    all_sources_feature_path,
    index=False,
    compression="snappy",
)

final_verification = pd.read_parquet(
    all_sources_feature_path,
    columns=[
        "SK_ID_CURR",
        "CC_HAS_HISTORY",
        "CC_CUSTOMER_ACCOUNT_COUNT",
        "CC_CUSTOMER_DPD_COUNT",
        "CC_CUSTOMER_90PLUS_DPD_COUNT",
    ],
)

assert len(final_verification) == base_train_rows
assert final_verification[
    "SK_ID_CURR"
].is_unique


# =========================================================
# 8. Feature manifest
# =========================================================
final_feature_count = (
    application_train_history_features
    .shape[1] - 1
)

feature_source_manifest = pd.DataFrame(
    {
        "feature_source": [
            "Bureau and bureau balance",
            "Previous applications",
            "POS CASH",
            "Installment payments",
            "Credit-card balance",
            "Total relational-history features",
        ],
        "feature_count": [
            88,
            73,
            67,
            81,
            len(credit_card_feature_columns) + 1,
            final_feature_count,
        ],
    }
)

assert (
    feature_source_manifest.iloc[
        :-1
    ]["feature_count"].sum()
    == final_feature_count
)

feature_source_manifest.to_csv(
    processed_data_path
    / "relational_feature_source_manifest.csv",
    index=False,
)


# =========================================================
# 9. Final join audit
# =========================================================
credit_card_train_join_check = pd.DataFrame(
    {
        "metric": [
            "Base train rows",
            "Final train rows",
            "Row count difference",
            "Unique train IDs",
            "Duplicate train IDs",
            "Unexpected overlapping features",
            "Customers with credit-card history",
            "Customers without credit-card history",
            "Previous history without credit card",
            "Credit-card history without previous history",
            "Customers with any card DPD",
            "Customers with 30+ card DPD",
            "Customers with 90+ card DPD",
            "Customers with multiple card accounts",
            "Customers without component information",
            "Nulls in zero-filled columns",
            "Base feature columns",
            "Credit-card features added",
            "Final relational feature columns",
            "Verified saved rows",
            "Saved file size (MB)",
            "Final table memory (MB)",
        ],
        "value": [
            base_train_rows,
            len(application_train_history_features),
            (
                len(application_train_history_features)
                - base_train_rows
            ),
            application_train_history_features[
                "SK_ID_CURR"
            ].nunique(),
            application_train_history_features[
                "SK_ID_CURR"
            ].duplicated().sum(),
            len(unexpected_overlap),
            credit_card_history_mask.sum(),
            (~credit_card_history_mask).sum(),
            previous_without_card_count,
            card_without_previous_count,
            application_train_history_features[
                "CC_CUSTOMER_DPD_COUNT"
            ].gt(0).sum(),
            application_train_history_features[
                "CC_CUSTOMER_30PLUS_DPD_COUNT"
            ].gt(0).sum(),
            application_train_history_features[
                "CC_CUSTOMER_90PLUS_DPD_COUNT"
            ].gt(0).sum(),
            application_train_history_features[
                "CC_CUSTOMER_ACCOUNT_COUNT"
            ].gt(1).sum(),
            (
                credit_card_history_mask
                & application_train_history_features[
                    "CC_CUSTOMER_DRAWING_COMPONENT_AVAILABLE_COUNT"
                ].eq(0)
            ).sum(),
            nulls_in_zero_filled_columns,
            base_feature_count,
            len(credit_card_feature_columns) + 1,
            final_feature_count,
            len(final_verification),
            (
                all_sources_feature_path
                .stat().st_size
                / 1024**2
            ),
            (
                application_train_history_features
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

credit_card_train_join_check.to_csv(
    processed_data_path
    / "credit_card_train_join_check.csv",
    index=False,
)


# =========================================================
# 10. Display results before cleanup
# =========================================================
display(
    credit_card_train_join_check.round(2)
)

display(
    feature_source_manifest
)


# =========================================================
# 11. Safe cleanup
# =========================================================
objects_to_remove = [
    "base_train_features",
    "credit_card_customer_data",
    "application_train_history_features",
    "final_verification",
    "previous_customer_ids",
]

for object_name in objects_to_remove:
    globals().pop(
        object_name,
        None,
    )

gc.collect()

In [ ]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd

# =========================================================
# 1. Paths
# =========================================================
project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

extract_path = (
    project_path
    / "data"
    / "raw"
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

application_train_path = (
    extract_path
    / "application_train.csv"
)

assert application_train_path.exists()


# =========================================================
# 2. Load clean source
# =========================================================
application_train = pd.read_csv(
    application_train_path,
    memory_map=True,
)

memory_before_optimization = (
    application_train
    .memory_usage(deep=True)
    .sum()
    / 1024**2
)

source_rows = len(
    application_train
)

source_columns = (
    application_train.shape[1]
)

assert application_train[
    "SK_ID_CURR"
].is_unique

assert application_train[
    "TARGET"
].notna().all()


# =========================================================
# 3. Optimize data types
# =========================================================
categorical_columns = (
    application_train
    .select_dtypes(include="object")
    .columns.tolist()
)

for column in categorical_columns:
    application_train[column] = (
        application_train[column]
        .astype("category")
    )

integer_columns = (
    application_train
    .select_dtypes(include="int64")
    .columns.tolist()
)

for column in integer_columns:
    application_train[column] = (
        pd.to_numeric(
            application_train[column],
            downcast="integer",
        )
    )

float_columns = (
    application_train
    .select_dtypes(include="float64")
    .columns.tolist()
)

for column in float_columns:
    application_train[column] = (
        pd.to_numeric(
            application_train[column],
            downcast="float",
        )
    )

memory_after_optimization = (
    application_train
    .memory_usage(deep=True)
    .sum()
    / 1024**2
)


# =========================================================
# 4. Record original missingness
# =========================================================
raw_predictor_columns = [
    column
    for column in application_train.columns
    if column not in [
        "SK_ID_CURR",
        "TARGET",
    ]
]

original_missing_values = int(
    application_train[
        raw_predictor_columns
    ].isna().sum().sum()
)

columns_before_engineering = set(
    application_train.columns
)


# =========================================================
# 5. DAYS_EMPLOYED sentinel treatment
# =========================================================
employment_sentinel_value = 365243

employment_sentinel_mask = (
    application_train[
        "DAYS_EMPLOYED"
    ].eq(employment_sentinel_value)
)

employment_sentinel_count = int(
    employment_sentinel_mask.sum()
)

application_train[
    "APP_EMPLOYMENT_ANOMALY_FLAG"
] = (
    employment_sentinel_mask
    .astype("int8")
)

application_train[
    "DAYS_EMPLOYED"
] = (
    application_train[
        "DAYS_EMPLOYED"
    ]
    .mask(employment_sentinel_mask)
    .astype("float32")
)


# =========================================================
# 6. Missing-information features
# =========================================================
application_train[
    "APP_MISSING_VALUE_COUNT"
] = (
    application_train[
        raw_predictor_columns
    ]
    .isna()
    .sum(axis=1)
    .astype("int16")
)

application_train[
    "APP_MISSING_VALUE_SHARE"
] = (
    application_train[
        "APP_MISSING_VALUE_COUNT"
    ]
    / len(raw_predictor_columns)
).astype("float32")


# =========================================================
# 7. Positive time and age features
# =========================================================
application_train[
    "APP_AGE_YEARS"
] = (
    -application_train[
        "DAYS_BIRTH"
    ]
    / 365.25
).astype("float32")

application_train[
    "APP_EMPLOYMENT_YEARS"
] = (
    -application_train[
        "DAYS_EMPLOYED"
    ]
    / 365.25
).astype("float32")

application_train[
    "APP_REGISTRATION_RECENCY_YEARS"
] = (
    -application_train[
        "DAYS_REGISTRATION"
    ]
    / 365.25
).astype("float32")

application_train[
    "APP_ID_PUBLISH_RECENCY_YEARS"
] = (
    -application_train[
        "DAYS_ID_PUBLISH"
    ]
    / 365.25
).astype("float32")

application_train[
    "APP_PHONE_CHANGE_RECENCY_YEARS"
] = (
    -application_train[
        "DAYS_LAST_PHONE_CHANGE"
    ]
    / 365.25
).astype("float32")

application_train[
    "APP_EMPLOYMENT_TO_AGE_RATIO"
] = (
    application_train[
        "APP_EMPLOYMENT_YEARS"
    ]
    / application_train[
        "APP_AGE_YEARS"
    ]
).astype("float32")


# =========================================================
# 8. Quality checks
# =========================================================
sentinel_values_after = int(
    application_train[
        "DAYS_EMPLOYED"
    ].eq(employment_sentinel_value).sum()
)

positive_employment_days_after = int(
    application_train[
        "DAYS_EMPLOYED"
    ].gt(0).sum()
)

invalid_age_records = int(
    (
        application_train[
            "APP_AGE_YEARS"
        ].le(0)
        | application_train[
            "APP_AGE_YEARS"
        ].gt(100)
    ).sum()
)

invalid_employment_ratio_records = int(
    application_train[
        "APP_EMPLOYMENT_TO_AGE_RATIO"
    ].gt(1).sum()
)

expected_missing_values = (
    original_missing_values
    + employment_sentinel_count
)

missing_values_after = int(
    application_train[
        raw_predictor_columns
    ].isna().sum().sum()
)

features_added = len(
    set(application_train.columns)
    - columns_before_engineering
)

application_stage1_check = pd.DataFrame(
    {
        "metric": [
            "Application rows",
            "Source columns",
            "Current columns",
            "Unique customer IDs",
            "Duplicate customer IDs",
            "Target missing values",
            "Target default rate (%)",
            "Employment sentinel values before",
            "Employment sentinel values after",
            "Positive employment-day records after",
            "Original missing values",
            "Missing values after transformation",
            "Expected missing values after",
            "Invalid age records",
            "Employment/age ratio greater than one",
            "Categorical columns",
            "Features added",
            "Memory before optimization (MB)",
            "Memory after engineering (MB)",
        ],
        "value": [
            len(application_train),
            source_columns,
            application_train.shape[1],
            application_train[
                "SK_ID_CURR"
            ].nunique(),
            application_train[
                "SK_ID_CURR"
            ].duplicated().sum(),
            application_train[
                "TARGET"
            ].isna().sum(),
            (
                application_train[
                    "TARGET"
                ].mean()
                * 100
            ),
            employment_sentinel_count,
            sentinel_values_after,
            positive_employment_days_after,
            original_missing_values,
            missing_values_after,
            expected_missing_values,
            invalid_age_records,
            invalid_employment_ratio_records,
            len(categorical_columns),
            features_added,
            memory_before_optimization,
            (
                application_train
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

assert len(application_train) == source_rows
assert sentinel_values_after == 0
assert positive_employment_days_after == 0
assert missing_values_after == expected_missing_values
assert invalid_age_records == 0

application_stage1_check.to_csv(
    processed_data_path
    / "application_stage1_quality_check.csv",
    index=False,
)

display(
    application_stage1_check.round(3)
)

In [ ]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd

project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

processed_data_path.mkdir(
    parents=True,
    exist_ok=True,
)

# Find application_train.csv recursively
application_train_candidates = [
    path
    for path in project_path.rglob(
        "application_train.csv"
    )
    if path.is_file()
]

assert application_train_candidates, (
    "application_train.csv could not be found "
    f"under {project_path}"
)

# If multiple copies exist, select the largest
application_train_path = max(
    application_train_candidates,
    key=lambda path: path.stat().st_size,
)

extract_path = (
    application_train_path.parent
)

path_discovery_check = pd.DataFrame(
    {
        "metric": [
            "Located file",
            "Detected raw-data folder",
            "File exists",
            "File size (MB)",
            "Candidate file count",
        ],
        "value": [
            str(application_train_path),
            str(extract_path),
            application_train_path.exists(),
            (
                application_train_path
                .stat().st_size
                / 1024**2
            ),
            len(application_train_candidates),
        ],
    }
)

display(path_discovery_check)

**Development data path:** `<PROJECT_ROOT>\data\raw\home_credit`

In [ ]:
application_train = pd.read_csv(
    application_train_path,
    memory_map=True,
)

In [ ]:
memory_before_optimization = (
    application_train
    .memory_usage(deep=True)
    .sum()
    / 1024**2
)

In [ ]:
display(
    application_stage1_check.round(3)
)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# =========================================================
# Complete, rerunnable Stage 1
# =========================================================
project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

application_train_candidates = list(
    project_path.rglob(
        "application_train.csv"
    )
)

assert application_train_candidates, (
    "application_train.csv not found."
)

application_train_path = max(
    application_train_candidates,
    key=lambda path: path.stat().st_size,
)

# Always reload the clean source
application_train = pd.read_csv(
    application_train_path,
    memory_map=True,
)

memory_before_optimization = (
    application_train
    .memory_usage(deep=True)
    .sum()
    / 1024**2
)

source_rows = len(application_train)
source_columns = application_train.shape[1]

assert application_train[
    "SK_ID_CURR"
].is_unique

assert application_train[
    "TARGET"
].notna().all()


# Optimize categorical columns
categorical_columns = (
    application_train
    .select_dtypes(include="object")
    .columns.tolist()
)

for column in categorical_columns:
    application_train[column] = (
        application_train[column]
        .astype("category")
    )


# Optimize integers
integer_columns = (
    application_train
    .select_dtypes(include="int64")
    .columns.tolist()
)

for column in integer_columns:
    application_train[column] = (
        pd.to_numeric(
            application_train[column],
            downcast="integer",
        )
    )


# Optimize floating-point columns
float_columns = (
    application_train
    .select_dtypes(include="float64")
    .columns.tolist()
)

for column in float_columns:
    application_train[column] = (
        pd.to_numeric(
            application_train[column],
            downcast="float",
        )
    )

memory_after_optimization = (
    application_train
    .memory_usage(deep=True)
    .sum()
    / 1024**2
)


# Original predictor columns
raw_predictor_columns = [
    column
    for column in application_train.columns
    if column not in [
        "SK_ID_CURR",
        "TARGET",
    ]
]

original_missing_values = int(
    application_train[
        raw_predictor_columns
    ].isna().sum().sum()
)

columns_before_engineering = set(
    application_train.columns
)


# DAYS_EMPLOYED sentinel
employment_sentinel_value = 365243

employment_sentinel_mask = (
    application_train[
        "DAYS_EMPLOYED"
    ].eq(employment_sentinel_value)
)

employment_sentinel_count = int(
    employment_sentinel_mask.sum()
)

application_train[
    "APP_EMPLOYMENT_ANOMALY_FLAG"
] = (
    employment_sentinel_mask
    .astype("int8")
)

application_train[
    "DAYS_EMPLOYED"
] = (
    application_train[
        "DAYS_EMPLOYED"
    ]
    .mask(employment_sentinel_mask)
    .astype("float32")
)


# Missing-information features
application_train[
    "APP_MISSING_VALUE_COUNT"
] = (
    application_train[
        raw_predictor_columns
    ]
    .isna()
    .sum(axis=1)
    .astype("int16")
)

application_train[
    "APP_MISSING_VALUE_SHARE"
] = (
    application_train[
        "APP_MISSING_VALUE_COUNT"
    ]
    / len(raw_predictor_columns)
).astype("float32")


# Positive time features
application_train[
    "APP_AGE_YEARS"
] = (
    -application_train[
        "DAYS_BIRTH"
    ]
    / 365.25
).astype("float32")

application_train[
    "APP_EMPLOYMENT_YEARS"
] = (
    -application_train[
        "DAYS_EMPLOYED"
    ]
    / 365.25
).astype("float32")

application_train[
    "APP_REGISTRATION_RECENCY_YEARS"
] = (
    -application_train[
        "DAYS_REGISTRATION"
    ]
    / 365.25
).astype("float32")

application_train[
    "APP_ID_PUBLISH_RECENCY_YEARS"
] = (
    -application_train[
        "DAYS_ID_PUBLISH"
    ]
    / 365.25
).astype("float32")

application_train[
    "APP_PHONE_CHANGE_RECENCY_YEARS"
] = (
    -application_train[
        "DAYS_LAST_PHONE_CHANGE"
    ]
    / 365.25
).astype("float32")

application_train[
    "APP_EMPLOYMENT_TO_AGE_RATIO"
] = (
    application_train[
        "APP_EMPLOYMENT_YEARS"
    ]
    / application_train[
        "APP_AGE_YEARS"
    ]
).astype("float32")


# Validation metrics
sentinel_values_after = int(
    application_train[
        "DAYS_EMPLOYED"
    ].eq(employment_sentinel_value).sum()
)

positive_employment_days_after = int(
    application_train[
        "DAYS_EMPLOYED"
    ].gt(0).sum()
)

invalid_age_records = int(
    (
        application_train[
            "APP_AGE_YEARS"
        ].le(0)
        | application_train[
            "APP_AGE_YEARS"
        ].gt(100)
    ).sum()
)

invalid_employment_ratio_records = int(
    application_train[
        "APP_EMPLOYMENT_TO_AGE_RATIO"
    ].gt(1).sum()
)

expected_missing_values = (
    original_missing_values
    + employment_sentinel_count
)

missing_values_after = int(
    application_train[
        raw_predictor_columns
    ].isna().sum().sum()
)

features_added = len(
    set(application_train.columns)
    - columns_before_engineering
)


# Final Stage 1 check
application_stage1_check = pd.DataFrame(
    {
        "metric": [
            "Application rows",
            "Source columns",
            "Current columns",
            "Unique customer IDs",
            "Duplicate customer IDs",
            "Target missing values",
            "Target default rate (%)",
            "Employment sentinel values before",
            "Employment sentinel values after",
            "Positive employment-day records after",
            "Original missing values",
            "Missing values after transformation",
            "Expected missing values after",
            "Invalid age records",
            "Employment/age ratio greater than one",
            "Categorical columns",
            "Features added",
            "Memory before optimization (MB)",
            "Memory after optimization (MB)",
            "Memory after engineering (MB)",
        ],
        "value": [
            len(application_train),
            source_columns,
            application_train.shape[1],
            application_train[
                "SK_ID_CURR"
            ].nunique(),
            application_train[
                "SK_ID_CURR"
            ].duplicated().sum(),
            application_train[
                "TARGET"
            ].isna().sum(),
            (
                application_train[
                    "TARGET"
                ].mean()
                * 100
            ),
            employment_sentinel_count,
            sentinel_values_after,
            positive_employment_days_after,
            original_missing_values,
            missing_values_after,
            expected_missing_values,
            invalid_age_records,
            invalid_employment_ratio_records,
            len(categorical_columns),
            features_added,
            memory_before_optimization,
            memory_after_optimization,
            (
                application_train
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

assert len(application_train) == source_rows
assert sentinel_values_after == 0
assert positive_employment_days_after == 0
assert missing_values_after == expected_missing_values
assert invalid_age_records == 0

application_stage1_check.to_csv(
    processed_data_path
    / "application_stage1_quality_check.csv",
    index=False,
)

display(
    application_stage1_check.round(3)
)

In [ ]:
import numpy as np
import pandas as pd

# Defragment existing table
application_train = application_train.copy()

columns_before_stage2 = set(
    application_train.columns
)


# =========================================================
# 1. Valid denominator masks
# =========================================================
positive_income_mask = (
    application_train[
        "AMT_INCOME_TOTAL"
    ] > 0
)

positive_credit_mask = (
    application_train[
        "AMT_CREDIT"
    ] > 0
)

positive_annuity_mask = (
    application_train[
        "AMT_ANNUITY"
    ] > 0
)

positive_family_mask = (
    application_train[
        "CNT_FAM_MEMBERS"
    ] > 0
)

adult_count = (
    application_train[
        "CNT_FAM_MEMBERS"
    ]
    - application_train[
        "CNT_CHILDREN"
    ]
).astype("float32")

positive_adult_mask = (
    adult_count > 0
)


# =========================================================
# 2. Build all new features separately
# =========================================================
new_application_features = pd.DataFrame(
    index=application_train.index
)


# Financial ratios
new_application_features[
    "APP_CREDIT_TO_INCOME_RATIO"
] = (
    application_train[
        "AMT_CREDIT"
    ]
    / application_train[
        "AMT_INCOME_TOTAL"
    ]
).where(
    positive_income_mask
).astype("float32")

new_application_features[
    "APP_ANNUITY_TO_INCOME_RATIO"
] = (
    application_train[
        "AMT_ANNUITY"
    ]
    / application_train[
        "AMT_INCOME_TOTAL"
    ]
).where(
    positive_income_mask
).astype("float32")

new_application_features[
    "APP_CREDIT_TO_ANNUITY_RATIO"
] = (
    application_train[
        "AMT_CREDIT"
    ]
    / application_train[
        "AMT_ANNUITY"
    ]
).where(
    positive_annuity_mask
).astype("float32")

new_application_features[
    "APP_ANNUITY_TO_CREDIT_RATIO"
] = (
    application_train[
        "AMT_ANNUITY"
    ]
    / application_train[
        "AMT_CREDIT"
    ]
).where(
    positive_credit_mask
).astype("float32")

new_application_features[
    "APP_GOODS_TO_CREDIT_RATIO"
] = (
    application_train[
        "AMT_GOODS_PRICE"
    ]
    / application_train[
        "AMT_CREDIT"
    ]
).where(
    positive_credit_mask
).astype("float32")

new_application_features[
    "APP_CREDIT_GOODS_DIFFERENCE"
] = (
    application_train[
        "AMT_CREDIT"
    ]
    - application_train[
        "AMT_GOODS_PRICE"
    ]
).astype("float32")


# Logarithmic monetary features
new_application_features[
    "APP_LOG_INCOME"
] = np.log1p(
    application_train[
        "AMT_INCOME_TOTAL"
    ].clip(lower=0)
).astype("float32")

new_application_features[
    "APP_LOG_CREDIT"
] = np.log1p(
    application_train[
        "AMT_CREDIT"
    ].clip(lower=0)
).astype("float32")

new_application_features[
    "APP_LOG_ANNUITY"
] = np.log1p(
    application_train[
        "AMT_ANNUITY"
    ].clip(lower=0)
).astype("float32")

new_application_features[
    "APP_LOG_GOODS_PRICE"
] = np.log1p(
    application_train[
        "AMT_GOODS_PRICE"
    ].clip(lower=0)
).astype("float32")


# Household features
new_application_features[
    "APP_ADULT_COUNT"
] = adult_count

new_application_features[
    "APP_INCOME_PER_FAMILY_MEMBER"
] = (
    application_train[
        "AMT_INCOME_TOTAL"
    ]
    / application_train[
        "CNT_FAM_MEMBERS"
    ]
).where(
    positive_family_mask
).astype("float32")

new_application_features[
    "APP_CREDIT_PER_FAMILY_MEMBER"
] = (
    application_train[
        "AMT_CREDIT"
    ]
    / application_train[
        "CNT_FAM_MEMBERS"
    ]
).where(
    positive_family_mask
).astype("float32")

new_application_features[
    "APP_ANNUITY_PER_FAMILY_MEMBER"
] = (
    application_train[
        "AMT_ANNUITY"
    ]
    / application_train[
        "CNT_FAM_MEMBERS"
    ]
).where(
    positive_family_mask
).astype("float32")

new_application_features[
    "APP_INCOME_PER_ADULT"
] = (
    application_train[
        "AMT_INCOME_TOTAL"
    ]
    / adult_count
).where(
    positive_adult_mask
).astype("float32")

new_application_features[
    "APP_CHILDREN_SHARE"
] = (
    application_train[
        "CNT_CHILDREN"
    ]
    / application_train[
        "CNT_FAM_MEMBERS"
    ]
).where(
    positive_family_mask
).astype("float32")

new_application_features[
    "APP_DEPENDENCY_RATIO"
] = (
    application_train[
        "CNT_CHILDREN"
    ]
    / adult_count
).where(
    positive_adult_mask
).astype("float32")

new_application_features[
    "APP_OWN_CAR_AGE_TO_AGE_RATIO"
] = (
    application_train[
        "OWN_CAR_AGE"
    ]
    / application_train[
        "APP_AGE_YEARS"
    ]
).where(
    application_train[
        "APP_AGE_YEARS"
    ].gt(0)
).astype("float32")


# External score features
external_source_columns = [
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
]

external_source_data = (
    application_train[
        external_source_columns
    ]
)

new_application_features[
    "APP_EXT_SOURCE_AVAILABLE_COUNT"
] = (
    external_source_data
    .notna()
    .sum(axis=1)
    .astype("int8")
)

new_application_features[
    "APP_EXT_SOURCE_MEAN"
] = (
    external_source_data
    .mean(axis=1)
    .astype("float32")
)

new_application_features[
    "APP_EXT_SOURCE_MIN"
] = (
    external_source_data
    .min(axis=1)
    .astype("float32")
)

new_application_features[
    "APP_EXT_SOURCE_MAX"
] = (
    external_source_data
    .max(axis=1)
    .astype("float32")
)

new_application_features[
    "APP_EXT_SOURCE_STD"
] = (
    external_source_data
    .std(axis=1)
    .astype("float32")
)

new_application_features[
    "APP_EXT_SOURCE_RANGE"
] = (
    new_application_features[
        "APP_EXT_SOURCE_MAX"
    ]
    - new_application_features[
        "APP_EXT_SOURCE_MIN"
    ]
).astype("float32")


# Social-circle delinquency ratios
new_application_features[
    "APP_SOCIAL_DEFAULT_30_RATIO"
] = (
    application_train[
        "DEF_30_CNT_SOCIAL_CIRCLE"
    ]
    / application_train[
        "OBS_30_CNT_SOCIAL_CIRCLE"
    ]
).where(
    application_train[
        "OBS_30_CNT_SOCIAL_CIRCLE"
    ].gt(0)
).astype("float32")

new_application_features[
    "APP_SOCIAL_DEFAULT_60_RATIO"
] = (
    application_train[
        "DEF_60_CNT_SOCIAL_CIRCLE"
    ]
    / application_train[
        "OBS_60_CNT_SOCIAL_CIRCLE"
    ]
).where(
    application_train[
        "OBS_60_CNT_SOCIAL_CIRCLE"
    ].gt(0)
).astype("float32")


# Credit-bureau request features
bureau_request_columns = [
    "AMT_REQ_CREDIT_BUREAU_HOUR",
    "AMT_REQ_CREDIT_BUREAU_DAY",
    "AMT_REQ_CREDIT_BUREAU_WEEK",
    "AMT_REQ_CREDIT_BUREAU_MON",
    "AMT_REQ_CREDIT_BUREAU_QRT",
    "AMT_REQ_CREDIT_BUREAU_YEAR",
]

bureau_request_data = (
    application_train[
        bureau_request_columns
    ]
)

new_application_features[
    "APP_BUREAU_REQUEST_TOTAL"
] = (
    bureau_request_data
    .sum(
        axis=1,
        min_count=1,
    )
    .astype("float32")
)

new_application_features[
    "APP_BUREAU_REQUEST_AVAILABLE_COUNT"
] = (
    bureau_request_data
    .notna()
    .sum(axis=1)
    .astype("int8")
)

new_application_features[
    "APP_BUREAU_REQUEST_MISSING_COUNT"
] = (
    bureau_request_data
    .isna()
    .sum(axis=1)
    .astype("int8")
)


# Document flags
document_columns = [
    column
    for column in application_train.columns
    if column.startswith(
        "FLAG_DOCUMENT_"
    )
]

new_application_features[
    "APP_DOCUMENT_FLAG_COUNT"
] = (
    application_train[
        document_columns
    ]
    .sum(axis=1)
    .astype("int8")
)


# Contactability flags
contact_columns = [
    "FLAG_MOBIL",
    "FLAG_EMP_PHONE",
    "FLAG_WORK_PHONE",
    "FLAG_CONT_MOBILE",
    "FLAG_PHONE",
    "FLAG_EMAIL",
]

new_application_features[
    "APP_CONTACTABILITY_COUNT"
] = (
    application_train[
        contact_columns
    ]
    .sum(axis=1)
    .astype("int8")
)


# Address mismatch indicators
address_mismatch_columns = [
    "REG_REGION_NOT_LIVE_REGION",
    "REG_REGION_NOT_WORK_REGION",
    "LIVE_REGION_NOT_WORK_REGION",
    "REG_CITY_NOT_LIVE_CITY",
    "REG_CITY_NOT_WORK_CITY",
    "LIVE_CITY_NOT_WORK_CITY",
]

new_application_features[
    "APP_ADDRESS_MISMATCH_COUNT"
] = (
    application_train[
        address_mismatch_columns
    ]
    .sum(axis=1)
    .astype("int8")
)


# =========================================================
# 3. Add all features in one operation
# =========================================================
application_train = pd.concat(
    [
        application_train,
        new_application_features,
    ],
    axis=1,
)

application_train = application_train.copy()


# =========================================================
# 4. Ratio distribution audit
# =========================================================
ratio_audit_columns = [
    "APP_CREDIT_TO_INCOME_RATIO",
    "APP_ANNUITY_TO_INCOME_RATIO",
    "APP_CREDIT_TO_ANNUITY_RATIO",
    "APP_ANNUITY_TO_CREDIT_RATIO",
    "APP_GOODS_TO_CREDIT_RATIO",
    "APP_INCOME_PER_FAMILY_MEMBER",
    "APP_INCOME_PER_ADULT",
    "APP_CHILDREN_SHARE",
    "APP_DEPENDENCY_RATIO",
    "APP_OWN_CAR_AGE_TO_AGE_RATIO",
    "APP_SOCIAL_DEFAULT_30_RATIO",
    "APP_SOCIAL_DEFAULT_60_RATIO",
]

application_ratio_rows = []

for column in ratio_audit_columns:
    values = application_train[column]

    quantiles = values.quantile(
        [0.50, 0.95, 0.99]
    )

    application_ratio_rows.append(
        {
            "feature": column,
            "valid_records": (
                values.notna().sum()
            ),
            "negative_records": (
                values.lt(0).sum()
            ),
            "median": quantiles.loc[0.50],
            "p95": quantiles.loc[0.95],
            "p99": quantiles.loc[0.99],
            "maximum": values.max(),
        }
    )

application_ratio_summary = pd.DataFrame(
    application_ratio_rows
)


# =========================================================
# 5. Stage validation
# =========================================================
stage2_features_added = len(
    set(application_train.columns)
    - columns_before_stage2
)

application_stage2_check = pd.DataFrame(
    {
        "metric": [
            "Application rows",
            "Unique customer IDs",
            "Duplicate customer IDs",
            "Target default rate (%)",
            "Current columns",
            "Stage 2 features added",
            "Missing annuity records",
            "Missing goods-price records",
            "Missing family-member records",
            "Nonpositive income records",
            "Nonpositive credit records",
            "Nonpositive adult-count records",
            "Children greater than family members",
            "All external sources missing",
            "Valid credit/income ratios",
            "Valid credit/annuity ratios",
            "Valid social 30-day ratios",
            "Valid social 60-day ratios",
            "Memory usage (MB)",
        ],
        "value": [
            len(application_train),
            application_train[
                "SK_ID_CURR"
            ].nunique(),
            application_train[
                "SK_ID_CURR"
            ].duplicated().sum(),
            application_train[
                "TARGET"
            ].mean() * 100,
            application_train.shape[1],
            stage2_features_added,
            application_train[
                "AMT_ANNUITY"
            ].isna().sum(),
            application_train[
                "AMT_GOODS_PRICE"
            ].isna().sum(),
            application_train[
                "CNT_FAM_MEMBERS"
            ].isna().sum(),
            application_train[
                "AMT_INCOME_TOTAL"
            ].le(0).sum(),
            application_train[
                "AMT_CREDIT"
            ].le(0).sum(),
            application_train[
                "APP_ADULT_COUNT"
            ].le(0).sum(),
            (
                application_train[
                    "CNT_CHILDREN"
                ]
                > application_train[
                    "CNT_FAM_MEMBERS"
                ]
            ).sum(),
            application_train[
                "APP_EXT_SOURCE_AVAILABLE_COUNT"
            ].eq(0).sum(),
            application_train[
                "APP_CREDIT_TO_INCOME_RATIO"
            ].notna().sum(),
            application_train[
                "APP_CREDIT_TO_ANNUITY_RATIO"
            ].notna().sum(),
            application_train[
                "APP_SOCIAL_DEFAULT_30_RATIO"
            ].notna().sum(),
            application_train[
                "APP_SOCIAL_DEFAULT_60_RATIO"
            ].notna().sum(),
            (
                application_train
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

assert len(application_train) == 307511
assert application_train[
    "SK_ID_CURR"
].is_unique
assert application_train[
    "TARGET"
].notna().all()


# Save checkpoint
application_base_feature_path = (
    processed_data_path
    / "application_train_base_features.parquet"
)

application_train.to_parquet(
    application_base_feature_path,
    index=False,
    compression="snappy",
)

application_stage2_check.to_csv(
    processed_data_path
    / "application_stage2_quality_check.csv",
    index=False,
)

application_ratio_summary.to_csv(
    processed_data_path
    / "application_ratio_summary.csv",
    index=False,
)

display(
    application_stage2_check.round(3)
)

display(
    application_ratio_summary.round(3)
)

In [ ]:
import numpy as np
import pandas as pd

# Robust feature names — remove first if rerunning
robust_feature_names = [
    "APP_CREDIT_TO_INCOME_CAP_15",
    "APP_CREDIT_TO_INCOME_EXTREME_FLAG",
    "APP_LOG1P_CREDIT_TO_INCOME_RATIO",
    "APP_ANNUITY_TO_INCOME_CAP_1",
    "APP_ANNUITY_TO_INCOME_EXTREME_FLAG",
    "APP_GOODS_TO_CREDIT_CAP_2",
    "APP_GOODS_TO_CREDIT_EXTREME_FLAG",
    "APP_DEPENDENCY_RATIO_CAP_5",
    "APP_DEPENDENCY_RATIO_EXTREME_FLAG",
    "APP_LOG1P_DEPENDENCY_RATIO",
    "APP_CAR_AGE_TO_AGE_CAP_2_5",
    "APP_CAR_OLDER_THAN_APPLICANT_FLAG",
    "APP_CAR_AGE_RATIO_EXTREME_FLAG",
    "APP_LOG_INCOME_PER_FAMILY_MEMBER",
    "APP_LOG_INCOME_PER_ADULT",
]

existing_robust_features = [
    column
    for column in robust_feature_names
    if column in application_train.columns
]

if existing_robust_features:
    application_train.drop(
        columns=existing_robust_features,
        inplace=True,
    )

columns_before_robust_features = set(
    application_train.columns
)


# Build in a separate DataFrame
robust_features = pd.DataFrame(
    index=application_train.index
)

credit_income_ratio = (
    application_train[
        "APP_CREDIT_TO_INCOME_RATIO"
    ]
)

annuity_income_ratio = (
    application_train[
        "APP_ANNUITY_TO_INCOME_RATIO"
    ]
)

goods_credit_ratio = (
    application_train[
        "APP_GOODS_TO_CREDIT_RATIO"
    ]
)

dependency_ratio = (
    application_train[
        "APP_DEPENDENCY_RATIO"
    ]
)

car_age_ratio = (
    application_train[
        "APP_OWN_CAR_AGE_TO_AGE_RATIO"
    ]
)


# Credit / income
robust_features[
    "APP_CREDIT_TO_INCOME_CAP_15"
] = (
    credit_income_ratio
    .clip(lower=0, upper=15)
    .astype("float32")
)

robust_features[
    "APP_CREDIT_TO_INCOME_EXTREME_FLAG"
] = (
    credit_income_ratio.gt(15)
    .astype("int8")
)

robust_features[
    "APP_LOG1P_CREDIT_TO_INCOME_RATIO"
] = np.log1p(
    credit_income_ratio.clip(lower=0)
).astype("float32")


# Annuity / income
robust_features[
    "APP_ANNUITY_TO_INCOME_CAP_1"
] = (
    annuity_income_ratio
    .clip(lower=0, upper=1)
    .astype("float32")
)

robust_features[
    "APP_ANNUITY_TO_INCOME_EXTREME_FLAG"
] = (
    annuity_income_ratio.gt(1)
    .astype("int8")
)


# Goods price / credit
robust_features[
    "APP_GOODS_TO_CREDIT_CAP_2"
] = (
    goods_credit_ratio
    .clip(lower=0, upper=2)
    .astype("float32")
)

robust_features[
    "APP_GOODS_TO_CREDIT_EXTREME_FLAG"
] = (
    goods_credit_ratio.gt(2)
    .astype("int8")
)


# Household dependency
robust_features[
    "APP_DEPENDENCY_RATIO_CAP_5"
] = (
    dependency_ratio
    .clip(lower=0, upper=5)
    .astype("float32")
)

robust_features[
    "APP_DEPENDENCY_RATIO_EXTREME_FLAG"
] = (
    dependency_ratio.gt(5)
    .astype("int8")
)

robust_features[
    "APP_LOG1P_DEPENDENCY_RATIO"
] = np.log1p(
    dependency_ratio.clip(lower=0)
).astype("float32")


# Car age / applicant age
robust_features[
    "APP_CAR_AGE_TO_AGE_CAP_2_5"
] = (
    car_age_ratio
    .clip(lower=0, upper=2.5)
    .astype("float32")
)

robust_features[
    "APP_CAR_OLDER_THAN_APPLICANT_FLAG"
] = (
    car_age_ratio.gt(1)
    .astype("int8")
)

robust_features[
    "APP_CAR_AGE_RATIO_EXTREME_FLAG"
] = (
    car_age_ratio.gt(2.5)
    .astype("int8")
)


# Robust household-income features
robust_features[
    "APP_LOG_INCOME_PER_FAMILY_MEMBER"
] = np.log1p(
    application_train[
        "APP_INCOME_PER_FAMILY_MEMBER"
    ].clip(lower=0)
).astype("float32")

robust_features[
    "APP_LOG_INCOME_PER_ADULT"
] = np.log1p(
    application_train[
        "APP_INCOME_PER_ADULT"
    ].clip(lower=0)
).astype("float32")


# Add features in one operation
application_train = pd.concat(
    [
        application_train,
        robust_features,
    ],
    axis=1,
)

application_train = application_train.copy()


# Validation
robust_features_added = len(
    set(application_train.columns)
    - columns_before_robust_features
)

application_robust_feature_check = pd.DataFrame(
    {
        "metric": [
            "Application rows",
            "Unique customer IDs",
            "Duplicate customer IDs",
            "Robust features added",
            "Current columns",
            "Credit/income ratio above 15",
            "Maximum capped credit/income ratio",
            "Annuity/income ratio above 1",
            "Maximum capped annuity/income ratio",
            "Goods/credit ratio above 2",
            "Maximum capped goods/credit ratio",
            "Dependency ratio above 5",
            "Maximum capped dependency ratio",
            "Car older than applicant",
            "Car-age ratio above 2.5",
            "Maximum capped car-age ratio",
            "Memory usage (MB)",
        ],
        "value": [
            len(application_train),
            application_train[
                "SK_ID_CURR"
            ].nunique(),
            application_train[
                "SK_ID_CURR"
            ].duplicated().sum(),
            robust_features_added,
            application_train.shape[1],
            application_train[
                "APP_CREDIT_TO_INCOME_EXTREME_FLAG"
            ].sum(),
            application_train[
                "APP_CREDIT_TO_INCOME_CAP_15"
            ].max(),
            application_train[
                "APP_ANNUITY_TO_INCOME_EXTREME_FLAG"
            ].sum(),
            application_train[
                "APP_ANNUITY_TO_INCOME_CAP_1"
            ].max(),
            application_train[
                "APP_GOODS_TO_CREDIT_EXTREME_FLAG"
            ].sum(),
            application_train[
                "APP_GOODS_TO_CREDIT_CAP_2"
            ].max(),
            application_train[
                "APP_DEPENDENCY_RATIO_EXTREME_FLAG"
            ].sum(),
            application_train[
                "APP_DEPENDENCY_RATIO_CAP_5"
            ].max(),
            application_train[
                "APP_CAR_OLDER_THAN_APPLICANT_FLAG"
            ].sum(),
            application_train[
                "APP_CAR_AGE_RATIO_EXTREME_FLAG"
            ].sum(),
            application_train[
                "APP_CAR_AGE_TO_AGE_CAP_2_5"
            ].max(),
            (
                application_train
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

assert robust_features_added == 15
assert application_train[
    "SK_ID_CURR"
].is_unique
assert application_train[
    "APP_CREDIT_TO_INCOME_CAP_15"
].max() <= 15
assert application_train[
    "APP_ANNUITY_TO_INCOME_CAP_1"
].max() <= 1
assert application_train[
    "APP_GOODS_TO_CREDIT_CAP_2"
].max() <= 2
assert application_train[
    "APP_DEPENDENCY_RATIO_CAP_5"
].max() <= 5
assert application_train[
    "APP_CAR_AGE_TO_AGE_CAP_2_5"
].max() <= 2.5


# Update checkpoint
application_base_feature_path = (
    processed_data_path
    / "application_train_base_features.parquet"
)

application_train.to_parquet(
    application_base_feature_path,
    index=False,
    compression="snappy",
)

application_robust_feature_check.to_csv(
    processed_data_path
    / "application_robust_feature_check.csv",
    index=False,
)

display(
    application_robust_feature_check.round(3)
)

In [ ]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd

# =========================================================
# 1. Paths
# =========================================================
project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

application_feature_path = (
    processed_data_path
    / "application_train_base_features.parquet"
)

history_feature_path = (
    processed_data_path
    / "application_train_features_all_sources.parquet"
)

modeling_table_path = (
    processed_data_path
    / "modeling_train_full.parquet"
)

assert application_feature_path.exists()
assert history_feature_path.exists()


# =========================================================
# 2. Load clean checkpoints
# =========================================================
application_features = pd.read_parquet(
    application_feature_path
)

history_features = pd.read_parquet(
    history_feature_path
)

assert application_features[
    "SK_ID_CURR"
].is_unique

assert history_features[
    "SK_ID_CURR"
].is_unique

assert "TARGET" in application_features.columns
assert "TARGET" not in history_features.columns


# =========================================================
# 3. Join-integrity checks
# =========================================================
application_ids = pd.Index(
    application_features[
        "SK_ID_CURR"
    ]
)

history_ids = pd.Index(
    history_features[
        "SK_ID_CURR"
    ]
)

application_without_history_ids = (
    application_ids.difference(
        history_ids
    )
)

history_without_application_ids = (
    history_ids.difference(
        application_ids
    )
)

overlapping_feature_columns = sorted(
    (
        set(application_features.columns)
        .intersection(
            history_features.columns
        )
    )
    - {"SK_ID_CURR"}
)

assert not overlapping_feature_columns
assert len(application_without_history_ids) == 0
assert len(history_without_application_ids) == 0


# =========================================================
# 4. Safe one-to-one merge
# =========================================================
modeling_train_full = (
    application_features.merge(
        history_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one",
    )
)

modeling_train_full = (
    modeling_train_full.copy()
)

assert len(modeling_train_full) == 307511
assert modeling_train_full[
    "SK_ID_CURR"
].is_unique
assert modeling_train_full[
    "TARGET"
].notna().all()


# =========================================================
# 5. Define predictors
# =========================================================
identifier_columns = [
    "SK_ID_CURR"
]

target_column = "TARGET"

predictor_columns = [
    column
    for column in modeling_train_full.columns
    if column not in (
        identifier_columns
        + [target_column]
    )
]

categorical_predictor_columns = (
    modeling_train_full[
        predictor_columns
    ]
    .select_dtypes(
        include=[
            "category",
            "object",
            "string",
        ]
    )
    .columns.tolist()
)

numeric_predictor_columns = [
    column
    for column in predictor_columns
    if column not in (
        categorical_predictor_columns
    )
]


# =========================================================
# 6. Feature-level quality profile
# =========================================================
def identify_feature_source(column):
    if column.startswith("APP_"):
        return "Application engineered"

    if column.startswith("BUREAU_"):
        return "Bureau and bureau balance"

    if (
        column.startswith("PREV_")
        or column.startswith("PREVIOUS_")
    ):
        return "Previous applications"

    if column.startswith("POS_"):
        return "POS CASH"

    if column.startswith("INST_"):
        return "Installment payments"

    if column.startswith("CC_"):
        return "Credit-card balance"

    return "Application raw"


feature_schema_rows = []
total_infinite_values = 0

for column in predictor_columns:
    values = modeling_train_full[column]

    missing_count = int(
        values.isna().sum()
    )

    unique_count = int(
        values.nunique(
            dropna=False
        )
    )

    infinite_count = 0

    if column in numeric_predictor_columns:
        infinite_count = int(
            np.isinf(
                values.to_numpy()
            ).sum()
        )

        total_infinite_values += (
            infinite_count
        )

    feature_schema_rows.append(
        {
            "feature": column,
            "source": (
                identify_feature_source(
                    column
                )
            ),
            "dtype": str(values.dtype),
            "missing_count": missing_count,
            "missing_share_pct": (
                missing_count
                / len(modeling_train_full)
                * 100
            ),
            "unique_values": unique_count,
            "infinite_values": (
                infinite_count
            ),
            "is_all_null": (
                missing_count
                == len(modeling_train_full)
            ),
            "is_constant": (
                unique_count <= 1
            ),
        }
    )

modeling_feature_schema = pd.DataFrame(
    feature_schema_rows
)

all_null_feature_columns = (
    modeling_feature_schema.loc[
        modeling_feature_schema[
            "is_all_null"
        ],
        "feature",
    ].tolist()
)

constant_feature_columns = (
    modeling_feature_schema.loc[
        modeling_feature_schema[
            "is_constant"
        ],
        "feature",
    ].tolist()
)

constant_feature_summary = (
    modeling_feature_schema[
        modeling_feature_schema[
            "is_constant"
        ]
    ]
    .sort_values(
        [
            "source",
            "feature",
        ]
    )
    .reset_index(drop=True)
)

total_missing_feature_values = int(
    modeling_feature_schema[
        "missing_count"
    ].sum()
)

total_feature_cells = (
    len(modeling_train_full)
    * len(predictor_columns)
)

missing_feature_share_pct = (
    total_missing_feature_values
    / total_feature_cells
    * 100
)


# =========================================================
# 7. Feature-source summary
# =========================================================
modeling_feature_source_summary = (
    modeling_feature_schema
    .groupby(
        "source",
        sort=False,
    )
    .agg(
        feature_count=(
            "feature",
            "size",
        ),
        categorical_features=(
            "dtype",
            lambda values: values.isin(
                [
                    "category",
                    "object",
                    "string",
                ]
            ).sum(),
        ),
        all_null_features=(
            "is_all_null",
            "sum",
        ),
        constant_features=(
            "is_constant",
            "sum",
        ),
        mean_missing_share_pct=(
            "missing_share_pct",
            "mean",
        ),
    )
    .reset_index()
)

assert (
    modeling_feature_source_summary[
        "feature_count"
    ].sum()
    == len(predictor_columns)
)


# =========================================================
# 8. Save and verify
# =========================================================
modeling_train_full.to_parquet(
    modeling_table_path,
    index=False,
    compression="snappy",
)

modeling_verification = pd.read_parquet(
    modeling_table_path,
    columns=[
        "SK_ID_CURR",
        "TARGET",
        "APP_AGE_YEARS",
        "BUREAU_HAS_HISTORY",
        "INST_HAS_HISTORY",
        "CC_HAS_HISTORY",
    ],
)

assert len(modeling_verification) == 307511
assert modeling_verification[
    "SK_ID_CURR"
].is_unique

assert np.isclose(
    modeling_verification[
        "TARGET"
    ].mean(),
    application_features[
        "TARGET"
    ].mean(),
)


# =========================================================
# 9. Save quality evidence
# =========================================================
modeling_feature_schema.to_csv(
    processed_data_path
    / "modeling_feature_schema.csv",
    index=False,
)

constant_feature_summary.to_csv(
    processed_data_path
    / "modeling_constant_features.csv",
    index=False,
)

modeling_feature_source_summary.to_csv(
    processed_data_path
    / "modeling_feature_source_summary.csv",
    index=False,
)


# =========================================================
# 10. Assembly summary
# =========================================================
modeling_assembly_check = pd.DataFrame(
    {
        "metric": [
            "Application feature rows",
            "History feature rows",
            "Final modeling rows",
            "Row count difference",
            "Unique customer IDs",
            "Duplicate customer IDs",
            "Application IDs without history row",
            "History IDs without application row",
            "Unexpected overlapping features",
            "Target missing values",
            "Target default rate (%)",
            "Application predictor features",
            "Relational-history features",
            "Final predictor features",
            "Categorical predictor features",
            "Numeric predictor features",
            "Duplicate column names",
            "Infinite numeric values",
            "All-null predictor features",
            "Constant predictor features",
            "Total missing feature values",
            "Missing feature-cell share (%)",
            "Verified saved rows",
            "Saved file size (MB)",
            "Final table memory (MB)",
        ],
        "value": [
            len(application_features),
            len(history_features),
            len(modeling_train_full),
            (
                len(modeling_train_full)
                - len(application_features)
            ),
            modeling_train_full[
                "SK_ID_CURR"
            ].nunique(),
            modeling_train_full[
                "SK_ID_CURR"
            ].duplicated().sum(),
            len(application_without_history_ids),
            len(history_without_application_ids),
            len(overlapping_feature_columns),
            modeling_train_full[
                "TARGET"
            ].isna().sum(),
            (
                modeling_train_full[
                    "TARGET"
                ].mean()
                * 100
            ),
            (
                application_features.shape[1]
                - 2
            ),
            (
                history_features.shape[1]
                - 1
            ),
            len(predictor_columns),
            len(
                categorical_predictor_columns
            ),
            len(numeric_predictor_columns),
            modeling_train_full.columns
            .duplicated()
            .sum(),
            total_infinite_values,
            len(all_null_feature_columns),
            len(constant_feature_columns),
            total_missing_feature_values,
            missing_feature_share_pct,
            len(modeling_verification),
            (
                modeling_table_path
                .stat().st_size
                / 1024**2
            ),
            (
                modeling_train_full
                .memory_usage(deep=True)
                .sum()
                / 1024**2
            ),
        ],
    }
)

modeling_assembly_check.to_csv(
    processed_data_path
    / "modeling_assembly_check.csv",
    index=False,
)


# =========================================================
# 11. Display bounded results
# =========================================================
display(
    modeling_assembly_check.round(3)
)

display(
    modeling_feature_source_summary.round(3)
)

display(
    constant_feature_summary.head(30)
)


# =========================================================
# 12. Safe cleanup
# =========================================================
objects_to_remove = [
    "application_features",
    "history_features",
    "modeling_train_full",
    "modeling_verification",
]

for object_name in objects_to_remove:
    globals().pop(
        object_name,
        None,
    )

gc.collect()

In [ ]:
from pathlib import Path
from importlib.util import find_spec
from importlib import metadata
import gc

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# =========================================================
# 1. Paths
# =========================================================
project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

full_model_path = (
    processed_data_path
    / "modeling_train_full.parquet"
)

application_feature_path = (
    processed_data_path
    / "application_train_base_features.parquet"
)

history_feature_path = (
    processed_data_path
    / "application_train_features_all_sources.parquet"
)

clean_model_path = (
    processed_data_path
    / "modeling_train_clean.parquet"
)

clean_schema_path = (
    processed_data_path
    / "modeling_feature_schema_clean.csv"
)

dropped_feature_path = (
    processed_data_path
    / "modeling_dropped_features.csv"
)

source_summary_path = (
    processed_data_path
    / "modeling_feature_source_summary_clean.csv"
)

required_paths = [
    full_model_path,
    application_feature_path,
    history_feature_path,
]

assert all(path.exists() for path in required_paths), {
    str(path): path.exists()
    for path in required_paths
}


# =========================================================
# 2. Load final table
# =========================================================
modeling_train_clean = pd.read_parquet(
    full_model_path
)

source_rows = len(modeling_train_clean)

predictors_before = [
    column
    for column in modeling_train_clean.columns
    if column not in ["SK_ID_CURR", "TARGET"]
]

assert len(predictors_before) == 642


# =========================================================
# 3. Validate and remove constant features
# =========================================================
constant_columns = [
    "CC_CUSTOMER_LATEST_STATUS_APPROVED_ACCOUNT_COUNT",
    "CC_CUSTOMER_LATEST_STATUS_REFUSED_ACCOUNT_COUNT",
    "CC_CUSTOMER_LATEST_STATUS_SENT_PROPOSAL_ACCOUNT_COUNT",
]

missing_constant_columns = (
    set(constant_columns)
    - set(modeling_train_clean.columns)
)

assert not missing_constant_columns, (
    f"Missing constant columns: "
    f"{missing_constant_columns}"
)

constant_feature_audit = pd.DataFrame({
    "feature": constant_columns,
    "dtype": [
        str(modeling_train_clean[column].dtype)
        for column in constant_columns
    ],
    "unique_non_null_values": [
        modeling_train_clean[column]
        .nunique(dropna=True)
        for column in constant_columns
    ],
    "minimum": [
        modeling_train_clean[column].min()
        for column in constant_columns
    ],
    "maximum": [
        modeling_train_clean[column].max()
        for column in constant_columns
    ],
    "missing_values": [
        modeling_train_clean[column]
        .isna().sum()
        for column in constant_columns
    ],
})

assert (
    constant_feature_audit[
        "unique_non_null_values"
    ].eq(1).all()
)

assert (
    constant_feature_audit[
        ["minimum", "maximum"]
    ].eq(0).all().all()
)

modeling_train_clean.drop(
    columns=constant_columns,
    inplace=True,
)

dropped_features = pd.DataFrame({
    "feature": constant_columns,
    "reason": (
        "Constant zero in application_train"
    ),
    "action": "Dropped before modeling",
})

dropped_features.to_csv(
    dropped_feature_path,
    index=False,
)


# =========================================================
# 4. Recover exact feature provenance
# =========================================================
application_schema_columns = (
    pq.ParquetFile(
        application_feature_path
    ).schema.names
)

history_schema_columns = (
    pq.ParquetFile(
        history_feature_path
    ).schema.names
)

application_predictors = [
    column
    for column in application_schema_columns
    if column not in [
        "SK_ID_CURR",
        "TARGET",
    ]
    and not column.startswith(
        "__index_level_"
    )
]

history_predictors = [
    column
    for column in history_schema_columns
    if column != "SK_ID_CURR"
    and not column.startswith(
        "__index_level_"
    )
]

assert len(application_predictors) == 176
assert len(history_predictors) == 466

assert (
    set(application_predictors)
    .isdisjoint(history_predictors)
)

assert (
    set(application_predictors)
    | set(history_predictors)
) == set(predictors_before)


feature_source_map = {}

for feature in application_predictors:
    if feature.startswith("APP_"):
        feature_source_map[
            feature
        ] = "Application engineered"
    else:
        feature_source_map[
            feature
        ] = "Application raw"


history_blocks = [
    ("Bureau and bureau balance", 88),
    ("Previous applications", 73),
    ("POS CASH", 67),
    ("Installment payments", 81),
    ("Credit-card balance", 157),
]

block_start = 0

for source_name, feature_count in history_blocks:
    block_end = block_start + feature_count

    for feature in history_predictors[
        block_start:block_end
    ]:
        feature_source_map[
            feature
        ] = source_name

    block_start = block_end

assert block_start == len(history_predictors)


# =========================================================
# 5. Clean predictor schema and validations
# =========================================================
predictor_columns = [
    column
    for column in modeling_train_clean.columns
    if column not in ["SK_ID_CURR", "TARGET"]
]

numeric_predictors = [
    column
    for column in predictor_columns
    if pd.api.types.is_numeric_dtype(
        modeling_train_clean[column]
    )
]

categorical_predictors = [
    column
    for column in predictor_columns
    if column not in numeric_predictors
]

assert len(predictor_columns) == 639
assert len(categorical_predictors) == 16
assert len(numeric_predictors) == 623

assert all(
    feature in feature_source_map
    for feature in predictor_columns
)


# Memory-safe missing/infinite checks
missing_by_feature = {}
infinite_numeric_values = 0

for feature in predictor_columns:
    missing_by_feature[feature] = int(
        modeling_train_clean[
            feature
        ].isna().sum()
    )

for feature in numeric_predictors:
    values = modeling_train_clean[
        feature
    ].to_numpy(copy=False)

    infinite_numeric_values += int(
        np.isinf(values).sum()
    )

total_missing_values = sum(
    missing_by_feature.values()
)

all_null_features = sum(
    missing_count
    == len(modeling_train_clean)
    for missing_count
    in missing_by_feature.values()
)

clean_feature_schema = pd.DataFrame({
    "feature": predictor_columns,
    "source": [
        feature_source_map[feature]
        for feature in predictor_columns
    ],
    "dtype": [
        str(modeling_train_clean[feature].dtype)
        for feature in predictor_columns
    ],
    "missing_values": [
        missing_by_feature[feature]
        for feature in predictor_columns
    ],
})

clean_feature_schema[
    "missing_share_pct"
] = (
    clean_feature_schema[
        "missing_values"
    ]
    / len(modeling_train_clean)
    * 100
)

clean_feature_schema.to_csv(
    clean_schema_path,
    index=False,
)


# =========================================================
# 6. Correct source summary
# =========================================================
source_order = [
    "Application raw",
    "Application engineered",
    "Bureau and bureau balance",
    "Previous applications",
    "POS CASH",
    "Installment payments",
    "Credit-card balance",
]

expected_source_counts = {
    "Application raw": 120,
    "Application engineered": 56,
    "Bureau and bureau balance": 88,
    "Previous applications": 73,
    "POS CASH": 67,
    "Installment payments": 81,
    "Credit-card balance": 154,
}

actual_source_counts = (
    clean_feature_schema[
        "source"
    ].value_counts()
)

feature_source_summary_clean = pd.DataFrame({
    "source": source_order,
    "expected_features": [
        expected_source_counts[source]
        for source in source_order
    ],
    "actual_features": [
        int(actual_source_counts.get(source, 0))
        for source in source_order
    ],
})

feature_source_summary_clean[
    "difference"
] = (
    feature_source_summary_clean[
        "actual_features"
    ]
    - feature_source_summary_clean[
        "expected_features"
    ]
)

assert (
    feature_source_summary_clean[
        "difference"
    ].eq(0).all()
)

feature_source_summary_clean.to_csv(
    source_summary_path,
    index=False,
)


# =========================================================
# 7. Save and verify cleaned modeling table
# =========================================================
modeling_train_clean.to_parquet(
    clean_model_path,
    index=False,
    compression="snappy",
)

saved_verification = pd.read_parquet(
    clean_model_path,
    columns=[
        "SK_ID_CURR",
        "TARGET",
    ],
)

modeling_cleanup_check = pd.DataFrame({
    "metric": [
        "Source modeling rows",
        "Clean modeling rows",
        "Row count difference",
        "Unique customer IDs",
        "Duplicate customer IDs",
        "Predictors before cleanup",
        "Constant predictors dropped",
        "Final predictor features",
        "Categorical predictors",
        "Numeric predictors",
        "Target missing values",
        "Target default rate (%)",
        "Infinite numeric values",
        "All-null predictors",
        "Total missing feature values",
        "Missing feature-cell share (%)",
        "Verified saved rows",
        "Clean file size (MB)",
        "Clean table memory (MB)",
    ],
    "value": [
        source_rows,
        len(modeling_train_clean),
        len(modeling_train_clean) - source_rows,
        modeling_train_clean[
            "SK_ID_CURR"
        ].nunique(),
        modeling_train_clean[
            "SK_ID_CURR"
        ].duplicated().sum(),
        len(predictors_before),
        len(constant_columns),
        len(predictor_columns),
        len(categorical_predictors),
        len(numeric_predictors),
        modeling_train_clean[
            "TARGET"
        ].isna().sum(),
        modeling_train_clean[
            "TARGET"
        ].mean() * 100,
        infinite_numeric_values,
        all_null_features,
        total_missing_values,
        (
            total_missing_values
            / (
                len(modeling_train_clean)
                * len(predictor_columns)
            )
            * 100
        ),
        len(saved_verification),
        (
            clean_model_path.stat().st_size
            / 1024**2
        ),
        (
            modeling_train_clean
            .memory_usage(deep=True)
            .sum()
            / 1024**2
        ),
    ],
})


# =========================================================
# 8. Detect available modeling packages
# =========================================================
package_specs = [
    ("scikit-learn", "sklearn", "scikit-learn"),
    ("LightGBM", "lightgbm", "lightgbm"),
    ("CatBoost", "catboost", "catboost"),
    ("XGBoost", "xgboost", "xgboost"),
    ("SHAP", "shap", "shap"),
    ("Optuna", "optuna", "optuna"),
    (
        "imbalanced-learn",
        "imblearn",
        "imbalanced-learn",
    ),
]

package_rows = []

for display_name, module_name, distribution_name in package_specs:
    available = (
        find_spec(module_name) is not None
    )

    try:
        version = metadata.version(
            distribution_name
        )
    except metadata.PackageNotFoundError:
        version = None

    package_rows.append({
        "package": display_name,
        "available": available,
        "version": version,
    })

modeling_package_check = pd.DataFrame(
    package_rows
)


# =========================================================
# 9. Display results and free memory
# =========================================================
display(
    modeling_cleanup_check.round(3)
)

display(
    feature_source_summary_clean
)

display(
    constant_feature_audit
)

display(
    modeling_package_check
)

del saved_verification
del modeling_train_clean
gc.collect()

In [ ]:
%pip install lightgbm

In [ ]:
import sys
import platform
import lightgbm as lgb
import sklearn
import pandas as pd
import numpy as np

lightgbm_environment_check = pd.DataFrame({
    "metric": [
        "Python executable",
        "Python version",
        "Operating system",
        "LightGBM version",
        "Scikit-learn version",
        "Pandas version",
        "NumPy version",
        "LightGBM import successful",
    ],
    "value": [
        sys.executable,
        platform.python_version(),
        platform.platform(),
        lgb.__version__,
        sklearn.__version__,
        pd.__version__,
        np.__version__,
        True,
    ],
})

display(lightgbm_environment_check)

In [ ]:
from pathlib import Path
from time import perf_counter
import gc
import json

import lightgbm as lgb
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    roc_curve,
)


# =========================================================
# 1. Paths
# =========================================================
project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

modeling_data_path = (
    processed_data_path
    / "modeling_train_clean.parquet"
)

model_output_path = (
    project_path
    / "models"
    / "lightgbm_baseline"
)

metric_output_path = (
    processed_data_path
    / "modeling"
)

model_output_path.mkdir(
    parents=True,
    exist_ok=True,
)

metric_output_path.mkdir(
    parents=True,
    exist_ok=True,
)

assert modeling_data_path.exists()


# =========================================================
# 2. Load clean modeling table
# =========================================================
modeling_features = pd.read_parquet(
    modeling_data_path
)

customer_ids = modeling_features.pop(
    "SK_ID_CURR"
)

target = (
    modeling_features.pop("TARGET")
    .astype("int8")
)

feature_columns = list(
    modeling_features.columns
)

categorical_features = [
    feature
    for feature in feature_columns
    if not pd.api.types.is_numeric_dtype(
        modeling_features[feature]
    )
]

numeric_features = [
    feature
    for feature in feature_columns
    if feature not in categorical_features
]

# Ensure consistent categorical metadata across folds
for feature in categorical_features:
    if not isinstance(
        modeling_features[feature].dtype,
        pd.CategoricalDtype,
    ):
        modeling_features[feature] = (
            modeling_features[feature]
            .astype("category")
        )


# =========================================================
# 3. Input validation
# =========================================================
assert len(modeling_features) == 307_511
assert len(feature_columns) == 639
assert len(categorical_features) == 16
assert len(numeric_features) == 623

assert customer_ids.nunique() == len(customer_ids)
assert not customer_ids.isna().any()
assert not target.isna().any()
assert set(target.unique()) == {0, 1}


# =========================================================
# 4. Reproducible baseline configuration
# =========================================================
random_seed = 42
number_of_folds = 5

model_parameters = {
    "objective": "binary",
    "boosting_type": "gbdt",
    "n_estimators": 3000,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 100,
    "subsample": 0.80,
    "subsample_freq": 1,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.10,
    "reg_lambda": 1.00,
    "class_weight": None,
    "random_state": random_seed,
    "n_jobs": -1,
    "verbosity": -1,
    "deterministic": True,
    "force_col_wise": True,
}

with open(
    model_output_path
    / "lightgbm_baseline_parameters.json",
    "w",
    encoding="utf-8",
) as parameter_file:
    json.dump(
        model_parameters,
        parameter_file,
        indent=2,
    )


# =========================================================
# 5. Stratified cross-validation
# =========================================================
cross_validator = StratifiedKFold(
    n_splits=number_of_folds,
    shuffle=True,
    random_state=random_seed,
)

oof_predictions = np.full(
    len(target),
    np.nan,
    dtype="float32",
)

fold_assignments = np.zeros(
    len(target),
    dtype="int8",
)

fold_metric_records = []
feature_importance_records = []

training_start_time = perf_counter()

for fold_number, (
    training_indices,
    validation_indices,
) in enumerate(
    cross_validator.split(
        modeling_features,
        target,
    ),
    start=1,
):
    fold_start_time = perf_counter()

    print(
        f"\n========== Fold "
        f"{fold_number}/{number_of_folds} =========="
    )

    training_features = (
        modeling_features.iloc[
            training_indices
        ]
    )

    validation_features = (
        modeling_features.iloc[
            validation_indices
        ]
    )

    training_target = target.iloc[
        training_indices
    ]

    validation_target = target.iloc[
        validation_indices
    ]

    model = lgb.LGBMClassifier(
        **model_parameters
    )

    model.fit(
        training_features,
        training_target,
        eval_set=[
            (
                validation_features,
                validation_target,
            )
        ],
        eval_metric="auc",
        categorical_feature=(
            categorical_features
        ),
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=150,
                first_metric_only=True,
                verbose=False,
            ),
            lgb.log_evaluation(
                period=200
            ),
        ],
    )

    validation_probabilities = (
        model.predict_proba(
            validation_features,
            num_iteration=(
                model.best_iteration_
            ),
        )[:, 1]
    )

    oof_predictions[
        validation_indices
    ] = validation_probabilities.astype(
        "float32"
    )

    fold_assignments[
        validation_indices
    ] = fold_number

    fold_auc = roc_auc_score(
        validation_target,
        validation_probabilities,
    )

    fold_average_precision = (
        average_precision_score(
            validation_target,
            validation_probabilities,
        )
    )

    fold_log_loss = log_loss(
        validation_target,
        validation_probabilities,
        labels=[0, 1],
    )

    fold_brier_score = brier_score_loss(
        validation_target,
        validation_probabilities,
    )

    fold_elapsed_seconds = (
        perf_counter()
        - fold_start_time
    )

    fold_metric_records.append({
        "fold": fold_number,
        "training_rows": len(
            training_indices
        ),
        "validation_rows": len(
            validation_indices
        ),
        "training_default_rate_pct": (
            training_target.mean() * 100
        ),
        "validation_default_rate_pct": (
            validation_target.mean() * 100
        ),
        "best_iteration": (
            model.best_iteration_
        ),
        "roc_auc": fold_auc,
        "average_precision": (
            fold_average_precision
        ),
        "log_loss": fold_log_loss,
        "brier_score": fold_brier_score,
        "elapsed_minutes": (
            fold_elapsed_seconds / 60
        ),
    })

    booster = model.booster_

    fold_importance = pd.DataFrame({
        "feature": (
            booster.feature_name()
        ),
        "fold": fold_number,
        "gain": (
            booster.feature_importance(
                importance_type="gain"
            )
        ),
        "split": (
            booster.feature_importance(
                importance_type="split"
            )
        ),
    })

    feature_importance_records.append(
        fold_importance
    )

    # Save trained fold model
    booster.save_model(
        str(
            model_output_path
            / (
                f"lightgbm_baseline_"
                f"fold_{fold_number}.txt"
            )
        ),
        num_iteration=model.best_iteration_,
    )

    # Save checkpoints after each fold
    fold_metrics_checkpoint = (
        pd.DataFrame(
            fold_metric_records
        )
    )

    fold_metrics_checkpoint.to_csv(
        metric_output_path
        / "lightgbm_baseline_fold_metrics.csv",
        index=False,
    )

    oof_checkpoint = pd.DataFrame({
        "SK_ID_CURR": customer_ids,
        "TARGET": target,
        "CV_FOLD": fold_assignments,
        "LIGHTGBM_BASELINE_OOF": (
            oof_predictions
        ),
    })

    oof_checkpoint.to_parquet(
        metric_output_path
        / "lightgbm_baseline_oof.parquet",
        index=False,
        compression="snappy",
    )

    print(
        f"Fold {fold_number} ROC-AUC: "
        f"{fold_auc:.6f}"
    )

    print(
        f"Fold {fold_number} "
        f"best iteration: "
        f"{model.best_iteration_}"
    )

    del training_features
    del validation_features
    del training_target
    del validation_target
    del validation_probabilities
    del model
    del booster
    gc.collect()


# =========================================================
# 6. OOF validation
# =========================================================
assert not np.isnan(
    oof_predictions
).any()

assert np.isfinite(
    oof_predictions
).all()

assert (
    (oof_predictions >= 0)
    & (oof_predictions <= 1)
).all()

assert set(
    np.unique(fold_assignments)
) == {1, 2, 3, 4, 5}


overall_auc = roc_auc_score(
    target,
    oof_predictions,
)

overall_average_precision = (
    average_precision_score(
        target,
        oof_predictions,
    )
)

overall_log_loss = log_loss(
    target,
    oof_predictions,
    labels=[0, 1],
)

overall_brier_score = brier_score_loss(
    target,
    oof_predictions,
)

false_positive_rate, true_positive_rate, _ = (
    roc_curve(
        target,
        oof_predictions,
    )
)

ks_statistic = np.max(
    true_positive_rate
    - false_positive_rate
)

gini_coefficient = (
    2 * overall_auc - 1
)

total_elapsed_minutes = (
    perf_counter()
    - training_start_time
) / 60


# =========================================================
# 7. Aggregate feature importance
# =========================================================
feature_importance_all_folds = pd.concat(
    feature_importance_records,
    ignore_index=True,
)

feature_importance_summary = (
    feature_importance_all_folds
    .groupby(
        "feature",
        as_index=False,
    )
    .agg(
        mean_gain=("gain", "mean"),
        total_gain=("gain", "sum"),
        gain_std=("gain", "std"),
        mean_split=("split", "mean"),
        total_split=("split", "sum"),
    )
    .sort_values(
        "total_gain",
        ascending=False,
    )
    .reset_index(drop=True)
)

feature_importance_summary[
    "gain_share_pct"
] = (
    feature_importance_summary[
        "total_gain"
    ]
    / feature_importance_summary[
        "total_gain"
    ].sum()
    * 100
)

feature_importance_summary[
    "importance_rank"
] = (
    np.arange(
        1,
        len(feature_importance_summary) + 1,
    )
)

feature_schema_path = (
    processed_data_path
    / "modeling_feature_schema_clean.csv"
)

if feature_schema_path.exists():
    feature_schema = pd.read_csv(
        feature_schema_path
    )[
        ["feature", "source"]
    ]

    feature_importance_summary = (
        feature_importance_summary.merge(
            feature_schema,
            on="feature",
            how="left",
            validate="one_to_one",
        )
    )

    assert (
        feature_importance_summary[
            "source"
        ].notna().all()
    )


# =========================================================
# 8. Save final baseline artifacts
# =========================================================
fold_metrics = pd.DataFrame(
    fold_metric_records
)

fold_metrics.to_csv(
    metric_output_path
    / "lightgbm_baseline_fold_metrics.csv",
    index=False,
)

feature_importance_summary.to_csv(
    metric_output_path
    / "lightgbm_baseline_feature_importance.csv",
    index=False,
)

oof_results = pd.DataFrame({
    "SK_ID_CURR": customer_ids,
    "TARGET": target,
    "CV_FOLD": fold_assignments,
    "LIGHTGBM_BASELINE_OOF": (
        oof_predictions
    ),
})

oof_results.to_parquet(
    metric_output_path
    / "lightgbm_baseline_oof.parquet",
    index=False,
    compression="snappy",
)


# =========================================================
# 9. Baseline summary
# =========================================================
baseline_summary = pd.DataFrame({
    "metric": [
        "Modeling rows",
        "Predictor features",
        "Categorical features",
        "Cross-validation folds",
        "Observed default rate (%)",
        "Mean predicted default (%)",
        "OOF ROC-AUC",
        "OOF Gini coefficient",
        "OOF average precision",
        "Average-precision lift",
        "OOF KS statistic",
        "OOF log loss",
        "OOF Brier score",
        "Fold ROC-AUC mean",
        "Fold ROC-AUC standard deviation",
        "Mean best iteration",
        "Total training time (minutes)",
        "Missing OOF predictions",
        "Duplicate OOF customer IDs",
        "Saved fold models",
    ],
    "value": [
        len(target),
        len(feature_columns),
        len(categorical_features),
        number_of_folds,
        target.mean() * 100,
        oof_predictions.mean() * 100,
        overall_auc,
        gini_coefficient,
        overall_average_precision,
        (
            overall_average_precision
            / target.mean()
        ),
        ks_statistic,
        overall_log_loss,
        overall_brier_score,
        fold_metrics[
            "roc_auc"
        ].mean(),
        fold_metrics[
            "roc_auc"
        ].std(ddof=1),
        fold_metrics[
            "best_iteration"
        ].mean(),
        total_elapsed_minutes,
        np.isnan(
            oof_predictions
        ).sum(),
        oof_results[
            "SK_ID_CURR"
        ].duplicated().sum(),
        len(
            list(
                model_output_path.glob(
                    "lightgbm_baseline_fold_*.txt"
                )
            )
        ),
    ],
})


# =========================================================
# 10. Display bounded results
# =========================================================
display(
    baseline_summary.round(6)
)

display(
    fold_metrics.round(6)
)

display(
    feature_importance_summary[
        [
            "importance_rank",
            "feature",
            "source",
            "mean_gain",
            "gain_share_pct",
            "mean_split",
        ]
    ]
    .head(20)
    .round(4)
)

In [ ]:
from pathlib import Path
from time import perf_counter
import gc
import json

import lightgbm as lgb
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    roc_curve,
)


# =========================================================
# 1. Paths
# =========================================================
project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

clean_model_path = (
    processed_data_path
    / "modeling_train_clean.parquet"
)

schema_path = (
    processed_data_path
    / "modeling_feature_schema_clean.csv"
)

modeling_output_path = (
    processed_data_path
    / "modeling"
)

baseline_oof_path = (
    modeling_output_path
    / "lightgbm_baseline_oof.parquet"
)

baseline_importance_path = (
    modeling_output_path
    / "lightgbm_baseline_feature_importance.csv"
)

baseline_parameter_path = (
    project_path
    / "models"
    / "lightgbm_baseline"
    / "lightgbm_baseline_parameters.json"
)

application_model_path = (
    project_path
    / "models"
    / "lightgbm_application_only"
)

application_model_path.mkdir(
    parents=True,
    exist_ok=True,
)

required_paths = [
    clean_model_path,
    schema_path,
    baseline_oof_path,
    baseline_importance_path,
    baseline_parameter_path,
]

assert all(
    path.exists()
    for path in required_paths
), {
    str(path): path.exists()
    for path in required_paths
}


# =========================================================
# 2. Identify application-only predictors
# =========================================================
feature_schema = pd.read_csv(
    schema_path
)

application_sources = [
    "Application raw",
    "Application engineered",
]

application_features = (
    feature_schema.loc[
        feature_schema["source"].isin(
            application_sources
        ),
        "feature",
    ]
    .tolist()
)

assert len(application_features) == 176

application_data = pd.read_parquet(
    clean_model_path,
    columns=[
        "SK_ID_CURR",
        "TARGET",
    ] + application_features,
)

baseline_oof = pd.read_parquet(
    baseline_oof_path
)

assert len(application_data) == len(
    baseline_oof
)

assert application_data[
    "SK_ID_CURR"
].equals(
    baseline_oof["SK_ID_CURR"]
)

assert application_data[
    "TARGET"
].equals(
    baseline_oof["TARGET"]
)


customer_ids = application_data.pop(
    "SK_ID_CURR"
)

target = (
    application_data.pop("TARGET")
    .astype("int8")
)

fold_assignments = (
    baseline_oof["CV_FOLD"]
    .astype("int8")
    .to_numpy()
)

application_categorical_features = [
    feature
    for feature in application_features
    if not pd.api.types.is_numeric_dtype(
        application_data[feature]
    )
]

for feature in application_categorical_features:
    if not isinstance(
        application_data[feature].dtype,
        pd.CategoricalDtype,
    ):
        application_data[feature] = (
            application_data[feature]
            .astype("category")
        )

assert len(application_categorical_features) == 16
assert set(np.unique(fold_assignments)) == {
    1, 2, 3, 4, 5
}


# =========================================================
# 3. Load identical baseline parameters
# =========================================================
with open(
    baseline_parameter_path,
    "r",
    encoding="utf-8",
) as parameter_file:
    model_parameters = json.load(
        parameter_file
    )

number_of_folds = 5

application_oof_predictions = np.full(
    len(target),
    np.nan,
    dtype="float32",
)

fold_metric_records = []
importance_records = []

training_start_time = perf_counter()


# =========================================================
# 4. Application-only cross-validation
# =========================================================
for fold_number in range(
    1,
    number_of_folds + 1,
):
    print(
        f"\n========== Application-only "
        f"Fold {fold_number}/5 =========="
    )

    fold_start_time = perf_counter()

    training_mask = (
        fold_assignments != fold_number
    )

    validation_mask = (
        fold_assignments == fold_number
    )

    training_features = (
        application_data.loc[
            training_mask
        ]
    )

    validation_features = (
        application_data.loc[
            validation_mask
        ]
    )

    training_target = target.loc[
        training_mask
    ]

    validation_target = target.loc[
        validation_mask
    ]

    model = lgb.LGBMClassifier(
        **model_parameters
    )

    model.fit(
        training_features,
        training_target,
        eval_X=validation_features,
        eval_y=validation_target,
        eval_metric="auc",
        categorical_feature=(
            application_categorical_features
        ),
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=150,
                first_metric_only=True,
                verbose=False,
            ),
            lgb.log_evaluation(
                period=200
            ),
        ],
    )

    validation_probabilities = (
        model.predict_proba(
            validation_features,
            num_iteration=(
                model.best_iteration_
            ),
        )[:, 1]
    )

    application_oof_predictions[
        validation_mask
    ] = validation_probabilities.astype(
        "float32"
    )

    fold_metric_records.append({
        "fold": fold_number,
        "training_rows": int(
            training_mask.sum()
        ),
        "validation_rows": int(
            validation_mask.sum()
        ),
        "validation_default_rate_pct": (
            validation_target.mean() * 100
        ),
        "best_iteration": (
            model.best_iteration_
        ),
        "roc_auc": roc_auc_score(
            validation_target,
            validation_probabilities,
        ),
        "average_precision": (
            average_precision_score(
                validation_target,
                validation_probabilities,
            )
        ),
        "log_loss": log_loss(
            validation_target,
            validation_probabilities,
            labels=[0, 1],
        ),
        "brier_score": brier_score_loss(
            validation_target,
            validation_probabilities,
        ),
        "elapsed_minutes": (
            perf_counter()
            - fold_start_time
        ) / 60,
    })

    booster = model.booster_

    importance_records.append(
        pd.DataFrame({
            "feature": (
                booster.feature_name()
            ),
            "fold": fold_number,
            "gain": (
                booster.feature_importance(
                    importance_type="gain"
                )
            ),
            "split": (
                booster.feature_importance(
                    importance_type="split"
                )
            ),
        })
    )

    booster.save_model(
        str(
            application_model_path
            / (
                "lightgbm_application_only_"
                f"fold_{fold_number}.txt"
            )
        ),
        num_iteration=model.best_iteration_,
    )

    # Fold checkpoint
    pd.DataFrame({
        "SK_ID_CURR": customer_ids,
        "TARGET": target,
        "CV_FOLD": fold_assignments,
        "APPLICATION_ONLY_OOF": (
            application_oof_predictions
        ),
    }).to_parquet(
        modeling_output_path
        / "lightgbm_application_only_oof.parquet",
        index=False,
        compression="snappy",
    )

    print(
        f"Fold {fold_number} ROC-AUC: "
        f"{fold_metric_records[-1]['roc_auc']:.6f}"
    )

    print(
        f"Best iteration: "
        f"{model.best_iteration_}"
    )

    del training_features
    del validation_features
    del training_target
    del validation_target
    del validation_probabilities
    del model
    del booster
    gc.collect()


# =========================================================
# 5. Validate and save application-only results
# =========================================================
assert not np.isnan(
    application_oof_predictions
).any()

assert np.isfinite(
    application_oof_predictions
).all()

application_fold_metrics = pd.DataFrame(
    fold_metric_records
)

application_fold_metrics.to_csv(
    modeling_output_path
    / "lightgbm_application_only_fold_metrics.csv",
    index=False,
)

application_oof_results = pd.DataFrame({
    "SK_ID_CURR": customer_ids,
    "TARGET": target,
    "CV_FOLD": fold_assignments,
    "APPLICATION_ONLY_OOF": (
        application_oof_predictions
    ),
})

application_oof_results.to_parquet(
    modeling_output_path
    / "lightgbm_application_only_oof.parquet",
    index=False,
    compression="snappy",
)


application_importance = (
    pd.concat(
        importance_records,
        ignore_index=True,
    )
    .groupby(
        "feature",
        as_index=False,
    )
    .agg(
        mean_gain=("gain", "mean"),
        total_gain=("gain", "sum"),
        mean_split=("split", "mean"),
    )
    .sort_values(
        "total_gain",
        ascending=False,
    )
    .reset_index(drop=True)
)

application_importance.to_csv(
    modeling_output_path
    / "lightgbm_application_only_feature_importance.csv",
    index=False,
)


# =========================================================
# 6. Metric helper
# =========================================================
def calculate_model_metrics(
    actual_target,
    probabilities,
):
    false_positive_rate, (
        true_positive_rate
    ), _ = roc_curve(
        actual_target,
        probabilities,
    )

    auc_value = roc_auc_score(
        actual_target,
        probabilities,
    )

    return {
        "roc_auc": auc_value,
        "gini": 2 * auc_value - 1,
        "average_precision": (
            average_precision_score(
                actual_target,
                probabilities,
            )
        ),
        "ks_statistic": np.max(
            true_positive_rate
            - false_positive_rate
        ),
        "log_loss": log_loss(
            actual_target,
            probabilities,
            labels=[0, 1],
        ),
        "brier_score": brier_score_loss(
            actual_target,
            probabilities,
        ),
        "mean_prediction_pct": (
            np.mean(probabilities) * 100
        ),
    }


application_metrics = (
    calculate_model_metrics(
        target,
        application_oof_predictions,
    )
)

full_model_probabilities = (
    baseline_oof[
        "LIGHTGBM_BASELINE_OOF"
    ].to_numpy()
)

full_metrics = calculate_model_metrics(
    target,
    full_model_probabilities,
)


# =========================================================
# 7. Fair model comparison
# =========================================================
model_comparison = pd.DataFrame([
    {
        "model": "Application only",
        "predictors": 176,
        **application_metrics,
    },
    {
        "model": "Application + relational history",
        "predictors": 639,
        **full_metrics,
    },
])

improvement_summary = pd.DataFrame({
    "metric": [
        "ROC-AUC gain",
        "Gini gain",
        "Average precision gain",
        "KS gain",
        "Log-loss reduction",
        "Brier-score reduction",
    ],
    "full_history_improvement": [
        (
            full_metrics["roc_auc"]
            - application_metrics["roc_auc"]
        ),
        (
            full_metrics["gini"]
            - application_metrics["gini"]
        ),
        (
            full_metrics["average_precision"]
            - application_metrics[
                "average_precision"
            ]
        ),
        (
            full_metrics["ks_statistic"]
            - application_metrics[
                "ks_statistic"
            ]
        ),
        (
            application_metrics["log_loss"]
            - full_metrics["log_loss"]
        ),
        (
            application_metrics["brier_score"]
            - full_metrics["brier_score"]
        ),
    ],
})


# =========================================================
# 8. Descriptive source importance
# =========================================================
baseline_importance = pd.read_csv(
    baseline_importance_path
)

source_importance_summary = (
    baseline_importance
    .groupby(
        "source",
        as_index=False,
    )
    .agg(
        feature_count=(
            "feature",
            "nunique",
        ),
        total_gain=(
            "total_gain",
            "sum",
        ),
    )
)

source_importance_summary[
    "gain_share_pct"
] = (
    source_importance_summary[
        "total_gain"
    ]
    / source_importance_summary[
        "total_gain"
    ].sum()
    * 100
)

source_importance_summary = (
    source_importance_summary
    .sort_values(
        "gain_share_pct",
        ascending=False,
    )
    .reset_index(drop=True)
)

source_importance_summary.to_csv(
    modeling_output_path
    / "lightgbm_baseline_source_importance.csv",
    index=False,
)

model_comparison.to_csv(
    modeling_output_path
    / "lightgbm_model_comparison.csv",
    index=False,
)

improvement_summary.to_csv(
    modeling_output_path
    / "lightgbm_relational_history_improvement.csv",
    index=False,
)


# =========================================================
# 9. Final checks
# =========================================================
comparison_check = pd.DataFrame({
    "metric": [
        "Application-only feature count",
        "Application categorical features",
        "Application OOF rows",
        "Missing application OOF predictions",
        "Duplicate application OOF IDs",
        "Saved application-only models",
        "Application-only training time (minutes)",
        "Full and application folds identical",
    ],
    "value": [
        len(application_features),
        len(application_categorical_features),
        len(application_oof_results),
        application_oof_results[
            "APPLICATION_ONLY_OOF"
        ].isna().sum(),
        application_oof_results[
            "SK_ID_CURR"
        ].duplicated().sum(),
        len(
            list(
                application_model_path.glob(
                    "lightgbm_application_only_"
                    "fold_*.txt"
                )
            )
        ),
        (
            perf_counter()
            - training_start_time
        ) / 60,
        np.array_equal(
            application_oof_results[
                "CV_FOLD"
            ].to_numpy(),
            baseline_oof[
                "CV_FOLD"
            ].to_numpy(),
        ),
    ],
})


display(
    comparison_check.round(4)
)

display(
    application_fold_metrics.round(6)
)

display(
    model_comparison.round(6)
)

display(
    improvement_summary.round(6)
)

display(
    source_importance_summary[
        [
            "source",
            "feature_count",
            "gain_share_pct",
        ]
    ].round(3)
)

In [17]:
%pip install optuna

In [18]:
import optuna
import pandas as pd

optuna_environment_check = pd.DataFrame({
    "metric": [
        "Optuna import successful",
        "Optuna version",
    ],
    "value": [
        True,
        optuna.__version__,
    ],
})

display(optuna_environment_check)

,metric,value
0,Optuna import successful,True
1,Optuna version,4.9.0


In [ ]:
from pathlib import Path
from time import perf_counter
import gc
import json

import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd

from optuna.trial import TrialState
from sklearn.metrics import roc_auc_score


# =========================================================
# 1. Paths and experiment settings
# =========================================================
project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

modeling_data_path = (
    processed_data_path
    / "modeling_train_clean.parquet"
)

modeling_output_path = (
    processed_data_path
    / "modeling"
)

baseline_oof_path = (
    modeling_output_path
    / "lightgbm_baseline_oof.parquet"
)

baseline_fold_metrics_path = (
    modeling_output_path
    / "lightgbm_baseline_fold_metrics.csv"
)

tuning_output_path = (
    modeling_output_path
    / "optuna_lightgbm"
)

tuning_output_path.mkdir(
    parents=True,
    exist_ok=True,
)

study_database_path = (
    tuning_output_path
    / "lightgbm_tuning.db"
)

storage_url = (
    "sqlite:///"
    + study_database_path.as_posix()
)

study_name = (
    "finguard360_lightgbm_tuning"
)

target_completed_trials = 8
tuning_folds = [1, 2]
random_seed = 42

required_paths = [
    modeling_data_path,
    baseline_oof_path,
    baseline_fold_metrics_path,
]

assert all(
    path.exists()
    for path in required_paths
)


# =========================================================
# 2. Load data and preserved fold assignments
# =========================================================
modeling_features = pd.read_parquet(
    modeling_data_path
)

customer_ids = modeling_features.pop(
    "SK_ID_CURR"
)

target = (
    modeling_features.pop("TARGET")
    .astype("int8")
)

baseline_oof = pd.read_parquet(
    baseline_oof_path
)

assert customer_ids.equals(
    baseline_oof["SK_ID_CURR"]
)

assert target.equals(
    baseline_oof["TARGET"]
)

fold_assignments = (
    baseline_oof["CV_FOLD"]
    .astype("int8")
    .to_numpy()
)

feature_columns = list(
    modeling_features.columns
)

categorical_features = [
    feature
    for feature in feature_columns
    if not pd.api.types.is_numeric_dtype(
        modeling_features[feature]
    )
]

for feature in categorical_features:
    if not isinstance(
        modeling_features[feature].dtype,
        pd.CategoricalDtype,
    ):
        modeling_features[feature] = (
            modeling_features[feature]
            .astype("category")
        )

assert len(modeling_features) == 307_511
assert len(feature_columns) == 639
assert len(categorical_features) == 16
assert set(np.unique(fold_assignments)) == {
    1, 2, 3, 4, 5
}


# =========================================================
# 3. Baseline score on identical tuning folds
# =========================================================
baseline_fold_metrics = pd.read_csv(
    baseline_fold_metrics_path
)

baseline_tuning_auc = (
    baseline_fold_metrics.loc[
        baseline_fold_metrics[
            "fold"
        ].isin(tuning_folds),
        "roc_auc",
    ].mean()
)


# =========================================================
# 4. Optuna objective
# =========================================================
def objective(trial):
    search_parameters = {
        "learning_rate": (
            trial.suggest_float(
                "learning_rate",
                0.015,
                0.060,
                log=True,
            )
        ),
        "num_leaves": (
            trial.suggest_int(
                "num_leaves",
                20,
                80,
            )
        ),
        "min_child_samples": (
            trial.suggest_int(
                "min_child_samples",
                40,
                250,
            )
        ),
        "subsample": (
            trial.suggest_float(
                "subsample",
                0.70,
                1.00,
            )
        ),
        "colsample_bytree": (
            trial.suggest_float(
                "colsample_bytree",
                0.65,
                1.00,
            )
        ),
        "reg_alpha": (
            trial.suggest_float(
                "reg_alpha",
                1e-4,
                5.0,
                log=True,
            )
        ),
        "reg_lambda": (
            trial.suggest_float(
                "reg_lambda",
                1e-3,
                20.0,
                log=True,
            )
        ),
        "min_split_gain": (
            trial.suggest_float(
                "min_split_gain",
                0.0,
                0.20,
            )
        ),
    }

    fixed_parameters = {
        "objective": "binary",
        "boosting_type": "gbdt",
        "n_estimators": 3000,
        "max_depth": -1,
        "subsample_freq": 1,
        "class_weight": None,
        "random_state": random_seed,
        "n_jobs": -1,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
    }

    fold_auc_values = []
    best_iterations = []

    for fold_number in tuning_folds:
        training_mask = (
            fold_assignments
            != fold_number
        )

        validation_mask = (
            fold_assignments
            == fold_number
        )

        training_features = (
            modeling_features.loc[
                training_mask
            ]
        )

        validation_features = (
            modeling_features.loc[
                validation_mask
            ]
        )

        training_target = target.loc[
            training_mask
        ]

        validation_target = target.loc[
            validation_mask
        ]

        model = lgb.LGBMClassifier(
            **fixed_parameters,
            **search_parameters,
        )

        model.fit(
            training_features,
            training_target,
            eval_X=validation_features,
            eval_y=validation_target,
            eval_metric="auc",
            categorical_feature=(
                categorical_features
            ),
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=120,
                    first_metric_only=True,
                    verbose=False,
                ),
                lgb.log_evaluation(
                    period=0
                ),
            ],
        )

        validation_probabilities = (
            model.predict_proba(
                validation_features,
                num_iteration=(
                    model.best_iteration_
                ),
            )[:, 1]
        )

        fold_auc = roc_auc_score(
            validation_target,
            validation_probabilities,
        )

        fold_auc_values.append(
            fold_auc
        )

        best_iterations.append(
            model.best_iteration_
        )

        trial.set_user_attr(
            f"fold_{fold_number}_auc",
            float(fold_auc),
        )

        del training_features
        del validation_features
        del training_target
        del validation_target
        del validation_probabilities
        del model
        gc.collect()

    mean_auc = float(
        np.mean(fold_auc_values)
    )

    trial.set_user_attr(
        "mean_best_iteration",
        float(
            np.mean(best_iterations)
        ),
    )

    trial.set_user_attr(
        "auc_standard_deviation",
        float(
            np.std(
                fold_auc_values,
                ddof=1,
            )
        ),
    )

    return mean_auc


# =========================================================
# 5. Persistent Optuna study
# =========================================================
sampler = optuna.samplers.TPESampler(
    seed=random_seed,
)

study = optuna.create_study(
    study_name=study_name,
    storage=storage_url,
    direction="maximize",
    sampler=sampler,
    load_if_exists=True,
)

baseline_search_parameters = {
    "learning_rate": 0.03,
    "num_leaves": 31,
    "min_child_samples": 100,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.10,
    "reg_lambda": 1.00,
    "min_split_gain": 0.0,
}

# Ensure the first trial is the existing baseline
if len(study.trials) == 0:
    study.enqueue_trial(
        baseline_search_parameters
    )

completed_before = sum(
    trial.state == TrialState.COMPLETE
    for trial in study.trials
)

remaining_trials = max(
    0,
    (
        target_completed_trials
        - completed_before
    ),
)

print(
    f"Completed trials before run: "
    f"{completed_before}"
)

print(
    f"Remaining trials: "
    f"{remaining_trials}"
)

tuning_start_time = perf_counter()

if remaining_trials > 0:
    study.optimize(
        objective,
        n_trials=remaining_trials,
        gc_after_trial=True,
        show_progress_bar=False,
    )

elapsed_minutes = (
    perf_counter()
    - tuning_start_time
) / 60


# =========================================================
# 6. Save study results
# =========================================================
completed_trials = [
    trial
    for trial in study.trials
    if trial.state == TrialState.COMPLETE
]

assert len(completed_trials) >= (
    target_completed_trials
)

trial_results = study.trials_dataframe(
    attrs=(
        "number",
        "value",
        "params",
        "user_attrs",
        "state",
    )
)

trial_results.to_csv(
    tuning_output_path
    / "optuna_trial_results.csv",
    index=False,
)

best_parameters = {
    "objective": "binary",
    "boosting_type": "gbdt",
    "n_estimators": 3000,
    "max_depth": -1,
    "subsample_freq": 1,
    "class_weight": None,
    "random_state": random_seed,
    "n_jobs": -1,
    "verbosity": -1,
    "deterministic": True,
    "force_col_wise": True,
    **study.best_params,
}

with open(
    tuning_output_path
    / "optuna_best_parameters.json",
    "w",
    encoding="utf-8",
) as best_parameter_file:
    json.dump(
        best_parameters,
        best_parameter_file,
        indent=2,
    )


# =========================================================
# 7. Parameter importance
# =========================================================
try:
    parameter_importance = (
        optuna.importance
        .get_param_importances(study)
    )

    parameter_importance_table = (
        pd.DataFrame({
            "parameter": list(
                parameter_importance.keys()
            ),
            "importance": list(
                parameter_importance.values()
            ),
        })
    )

except Exception as importance_error:
    parameter_importance_table = (
        pd.DataFrame({
            "parameter": [
                "Unavailable"
            ],
            "importance": [np.nan],
        })
    )

    print(
        "Parameter importance could "
        "not be calculated:",
        importance_error,
    )

parameter_importance_table.to_csv(
    tuning_output_path
    / "optuna_parameter_importance.csv",
    index=False,
)


# =========================================================
# 8. Tuning summary
# =========================================================
best_trial = study.best_trial

best_fold_1_auc = (
    best_trial.user_attrs.get(
        "fold_1_auc",
        np.nan,
    )
)

best_fold_2_auc = (
    best_trial.user_attrs.get(
        "fold_2_auc",
        np.nan,
    )
)

tuning_summary = pd.DataFrame({
    "metric": [
        "Target completed trials",
        "Completed trials",
        "Tuning folds",
        "Predictor features",
        "Baseline mean tuning ROC-AUC",
        "Best tuning ROC-AUC",
        "Tuning ROC-AUC gain",
        "Best trial number",
        "Best trial Fold 1 ROC-AUC",
        "Best trial Fold 2 ROC-AUC",
        "Best mean iteration",
        "Trials run in this execution",
        "Execution time (minutes)",
        "Persistent database exists",
    ],
    "value": [
        target_completed_trials,
        len(completed_trials),
        len(tuning_folds),
        len(feature_columns),
        baseline_tuning_auc,
        study.best_value,
        (
            study.best_value
            - baseline_tuning_auc
        ),
        best_trial.number,
        best_fold_1_auc,
        best_fold_2_auc,
        best_trial.user_attrs.get(
            "mean_best_iteration",
            np.nan,
        ),
        remaining_trials,
        elapsed_minutes,
        study_database_path.exists(),
    ],
})


# Bounded trial display
preferred_trial_columns = [
    "number",
    "value",
    "user_attrs_fold_1_auc",
    "user_attrs_fold_2_auc",
    "user_attrs_mean_best_iteration",
    "params_learning_rate",
    "params_num_leaves",
    "params_min_child_samples",
    "params_subsample",
    "params_colsample_bytree",
    "params_reg_alpha",
    "params_reg_lambda",
    "params_min_split_gain",
    "state",
]

available_trial_columns = [
    column
    for column in preferred_trial_columns
    if column in trial_results.columns
]

top_trials = (
    trial_results.loc[
        trial_results["state"]
        == "COMPLETE",
        available_trial_columns,
    ]
    .sort_values(
        "value",
        ascending=False,
    )
    .head(8)
)

best_parameter_table = pd.DataFrame({
    "parameter": list(
        study.best_params.keys()
    ),
    "best_value": list(
        study.best_params.values()
    ),
})


display(
    tuning_summary.round(6)
)

display(
    top_trials.round(6)
)

display(
    best_parameter_table
)

display(
    parameter_importance_table.round(4)
)


# Free the large table after persistence
del modeling_features
del baseline_oof
gc.collect()

In [ ]:
from pathlib import Path
from time import perf_counter
import gc
import json

import lightgbm as lgb
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    roc_curve,
)


# =========================================================
# 1. Paths
# =========================================================
project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

modeling_data_path = (
    processed_data_path
    / "modeling_train_clean.parquet"
)

feature_schema_path = (
    processed_data_path
    / "modeling_feature_schema_clean.csv"
)

modeling_output_path = (
    processed_data_path
    / "modeling"
)

baseline_oof_path = (
    modeling_output_path
    / "lightgbm_baseline_oof.parquet"
)

baseline_fold_metrics_path = (
    modeling_output_path
    / "lightgbm_baseline_fold_metrics.csv"
)

best_parameter_path = (
    modeling_output_path
    / "optuna_lightgbm"
    / "optuna_best_parameters.json"
)

tuned_model_path = (
    project_path
    / "models"
    / "lightgbm_tuned"
)

tuned_model_path.mkdir(
    parents=True,
    exist_ok=True,
)

required_paths = [
    modeling_data_path,
    feature_schema_path,
    baseline_oof_path,
    baseline_fold_metrics_path,
    best_parameter_path,
]

assert all(
    path.exists()
    for path in required_paths
), {
    str(path): path.exists()
    for path in required_paths
}


# =========================================================
# 2. Load model data and preserved folds
# =========================================================
modeling_features = pd.read_parquet(
    modeling_data_path
)

customer_ids = modeling_features.pop(
    "SK_ID_CURR"
)

target = (
    modeling_features.pop("TARGET")
    .astype("int8")
)

baseline_oof = pd.read_parquet(
    baseline_oof_path
)

assert np.array_equal(
    customer_ids.to_numpy(),
    baseline_oof[
        "SK_ID_CURR"
    ].to_numpy(),
)

assert np.array_equal(
    target.to_numpy(),
    baseline_oof[
        "TARGET"
    ].to_numpy(),
)

fold_assignments = (
    baseline_oof["CV_FOLD"]
    .astype("int8")
    .to_numpy()
)

feature_columns = list(
    modeling_features.columns
)

categorical_features = [
    feature
    for feature in feature_columns
    if not pd.api.types.is_numeric_dtype(
        modeling_features[feature]
    )
]

for feature in categorical_features:
    if not isinstance(
        modeling_features[feature].dtype,
        pd.CategoricalDtype,
    ):
        modeling_features[feature] = (
            modeling_features[feature]
            .astype("category")
        )

assert len(modeling_features) == 307_511
assert len(feature_columns) == 639
assert len(categorical_features) == 16
assert set(np.unique(fold_assignments)) == {
    1, 2, 3, 4, 5
}


# =========================================================
# 3. Load Optuna best parameters
# =========================================================
with open(
    best_parameter_path,
    "r",
    encoding="utf-8",
) as parameter_file:
    tuned_parameters = json.load(
        parameter_file
    )

tuned_parameter_check = pd.DataFrame({
    "parameter": list(
        tuned_parameters.keys()
    ),
    "value": list(
        tuned_parameters.values()
    ),
})


# =========================================================
# 4. Five-fold tuned model
# =========================================================
number_of_folds = 5

tuned_oof_predictions = np.full(
    len(target),
    np.nan,
    dtype="float32",
)

fold_metric_records = []
importance_records = []

training_start_time = perf_counter()

for fold_number in range(
    1,
    number_of_folds + 1,
):
    print(
        f"\n========== Tuned Fold "
        f"{fold_number}/5 =========="
    )

    fold_start_time = perf_counter()

    training_mask = (
        fold_assignments != fold_number
    )

    validation_mask = (
        fold_assignments == fold_number
    )

    training_features = (
        modeling_features.loc[
            training_mask
        ]
    )

    validation_features = (
        modeling_features.loc[
            validation_mask
        ]
    )

    training_target = target.loc[
        training_mask
    ]

    validation_target = target.loc[
        validation_mask
    ]

    model = lgb.LGBMClassifier(
        **tuned_parameters
    )

    model.fit(
        training_features,
        training_target,
        eval_X=validation_features,
        eval_y=validation_target,
        eval_metric="auc",
        categorical_feature=(
            categorical_features
        ),
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=150,
                first_metric_only=True,
                verbose=False,
            ),
            lgb.log_evaluation(
                period=200
            ),
        ],
    )

    validation_probabilities = (
        model.predict_proba(
            validation_features,
            num_iteration=(
                model.best_iteration_
            ),
        )[:, 1]
    )

    tuned_oof_predictions[
        validation_mask
    ] = validation_probabilities.astype(
        "float32"
    )

    fold_auc = roc_auc_score(
        validation_target,
        validation_probabilities,
    )

    fold_metric_records.append({
        "fold": fold_number,
        "training_rows": int(
            training_mask.sum()
        ),
        "validation_rows": int(
            validation_mask.sum()
        ),
        "validation_default_rate_pct": (
            validation_target.mean() * 100
        ),
        "best_iteration": (
            model.best_iteration_
        ),
        "roc_auc": fold_auc,
        "average_precision": (
            average_precision_score(
                validation_target,
                validation_probabilities,
            )
        ),
        "log_loss": log_loss(
            validation_target,
            validation_probabilities,
            labels=[0, 1],
        ),
        "brier_score": brier_score_loss(
            validation_target,
            validation_probabilities,
        ),
        "elapsed_minutes": (
            perf_counter()
            - fold_start_time
        ) / 60,
    })

    booster = model.booster_

    importance_records.append(
        pd.DataFrame({
            "feature": (
                booster.feature_name()
            ),
            "fold": fold_number,
            "gain": (
                booster.feature_importance(
                    importance_type="gain"
                )
            ),
            "split": (
                booster.feature_importance(
                    importance_type="split"
                )
            ),
        })
    )

    booster.save_model(
        str(
            tuned_model_path
            / (
                "lightgbm_tuned_"
                f"fold_{fold_number}.txt"
            )
        ),
        num_iteration=model.best_iteration_,
    )

    # Recoverable checkpoint
    pd.DataFrame({
        "SK_ID_CURR": customer_ids,
        "TARGET": target,
        "CV_FOLD": fold_assignments,
        "LIGHTGBM_TUNED_OOF": (
            tuned_oof_predictions
        ),
    }).to_parquet(
        modeling_output_path
        / "lightgbm_tuned_oof.parquet",
        index=False,
        compression="snappy",
    )

    print(
        f"Fold {fold_number} ROC-AUC: "
        f"{fold_auc:.6f}"
    )

    print(
        f"Fold {fold_number} "
        f"best iteration: "
        f"{model.best_iteration_}"
    )

    del training_features
    del validation_features
    del training_target
    del validation_target
    del validation_probabilities
    del model
    del booster
    gc.collect()


# =========================================================
# 5. OOF validation
# =========================================================
assert not np.isnan(
    tuned_oof_predictions
).any()

assert np.isfinite(
    tuned_oof_predictions
).all()

assert (
    (tuned_oof_predictions >= 0)
    & (tuned_oof_predictions <= 1)
).all()

tuned_fold_metrics = pd.DataFrame(
    fold_metric_records
)

tuned_fold_metrics.to_csv(
    modeling_output_path
    / "lightgbm_tuned_fold_metrics.csv",
    index=False,
)

tuned_oof_results = pd.DataFrame({
    "SK_ID_CURR": customer_ids,
    "TARGET": target,
    "CV_FOLD": fold_assignments,
    "LIGHTGBM_TUNED_OOF": (
        tuned_oof_predictions
    ),
})

tuned_oof_results.to_parquet(
    modeling_output_path
    / "lightgbm_tuned_oof.parquet",
    index=False,
    compression="snappy",
)


# =========================================================
# 6. Aggregate tuned feature importance
# =========================================================
tuned_importance = (
    pd.concat(
        importance_records,
        ignore_index=True,
    )
    .groupby(
        "feature",
        as_index=False,
    )
    .agg(
        mean_gain=("gain", "mean"),
        total_gain=("gain", "sum"),
        gain_std=("gain", "std"),
        mean_split=("split", "mean"),
        total_split=("split", "sum"),
    )
    .sort_values(
        "total_gain",
        ascending=False,
    )
    .reset_index(drop=True)
)

tuned_importance[
    "gain_share_pct"
] = (
    tuned_importance["total_gain"]
    / tuned_importance[
        "total_gain"
    ].sum()
    * 100
)

tuned_importance[
    "importance_rank"
] = np.arange(
    1,
    len(tuned_importance) + 1,
)

feature_schema = pd.read_csv(
    feature_schema_path
)[
    ["feature", "source"]
]

tuned_importance = (
    tuned_importance.merge(
        feature_schema,
        on="feature",
        how="left",
        validate="one_to_one",
    )
)

assert tuned_importance[
    "source"
].notna().all()

tuned_importance.to_csv(
    modeling_output_path
    / "lightgbm_tuned_feature_importance.csv",
    index=False,
)


# =========================================================
# 7. Common metric function
# =========================================================
def calculate_metrics(
    actual_target,
    probabilities,
):
    false_positive_rate, (
        true_positive_rate
    ), _ = roc_curve(
        actual_target,
        probabilities,
    )

    auc_value = roc_auc_score(
        actual_target,
        probabilities,
    )

    return {
        "roc_auc": auc_value,
        "gini": 2 * auc_value - 1,
        "average_precision": (
            average_precision_score(
                actual_target,
                probabilities,
            )
        ),
        "ks_statistic": float(
            np.max(
                true_positive_rate
                - false_positive_rate
            )
        ),
        "log_loss": log_loss(
            actual_target,
            probabilities,
            labels=[0, 1],
        ),
        "brier_score": brier_score_loss(
            actual_target,
            probabilities,
        ),
        "mean_prediction_pct": (
            np.mean(probabilities) * 100
        ),
    }


baseline_predictions = (
    baseline_oof[
        "LIGHTGBM_BASELINE_OOF"
    ].to_numpy()
)

baseline_metrics = calculate_metrics(
    target,
    baseline_predictions,
)

tuned_metrics = calculate_metrics(
    target,
    tuned_oof_predictions,
)


# =========================================================
# 8. Overall model comparison
# =========================================================
model_comparison = pd.DataFrame([
    {
        "model": "Baseline LightGBM",
        **baseline_metrics,
    },
    {
        "model": "Optuna-tuned LightGBM",
        **tuned_metrics,
    },
])

metric_improvement = pd.DataFrame({
    "metric": [
        "ROC-AUC gain",
        "Gini gain",
        "Average precision gain",
        "KS gain",
        "Log-loss reduction",
        "Brier-score reduction",
    ],
    "improvement": [
        (
            tuned_metrics["roc_auc"]
            - baseline_metrics["roc_auc"]
        ),
        (
            tuned_metrics["gini"]
            - baseline_metrics["gini"]
        ),
        (
            tuned_metrics[
                "average_precision"
            ]
            - baseline_metrics[
                "average_precision"
            ]
        ),
        (
            tuned_metrics["ks_statistic"]
            - baseline_metrics[
                "ks_statistic"
            ]
        ),
        (
            baseline_metrics["log_loss"]
            - tuned_metrics["log_loss"]
        ),
        (
            baseline_metrics["brier_score"]
            - tuned_metrics["brier_score"]
        ),
    ],
})


# =========================================================
# 9. Fold-by-fold comparison
# =========================================================
baseline_fold_metrics = pd.read_csv(
    baseline_fold_metrics_path
)

fold_comparison = (
    baseline_fold_metrics[
        [
            "fold",
            "roc_auc",
            "average_precision",
            "log_loss",
            "brier_score",
            "best_iteration",
        ]
    ]
    .rename(columns={
        "roc_auc": "baseline_auc",
        "average_precision": (
            "baseline_average_precision"
        ),
        "log_loss": "baseline_log_loss",
        "brier_score": (
            "baseline_brier_score"
        ),
        "best_iteration": (
            "baseline_best_iteration"
        ),
    })
    .merge(
        tuned_fold_metrics[
            [
                "fold",
                "roc_auc",
                "average_precision",
                "log_loss",
                "brier_score",
                "best_iteration",
            ]
        ].rename(columns={
            "roc_auc": "tuned_auc",
            "average_precision": (
                "tuned_average_precision"
            ),
            "log_loss": "tuned_log_loss",
            "brier_score": (
                "tuned_brier_score"
            ),
            "best_iteration": (
                "tuned_best_iteration"
            ),
        }),
        on="fold",
        how="inner",
        validate="one_to_one",
    )
)

fold_comparison[
    "auc_gain"
] = (
    fold_comparison["tuned_auc"]
    - fold_comparison["baseline_auc"]
)

fold_comparison[
    "average_precision_gain"
] = (
    fold_comparison[
        "tuned_average_precision"
    ]
    - fold_comparison[
        "baseline_average_precision"
    ]
)

fold_comparison[
    "log_loss_reduction"
] = (
    fold_comparison[
        "baseline_log_loss"
    ]
    - fold_comparison[
        "tuned_log_loss"
    ]
)

tuning_fold_auc_gain = (
    fold_comparison.loc[
        fold_comparison["fold"].isin(
            [1, 2]
        ),
        "auc_gain",
    ].mean()
)

untouched_fold_auc_gain = (
    fold_comparison.loc[
        fold_comparison["fold"].isin(
            [3, 4, 5]
        ),
        "auc_gain",
    ].mean()
)

positive_auc_folds = int(
    fold_comparison[
        "auc_gain"
    ].gt(0).sum()
)

overall_auc_gain = (
    tuned_metrics["roc_auc"]
    - baseline_metrics["roc_auc"]
)


# =========================================================
# 10. Champion selection
# =========================================================
tuned_model_selected = bool(
    overall_auc_gain > 0
    and untouched_fold_auc_gain > 0
    and positive_auc_folds >= 3
)

champion_name = (
    "Optuna-tuned LightGBM"
    if tuned_model_selected
    else "Baseline LightGBM"
)

champion_manifest = {
    "champion_model": champion_name,
    "selection_primary_metric": (
        "OOF ROC-AUC"
    ),
    "baseline_oof_auc": float(
        baseline_metrics["roc_auc"]
    ),
    "tuned_oof_auc": float(
        tuned_metrics["roc_auc"]
    ),
    "overall_auc_gain": float(
        overall_auc_gain
    ),
    "untouched_fold_auc_gain": float(
        untouched_fold_auc_gain
    ),
    "positive_auc_folds": (
        positive_auc_folds
    ),
    "selected_parameters_file": str(
        best_parameter_path
        if tuned_model_selected
        else (
            project_path
            / "models"
            / "lightgbm_baseline"
            / (
                "lightgbm_baseline_"
                "parameters.json"
            )
        )
    ),
}

with open(
    modeling_output_path
    / "champion_model_manifest.json",
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(
        champion_manifest,
        manifest_file,
        indent=2,
    )


# =========================================================
# 11. Final validation summary
# =========================================================
validation_summary = pd.DataFrame({
    "metric": [
        "Modeling rows",
        "Predictor features",
        "Cross-validation folds",
        "Missing tuned OOF predictions",
        "Duplicate tuned OOF IDs",
        "Saved tuned fold models",
        "Baseline OOF ROC-AUC",
        "Tuned OOF ROC-AUC",
        "Overall ROC-AUC gain",
        "Tuning-fold mean AUC gain",
        "Untouched-fold mean AUC gain",
        "Folds with positive AUC gain",
        "Tuned average precision",
        "Tuned KS statistic",
        "Tuned log loss",
        "Tuned Brier score",
        "Observed default rate (%)",
        "Mean tuned prediction (%)",
        "Mean tuned best iteration",
        "Training time (minutes)",
        "Selected champion",
    ],
    "value": [
        len(target),
        len(feature_columns),
        number_of_folds,
        np.isnan(
            tuned_oof_predictions
        ).sum(),
        tuned_oof_results[
            "SK_ID_CURR"
        ].duplicated().sum(),
        len(
            list(
                tuned_model_path.glob(
                    "lightgbm_tuned_fold_*.txt"
                )
            )
        ),
        baseline_metrics["roc_auc"],
        tuned_metrics["roc_auc"],
        overall_auc_gain,
        tuning_fold_auc_gain,
        untouched_fold_auc_gain,
        positive_auc_folds,
        tuned_metrics[
            "average_precision"
        ],
        tuned_metrics["ks_statistic"],
        tuned_metrics["log_loss"],
        tuned_metrics["brier_score"],
        target.mean() * 100,
        tuned_metrics[
            "mean_prediction_pct"
        ],
        tuned_fold_metrics[
            "best_iteration"
        ].mean(),
        (
            perf_counter()
            - training_start_time
        ) / 60,
        champion_name,
    ],
})


# =========================================================
# 12. Save comparisons
# =========================================================
model_comparison.to_csv(
    modeling_output_path
    / "baseline_vs_tuned_model.csv",
    index=False,
)

metric_improvement.to_csv(
    modeling_output_path
    / "tuned_metric_improvement.csv",
    index=False,
)

fold_comparison.to_csv(
    modeling_output_path
    / "baseline_vs_tuned_folds.csv",
    index=False,
)

validation_summary.to_csv(
    modeling_output_path
    / "tuned_model_validation_summary.csv",
    index=False,
)


# =========================================================
# 13. Display bounded outputs
# =========================================================
display(
    validation_summary.round(6)
)

display(
    fold_comparison[
        [
            "fold",
            "baseline_auc",
            "tuned_auc",
            "auc_gain",
            "average_precision_gain",
            "log_loss_reduction",
            "baseline_best_iteration",
            "tuned_best_iteration",
        ]
    ].round(6)
)

display(
    model_comparison.round(6)
)

display(
    metric_improvement.round(6)
)

display(
    tuned_importance[
        [
            "importance_rank",
            "feature",
            "source",
            "gain_share_pct",
            "mean_split",
        ]
    ]
    .head(20)
    .round(4)
)


del modeling_features
del baseline_oof
gc.collect()

In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd

from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    roc_curve,
)


# =========================================================
# 1. Paths
# =========================================================
project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

modeling_output_path = (
    processed_data_path
    / "modeling"
)

baseline_oof_path = (
    modeling_output_path
    / "lightgbm_baseline_oof.parquet"
)

tuned_oof_path = (
    modeling_output_path
    / "lightgbm_tuned_oof.parquet"
)

calibration_model_path = (
    project_path
    / "models"
    / "calibration"
)

calibration_model_path.mkdir(
    parents=True,
    exist_ok=True,
)

assert baseline_oof_path.exists()
assert tuned_oof_path.exists()


# =========================================================
# 2. Load OOF predictions
# =========================================================
baseline_oof = pd.read_parquet(
    baseline_oof_path
)

tuned_oof = pd.read_parquet(
    tuned_oof_path
)

assert np.array_equal(
    baseline_oof["SK_ID_CURR"],
    tuned_oof["SK_ID_CURR"],
)

assert np.array_equal(
    baseline_oof["TARGET"],
    tuned_oof["TARGET"],
)

assert np.array_equal(
    baseline_oof["CV_FOLD"],
    tuned_oof["CV_FOLD"],
)

customer_ids = tuned_oof[
    "SK_ID_CURR"
].to_numpy()

target = (
    tuned_oof["TARGET"]
    .astype("int8")
    .to_numpy()
)

fold_assignments = (
    tuned_oof["CV_FOLD"]
    .astype("int8")
    .to_numpy()
)

baseline_probabilities = (
    baseline_oof[
        "LIGHTGBM_BASELINE_OOF"
    ]
    .astype("float64")
    .to_numpy()
)

tuned_probabilities = (
    tuned_oof[
        "LIGHTGBM_TUNED_OOF"
    ]
    .astype("float64")
    .to_numpy()
)

assert not np.isnan(
    tuned_probabilities
).any()


# =========================================================
# 3. Helper functions
# =========================================================
def probability_to_logit(
    probabilities,
):
    clipped = np.clip(
        probabilities,
        1e-6,
        1 - 1e-6,
    )

    return np.log(
        clipped / (1 - clipped)
    ).reshape(-1, 1)


def equal_frequency_calibration_error(
    actual_target,
    probabilities,
    number_of_bins=10,
):
    sorted_indices = np.argsort(
        probabilities
    )

    bin_indices = np.array_split(
        sorted_indices,
        number_of_bins,
    )

    weighted_gap = 0.0
    maximum_gap = 0.0

    for indices in bin_indices:
        observed_rate = np.mean(
            actual_target[indices]
        )

        predicted_rate = np.mean(
            probabilities[indices]
        )

        absolute_gap = abs(
            observed_rate
            - predicted_rate
        )

        weighted_gap += (
            len(indices)
            / len(actual_target)
            * absolute_gap
        )

        maximum_gap = max(
            maximum_gap,
            absolute_gap,
        )

    return weighted_gap, maximum_gap


def calculate_probability_metrics(
    model_name,
    actual_target,
    probabilities,
):
    false_positive_rate, (
        true_positive_rate
    ), _ = roc_curve(
        actual_target,
        probabilities,
    )

    auc_value = roc_auc_score(
        actual_target,
        probabilities,
    )

    calibration_error, (
        maximum_bin_gap
    ) = equal_frequency_calibration_error(
        actual_target,
        probabilities,
    )

    # Descriptive calibration intercept/slope
    calibration_diagnostic = (
        LogisticRegression(
            C=1e6,
            solver="lbfgs",
            max_iter=1000,
        )
    )

    calibration_diagnostic.fit(
        probability_to_logit(
            probabilities
        ),
        actual_target,
    )

    return {
        "model": model_name,
        "roc_auc": auc_value,
        "gini": 2 * auc_value - 1,
        "average_precision": (
            average_precision_score(
                actual_target,
                probabilities,
            )
        ),
        "ks_statistic": np.max(
            true_positive_rate
            - false_positive_rate
        ),
        "log_loss": log_loss(
            actual_target,
            probabilities,
            labels=[0, 1],
        ),
        "brier_score": brier_score_loss(
            actual_target,
            probabilities,
        ),
        "mean_prediction_pct": (
            np.mean(probabilities) * 100
        ),
        "calibration_gap_pp": (
            (
                np.mean(probabilities)
                - np.mean(actual_target)
            )
            * 100
        ),
        "equal_frequency_ece": (
            calibration_error
        ),
        "maximum_decile_gap": (
            maximum_bin_gap
        ),
        "calibration_intercept": (
            calibration_diagnostic
            .intercept_[0]
        ),
        "calibration_slope": (
            calibration_diagnostic
            .coef_[0, 0]
        ),
    }


# =========================================================
# 4. Cross-fitted calibration
# =========================================================
platt_oof_probabilities = np.full(
    len(target),
    np.nan,
    dtype="float64",
)

isotonic_oof_probabilities = np.full(
    len(target),
    np.nan,
    dtype="float64",
)

calibration_fold_records = []

for fold_number in range(1, 6):
    calibration_training_mask = (
        fold_assignments != fold_number
    )

    calibration_validation_mask = (
        fold_assignments == fold_number
    )

    calibration_training_target = (
        target[
            calibration_training_mask
        ]
    )

    calibration_validation_target = (
        target[
            calibration_validation_mask
        ]
    )

    training_probabilities = (
        tuned_probabilities[
            calibration_training_mask
        ]
    )

    validation_probabilities = (
        tuned_probabilities[
            calibration_validation_mask
        ]
    )

    # Platt scaling
    platt_calibrator = (
        LogisticRegression(
            C=1e6,
            solver="lbfgs",
            max_iter=1000,
        )
    )

    platt_calibrator.fit(
        probability_to_logit(
            training_probabilities
        ),
        calibration_training_target,
    )

    platt_fold_probabilities = (
        platt_calibrator.predict_proba(
            probability_to_logit(
                validation_probabilities
            )
        )[:, 1]
    )

    platt_oof_probabilities[
        calibration_validation_mask
    ] = platt_fold_probabilities

    # Isotonic regression
    isotonic_calibrator = (
        IsotonicRegression(
            y_min=0,
            y_max=1,
            increasing=True,
            out_of_bounds="clip",
        )
    )

    isotonic_calibrator.fit(
        training_probabilities,
        calibration_training_target,
    )

    isotonic_fold_probabilities = (
        isotonic_calibrator.predict(
            validation_probabilities
        )
    )

    isotonic_oof_probabilities[
        calibration_validation_mask
    ] = isotonic_fold_probabilities

    calibration_fold_records.append({
        "fold": fold_number,
        "training_rows": int(
            calibration_training_mask.sum()
        ),
        "validation_rows": int(
            calibration_validation_mask.sum()
        ),
        "validation_default_rate_pct": (
            calibration_validation_target
            .mean()
            * 100
        ),
        "raw_mean_prediction_pct": (
            validation_probabilities.mean()
            * 100
        ),
        "platt_mean_prediction_pct": (
            platt_fold_probabilities.mean()
            * 100
        ),
        "isotonic_mean_prediction_pct": (
            isotonic_fold_probabilities
            .mean()
            * 100
        ),
        "platt_intercept": (
            platt_calibrator.intercept_[0]
        ),
        "platt_slope": (
            platt_calibrator.coef_[0, 0]
        ),
        "isotonic_threshold_count": (
            len(
                isotonic_calibrator.X_thresholds_
            )
        ),
    })


assert not np.isnan(
    platt_oof_probabilities
).any()

assert not np.isnan(
    isotonic_oof_probabilities
).any()

assert (
    (platt_oof_probabilities >= 0)
    & (platt_oof_probabilities <= 1)
).all()

assert (
    (isotonic_oof_probabilities >= 0)
    & (isotonic_oof_probabilities <= 1)
).all()


# =========================================================
# 5. Compare raw and calibrated probabilities
# =========================================================
calibration_comparison = pd.DataFrame([
    calculate_probability_metrics(
        "Baseline raw",
        target,
        baseline_probabilities,
    ),
    calculate_probability_metrics(
        "Tuned raw",
        target,
        tuned_probabilities,
    ),
    calculate_probability_metrics(
        "Tuned + Platt",
        target,
        platt_oof_probabilities,
    ),
    calculate_probability_metrics(
        "Tuned + Isotonic",
        target,
        isotonic_oof_probabilities,
    ),
])

raw_tuned_auc = float(
    calibration_comparison.loc[
        calibration_comparison[
            "model"
        ].eq("Tuned raw"),
        "roc_auc",
    ].iloc[0]
)

# Calibration candidates must preserve ranking
eligible_candidates = (
    calibration_comparison.loc[
        calibration_comparison[
            "model"
        ].isin([
            "Tuned raw",
            "Tuned + Platt",
            "Tuned + Isotonic",
        ])
        & (
            calibration_comparison[
                "roc_auc"
            ]
            >= raw_tuned_auc - 0.0005
        )
    ]
    .sort_values(
        [
            "log_loss",
            "brier_score",
        ],
        ascending=True,
    )
)

selected_probability_model = (
    eligible_candidates.iloc[0][
        "model"
    ]
)

probability_map = {
    "Tuned raw": tuned_probabilities,
    "Tuned + Platt": (
        platt_oof_probabilities
    ),
    "Tuned + Isotonic": (
        isotonic_oof_probabilities
    ),
}

selected_probabilities = (
    probability_map[
        selected_probability_model
    ]
)


# =========================================================
# 6. Fit deployable calibrators on all OOF data
# =========================================================
final_platt_calibrator = LogisticRegression(
    C=1e6,
    solver="lbfgs",
    max_iter=1000,
)

final_platt_calibrator.fit(
    probability_to_logit(
        tuned_probabilities
    ),
    target,
)

final_isotonic_calibrator = (
    IsotonicRegression(
        y_min=0,
        y_max=1,
        increasing=True,
        out_of_bounds="clip",
    )
)

final_isotonic_calibrator.fit(
    tuned_probabilities,
    target,
)

joblib.dump(
    final_platt_calibrator,
    calibration_model_path
    / "platt_calibrator.joblib",
)

joblib.dump(
    final_isotonic_calibrator,
    calibration_model_path
    / "isotonic_calibrator.joblib",
)


# =========================================================
# 7. Save calibrated OOF predictions
# =========================================================
calibrated_oof = pd.DataFrame({
    "SK_ID_CURR": customer_ids,
    "TARGET": target,
    "CV_FOLD": fold_assignments,
    "TUNED_RAW_PROBABILITY": (
        tuned_probabilities
    ),
    "PLATT_CROSSFIT_PROBABILITY": (
        platt_oof_probabilities
    ),
    "ISOTONIC_CROSSFIT_PROBABILITY": (
        isotonic_oof_probabilities
    ),
    "SELECTED_PROBABILITY": (
        selected_probabilities
    ),
})

calibrated_oof.to_parquet(
    modeling_output_path
    / "lightgbm_calibrated_oof.parquet",
    index=False,
    compression="snappy",
)


# =========================================================
# 8. Risk-decile analysis
# Decile 1 = highest predicted risk
# =========================================================
risk_rank = pd.Series(
    selected_probabilities
).rank(
    method="first",
    ascending=False,
)

risk_deciles = (
    pd.qcut(
        risk_rank,
        q=10,
        labels=False,
    )
    .astype("int8")
    + 1
)

risk_decile_records = pd.DataFrame({
    "risk_decile": risk_deciles,
    "TARGET": target,
    "probability": (
        selected_probabilities
    ),
})

risk_decile_summary = (
    risk_decile_records
    .groupby(
        "risk_decile",
        as_index=False,
        observed=True,
    )
    .agg(
        applicants=("TARGET", "size"),
        defaults=("TARGET", "sum"),
        minimum_score=(
            "probability",
            "min",
        ),
        maximum_score=(
            "probability",
            "max",
        ),
        mean_predicted_rate=(
            "probability",
            "mean",
        ),
    )
    .sort_values("risk_decile")
    .reset_index(drop=True)
)

risk_decile_summary[
    "observed_default_rate"
] = (
    risk_decile_summary["defaults"]
    / risk_decile_summary["applicants"]
)

overall_default_rate = target.mean()

risk_decile_summary[
    "default_capture_pct"
] = (
    risk_decile_summary["defaults"]
    / target.sum()
    * 100
)

risk_decile_summary[
    "cumulative_default_capture_pct"
] = (
    risk_decile_summary[
        "default_capture_pct"
    ].cumsum()
)

risk_decile_summary[
    "population_share_pct"
] = (
    risk_decile_summary["applicants"]
    / len(target)
    * 100
)

risk_decile_summary[
    "cumulative_population_share_pct"
] = (
    risk_decile_summary[
        "population_share_pct"
    ].cumsum()
)

risk_decile_summary[
    "risk_lift"
] = (
    risk_decile_summary[
        "observed_default_rate"
    ]
    / overall_default_rate
)

risk_decile_summary[
    "cumulative_default_rate"
] = (
    risk_decile_summary[
        "defaults"
    ].cumsum()
    / risk_decile_summary[
        "applicants"
    ].cumsum()
)

risk_decile_summary[
    "cumulative_lift"
] = (
    risk_decile_summary[
        "cumulative_default_rate"
    ]
    / overall_default_rate
)

risk_decile_summary[
    "calibration_gap_pp"
] = (
    (
        risk_decile_summary[
            "mean_predicted_rate"
        ]
        - risk_decile_summary[
            "observed_default_rate"
        ]
    )
    * 100
)


# Convert selected rates to percentage display
percentage_columns = [
    "minimum_score",
    "maximum_score",
    "mean_predicted_rate",
    "observed_default_rate",
    "cumulative_default_rate",
]

risk_decile_display = (
    risk_decile_summary.copy()
)

for column in percentage_columns:
    risk_decile_display[
        column
    ] = (
        risk_decile_display[column]
        * 100
    )


# =========================================================
# 9. Save audit artifacts
# =========================================================
calibration_fold_details = pd.DataFrame(
    calibration_fold_records
)

calibration_comparison.to_csv(
    modeling_output_path
    / "probability_calibration_comparison.csv",
    index=False,
)

calibration_fold_details.to_csv(
    modeling_output_path
    / "probability_calibration_folds.csv",
    index=False,
)

risk_decile_summary.to_csv(
    modeling_output_path
    / "champion_risk_deciles.csv",
    index=False,
)

selected_metrics = (
    calibration_comparison.loc[
        calibration_comparison[
            "model"
        ].eq(
            selected_probability_model
        )
    ].iloc[0]
)

calibration_manifest = {
    "ranking_champion": (
        "Optuna-tuned LightGBM"
    ),
    "probability_champion": (
        selected_probability_model
    ),
    "selection_method": (
        "Cross-fitted calibration"
    ),
    "raw_tuned_auc": float(
        raw_tuned_auc
    ),
    "selected_auc": float(
        selected_metrics["roc_auc"]
    ),
    "selected_log_loss": float(
        selected_metrics["log_loss"]
    ),
    "selected_brier_score": float(
        selected_metrics["brier_score"]
    ),
    "selected_calibration_gap_pp": float(
        selected_metrics[
            "calibration_gap_pp"
        ]
    ),
}

with open(
    modeling_output_path
    / "calibration_manifest.json",
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(
        calibration_manifest,
        manifest_file,
        indent=2,
    )


calibration_validation = pd.DataFrame({
    "metric": [
        "OOF rows",
        "Unique customer IDs",
        "Duplicate customer IDs",
        "Missing selected probabilities",
        "Observed default rate (%)",
        "Raw tuned prediction mean (%)",
        "Selected prediction mean (%)",
        "Raw tuned log loss",
        "Selected log loss",
        "Raw tuned Brier score",
        "Selected Brier score",
        "Raw tuned calibration gap (pp)",
        "Selected calibration gap (pp)",
        "Raw tuned ECE",
        "Selected ECE",
        "Ranking champion",
        "Probability champion",
        "Platt model saved",
        "Isotonic model saved",
    ],
    "value": [
        len(calibrated_oof),
        calibrated_oof[
            "SK_ID_CURR"
        ].nunique(),
        calibrated_oof[
            "SK_ID_CURR"
        ].duplicated().sum(),
        calibrated_oof[
            "SELECTED_PROBABILITY"
        ].isna().sum(),
        overall_default_rate * 100,
        tuned_probabilities.mean() * 100,
        selected_probabilities.mean() * 100,
        calculate_probability_metrics(
            "raw",
            target,
            tuned_probabilities,
        )["log_loss"],
        selected_metrics["log_loss"],
        calculate_probability_metrics(
            "raw",
            target,
            tuned_probabilities,
        )["brier_score"],
        selected_metrics["brier_score"],
        (
            tuned_probabilities.mean()
            - overall_default_rate
        ) * 100,
        selected_metrics[
            "calibration_gap_pp"
        ],
        calculate_probability_metrics(
            "raw",
            target,
            tuned_probabilities,
        )["equal_frequency_ece"],
        selected_metrics[
            "equal_frequency_ece"
        ],
        "Optuna-tuned LightGBM",
        selected_probability_model,
        (
            calibration_model_path
            / "platt_calibrator.joblib"
        ).exists(),
        (
            calibration_model_path
            / "isotonic_calibrator.joblib"
        ).exists(),
    ],
})


display(
    calibration_validation.round(6)
)

display(
    calibration_comparison.round(6)
)

display(
    calibration_fold_details.round(6)
)

display(
    risk_decile_display[
        [
            "risk_decile",
            "applicants",
            "defaults",
            "mean_predicted_rate",
            "observed_default_rate",
            "default_capture_pct",
            "cumulative_default_capture_pct",
            "risk_lift",
            "cumulative_lift",
            "calibration_gap_pp",
        ]
    ].round(3)
)

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


# =========================================================
# 1. Paths
# =========================================================
project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

modeling_output_path = (
    processed_data_path
    / "modeling"
)

calibrated_oof_path = (
    modeling_output_path
    / "lightgbm_calibrated_oof.parquet"
)

calibration_manifest_path = (
    modeling_output_path
    / "calibration_manifest.json"
)

scorecard_path = (
    modeling_output_path
    / "credit_risk_scorecard_train.parquet"
)

assert calibrated_oof_path.exists()
assert calibration_manifest_path.exists()


# =========================================================
# 2. Load calibrated predictions
# =========================================================
calibrated_oof = pd.read_parquet(
    calibrated_oof_path
)

with open(
    calibration_manifest_path,
    "r",
    encoding="utf-8",
) as manifest_file:
    calibration_manifest = json.load(
        manifest_file
    )

assert (
    calibration_manifest[
        "probability_champion"
    ]
    == "Tuned + Platt"
)

customer_ids = calibrated_oof[
    "SK_ID_CURR"
].to_numpy()

target = (
    calibrated_oof["TARGET"]
    .astype("int8")
    .to_numpy()
)

raw_probabilities = (
    calibrated_oof[
        "TUNED_RAW_PROBABILITY"
    ].to_numpy()
)

selected_probabilities = (
    calibrated_oof[
        "SELECTED_PROBABILITY"
    ].to_numpy()
)

number_of_customers = len(target)
number_of_defaults = int(target.sum())
number_of_good_customers = int(
    number_of_customers
    - number_of_defaults
)

overall_default_rate = target.mean()

assert number_of_customers == 307_511
assert not np.isnan(
    selected_probabilities
).any()


# =========================================================
# 3. Exact risk ranking
# Rank 1 = highest predicted risk
# =========================================================
risk_rank = (
    pd.Series(
        selected_probabilities
    )
    .rank(
        method="first",
        ascending=False,
    )
    .astype("int32")
    .to_numpy()
)

top_population_share_pct = (
    risk_rank
    / number_of_customers
    * 100
)


# =========================================================
# 4. Threshold trade-off table
# =========================================================
target_review_shares = [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50,
]

threshold_records = []

for target_review_share in (
    target_review_shares
):
    review_count = int(
        np.ceil(
            number_of_customers
            * target_review_share
        )
    )

    high_risk_flag = (
        risk_rank <= review_count
    )

    remaining_flag = (
        ~high_risk_flag
    )

    true_positive = int(
        (
            high_risk_flag
            & (target == 1)
        ).sum()
    )

    false_positive = int(
        (
            high_risk_flag
            & (target == 0)
        ).sum()
    )

    false_negative = int(
        (
            remaining_flag
            & (target == 1)
        ).sum()
    )

    true_negative = int(
        (
            remaining_flag
            & (target == 0)
        ).sum()
    )

    reviewed_count = int(
        high_risk_flag.sum()
    )

    remaining_count = int(
        remaining_flag.sum()
    )

    score_threshold = float(
        selected_probabilities[
            high_risk_flag
        ].min()
    )

    precision = (
        true_positive
        / reviewed_count
    )

    recall = (
        true_positive
        / number_of_defaults
    )

    specificity = (
        true_negative
        / number_of_good_customers
    )

    remaining_default_rate = (
        false_negative
        / remaining_count
    )

    threshold_records.append({
        "target_review_share_pct": (
            target_review_share * 100
        ),
        "score_threshold_pct": (
            score_threshold * 100
        ),
        "reviewed_customers": (
            reviewed_count
        ),
        "remaining_customers": (
            remaining_count
        ),
        "defaults_captured": (
            true_positive
        ),
        "good_customers_flagged": (
            false_positive
        ),
        "defaults_not_flagged": (
            false_negative
        ),
        "default_capture_pct": (
            recall * 100
        ),
        "reviewed_default_rate_pct": (
            precision * 100
        ),
        "reviewed_risk_lift": (
            precision
            / overall_default_rate
        ),
        "good_customer_flag_rate_pct": (
            false_positive
            / number_of_good_customers
            * 100
        ),
        "remaining_population_pct": (
            remaining_count
            / number_of_customers
            * 100
        ),
        "remaining_default_rate_pct": (
            remaining_default_rate
            * 100
        ),
        "remaining_default_rate_reduction_pct": (
            (
                1
                - remaining_default_rate
                / overall_default_rate
            )
            * 100
        ),
        "good_flagged_per_default_captured": (
            false_positive
            / true_positive
        ),
        "specificity_pct": (
            specificity * 100
        ),
    })

threshold_tradeoff = pd.DataFrame(
    threshold_records
)


# =========================================================
# 5. Operational risk bands
# These are analysis bands, not automatic
# credit decisions.
# =========================================================
risk_band_values = np.select(
    [
        top_population_share_pct <= 5,
        top_population_share_pct <= 10,
        top_population_share_pct <= 20,
        top_population_share_pct <= 40,
    ],
    [
        "Very High: Top 5%",
        "High: 5–10%",
        "Elevated: 10–20%",
        "Moderate: 20–40%",
    ],
    default="Lower: Bottom 60%",
)

risk_band_order = [
    "Very High: Top 5%",
    "High: 5–10%",
    "Elevated: 10–20%",
    "Moderate: 20–40%",
    "Lower: Bottom 60%",
]

risk_band = pd.Categorical(
    risk_band_values,
    categories=risk_band_order,
    ordered=True,
)

scorecard = pd.DataFrame({
    "SK_ID_CURR": customer_ids,
    "TARGET": target,
    "CV_FOLD": calibrated_oof[
        "CV_FOLD"
    ].to_numpy(),
    "RAW_RISK_PROBABILITY": (
        raw_probabilities
    ),
    "CALIBRATED_RISK_PROBABILITY": (
        selected_probabilities
    ),
    "RISK_RANK": risk_rank,
    "TOP_POPULATION_SHARE_PCT": (
        top_population_share_pct
    ),
    "RISK_BAND": risk_band,
})

assert scorecard[
    "SK_ID_CURR"
].nunique() == len(scorecard)

assert not scorecard[
    "CALIBRATED_RISK_PROBABILITY"
].isna().any()

scorecard.to_parquet(
    scorecard_path,
    index=False,
    compression="snappy",
)


# =========================================================
# 6. Risk-band performance
# =========================================================
risk_band_summary = (
    scorecard
    .groupby(
        "RISK_BAND",
        observed=True,
        sort=False,
    )
    .agg(
        customers=(
            "SK_ID_CURR",
            "size",
        ),
        defaults=(
            "TARGET",
            "sum",
        ),
        minimum_probability=(
            "CALIBRATED_RISK_PROBABILITY",
            "min",
        ),
        maximum_probability=(
            "CALIBRATED_RISK_PROBABILITY",
            "max",
        ),
        mean_predicted_rate=(
            "CALIBRATED_RISK_PROBABILITY",
            "mean",
        ),
    )
    .reset_index()
)

risk_band_summary[
    "population_share_pct"
] = (
    risk_band_summary["customers"]
    / number_of_customers
    * 100
)

risk_band_summary[
    "observed_default_rate_pct"
] = (
    risk_band_summary["defaults"]
    / risk_band_summary["customers"]
    * 100
)

risk_band_summary[
    "mean_predicted_rate_pct"
] = (
    risk_band_summary[
        "mean_predicted_rate"
    ]
    * 100
)

risk_band_summary[
    "default_capture_pct"
] = (
    risk_band_summary["defaults"]
    / number_of_defaults
    * 100
)

risk_band_summary[
    "cumulative_default_capture_pct"
] = (
    risk_band_summary[
        "default_capture_pct"
    ].cumsum()
)

risk_band_summary[
    "risk_lift"
] = (
    risk_band_summary[
        "observed_default_rate_pct"
    ]
    / (
        overall_default_rate
        * 100
    )
)

risk_band_summary[
    "calibration_gap_pp"
] = (
    risk_band_summary[
        "mean_predicted_rate_pct"
    ]
    - risk_band_summary[
        "observed_default_rate_pct"
    ]
)

risk_band_summary[
    "minimum_probability_pct"
] = (
    risk_band_summary[
        "minimum_probability"
    ]
    * 100
)

risk_band_summary[
    "maximum_probability_pct"
] = (
    risk_band_summary[
        "maximum_probability"
    ]
    * 100
)


# =========================================================
# 7. Save outputs
# =========================================================
threshold_tradeoff.to_csv(
    modeling_output_path
    / "credit_risk_threshold_tradeoff.csv",
    index=False,
)

risk_band_summary.to_csv(
    modeling_output_path
    / "credit_risk_band_summary.csv",
    index=False,
)


def threshold_value(
    review_share_pct,
    column,
):
    return threshold_tradeoff.loc[
        threshold_tradeoff[
            "target_review_share_pct"
        ].eq(review_share_pct),
        column,
    ].iloc[0]


risk_policy_check = pd.DataFrame({
    "metric": [
        "Scorecard rows",
        "Unique customer IDs",
        "Duplicate customer IDs",
        "Missing calibrated probabilities",
        "Observed default rate (%)",
        "Mean calibrated probability (%)",
        "Risk bands",
        "Top 10% default capture (%)",
        "Top 20% default capture (%)",
        "Top 30% default capture (%)",
        "Top 10% score threshold (%)",
        "Top 20% score threshold (%)",
        "Saved scorecard rows",
        "Scorecard file size (MB)",
    ],
    "value": [
        len(scorecard),
        scorecard[
            "SK_ID_CURR"
        ].nunique(),
        scorecard[
            "SK_ID_CURR"
        ].duplicated().sum(),
        scorecard[
            "CALIBRATED_RISK_PROBABILITY"
        ].isna().sum(),
        overall_default_rate * 100,
        selected_probabilities.mean() * 100,
        risk_band_summary.shape[0],
        threshold_value(
            10,
            "default_capture_pct",
        ),
        threshold_value(
            20,
            "default_capture_pct",
        ),
        threshold_value(
            30,
            "default_capture_pct",
        ),
        threshold_value(
            10,
            "score_threshold_pct",
        ),
        threshold_value(
            20,
            "score_threshold_pct",
        ),
        len(
            pd.read_parquet(
                scorecard_path,
                columns=["SK_ID_CURR"],
            )
        ),
        (
            scorecard_path.stat().st_size
            / 1024**2
        ),
    ],
})


display(
    risk_policy_check.round(4)
)

display(
    threshold_tradeoff.round(3)
)

display(
    risk_band_summary[
        [
            "RISK_BAND",
            "customers",
            "defaults",
            "population_share_pct",
            "minimum_probability_pct",
            "maximum_probability_pct",
            "mean_predicted_rate_pct",
            "observed_default_rate_pct",
            "default_capture_pct",
            "cumulative_default_capture_pct",
            "risk_lift",
            "calibration_gap_pp",
        ]
    ].round(3)
)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


# =========================================================
# 1. Paths and data
# =========================================================
project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

modeling_output_path = (
    project_path
    / "data"
    / "processed"
    / "modeling"
)

scorecard_path = (
    modeling_output_path
    / "credit_risk_scorecard_train.parquet"
)

assert scorecard_path.exists()

scorecard = pd.read_parquet(
    scorecard_path
)

risk_band_order = [
    "Very High: Top 5%",
    "High: 5–10%",
    "Elevated: 10–20%",
    "Moderate: 20–40%",
    "Lower: Bottom 60%",
]

band_order_map = {
    band: order
    for order, band in enumerate(
        risk_band_order,
        start=1,
    )
}

scorecard["RISK_BAND"] = (
    scorecard["RISK_BAND"]
    .astype("string")
)

unexpected_bands = (
    set(scorecard["RISK_BAND"].dropna())
    - set(risk_band_order)
)

assert not unexpected_bands, (
    f"Unexpected risk bands: "
    f"{unexpected_bands}"
)


# =========================================================
# 2. Rebuild summary in correct risk order
# =========================================================
customer_count = len(scorecard)
default_count = int(
    scorecard["TARGET"].sum()
)

overall_default_rate = (
    scorecard["TARGET"].mean()
)

risk_band_summary_corrected = (
    scorecard
    .groupby(
        "RISK_BAND",
        observed=True,
    )
    .agg(
        customers=(
            "SK_ID_CURR",
            "size",
        ),
        defaults=(
            "TARGET",
            "sum",
        ),
        minimum_probability=(
            "CALIBRATED_RISK_PROBABILITY",
            "min",
        ),
        maximum_probability=(
            "CALIBRATED_RISK_PROBABILITY",
            "max",
        ),
        mean_predicted_rate=(
            "CALIBRATED_RISK_PROBABILITY",
            "mean",
        ),
    )
    .reset_index()
)

risk_band_summary_corrected[
    "band_order"
] = (
    risk_band_summary_corrected[
        "RISK_BAND"
    ].map(band_order_map)
)

risk_band_summary_corrected = (
    risk_band_summary_corrected
    .sort_values("band_order")
    .reset_index(drop=True)
)

risk_band_summary_corrected[
    "population_share_pct"
] = (
    risk_band_summary_corrected[
        "customers"
    ]
    / customer_count
    * 100
)

risk_band_summary_corrected[
    "observed_default_rate_pct"
] = (
    risk_band_summary_corrected[
        "defaults"
    ]
    / risk_band_summary_corrected[
        "customers"
    ]
    * 100
)

risk_band_summary_corrected[
    "mean_predicted_rate_pct"
] = (
    risk_band_summary_corrected[
        "mean_predicted_rate"
    ]
    * 100
)

risk_band_summary_corrected[
    "default_capture_pct"
] = (
    risk_band_summary_corrected[
        "defaults"
    ]
    / default_count
    * 100
)

# Calculated only after correct sorting
risk_band_summary_corrected[
    "cumulative_default_capture_pct"
] = (
    risk_band_summary_corrected[
        "default_capture_pct"
    ].cumsum()
)

risk_band_summary_corrected[
    "cumulative_population_share_pct"
] = (
    risk_band_summary_corrected[
        "population_share_pct"
    ].cumsum()
)

risk_band_summary_corrected[
    "risk_lift"
] = (
    risk_band_summary_corrected[
        "observed_default_rate_pct"
    ]
    / (
        overall_default_rate * 100
    )
)

risk_band_summary_corrected[
    "calibration_gap_pp"
] = (
    risk_band_summary_corrected[
        "mean_predicted_rate_pct"
    ]
    - risk_band_summary_corrected[
        "observed_default_rate_pct"
    ]
)

risk_band_summary_corrected[
    "minimum_probability_pct"
] = (
    risk_band_summary_corrected[
        "minimum_probability"
    ]
    * 100
)

risk_band_summary_corrected[
    "maximum_probability_pct"
] = (
    risk_band_summary_corrected[
        "maximum_probability"
    ]
    * 100
)


# =========================================================
# 3. Reconciliation checks
# =========================================================
risk_band_correction_check = pd.DataFrame({
    "metric": [
        "Scorecard rows",
        "Reconciled band customers",
        "Customer difference",
        "Source defaults",
        "Reconciled band defaults",
        "Default difference",
        "Risk bands",
        "Final cumulative population (%)",
        "Final cumulative default capture (%)",
        "Top 10% cumulative capture (%)",
        "Top 20% cumulative capture (%)",
        "Top 40% cumulative capture (%)",
    ],
    "value": [
        customer_count,
        risk_band_summary_corrected[
            "customers"
        ].sum(),
        (
            risk_band_summary_corrected[
                "customers"
            ].sum()
            - customer_count
        ),
        default_count,
        risk_band_summary_corrected[
            "defaults"
        ].sum(),
        (
            risk_band_summary_corrected[
                "defaults"
            ].sum()
            - default_count
        ),
        len(
            risk_band_summary_corrected
        ),
        risk_band_summary_corrected[
            "cumulative_population_share_pct"
        ].iloc[-1],
        risk_band_summary_corrected[
            "cumulative_default_capture_pct"
        ].iloc[-1],
        risk_band_summary_corrected[
            "cumulative_default_capture_pct"
        ].iloc[1],
        risk_band_summary_corrected[
            "cumulative_default_capture_pct"
        ].iloc[2],
        risk_band_summary_corrected[
            "cumulative_default_capture_pct"
        ].iloc[3],
    ],
})

assert (
    risk_band_correction_check.loc[
        risk_band_correction_check[
            "metric"
        ].isin([
            "Customer difference",
            "Default difference",
        ]),
        "value",
    ].eq(0).all()
)

assert np.isclose(
    risk_band_summary_corrected[
        "cumulative_default_capture_pct"
    ].iloc[-1],
    100,
)

risk_band_summary_corrected.drop(
    columns="band_order"
).to_csv(
    modeling_output_path
    / "credit_risk_band_summary.csv",
    index=False,
)


# =========================================================
# 4. Display corrected results
# =========================================================
display(
    risk_band_correction_check.round(4)
)

display(
    risk_band_summary_corrected[
        [
            "RISK_BAND",
            "customers",
            "defaults",
            "population_share_pct",
            "minimum_probability_pct",
            "maximum_probability_pct",
            "mean_predicted_rate_pct",
            "observed_default_rate_pct",
            "default_capture_pct",
            "cumulative_default_capture_pct",
            "risk_lift",
            "calibration_gap_pp",
        ]
    ].round(3)
)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)


# =========================================================
# 1. Paths
# =========================================================
project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

modeling_data_path = (
    processed_data_path
    / "modeling_train_clean.parquet"
)

modeling_output_path = (
    processed_data_path
    / "modeling"
)

scorecard_path = (
    modeling_output_path
    / "credit_risk_scorecard_train.parquet"
)

assert modeling_data_path.exists()
assert scorecard_path.exists()


# =========================================================
# 2. Load required columns only
# =========================================================
application_segments = pd.read_parquet(
    modeling_data_path,
    columns=[
        "SK_ID_CURR",
        "CODE_GENDER",
        "APP_AGE_YEARS",
        "AMT_INCOME_TOTAL",
        "NAME_EDUCATION_TYPE",
    ],
)

scorecard = pd.read_parquet(
    scorecard_path,
    columns=[
        "SK_ID_CURR",
        "TARGET",
        "CALIBRATED_RISK_PROBABILITY",
        "RISK_RANK",
    ],
)

analysis_data = scorecard.merge(
    application_segments,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one",
)

assert len(analysis_data) == 307_511
assert (
    analysis_data["SK_ID_CURR"]
    .duplicated()
    .sum()
    == 0
)


# =========================================================
# 3. Exact global top-20% flag
# =========================================================
top_20_count = int(
    np.ceil(
        len(analysis_data) * 0.20
    )
)

analysis_data[
    "HIGH_RISK_TOP20_FLAG"
] = (
    analysis_data["RISK_RANK"]
    <= top_20_count
).astype("int8")

top_20_threshold = (
    analysis_data.loc[
        analysis_data[
            "HIGH_RISK_TOP20_FLAG"
        ].eq(1),
        "CALIBRATED_RISK_PROBABILITY",
    ].min()
)


# =========================================================
# 4. Build audit segments
# =========================================================
analysis_data["GENDER_GROUP"] = (
    analysis_data["CODE_GENDER"]
    .astype("string")
    .fillna("Missing")
)

analysis_data["AGE_BAND"] = (
    pd.cut(
        analysis_data[
            "APP_AGE_YEARS"
        ],
        bins=[
            20,
            30,
            40,
            50,
            60,
            70,
        ],
        labels=[
            "20–29",
            "30–39",
            "40–49",
            "50–59",
            "60–69",
        ],
        right=False,
        include_lowest=True,
    )
    .astype("string")
    .fillna("Missing/Outside range")
)

income_rank = (
    analysis_data[
        "AMT_INCOME_TOTAL"
    ].rank(
        method="first"
    )
)

analysis_data[
    "INCOME_QUINTILE"
] = (
    pd.qcut(
        income_rank,
        q=5,
        labels=[
            "Q1: Lowest",
            "Q2",
            "Q3",
            "Q4",
            "Q5: Highest",
        ],
    )
    .astype("string")
    .fillna("Missing")
)

analysis_data[
    "EDUCATION_GROUP"
] = (
    analysis_data[
        "NAME_EDUCATION_TYPE"
    ]
    .astype("string")
    .fillna("Missing")
)

segment_dimensions = {
    "Gender": "GENDER_GROUP",
    "Age band": "AGE_BAND",
    "Income quintile": (
        "INCOME_QUINTILE"
    ),
    "Education": "EDUCATION_GROUP",
}


# =========================================================
# 5. Calculate subgroup performance
# =========================================================
subgroup_records = []

for dimension_name, (
    segment_column
) in segment_dimensions.items():

    for segment_name, group in (
        analysis_data.groupby(
            segment_column,
            observed=True,
            dropna=False,
        )
    ):
        group_target = (
            group["TARGET"]
            .astype("int8")
            .to_numpy()
        )

        group_probability = (
            group[
                "CALIBRATED_RISK_PROBABILITY"
            ].to_numpy()
        )

        group_flag = (
            group[
                "HIGH_RISK_TOP20_FLAG"
            ].astype(bool).to_numpy()
        )

        group_defaults = int(
            group_target.sum()
        )

        group_good_customers = int(
            len(group_target)
            - group_defaults
        )

        captured_defaults = int(
            (
                group_flag
                & (group_target == 1)
            ).sum()
        )

        flagged_good_customers = int(
            (
                group_flag
                & (group_target == 0)
            ).sum()
        )

        flagged_customers = int(
            group_flag.sum()
        )

        has_both_classes = (
            group_defaults > 0
            and group_good_customers > 0
        )

        if has_both_classes:
            group_auc = roc_auc_score(
                group_target,
                group_probability,
            )

            group_average_precision = (
                average_precision_score(
                    group_target,
                    group_probability,
                )
            )
        else:
            group_auc = np.nan
            group_average_precision = (
                np.nan
            )

        subgroup_records.append({
            "dimension": dimension_name,
            "segment": str(
                segment_name
            ),
            "customers": len(group),
            "defaults": group_defaults,
            "good_customers": (
                group_good_customers
            ),
            "observed_default_rate_pct": (
                group_target.mean() * 100
            ),
            "mean_predicted_rate_pct": (
                group_probability.mean()
                * 100
            ),
            "calibration_gap_pp": (
                (
                    group_probability.mean()
                    - group_target.mean()
                )
                * 100
            ),
            "roc_auc": group_auc,
            "average_precision": (
                group_average_precision
            ),
            "brier_score": (
                brier_score_loss(
                    group_target,
                    group_probability,
                )
            ),
            "top20_flagged_customers": (
                flagged_customers
            ),
            "top20_flag_rate_pct": (
                flagged_customers
                / len(group)
                * 100
            ),
            "top20_default_recall_pct": (
                captured_defaults
                / group_defaults
                * 100
                if group_defaults > 0
                else np.nan
            ),
            "top20_good_flag_rate_pct": (
                flagged_good_customers
                / group_good_customers
                * 100
                if group_good_customers > 0
                else np.nan
            ),
            "top20_flagged_default_rate_pct": (
                captured_defaults
                / flagged_customers
                * 100
                if flagged_customers > 0
                else np.nan
            ),
            "eligible_for_comparison": (
                len(group) >= 1000
                and group_defaults >= 50
                and group_good_customers
                >= 50
            ),
        })

subgroup_audit = pd.DataFrame(
    subgroup_records
)


# =========================================================
# 6. Dimension-level disparity summary
# Descriptive audit only—not a causal or
# legal discrimination conclusion.
# =========================================================
disparity_records = []

for dimension_name in (
    segment_dimensions.keys()
):
    eligible_groups = (
        subgroup_audit.loc[
            subgroup_audit[
                "dimension"
            ].eq(dimension_name)
            & subgroup_audit[
                "eligible_for_comparison"
            ]
        ]
    )

    auc_values = (
        eligible_groups[
            "roc_auc"
        ].dropna()
    )

    disparity_records.append({
        "dimension": dimension_name,
        "total_segments": int(
            subgroup_audit[
                "dimension"
            ].eq(
                dimension_name
            ).sum()
        ),
        "eligible_segments": len(
            eligible_groups
        ),
        "customer_coverage_pct": (
            eligible_groups[
                "customers"
            ].sum()
            / len(analysis_data)
            * 100
        ),
        "default_rate_range_pp": (
            eligible_groups[
                "observed_default_rate_pct"
            ].max()
            - eligible_groups[
                "observed_default_rate_pct"
            ].min()
        ),
        "top20_flag_rate_range_pp": (
            eligible_groups[
                "top20_flag_rate_pct"
            ].max()
            - eligible_groups[
                "top20_flag_rate_pct"
            ].min()
        ),
        "top20_recall_range_pp": (
            eligible_groups[
                "top20_default_recall_pct"
            ].max()
            - eligible_groups[
                "top20_default_recall_pct"
            ].min()
        ),
        "top20_good_flag_range_pp": (
            eligible_groups[
                "top20_good_flag_rate_pct"
            ].max()
            - eligible_groups[
                "top20_good_flag_rate_pct"
            ].min()
        ),
        "maximum_absolute_calibration_gap_pp": (
            eligible_groups[
                "calibration_gap_pp"
            ].abs().max()
        ),
        "minimum_group_auc": (
            auc_values.min()
            if len(auc_values) > 0
            else np.nan
        ),
        "maximum_group_auc": (
            auc_values.max()
            if len(auc_values) > 0
            else np.nan
        ),
        "auc_range": (
            auc_values.max()
            - auc_values.min()
            if len(auc_values) > 0
            else np.nan
        ),
    })

subgroup_disparity_summary = (
    pd.DataFrame(
        disparity_records
    )
)


# =========================================================
# 7. Reconciliation checks
# =========================================================
dimension_reconciliation = (
    subgroup_audit
    .groupby("dimension")
    .agg(
        reconciled_customers=(
            "customers",
            "sum",
        ),
        reconciled_defaults=(
            "defaults",
            "sum",
        ),
    )
    .reset_index()
)

dimension_reconciliation[
    "customer_difference"
] = (
    dimension_reconciliation[
        "reconciled_customers"
    ]
    - len(analysis_data)
)

dimension_reconciliation[
    "default_difference"
] = (
    dimension_reconciliation[
        "reconciled_defaults"
    ]
    - analysis_data[
        "TARGET"
    ].sum()
)

assert dimension_reconciliation[
    "customer_difference"
].eq(0).all()

assert dimension_reconciliation[
    "default_difference"
].eq(0).all()


subgroup_audit_check = pd.DataFrame({
    "metric": [
        "Audit rows",
        "Unique customer IDs",
        "Duplicate customer IDs",
        "Audit dimensions",
        "Subgroup result rows",
        "Reconciliation errors",
        "Global top-20 count",
        "Actual top-20 share (%)",
        "Top-20 score threshold (%)",
        "Top-20 default capture (%)",
        "Eligible comparison groups",
    ],
    "value": [
        len(analysis_data),
        analysis_data[
            "SK_ID_CURR"
        ].nunique(),
        analysis_data[
            "SK_ID_CURR"
        ].duplicated().sum(),
        len(segment_dimensions),
        len(subgroup_audit),
        (
            dimension_reconciliation[
                [
                    "customer_difference",
                    "default_difference",
                ]
            ].ne(0).sum().sum()
        ),
        int(
            analysis_data[
                "HIGH_RISK_TOP20_FLAG"
            ].sum()
        ),
        (
            analysis_data[
                "HIGH_RISK_TOP20_FLAG"
            ].mean()
            * 100
        ),
        top_20_threshold * 100,
        (
            analysis_data.loc[
                analysis_data[
                    "HIGH_RISK_TOP20_FLAG"
                ].eq(1),
                "TARGET",
            ].sum()
            / analysis_data[
                "TARGET"
            ].sum()
            * 100
        ),
        int(
            subgroup_audit[
                "eligible_for_comparison"
            ].sum()
        ),
    ],
})


# =========================================================
# 8. Save audit evidence
# =========================================================
subgroup_audit.to_csv(
    modeling_output_path
    / "credit_risk_subgroup_audit.csv",
    index=False,
)

subgroup_disparity_summary.to_csv(
    modeling_output_path
    / "credit_risk_subgroup_disparity.csv",
    index=False,
)

dimension_reconciliation.to_csv(
    modeling_output_path
    / "credit_risk_subgroup_reconciliation.csv",
    index=False,
)


# =========================================================
# 9. Display bounded outputs
# =========================================================
display(
    subgroup_audit_check.round(4)
)

display(
    subgroup_disparity_summary.round(4)
)

display(
    subgroup_audit[
        [
            "dimension",
            "segment",
            "customers",
            "defaults",
            "observed_default_rate_pct",
            "mean_predicted_rate_pct",
            "calibration_gap_pp",
            "roc_auc",
            "top20_flag_rate_pct",
            "top20_default_recall_pct",
            "top20_good_flag_rate_pct",
            "eligible_for_comparison",
        ]
    ].round(4)
)

In [ ]:
# =========================================================
# Gender-blind LightGBM challenger
# =========================================================

import gc
import time
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
    roc_curve,
)


# =========================================================
# 1. Paths
# =========================================================

project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

modeling_output_path = (
    processed_data_path
    / "modeling"
)

challenger_model_path = (
    project_path
    / "models"
    / "lightgbm_gender_blind"
)

modeling_output_path.mkdir(
    parents=True,
    exist_ok=True,
)

challenger_model_path.mkdir(
    parents=True,
    exist_ok=True,
)

clean_modeling_path = (
    processed_data_path
    / "modeling_train_clean.parquet"
)

tuned_oof_path = (
    modeling_output_path
    / "lightgbm_tuned_oof.parquet"
)

calibrated_oof_path = (
    modeling_output_path
    / "lightgbm_calibrated_oof.parquet"
)

assert clean_modeling_path.exists()
assert tuned_oof_path.exists()
assert calibrated_oof_path.exists()


# =========================================================
# 2. Helpers for safely locating existing columns
# =========================================================

def find_column(
    frame,
    candidates,
    description,
):
    for candidate in candidates:
        if candidate in frame.columns:
            return candidate

    raise KeyError(
        f"{description} column not found. "
        f"Available columns: {list(frame.columns)}"
    )


def find_probability_column(
    frame,
    candidates,
    description,
):
    for candidate in candidates:
        if candidate in frame.columns:
            return candidate

    possible_columns = []

    for column in frame.columns:
        upper_name = column.upper()

        if (
            "PROB" in upper_name
            or "PRED" in upper_name
        ):
            if pd.api.types.is_numeric_dtype(
                frame[column]
            ):
                nonmissing = frame[column].dropna()

                if (
                    len(nonmissing) > 0
                    and nonmissing.between(0, 1).all()
                ):
                    possible_columns.append(column)

    if len(possible_columns) == 1:
        return possible_columns[0]

    raise KeyError(
        f"{description} probability column could not "
        f"be uniquely identified. Candidates found: "
        f"{possible_columns}. All columns: "
        f"{list(frame.columns)}"
    )


def safe_logit(probability):
    clipped = np.clip(
        np.asarray(probability, dtype="float64"),
        1e-6,
        1 - 1e-6,
    )

    return np.log(
        clipped / (1 - clipped)
    ).reshape(-1, 1)


def ks_statistic(y_true, probability):
    false_positive_rate, true_positive_rate, _ = (
        roc_curve(y_true, probability)
    )

    return float(
        np.max(
            true_positive_rate
            - false_positive_rate
        )
    )


def exact_top_share_flag(
    probability,
    share=0.20,
):
    probability = pd.Series(
        probability
    ).reset_index(drop=True)

    top_count = int(
        np.ceil(len(probability) * share)
    )

    flag = pd.Series(
        np.zeros(len(probability), dtype="int8")
    )

    selected_positions = (
        probability
        .sort_values(
            ascending=False,
            kind="mergesort",
        )
        .index[:top_count]
    )

    flag.iloc[selected_positions] = 1

    return flag.to_numpy()


# =========================================================
# 3. Load modeling table
# =========================================================

modeling_table = pd.read_parquet(
    clean_modeling_path
)

assert {
    "SK_ID_CURR",
    "TARGET",
    "CODE_GENDER",
}.issubset(modeling_table.columns)

customer_ids = (
    modeling_table["SK_ID_CURR"]
    .astype("int64")
    .copy()
)

target = (
    modeling_table["TARGET"]
    .astype("int8")
    .copy()
)

gender = (
    modeling_table["CODE_GENDER"]
    .astype("string")
    .fillna("Missing")
    .copy()
)

gender_blind_features = (
    modeling_table.drop(
        columns=[
            "SK_ID_CURR",
            "TARGET",
            "CODE_GENDER",
        ]
    )
)

categorical_features = [
    column
    for column in gender_blind_features.columns
    if (
        str(
            gender_blind_features[column].dtype
        ) == "category"
        or pd.api.types.is_object_dtype(
            gender_blind_features[column]
        )
        or pd.api.types.is_string_dtype(
            gender_blind_features[column]
        )
    )
]

for column in categorical_features:
    if (
        str(
            gender_blind_features[column].dtype
        )
        != "category"
    ):
        gender_blind_features[column] = (
            gender_blind_features[column]
            .astype("category")
        )

assert "CODE_GENDER" not in (
    gender_blind_features.columns
)

assert (
    gender_blind_features.shape[1]
    == 638
), gender_blind_features.shape

# =========================================================
# 4. Load existing tuned OOF and fold assignments
# =========================================================

existing_tuned_oof = pd.read_parquet(
    tuned_oof_path
)

tuned_id_column = find_column(
    existing_tuned_oof,
    ["SK_ID_CURR"],
    "Customer ID",
)

fold_column = find_column(
    existing_tuned_oof,
    [
        "CV_FOLD",
        "FOLD",
        "fold",
    ],
    "Cross-validation fold",
)

raw_probability_column = (
    find_probability_column(
        existing_tuned_oof,
        [
            # Actual column in your file
            "LIGHTGBM_TUNED_OOF",
            "TUNED_OOF_PROBABILITY",
            "OOF_PROBABILITY",
            "TUNED_PROBABILITY",
            "PREDICTION",
            "DEFAULT_PROBABILITY",
        ],
        "Tuned raw OOF",
    )
)

current_raw_oof = (
    existing_tuned_oof[
        [
            tuned_id_column,
            fold_column,
            raw_probability_column,
        ]
    ]
    .rename(
        columns={
            tuned_id_column: "SK_ID_CURR",
            fold_column: "FOLD",
            raw_probability_column:
                "CURRENT_RAW_PROBABILITY",
        }
    )
)


# Load the existing calibrated OOF
existing_calibrated_oof = pd.read_parquet(
    calibrated_oof_path
)

calibrated_id_column = find_column(
    existing_calibrated_oof,
    ["SK_ID_CURR"],
    "Calibrated customer ID",
)

selected_probability_column = (
    find_probability_column(
        existing_calibrated_oof,
        [
            "LIGHTGBM_SELECTED_OOF",
            "LIGHTGBM_PLATT_OOF",
            "SELECTED_CALIBRATED_OOF",
            "SELECTED_OOF_PROBABILITY",
            "PLATT_CALIBRATED_OOF",
            "SELECTED_PROBABILITY",
            "CALIBRATED_PROBABILITY",
            "PLATT_PROBABILITY",
            "DEFAULT_PROBABILITY",
        ],
        "Selected calibrated OOF",
    )
)

current_calibrated_oof = (
    existing_calibrated_oof[
        [
            calibrated_id_column,
            selected_probability_column,
        ]
    ]
    .rename(
        columns={
            calibrated_id_column: "SK_ID_CURR",
            selected_probability_column:
                "CURRENT_CALIBRATED_PROBABILITY",
        }
    )
)

print(
    "Raw probability column:",
    raw_probability_column,
)

print(
    "Selected calibrated column:",
    selected_probability_column,
)

print(
    "Fold column:",
    fold_column,
)


# =========================================================
# 5. Align all existing evidence by customer ID
# =========================================================

evaluation_table = pd.DataFrame(
    {
        "SK_ID_CURR": customer_ids,
        "TARGET": target,
        "CODE_GENDER": gender,
    }
)

evaluation_table = (
    evaluation_table
    .merge(
        current_raw_oof,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one",
    )
    .merge(
        current_calibrated_oof,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one",
    )
)

assert len(evaluation_table) == len(
    modeling_table
)

assert (
    evaluation_table["SK_ID_CURR"]
    .duplicated()
    .sum()
    == 0
)

assert (
    evaluation_table[
        [
            "FOLD",
            "CURRENT_RAW_PROBABILITY",
            "CURRENT_CALIBRATED_PROBABILITY",
        ]
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

fold_values = sorted(
    evaluation_table["FOLD"].unique()
)

assert len(fold_values) == 5


# =========================================================
# 6. Best Optuna parameters
# =========================================================

gender_blind_parameters = {
    "objective": "binary",
    "boosting_type": "gbdt",
    "n_estimators": 3000,
    "learning_rate": 0.016415579569482176,
    "num_leaves": 77,
    "min_child_samples": 243,
    "subsample": 0.9425192044349383,
    "subsample_freq": 1,
    "colsample_bytree": 0.7566148192106797,
    "reg_alpha": 0.00028771223716237863,
    "reg_lambda": 0.8768125777016167,
    "min_split_gain": 0.08803049874792027,
    "random_state": 42,
    "n_jobs": -1,
    "importance_type": "gain",
}


# =========================================================
# 7. Five-fold gender-blind training
# =========================================================

gender_blind_raw_probability = np.full(
    len(evaluation_table),
    np.nan,
    dtype="float32",
)

gender_blind_fold_rows = []
training_start_time = time.time()

for fold_position, fold_value in enumerate(
    fold_values,
    start=1,
):
    print(
        "\n"
        + "=" * 10
        + f" Gender-blind Fold "
        f"{fold_position}/5 "
        + "=" * 10
    )

    validation_mask = (
        evaluation_table["FOLD"]
        == fold_value
    ).to_numpy()

    training_mask = ~validation_mask

    X_train = gender_blind_features.loc[
        training_mask
    ]

    X_validation = gender_blind_features.loc[
        validation_mask
    ]

    y_train = target.loc[
        training_mask
    ]

    y_validation = target.loc[
        validation_mask
    ]

    fold_model = lgb.LGBMClassifier(
        **gender_blind_parameters
    )

    fold_start_time = time.time()

    fold_model.fit(
        X_train,
        y_train,
        eval_X=X_validation,
        eval_y=y_validation,
        eval_names=["validation"],
        eval_metric=[
            "auc",
            "binary_logloss",
        ],
        categorical_feature=(
            categorical_features
        ),
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=150,
                verbose=False,
            ),
            lgb.log_evaluation(
                period=200
            ),
        ],
    )

    fold_probability = (
        fold_model.predict_proba(
            X_validation,
            num_iteration=(
                fold_model.best_iteration_
            ),
        )[:, 1]
    )

    gender_blind_raw_probability[
        validation_mask
    ] = fold_probability.astype("float32")

    fold_auc = roc_auc_score(
        y_validation,
        fold_probability,
    )

    gender_blind_fold_rows.append(
        {
            "fold": fold_value,
            "training_rows": int(
                training_mask.sum()
            ),
            "validation_rows": int(
                validation_mask.sum()
            ),
            "validation_default_rate_pct": (
                float(y_validation.mean() * 100)
            ),
            "best_iteration": int(
                fold_model.best_iteration_
            ),
            "roc_auc": float(fold_auc),
            "average_precision": float(
                average_precision_score(
                    y_validation,
                    fold_probability,
                )
            ),
            "log_loss": float(
                log_loss(
                    y_validation,
                    fold_probability,
                )
            ),
            "brier_score": float(
                brier_score_loss(
                    y_validation,
                    fold_probability,
                )
            ),
            "training_minutes": (
                time.time()
                - fold_start_time
            ) / 60,
        }
    )

    fold_model.booster_.save_model(
        challenger_model_path
        / (
            "lightgbm_gender_blind_"
            f"fold_{fold_position}.txt"
        )
    )

    checkpoint = pd.DataFrame(
        {
            "SK_ID_CURR":
                evaluation_table["SK_ID_CURR"],
            "TARGET":
                evaluation_table["TARGET"],
            "FOLD":
                evaluation_table["FOLD"],
            "GENDER_BLIND_RAW_PROBABILITY":
                gender_blind_raw_probability,
        }
    )

    checkpoint.to_parquet(
        modeling_output_path
        / "gender_blind_lightgbm_oof_checkpoint.parquet",
        index=False,
    )

    print(
        f"Fold {fold_position} ROC-AUC: "
        f"{fold_auc:.6f}"
    )

    print(
        "Best iteration: "
        f"{fold_model.best_iteration_}"
    )

    del (
        X_train,
        X_validation,
        y_train,
        y_validation,
        fold_model,
        fold_probability,
        checkpoint,
    )

    gc.collect()


assert not np.isnan(
    gender_blind_raw_probability
).any()

evaluation_table[
    "GENDER_BLIND_RAW_PROBABILITY"
] = gender_blind_raw_probability

gender_blind_fold_metrics = pd.DataFrame(
    gender_blind_fold_rows
)

gender_blind_fold_metrics.to_csv(
    modeling_output_path
    / "gender_blind_fold_metrics.csv",
    index=False,
)


# =========================================================
# 8. Cross-fitted Platt calibration
# =========================================================

gender_blind_calibrated_probability = (
    np.full(
        len(evaluation_table),
        np.nan,
        dtype="float64",
    )
)

for fold_value in fold_values:
    validation_mask = (
        evaluation_table["FOLD"]
        == fold_value
    ).to_numpy()

    calibration_training_mask = (
        ~validation_mask
    )

    fold_calibrator = LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=2000,
        random_state=42,
    )

    fold_calibrator.fit(
        safe_logit(
            evaluation_table.loc[
                calibration_training_mask,
                "GENDER_BLIND_RAW_PROBABILITY",
            ]
        ),
        evaluation_table.loc[
            calibration_training_mask,
            "TARGET",
        ],
    )

    gender_blind_calibrated_probability[
        validation_mask
    ] = fold_calibrator.predict_proba(
        safe_logit(
            evaluation_table.loc[
                validation_mask,
                "GENDER_BLIND_RAW_PROBABILITY",
            ]
        )
    )[:, 1]


assert not np.isnan(
    gender_blind_calibrated_probability
).any()

evaluation_table[
    "GENDER_BLIND_CALIBRATED_PROBABILITY"
] = (
    gender_blind_calibrated_probability
    .astype("float32")
)

final_gender_blind_calibrator = (
    LogisticRegression(
        C=1e6,
        solver="lbfgs",
        max_iter=2000,
        random_state=42,
    )
)

final_gender_blind_calibrator.fit(
    safe_logit(
        evaluation_table[
            "GENDER_BLIND_RAW_PROBABILITY"
        ]
    ),
    evaluation_table["TARGET"],
)

joblib.dump(
    final_gender_blind_calibrator,
    challenger_model_path
    / "gender_blind_platt_calibrator.joblib",
)


# =========================================================
# 9. Overall model comparison
# =========================================================

def calculate_model_metrics(
    model_name,
    probability_column,
):
    probability = evaluation_table[
        probability_column
    ]

    return {
        "model": model_name,
        "roc_auc": roc_auc_score(
            target,
            probability,
        ),
        "average_precision":
            average_precision_score(
                target,
                probability,
            ),
        "ks_statistic": ks_statistic(
            target,
            probability,
        ),
        "log_loss": log_loss(
            target,
            probability,
        ),
        "brier_score": brier_score_loss(
            target,
            probability,
        ),
        "mean_probability_pct":
            probability.mean() * 100,
        "calibration_gap_pp": (
            probability.mean()
            - target.mean()
        ) * 100,
    }


overall_model_comparison = pd.DataFrame(
    [
        calculate_model_metrics(
            "Current tuned raw",
            "CURRENT_RAW_PROBABILITY",
        ),
        calculate_model_metrics(
            "Gender-blind tuned raw",
            "GENDER_BLIND_RAW_PROBABILITY",
        ),
        calculate_model_metrics(
            "Current tuned + Platt",
            "CURRENT_CALIBRATED_PROBABILITY",
        ),
        calculate_model_metrics(
            "Gender-blind tuned + Platt",
            "GENDER_BLIND_CALIBRATED_PROBABILITY",
        ),
    ]
)


# =========================================================
# 10. Gender subgroup comparison at exact global top 20%
# =========================================================

def build_gender_audit(
    model_name,
    probability_column,
):
    audit_frame = evaluation_table[
        [
            "TARGET",
            "CODE_GENDER",
            probability_column,
        ]
    ].copy()

    audit_frame["TOP_20_FLAG"] = (
        exact_top_share_flag(
            audit_frame[
                probability_column
            ],
            share=0.20,
        )
    )

    rows = []

    for group_value, group in (
        audit_frame.groupby(
            "CODE_GENDER",
            dropna=False,
            observed=True,
        )
    ):
        group_target = group["TARGET"]
        group_probability = group[
            probability_column
        ]
        group_flag = group["TOP_20_FLAG"]

        defaults = int(
            group_target.sum()
        )

        nondefaults = int(
            len(group) - defaults
        )

        true_positives = int(
            (
                (group_flag == 1)
                & (group_target == 1)
            ).sum()
        )

        false_positives = int(
            (
                (group_flag == 1)
                & (group_target == 0)
            ).sum()
        )

        eligible = (
            len(group) >= 1000
            and defaults >= 50
            and nondefaults >= 50
        )

        rows.append(
            {
                "model": model_name,
                "gender": str(group_value),
                "customers": len(group),
                "defaults": defaults,
                "observed_default_pct":
                    group_target.mean() * 100,
                "mean_predicted_pct":
                    group_probability.mean() * 100,
                "calibration_gap_pp": (
                    group_probability.mean()
                    - group_target.mean()
                ) * 100,
                "roc_auc": (
                    roc_auc_score(
                        group_target,
                        group_probability,
                    )
                    if group_target.nunique() == 2
                    else np.nan
                ),
                "top_20_flag_rate_pct":
                    group_flag.mean() * 100,
                "default_capture_pct": (
                    true_positives
                    / defaults
                    * 100
                    if defaults > 0
                    else np.nan
                ),
                "nondefault_flag_rate_pct": (
                    false_positives
                    / nondefaults
                    * 100
                    if nondefaults > 0
                    else np.nan
                ),
                "review_precision_pct": (
                    true_positives
                    / int(group_flag.sum())
                    * 100
                    if group_flag.sum() > 0
                    else np.nan
                ),
                "eligible_comparison_group":
                    eligible,
            }
        )

    return pd.DataFrame(rows)


gender_subgroup_comparison = pd.concat(
    [
        build_gender_audit(
            "Current tuned + Platt",
            "CURRENT_CALIBRATED_PROBABILITY",
        ),
        build_gender_audit(
            "Gender-blind tuned + Platt",
            "GENDER_BLIND_CALIBRATED_PROBABILITY",
        ),
    ],
    ignore_index=True,
)


def build_gender_disparity(
    subgroup_table,
):
    rows = []

    for model_name, model_groups in (
        subgroup_table.groupby("model")
    ):
        eligible_groups = model_groups.loc[
            model_groups[
                "eligible_comparison_group"
            ]
        ]

        rows.append(
            {
                "model": model_name,
                "eligible_groups":
                    len(eligible_groups),
                "auc_range": (
                    eligible_groups["roc_auc"].max()
                    - eligible_groups["roc_auc"].min()
                ),
                "flag_rate_range_pp": (
                    eligible_groups[
                        "top_20_flag_rate_pct"
                    ].max()
                    - eligible_groups[
                        "top_20_flag_rate_pct"
                    ].min()
                ),
                "default_capture_range_pp": (
                    eligible_groups[
                        "default_capture_pct"
                    ].max()
                    - eligible_groups[
                        "default_capture_pct"
                    ].min()
                ),
                "nondefault_flag_range_pp": (
                    eligible_groups[
                        "nondefault_flag_rate_pct"
                    ].max()
                    - eligible_groups[
                        "nondefault_flag_rate_pct"
                    ].min()
                ),
                "maximum_absolute_calibration_gap_pp":
                    eligible_groups[
                        "calibration_gap_pp"
                    ].abs().max(),
            }
        )

    return pd.DataFrame(rows)


gender_disparity_comparison = (
    build_gender_disparity(
        gender_subgroup_comparison
    )
)


# =========================================================
# 11. Performance-tolerance decision
# =========================================================

current_raw_auc = float(
    overall_model_comparison.loc[
        overall_model_comparison["model"]
        == "Current tuned raw",
        "roc_auc",
    ].iloc[0]
)

gender_blind_raw_auc = float(
    overall_model_comparison.loc[
        overall_model_comparison["model"]
        == "Gender-blind tuned raw",
        "roc_auc",
    ].iloc[0]
)

current_calibrated_auc = float(
    overall_model_comparison.loc[
        overall_model_comparison["model"]
        == "Current tuned + Platt",
        "roc_auc",
    ].iloc[0]
)

gender_blind_calibrated_auc = float(
    overall_model_comparison.loc[
        overall_model_comparison["model"]
        == "Gender-blind tuned + Platt",
        "roc_auc",
    ].iloc[0]
)

raw_auc_loss = (
    current_raw_auc
    - gender_blind_raw_auc
)

calibrated_auc_loss = (
    current_calibrated_auc
    - gender_blind_calibrated_auc
)

performance_tolerance = 0.0005

gender_blind_validation_summary = pd.DataFrame(
    {
        "metric": [
            "Modeling rows",
            "Gender-blind predictors",
            "Gender-blind categorical features",
            "Cross-validation folds",
            "Missing raw OOF predictions",
            "Missing calibrated predictions",
            "Saved fold models",
            "Current raw ROC-AUC",
            "Gender-blind raw ROC-AUC",
            "Raw ROC-AUC loss",
            "Current calibrated ROC-AUC",
            "Gender-blind calibrated ROC-AUC",
            "Calibrated ROC-AUC loss",
            "Performance-loss tolerance",
            "Passes performance tolerance",
            "Training time (minutes)",
        ],
        "value": [
            len(evaluation_table),
            gender_blind_features.shape[1],
            len(categorical_features),
            len(fold_values),
            evaluation_table[
                "GENDER_BLIND_RAW_PROBABILITY"
            ].isna().sum(),
            evaluation_table[
                "GENDER_BLIND_CALIBRATED_PROBABILITY"
            ].isna().sum(),
            len(
                list(
                    challenger_model_path.glob(
                        "lightgbm_gender_blind_"
                        "fold_*.txt"
                    )
                )
            ),
            current_raw_auc,
            gender_blind_raw_auc,
            raw_auc_loss,
            current_calibrated_auc,
            gender_blind_calibrated_auc,
            calibrated_auc_loss,
            performance_tolerance,
            (
                raw_auc_loss
                <= performance_tolerance
            ),
            (
                time.time()
                - training_start_time
            ) / 60,
        ],
    }
)


# =========================================================
# 12. Save challenger evidence
# =========================================================

evaluation_table[
    [
        "SK_ID_CURR",
        "TARGET",
        "FOLD",
        "CODE_GENDER",
        "GENDER_BLIND_RAW_PROBABILITY",
        "GENDER_BLIND_CALIBRATED_PROBABILITY",
    ]
].to_parquet(
    modeling_output_path
    / "gender_blind_lightgbm_oof.parquet",
    index=False,
)

overall_model_comparison.to_csv(
    modeling_output_path
    / "gender_blind_overall_comparison.csv",
    index=False,
)

gender_subgroup_comparison.to_csv(
    modeling_output_path
    / "gender_blind_gender_subgroup_audit.csv",
    index=False,
)

gender_disparity_comparison.to_csv(
    modeling_output_path
    / "gender_blind_gender_disparity.csv",
    index=False,
)

gender_blind_validation_summary.to_csv(
    modeling_output_path
    / "gender_blind_validation_summary.csv",
    index=False,
)


# =========================================================
# 13. Display
# =========================================================

display(
    gender_blind_validation_summary.round(6)
)

display(
    gender_blind_fold_metrics.round(6)
)

display(
    overall_model_comparison.round(6)
)

display(
    gender_subgroup_comparison.round(4)
)

display(
    gender_disparity_comparison.round(4)
)


In [ ]:
# =========================================================
# Final model-governance decision
# =========================================================

import gc
import json
from pathlib import Path

import pandas as pd


# =========================================================
# 1. Paths
# =========================================================

project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

modeling_output_path = (
    processed_data_path
    / "modeling"
)

overall_comparison_path = (
    modeling_output_path
    / "gender_blind_overall_comparison.csv"
)

gender_disparity_path = (
    modeling_output_path
    / "gender_blind_gender_disparity.csv"
)

fold_metrics_path = (
    modeling_output_path
    / "gender_blind_fold_metrics.csv"
)

assert overall_comparison_path.exists()
assert gender_disparity_path.exists()
assert fold_metrics_path.exists()


# =========================================================
# 2. Load saved comparison evidence
# =========================================================

overall_comparison = pd.read_csv(
    overall_comparison_path
)

gender_disparity = pd.read_csv(
    gender_disparity_path
)

gender_blind_fold_metrics = pd.read_csv(
    fold_metrics_path
)


def select_model_row(
    frame,
    model_name,
):
    result = frame.loc[
        frame["model"] == model_name
    ]

    assert len(result) == 1, (
        f"Model row not unique: {model_name}"
    )

    return result.iloc[0]


current_model = select_model_row(
    overall_comparison,
    "Current tuned + Platt",
)

gender_blind_model = select_model_row(
    overall_comparison,
    "Gender-blind tuned + Platt",
)

current_raw_model = select_model_row(
    overall_comparison,
    "Current tuned raw",
)

gender_blind_raw_model = select_model_row(
    overall_comparison,
    "Gender-blind tuned raw",
)

current_disparity = select_model_row(
    gender_disparity,
    "Current tuned + Platt",
)

gender_blind_disparity = select_model_row(
    gender_disparity,
    "Gender-blind tuned + Platt",
)


# =========================================================
# 3. Calculate decision metrics
# =========================================================

performance_tolerance = 0.0005

raw_auc_loss = float(
    current_raw_model["roc_auc"]
    - gender_blind_raw_model["roc_auc"]
)

calibrated_auc_loss = float(
    current_model["roc_auc"]
    - gender_blind_model["roc_auc"]
)

average_precision_loss = float(
    current_model["average_precision"]
    - gender_blind_model[
        "average_precision"
    ]
)

ks_loss = float(
    current_model["ks_statistic"]
    - gender_blind_model["ks_statistic"]
)

log_loss_increase = float(
    gender_blind_model["log_loss"]
    - current_model["log_loss"]
)

brier_increase = float(
    gender_blind_model["brier_score"]
    - current_model["brier_score"]
)

flag_rate_gap_reduction = float(
    current_disparity[
        "flag_rate_range_pp"
    ]
    - gender_blind_disparity[
        "flag_rate_range_pp"
    ]
)

capture_gap_reduction = float(
    current_disparity[
        "default_capture_range_pp"
    ]
    - gender_blind_disparity[
        "default_capture_range_pp"
    ]
)

nondefault_gap_reduction = float(
    current_disparity[
        "nondefault_flag_range_pp"
    ]
    - gender_blind_disparity[
        "nondefault_flag_range_pp"
    ]
)

calibration_gap_increase = float(
    gender_blind_disparity[
        "maximum_absolute_calibration_gap_pp"
    ]
    - current_disparity[
        "maximum_absolute_calibration_gap_pp"
    ]
)

passes_performance_tolerance = (
    calibrated_auc_loss
    <= performance_tolerance
)


# =========================================================
# 4. Decision and trade-off tables
# =========================================================

model_governance_decision = pd.DataFrame(
    {
        "item": [
            "Technical ranking champion",
            "Probability champion",
            "Gender-blind challenger selected",
            "Performance-loss tolerance",
            "Gender-blind calibrated AUC loss",
            "Passes performance tolerance",
            "Protected feature in champion",
            "Production deployment status",
            "Required governance action",
        ],
        "decision": [
            "Optuna-tuned LightGBM",
            "Optuna-tuned LightGBM + Platt",
            False,
            performance_tolerance,
            calibrated_auc_loss,
            passes_performance_tolerance,
            "CODE_GENDER",
            "Governance review required",
            (
                "Review protected-feature policy "
                "before operational deployment"
            ),
        ],
    }
)

model_tradeoff_summary = pd.DataFrame(
    {
        "metric": [
            "Raw ROC-AUC loss",
            "Calibrated ROC-AUC loss",
            "Average-precision loss",
            "KS loss",
            "Log-loss increase",
            "Brier-score increase",
            "Flag-rate gap reduction (pp)",
            "Default-capture gap reduction (pp)",
            "Non-default flag gap reduction (pp)",
            "Maximum calibration-gap increase (pp)",
            "Gender-blind folds evaluated",
        ],
        "value": [
            raw_auc_loss,
            calibrated_auc_loss,
            average_precision_loss,
            ks_loss,
            log_loss_increase,
            brier_increase,
            flag_rate_gap_reduction,
            capture_gap_reduction,
            nondefault_gap_reduction,
            calibration_gap_increase,
            len(gender_blind_fold_metrics),
        ],
    }
)


# =========================================================
# 5. Save machine-readable governance manifest
# =========================================================

governance_manifest = {
    "technical_ranking_champion":
        "Optuna-tuned LightGBM",
    "probability_champion":
        "Optuna-tuned LightGBM + Platt",
    "champion_uses_code_gender": True,
    "gender_blind_challenger_selected":
        bool(passes_performance_tolerance),
    "performance_loss_tolerance":
        performance_tolerance,
    "gender_blind_raw_auc_loss":
        raw_auc_loss,
    "gender_blind_calibrated_auc_loss":
        calibrated_auc_loss,
    "gender_blind_average_precision_loss":
        average_precision_loss,
    "gender_blind_ks_loss":
        ks_loss,
    "flag_rate_gap_reduction_pp":
        flag_rate_gap_reduction,
    "default_capture_gap_reduction_pp":
        capture_gap_reduction,
    "nondefault_flag_gap_reduction_pp":
        nondefault_gap_reduction,
    "maximum_calibration_gap_increase_pp":
        calibration_gap_increase,
    "deployment_status":
        "governance_review_required",
    "decision_reason": (
        "Gender-blind challenger reduced several "
        "operational disparity measures, but exceeded "
        "the predefined ROC-AUC loss tolerance and "
        "worsened subgroup calibration."
    ),
    "important_caveat": (
        "This is a technical model-governance record, "
        "not a legal or regulatory approval."
    ),
}

decision_csv_path = (
    modeling_output_path
    / "final_model_governance_decision.csv"
)

tradeoff_csv_path = (
    modeling_output_path
    / "final_model_tradeoff_summary.csv"
)

manifest_json_path = (
    modeling_output_path
    / "final_model_governance_manifest.json"
)

model_governance_decision.to_csv(
    decision_csv_path,
    index=False,
)

model_tradeoff_summary.to_csv(
    tradeoff_csv_path,
    index=False,
)

with open(
    manifest_json_path,
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(
        governance_manifest,
        manifest_file,
        indent=4,
        ensure_ascii=False,
    )


# =========================================================
# 6. Storage verification
# =========================================================

governance_storage_check = pd.DataFrame(
    {
        "metric": [
            "Decision file exists",
            "Trade-off file exists",
            "Manifest file exists",
            "Gender-blind challenger selected",
            "Production deployment approved",
        ],
        "value": [
            decision_csv_path.exists(),
            tradeoff_csv_path.exists(),
            manifest_json_path.exists(),
            passes_performance_tolerance,
            False,
        ],
    }
)

display(model_governance_decision)
display(model_tradeoff_summary.round(6))
display(governance_storage_check)


# =========================================================
# 7. Release large training objects from memory
# =========================================================

objects_to_remove = [
    "modeling_table",
    "gender_blind_features",
    "evaluation_table",
    "existing_tuned_oof",
    "existing_calibrated_oof",
    "current_raw_oof",
    "current_calibrated_oof",
    "gender_blind_raw_probability",
    "gender_blind_calibrated_probability",
    "final_gender_blind_calibrator",
]

for object_name in objects_to_remove:
    globals().pop(
        object_name,
        None,
    )

gc.collect()

In [ ]:
# =========================================================
# Global out-of-fold TreeSHAP explainability
# =========================================================

import gc
import re
import time
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# =========================================================
# 1. Paths and parameters
# =========================================================

project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

modeling_output_path = (
    processed_data_path
    / "modeling"
)

explainability_path = (
    modeling_output_path
    / "explainability"
)

explainability_path.mkdir(
    parents=True,
    exist_ok=True,
)

clean_modeling_path = (
    processed_data_path
    / "modeling_train_clean.parquet"
)

tuned_oof_path = (
    modeling_output_path
    / "lightgbm_tuned_oof.parquet"
)

sample_size = 10_000
random_seed = 42

assert clean_modeling_path.exists()
assert tuned_oof_path.exists()


# =========================================================
# 2. Locate the five tuned champion models
# =========================================================

all_text_models = list(
    project_path.rglob("*.txt")
)

tuned_model_candidates = [
    path
    for path in all_text_models
    if (
        "fold" in path.stem.lower()
        and "gender_blind"
        not in str(path).lower()
        and (
            "tuned" in path.stem.lower()
            or "tuned" in str(path.parent).lower()
        )
    )
]


def extract_fold_number(path):
    match = re.search(
        r"fold[_\-\s]*(\d+)",
        path.stem.lower(),
    )

    if match is None:
        return None

    return int(match.group(1))


# If duplicate model files exist, keep the newest one
model_files_by_fold = {}

for model_path in tuned_model_candidates:
    fold_number = extract_fold_number(
        model_path
    )

    if fold_number is None:
        continue

    existing_path = model_files_by_fold.get(
        fold_number
    )

    if (
        existing_path is None
        or model_path.stat().st_mtime
        > existing_path.stat().st_mtime
    ):
        model_files_by_fold[
            fold_number
        ] = model_path

if len(model_files_by_fold) != 5:
    raise FileNotFoundError(
        "Five tuned fold models could not be "
        "identified. Candidate files:\n"
        + "\n".join(
            str(path)
            for path in tuned_model_candidates
        )
    )

model_discovery = pd.DataFrame(
    {
        "fold": sorted(
            model_files_by_fold
        ),
        "model_file": [
            str(
                model_files_by_fold[fold]
            )
            for fold in sorted(
                model_files_by_fold
            )
        ],
    }
)

display(model_discovery)


# =========================================================
# 3. Load one model and recover exact feature order
# =========================================================

first_fold = sorted(
    model_files_by_fold
)[0]

reference_booster = lgb.Booster(
    model_file=str(
        model_files_by_fold[
            first_fold
        ]
    )
)

model_feature_names = (
    reference_booster.feature_name()
)

model_feature_count = (
    reference_booster.num_feature()
)

assert (
    len(model_feature_names)
    == model_feature_count
)

assert model_feature_count == 639


# =========================================================
# 4. Load modeling data and OOF fold assignment
# =========================================================

required_modeling_columns = [
    "SK_ID_CURR",
    "TARGET",
] + model_feature_names

modeling_table = pd.read_parquet(
    clean_modeling_path,
    columns=required_modeling_columns,
)

tuned_oof = pd.read_parquet(
    tuned_oof_path,
    columns=[
        "SK_ID_CURR",
        "CV_FOLD",
    ],
)

assert (
    modeling_table["SK_ID_CURR"]
    .duplicated()
    .sum()
    == 0
)

assert (
    tuned_oof["SK_ID_CURR"]
    .duplicated()
    .sum()
    == 0
)

explanation_population = (
    modeling_table.merge(
        tuned_oof,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one",
    )
)

assert len(explanation_population) == len(
    modeling_table
)

assert (
    explanation_population[
        "CV_FOLD"
    ].isna().sum()
    == 0
)

explanation_population[
    "CV_FOLD"
] = explanation_population[
    "CV_FOLD"
].astype("int8")

observed_folds = set(
    explanation_population[
        "CV_FOLD"
    ].unique()
)

assert observed_folds == set(
    model_files_by_fold
), (
    observed_folds,
    set(model_files_by_fold),
)


# =========================================================
# 5. Select a reproducible population sample
# =========================================================

actual_sample_size = min(
    sample_size,
    len(explanation_population),
)

explanation_sample = (
    explanation_population
    .sample(
        n=actual_sample_size,
        random_state=random_seed,
    )
    .reset_index(drop=True)
)

full_default_rate = (
    explanation_population[
        "TARGET"
    ].mean()
)

sample_default_rate = (
    explanation_sample[
        "TARGET"
    ].mean()
)

# Release the full 307k-row table before SHAP calculation
del modeling_table
del explanation_population
del tuned_oof
del reference_booster

gc.collect()


# =========================================================
# 6. Calculate strictly out-of-fold SHAP contributions
# =========================================================

calculation_start = time.time()

shap_contributions = np.zeros(
    (
        len(explanation_sample),
        model_feature_count,
    ),
    dtype="float32",
)

expected_values = np.zeros(
    len(explanation_sample),
    dtype="float32",
)

raw_model_scores = np.zeros(
    len(explanation_sample),
    dtype="float32",
)

fold_explanation_rows = []

for fold_number in sorted(
    model_files_by_fold
):
    fold_mask = (
        explanation_sample[
            "CV_FOLD"
        ].to_numpy()
        == fold_number
    )

    fold_positions = np.flatnonzero(
        fold_mask
    )

    fold_features = (
        explanation_sample.loc[
            fold_mask,
            model_feature_names,
        ]
    )

    fold_booster = lgb.Booster(
        model_file=str(
            model_files_by_fold[
                fold_number
            ]
        )
    )

    assert (
        fold_booster.feature_name()
        == model_feature_names
    )

    fold_contributions = (
        fold_booster.predict(
            fold_features,
            pred_contrib=True,
            validate_features=True,
        )
    )

    fold_raw_scores = (
        fold_booster.predict(
            fold_features,
            raw_score=True,
            validate_features=True,
        )
    )

    expected_shape = (
        len(fold_features),
        model_feature_count + 1,
    )

    assert (
        fold_contributions.shape
        == expected_shape
    ), (
        fold_contributions.shape,
        expected_shape,
    )

    # Last column is the expected/base value
    shap_contributions[
        fold_positions,
        :,
    ] = fold_contributions[
        :,
        :-1,
    ].astype("float32")

    expected_values[
        fold_positions
    ] = fold_contributions[
        :,
        -1,
    ].astype("float32")

    raw_model_scores[
        fold_positions
    ] = np.asarray(
        fold_raw_scores
    ).astype("float32")

    fold_reconstructed_scores = (
        fold_contributions[
            :,
            :-1,
        ].sum(axis=1)
        + fold_contributions[
            :,
            -1,
        ]
    )

    fold_max_error = float(
        np.max(
            np.abs(
                fold_reconstructed_scores
                - fold_raw_scores
            )
        )
    )

    fold_explanation_rows.append(
        {
            "fold": fold_number,
            "sample_rows":
                len(fold_features),
            "default_rate_pct": (
                explanation_sample.loc[
                    fold_mask,
                    "TARGET",
                ].mean()
                * 100
            ),
            "maximum_reconciliation_error":
                fold_max_error,
        }
    )

    print(
        f"Fold {fold_number}: "
        f"{len(fold_features):,} customers explained"
    )

    del (
        fold_features,
        fold_booster,
        fold_contributions,
        fold_raw_scores,
        fold_reconstructed_scores,
    )

    gc.collect()


# =========================================================
# 7. Additivity reconciliation
# =========================================================

reconstructed_raw_scores = (
    expected_values.astype("float64")
    + shap_contributions
    .astype("float64")
    .sum(axis=1)
)

reconciliation_errors = np.abs(
    reconstructed_raw_scores
    - raw_model_scores
)

maximum_reconciliation_error = float(
    reconciliation_errors.max()
)

mean_reconciliation_error = float(
    reconciliation_errors.mean()
)

assert (
    maximum_reconciliation_error
    < 1e-4
), maximum_reconciliation_error


# =========================================================
# 8. Feature-source classification
# =========================================================

def identify_feature_source(
    feature_name,
):
    if feature_name.startswith("APP_"):
        return "Application engineered"

    if feature_name.startswith("BUREAU_"):
        return "Bureau and bureau balance"

    if feature_name.startswith("PREV_"):
        return "Previous applications"

    if feature_name.startswith("POS_"):
        return "POS CASH"

    if feature_name.startswith("INST_"):
        return "Installment payments"

    if feature_name.startswith("CC_"):
        return "Credit-card balance"

    return "Application raw"


# =========================================================
# 9. Global feature importance
# =========================================================

mean_absolute_shap = (
    np.abs(shap_contributions)
    .mean(axis=0)
)

mean_signed_shap = (
    shap_contributions.mean(axis=0)
)

positive_contribution_share = (
    (
        shap_contributions > 0
    ).mean(axis=0)
    * 100
)

global_shap_importance = pd.DataFrame(
    {
        "feature": model_feature_names,
        "feature_source": [
            identify_feature_source(
                feature_name
            )
            for feature_name
            in model_feature_names
        ],
        "mean_absolute_shap":
            mean_absolute_shap,
        "mean_signed_shap":
            mean_signed_shap,
        "positive_contribution_share_pct":
            positive_contribution_share,
    }
)

total_mean_absolute_shap = float(
    global_shap_importance[
        "mean_absolute_shap"
    ].sum()
)

assert total_mean_absolute_shap > 0

global_shap_importance[
    "importance_share_pct"
] = (
    global_shap_importance[
        "mean_absolute_shap"
    ]
    / total_mean_absolute_shap
    * 100
)

global_shap_importance = (
    global_shap_importance
    .sort_values(
        "mean_absolute_shap",
        ascending=False,
    )
    .reset_index(drop=True)
)

global_shap_importance.insert(
    0,
    "rank",
    np.arange(
        1,
        len(global_shap_importance) + 1,
    ),
)


# =========================================================
# 10. Source-level importance
# =========================================================

shap_source_importance = (
    global_shap_importance
    .groupby(
        "feature_source",
        as_index=False,
    )
    .agg(
        features=("feature", "size"),
        total_mean_absolute_shap=(
            "mean_absolute_shap",
            "sum",
        ),
    )
)

shap_source_importance[
    "importance_share_pct"
] = (
    shap_source_importance[
        "total_mean_absolute_shap"
    ]
    / shap_source_importance[
        "total_mean_absolute_shap"
    ].sum()
    * 100
)

shap_source_importance = (
    shap_source_importance
    .sort_values(
        "importance_share_pct",
        ascending=False,
    )
    .reset_index(drop=True)
)

assert np.isclose(
    shap_source_importance[
        "importance_share_pct"
    ].sum(),
    100,
)


# =========================================================
# 11. Save sample-level SHAP evidence
# =========================================================

shap_column_names = [
    f"SHAP__{feature_name}"
    for feature_name
    in model_feature_names
]

shap_sample_output = pd.DataFrame(
    shap_contributions,
    columns=shap_column_names,
)

shap_sample_output.insert(
    0,
    "RAW_MODEL_SCORE",
    raw_model_scores,
)

shap_sample_output.insert(
    0,
    "EXPECTED_VALUE",
    expected_values,
)

shap_sample_output.insert(
    0,
    "CV_FOLD",
    explanation_sample[
        "CV_FOLD"
    ].to_numpy(),
)

shap_sample_output.insert(
    0,
    "TARGET",
    explanation_sample[
        "TARGET"
    ].to_numpy(),
)

shap_sample_output.insert(
    0,
    "SK_ID_CURR",
    explanation_sample[
        "SK_ID_CURR"
    ].to_numpy(),
)

global_importance_path = (
    explainability_path
    / "shap_global_feature_importance.csv"
)

source_importance_path = (
    explainability_path
    / "shap_source_importance.csv"
)

sample_contribution_path = (
    explainability_path
    / "shap_oof_sample_contributions.parquet"
)

global_shap_importance.to_csv(
    global_importance_path,
    index=False,
)

shap_source_importance.to_csv(
    source_importance_path,
    index=False,
)

shap_sample_output.to_parquet(
    sample_contribution_path,
    index=False,
)


# =========================================================
# 12. Create global importance chart
# =========================================================

source_colors = {
    "Application raw": "#355C7D",
    "Application engineered": "#2A9D8F",
    "Bureau and bureau balance": "#E9C46A",
    "Previous applications": "#F4A261",
    "POS CASH": "#E76F51",
    "Installment payments": "#6A4C93",
    "Credit-card balance": "#457B9D",
}

top_feature_count = 20

top_features = (
    global_shap_importance
    .head(top_feature_count)
    .sort_values(
        "mean_absolute_shap",
        ascending=True,
    )
)

figure, axes = plt.subplots(
    1,
    2,
    figsize=(18, 9),
    gridspec_kw={
        "width_ratios": [1.7, 1]
    },
)

axes[0].barh(
    top_features["feature"],
    top_features[
        "mean_absolute_shap"
    ],
    color=[
        source_colors.get(
            source,
            "#777777",
        )
        for source in top_features[
            "feature_source"
        ]
    ],
)

axes[0].set_title(
    "Top 20 Global TreeSHAP Features\n"
    "Strictly Out-of-Fold Sample"
)

axes[0].set_xlabel(
    "Mean absolute SHAP contribution "
    "(raw log-odds)"
)

source_plot = (
    shap_source_importance
    .sort_values(
        "importance_share_pct",
        ascending=True,
    )
)

axes[1].barh(
    source_plot["feature_source"],
    source_plot[
        "importance_share_pct"
    ],
    color=[
        source_colors.get(
            source,
            "#777777",
        )
        for source in source_plot[
            "feature_source"
        ]
    ],
)

axes[1].set_title(
    "Importance Share by Feature Source"
)

axes[1].set_xlabel(
    "Total SHAP importance share (%)"
)

for axis in axes:
    axis.grid(
        axis="x",
        alpha=0.2,
    )

figure.suptitle(
    "FinGuard360 Champion Model Explainability",
    fontsize=16,
    fontweight="bold",
)

figure.tight_layout()

chart_path = (
    explainability_path
    / "shap_global_importance.png"
)

figure.savefig(
    chart_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()


# =========================================================
# 13. Validation summary
# =========================================================

fold_explanation_check = pd.DataFrame(
    fold_explanation_rows
)

shap_validation_summary = pd.DataFrame(
    {
        "metric": [
            "Model feature count",
            "OOF explanation sample rows",
            "Unique sample customers",
            "Duplicate sample customers",
            "Folds represented",
            "Full population default rate (%)",
            "Sample default rate (%)",
            "Default-rate difference (pp)",
            "Maximum additivity error",
            "Mean additivity error",
            "Importance-share total (%)",
            "Global feature rows",
            "Feature sources",
            "Saved contribution rows",
            "Contribution file size (MB)",
            "Calculation time (minutes)",
        ],
        "value": [
            model_feature_count,
            len(explanation_sample),
            explanation_sample[
                "SK_ID_CURR"
            ].nunique(),
            explanation_sample[
                "SK_ID_CURR"
            ].duplicated().sum(),
            explanation_sample[
                "CV_FOLD"
            ].nunique(),
            full_default_rate * 100,
            sample_default_rate * 100,
            (
                sample_default_rate
                - full_default_rate
            ) * 100,
            maximum_reconciliation_error,
            mean_reconciliation_error,
            shap_source_importance[
                "importance_share_pct"
            ].sum(),
            len(global_shap_importance),
            len(shap_source_importance),
            len(shap_sample_output),
            (
                sample_contribution_path
                .stat()
                .st_size
                / 1024**2
            ),
            (
                time.time()
                - calculation_start
            ) / 60,
        ],
    }
)

shap_validation_summary.to_csv(
    explainability_path
    / "shap_validation_summary.csv",
    index=False,
)

display(
    shap_validation_summary.round(6)
)

display(
    fold_explanation_check.round(8)
)

display(
    global_shap_importance
    .head(30)
    .round(6)
)

display(
    shap_source_importance.round(4)
)


# =========================================================
# 14. Release large objects
# =========================================================

del shap_sample_output
del shap_contributions
del explanation_sample

gc.collect()

In [ ]:
# =========================================================
# Correct SHAP feature-source classification
# =========================================================

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# =========================================================
# 1. Paths
# =========================================================

project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

explainability_path = (
    project_path
    / "data"
    / "processed"
    / "modeling"
    / "explainability"
)

original_importance_path = (
    explainability_path
    / "shap_global_feature_importance.csv"
)

assert original_importance_path.exists()


# =========================================================
# 2. Load the correct SHAP values
# =========================================================

global_shap_importance = pd.read_csv(
    original_importance_path
)

assert len(global_shap_importance) == 639

assert np.isclose(
    global_shap_importance[
        "importance_share_pct"
    ].sum(),
    100,
    atol=1e-3,
)


# =========================================================
# 3. Correct source rules
# =========================================================

def identify_corrected_feature_source(
    feature_name,
):
    # Bureau balance features
    if feature_name.startswith(
        (
            "BB_",
            "BUREAU_",
        )
    ):
        return "Bureau and bureau balance"

    # Previous-application features
    if feature_name.startswith(
        (
            "PREV_",
            "APPROVED_PREV_",
        )
    ):
        return "Previous applications"

    if feature_name.startswith("POS_"):
        return "POS CASH"

    if feature_name.startswith("INST_"):
        return "Installment payments"

    if feature_name.startswith("CC_"):
        return "Credit-card balance"

    if feature_name.startswith("APP_"):
        return "Application engineered"

    return "Application raw"


global_shap_importance[
    "feature_source"
] = (
    global_shap_importance["feature"]
    .map(
        identify_corrected_feature_source
    )
)


# =========================================================
# 4. Validate expected feature counts
# =========================================================

expected_source_counts = {
    "Application raw": 120,
    "Application engineered": 56,
    "Bureau and bureau balance": 88,
    "Previous applications": 73,
    "POS CASH": 67,
    "Installment payments": 81,
    "Credit-card balance": 154,
}

actual_source_counts = (
    global_shap_importance[
        "feature_source"
    ]
    .value_counts()
    .to_dict()
)

source_count_check = pd.DataFrame(
    {
        "feature_source":
            list(expected_source_counts),
        "expected_features": [
            expected_source_counts[source]
            for source
            in expected_source_counts
        ],
        "actual_features": [
            actual_source_counts.get(
                source,
                0,
            )
            for source
            in expected_source_counts
        ],
    }
)

source_count_check[
    "difference"
] = (
    source_count_check[
        "actual_features"
    ]
    - source_count_check[
        "expected_features"
    ]
)

assert (
    source_count_check[
        "difference"
    ].eq(0).all()
)

assert (
    source_count_check[
        "actual_features"
    ].sum()
    == 639
)


# =========================================================
# 5. Recalculate source-level SHAP importance
# =========================================================

corrected_source_importance = (
    global_shap_importance
    .groupby(
        "feature_source",
        as_index=False,
    )
    .agg(
        features=("feature", "size"),
        total_mean_absolute_shap=(
            "mean_absolute_shap",
            "sum",
        ),
    )
)

corrected_source_importance[
    "importance_share_pct"
] = (
    corrected_source_importance[
        "total_mean_absolute_shap"
    ]
    / corrected_source_importance[
        "total_mean_absolute_shap"
    ].sum()
    * 100
)

corrected_source_importance = (
    corrected_source_importance
    .sort_values(
        "importance_share_pct",
        ascending=False,
    )
    .reset_index(drop=True)
)

assert np.isclose(
    corrected_source_importance[
        "importance_share_pct"
    ].sum(),
    100,
)


# =========================================================
# 6. Governance and interpretation summary
# =========================================================

gender_importance = (
    global_shap_importance.loc[
        global_shap_importance["feature"]
        == "CODE_GENDER"
    ]
)

assert len(gender_importance) == 1

gender_rank = int(
    gender_importance["rank"].iloc[0]
)

gender_importance_share = float(
    gender_importance[
        "importance_share_pct"
    ].iloc[0]
)

application_sources = [
    "Application raw",
    "Application engineered",
]

history_sources = [
    "Bureau and bureau balance",
    "Previous applications",
    "POS CASH",
    "Installment payments",
    "Credit-card balance",
]

application_importance_share = float(
    corrected_source_importance.loc[
        corrected_source_importance[
            "feature_source"
        ].isin(application_sources),
        "importance_share_pct",
    ].sum()
)

history_importance_share = float(
    corrected_source_importance.loc[
        corrected_source_importance[
            "feature_source"
        ].isin(history_sources),
        "importance_share_pct",
    ].sum()
)

interpretability_summary = pd.DataFrame(
    {
        "metric": [
            "Global SHAP feature rows",
            "Feature sources",
            "Feature-count reconciliation errors",
            "Importance-share total (%)",
            "Application importance share (%)",
            "Relational-history importance share (%)",
            "CODE_GENDER SHAP rank",
            "CODE_GENDER importance share (%)",
            "Protected feature materially used",
            "Production governance review required",
        ],
        "value": [
            len(global_shap_importance),
            corrected_source_importance[
                "feature_source"
            ].nunique(),
            source_count_check[
                "difference"
            ].ne(0).sum(),
            corrected_source_importance[
                "importance_share_pct"
            ].sum(),
            application_importance_share,
            history_importance_share,
            gender_rank,
            gender_importance_share,
            gender_rank <= 10,
            True,
        ],
    }
)


# =========================================================
# 7. Save corrected evidence
# =========================================================

corrected_global_path = (
    explainability_path
    / "shap_global_feature_importance_corrected.csv"
)

corrected_source_path = (
    explainability_path
    / "shap_source_importance_corrected.csv"
)

source_check_path = (
    explainability_path
    / "shap_source_classification_check.csv"
)

global_shap_importance.to_csv(
    corrected_global_path,
    index=False,
)

corrected_source_importance.to_csv(
    corrected_source_path,
    index=False,
)

source_count_check.to_csv(
    source_check_path,
    index=False,
)

interpretability_summary.to_csv(
    explainability_path
    / "shap_interpretability_summary.csv",
    index=False,
)


# =========================================================
# 8. Recreate corrected chart
# =========================================================

source_colors = {
    "Application raw": "#355C7D",
    "Application engineered": "#2A9D8F",
    "Bureau and bureau balance": "#E9C46A",
    "Previous applications": "#F4A261",
    "POS CASH": "#E76F51",
    "Installment payments": "#6A4C93",
    "Credit-card balance": "#457B9D",
}

top_features = (
    global_shap_importance
    .head(20)
    .sort_values(
        "mean_absolute_shap",
        ascending=True,
    )
)

source_plot = (
    corrected_source_importance
    .sort_values(
        "importance_share_pct",
        ascending=True,
    )
)

figure, axes = plt.subplots(
    1,
    2,
    figsize=(18, 9),
    gridspec_kw={
        "width_ratios": [1.7, 1]
    },
)

axes[0].barh(
    top_features["feature"],
    top_features[
        "mean_absolute_shap"
    ],
    color=[
        source_colors.get(
            source,
            "#777777",
        )
        for source in top_features[
            "feature_source"
        ]
    ],
)

axes[0].set_title(
    "Top 20 Global TreeSHAP Features\n"
    "Out-of-Fold Sample"
)

axes[0].set_xlabel(
    "Mean absolute SHAP contribution "
    "(raw log-odds)"
)

axes[1].barh(
    source_plot["feature_source"],
    source_plot[
        "importance_share_pct"
    ],
    color=[
        source_colors.get(
            source,
            "#777777",
        )
        for source in source_plot[
            "feature_source"
        ]
    ],
)

axes[1].set_title(
    "Corrected Importance Share "
    "by Feature Source"
)

axes[1].set_xlabel(
    "SHAP importance share (%)"
)

for axis in axes:
    axis.grid(
        axis="x",
        alpha=0.2,
    )

figure.suptitle(
    "FinGuard360 Champion Model Explainability",
    fontsize=16,
    fontweight="bold",
)

figure.tight_layout()

corrected_chart_path = (
    explainability_path
    / "shap_global_importance_corrected.png"
)

figure.savefig(
    corrected_chart_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()


# =========================================================
# 9. Verification
# =========================================================

corrected_storage_check = pd.DataFrame(
    {
        "metric": [
            "Corrected global file exists",
            "Corrected source file exists",
            "Source check file exists",
            "Corrected chart exists",
        ],
        "value": [
            corrected_global_path.exists(),
            corrected_source_path.exists(),
            source_check_path.exists(),
            corrected_chart_path.exists(),
        ],
    }
)

display(source_count_check)
display(corrected_source_importance.round(4))
display(interpretability_summary.round(4))
display(corrected_storage_check)

In [ ]:
# =========================================================
# SHAP directionality and technical reason codes
# =========================================================

import gc
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# =========================================================
# 1. Paths
# =========================================================

project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

modeling_output_path = (
    processed_data_path
    / "modeling"
)

explainability_path = (
    modeling_output_path
    / "explainability"
)

clean_modeling_path = (
    processed_data_path
    / "modeling_train_clean.parquet"
)

global_importance_path = (
    explainability_path
    / "shap_global_feature_importance_corrected.csv"
)

shap_sample_path = (
    explainability_path
    / "shap_oof_sample_contributions.parquet"
)

calibrated_oof_path = (
    modeling_output_path
    / "lightgbm_calibrated_oof.parquet"
)

for required_path in [
    clean_modeling_path,
    global_importance_path,
    shap_sample_path,
    calibrated_oof_path,
]:
    assert required_path.exists(), required_path


# =========================================================
# 2. Load saved SHAP evidence
# =========================================================

global_importance = pd.read_csv(
    global_importance_path
)

shap_sample = pd.read_parquet(
    shap_sample_path
)

assert len(global_importance) == 639
assert len(shap_sample) == 10_000

assert (
    shap_sample["SK_ID_CURR"]
    .duplicated()
    .sum()
    == 0
)

feature_source_map = (
    global_importance.set_index(
        "feature"
    )["feature_source"].to_dict()
)

global_rank_map = (
    global_importance.set_index(
        "feature"
    )["rank"].to_dict()
)


# =========================================================
# 3. Load values of the globally important features
# =========================================================

top_feature_count = 20

top_features = (
    global_importance
    .head(top_feature_count)
    .copy()
)

top_feature_names = (
    top_features["feature"].tolist()
)

top_shap_columns = [
    f"SHAP__{feature}"
    for feature in top_feature_names
]

assert set(top_shap_columns).issubset(
    shap_sample.columns
)

top_feature_values = pd.read_parquet(
    clean_modeling_path,
    columns=[
        "SK_ID_CURR",
    ] + top_feature_names,
)

assert (
    top_feature_values["SK_ID_CURR"]
    .duplicated()
    .sum()
    == 0
)

direction_sample = (
    shap_sample[
        [
            "SK_ID_CURR",
            "TARGET",
            "CV_FOLD",
        ] + top_shap_columns
    ]
    .merge(
        top_feature_values,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one",
    )
)

assert len(direction_sample) == len(
    shap_sample
)


# =========================================================
# 4. Calculate numeric and categorical directionality
# =========================================================

directionality_rows = []
numeric_bin_tables = []
categorical_effect_tables = []

for feature_name in top_feature_names:
    shap_column = (
        f"SHAP__{feature_name}"
    )

    feature_values = direction_sample[
        feature_name
    ]

    shap_values = direction_sample[
        shap_column
    ]

    unique_values = (
        feature_values.nunique(
            dropna=True
        )
    )

    dtype_name = str(
        feature_values.dtype
    )

    is_categorical = (
        dtype_name == "category"
        or pd.api.types.is_object_dtype(
            feature_values
        )
        or pd.api.types.is_string_dtype(
            feature_values
        )
        or pd.api.types.is_bool_dtype(
            feature_values
        )
        or unique_values <= 10
    )

    if not is_categorical:
        numeric_values = pd.to_numeric(
            feature_values,
            errors="coerce",
        )

        valid_mask = (
            numeric_values.notna()
            & shap_values.notna()
        )

        valid_values = numeric_values.loc[
            valid_mask
        ]

        valid_shap = shap_values.loc[
            valid_mask
        ]

        spearman_correlation = (
            valid_values.corr(
                valid_shap,
                method="spearman",
            )
        )

        lower_quartile = (
            valid_values.quantile(0.25)
        )

        upper_quartile = (
            valid_values.quantile(0.75)
        )

        low_value_mean_shap = (
            valid_shap.loc[
                valid_values
                <= lower_quartile
            ].mean()
        )

        high_value_mean_shap = (
            valid_shap.loc[
                valid_values
                >= upper_quartile
            ].mean()
        )

        if pd.isna(spearman_correlation):
            general_direction = (
                "Insufficient variation"
            )
        elif spearman_correlation >= 0.10:
            general_direction = (
                "Higher values generally "
                "increase predicted risk"
            )
        elif spearman_correlation <= -0.10:
            general_direction = (
                "Higher values generally "
                "reduce predicted risk"
            )
        else:
            general_direction = (
                "Nonlinear or mixed effect"
            )

        directionality_rows.append(
            {
                "global_rank":
                    global_rank_map[
                        feature_name
                    ],
                "feature": feature_name,
                "feature_source":
                    feature_source_map[
                        feature_name
                    ],
                "feature_type": "Numeric",
                "valid_records":
                    int(valid_mask.sum()),
                "missing_records":
                    int((~valid_mask).sum()),
                "unique_values":
                    int(unique_values),
                "spearman_value_shap":
                    spearman_correlation,
                "low_value_mean_shap":
                    low_value_mean_shap,
                "high_value_mean_shap":
                    high_value_mean_shap,
                "highest_risk_category":
                    None,
                "lowest_risk_category":
                    None,
                "general_direction":
                    general_direction,
            }
        )

        # Decile-level effect table
        bin_frame = pd.DataFrame(
            {
                "feature_value":
                    valid_values,
                "shap_value":
                    valid_shap,
            }
        )

        try:
            bin_frame["value_bin"] = (
                pd.qcut(
                    bin_frame[
                        "feature_value"
                    ],
                    q=10,
                    duplicates="drop",
                )
            )

            bin_summary = (
                bin_frame.groupby(
                    "value_bin",
                    observed=True,
                )
                .agg(
                    records=(
                        "feature_value",
                        "size",
                    ),
                    median_feature_value=(
                        "feature_value",
                        "median",
                    ),
                    mean_shap=(
                        "shap_value",
                        "mean",
                    ),
                    mean_absolute_shap=(
                        "shap_value",
                        lambda values:
                            np.abs(values).mean(),
                    ),
                )
                .reset_index(drop=True)
            )

            bin_summary.insert(
                0,
                "bin_number",
                np.arange(
                    1,
                    len(bin_summary) + 1,
                ),
            )

            bin_summary.insert(
                0,
                "feature",
                feature_name,
            )

            numeric_bin_tables.append(
                bin_summary
            )

        except ValueError:
            pass

    else:
        category_values = (
            feature_values
            .astype("string")
            .fillna("Missing")
        )

        category_frame = pd.DataFrame(
            {
                "category":
                    category_values,
                "shap_value":
                    shap_values,
            }
        )

        category_summary = (
            category_frame.groupby(
                "category",
                observed=True,
            )
            .agg(
                records=(
                    "shap_value",
                    "size",
                ),
                mean_shap=(
                    "shap_value",
                    "mean",
                ),
                mean_absolute_shap=(
                    "shap_value",
                    lambda values:
                        np.abs(values).mean(),
                ),
                positive_shap_share_pct=(
                    "shap_value",
                    lambda values:
                        (values > 0).mean()
                        * 100,
                ),
            )
            .reset_index()
        )

        category_summary[
            "eligible_category"
        ] = (
            category_summary["records"]
            >= 20
        )

        eligible_categories = (
            category_summary.loc[
                category_summary[
                    "eligible_category"
                ]
            ]
        )

        if len(eligible_categories) > 0:
            highest_risk_category = (
                eligible_categories
                .sort_values(
                    "mean_shap",
                    ascending=False,
                )
                .iloc[0]["category"]
            )

            lowest_risk_category = (
                eligible_categories
                .sort_values(
                    "mean_shap",
                    ascending=True,
                )
                .iloc[0]["category"]
            )
        else:
            highest_risk_category = None
            lowest_risk_category = None

        directionality_rows.append(
            {
                "global_rank":
                    global_rank_map[
                        feature_name
                    ],
                "feature": feature_name,
                "feature_source":
                    feature_source_map[
                        feature_name
                    ],
                "feature_type":
                    "Categorical",
                "valid_records":
                    int(len(category_values)),
                "missing_records":
                    int(
                        feature_values
                        .isna()
                        .sum()
                    ),
                "unique_values":
                    int(unique_values),
                "spearman_value_shap":
                    np.nan,
                "low_value_mean_shap":
                    np.nan,
                "high_value_mean_shap":
                    np.nan,
                "highest_risk_category":
                    highest_risk_category,
                "lowest_risk_category":
                    lowest_risk_category,
                "general_direction": (
                    "Category-specific effect"
                ),
            }
        )

        category_summary.insert(
            0,
            "feature",
            feature_name,
        )

        categorical_effect_tables.append(
            category_summary
        )


directionality_summary = (
    pd.DataFrame(directionality_rows)
    .sort_values("global_rank")
    .reset_index(drop=True)
)

numeric_effect_bins = (
    pd.concat(
        numeric_bin_tables,
        ignore_index=True,
    )
    if numeric_bin_tables
    else pd.DataFrame()
)

categorical_effects = (
    pd.concat(
        categorical_effect_tables,
        ignore_index=True,
    )
    if categorical_effect_tables
    else pd.DataFrame()
)


# =========================================================
# 5. Load calibrated probabilities for reason-code examples
# =========================================================

calibrated_oof = pd.read_parquet(
    calibrated_oof_path,
    columns=[
        "SK_ID_CURR",
        "SELECTED_PROBABILITY",
    ],
)

assert (
    calibrated_oof["SK_ID_CURR"]
    .duplicated()
    .sum()
    == 0
)

reason_sample = (
    shap_sample.merge(
        calibrated_oof,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one",
    )
)

assert (
    reason_sample[
        "SELECTED_PROBABILITY"
    ].isna().sum()
    == 0
)


# =========================================================
# 6. Select ten highest-risk sample customers
# =========================================================

reason_customer_count = 10
reasons_per_customer = 5

top_risk_customers = (
    reason_sample
    .nlargest(
        reason_customer_count,
        "SELECTED_PROBABILITY",
    )
    .copy()
)


def assign_risk_band(probability):
    if probability >= 0.28197:
        return "Very High: Top 5%"

    if probability >= 0.19538:
        return "High: 5–10%"

    if probability >= 0.11914:
        return "Elevated: 10–20%"

    if probability >= 0.05979:
        return "Moderate: 20–40%"

    return "Lower: Bottom 60%"


protected_or_sensitive_features = {
    "CODE_GENDER",
    "DAYS_BIRTH",
    "APP_AGE_YEARS",
}

reason_rows = []

for _, customer_row in (
    top_risk_customers.iterrows()
):
    customer_contributions = (
        customer_row[
            [
                column
                for column
                in shap_sample.columns
                if column.startswith(
                    "SHAP__"
                )
            ]
        ]
        .astype("float64")
    )

    customer_contributions.index = [
        column.replace(
            "SHAP__",
            "",
            1,
        )
        for column
        in customer_contributions.index
    ]

    positive_reasons = (
        customer_contributions.loc[
            customer_contributions > 0
        ]
        .sort_values(
            ascending=False
        )
        .head(reasons_per_customer)
    )

    for reason_rank, (
        feature_name,
        contribution,
    ) in enumerate(
        positive_reasons.items(),
        start=1,
    ):
        reason_rows.append(
            {
                "SK_ID_CURR": int(
                    customer_row[
                        "SK_ID_CURR"
                    ]
                ),
                "TARGET": int(
                    customer_row["TARGET"]
                ),
                "calibrated_probability":
                    float(
                        customer_row[
                            "SELECTED_PROBABILITY"
                        ]
                    ),
                "risk_band":
                    assign_risk_band(
                        customer_row[
                            "SELECTED_PROBABILITY"
                        ]
                    ),
                "reason_rank": reason_rank,
                "feature": feature_name,
                "feature_source":
                    feature_source_map[
                        feature_name
                    ],
                "global_shap_rank": int(
                    global_rank_map[
                        feature_name
                    ]
                ),
                "shap_contribution":
                    float(contribution),
                "risk_effect":
                    "Increases predicted risk",
                "protected_or_sensitive":
                    feature_name
                    in protected_or_sensitive_features,
                "technical_use_only": True,
            }
        )


technical_reason_codes = pd.DataFrame(
    reason_rows
)

assert len(technical_reason_codes) == (
    reason_customer_count
    * reasons_per_customer
)


# =========================================================
# 7. Attach original feature values to reason codes
# =========================================================

reason_feature_names = sorted(
    technical_reason_codes[
        "feature"
    ].unique()
)

reason_customer_ids = (
    technical_reason_codes[
        "SK_ID_CURR"
    ].unique()
)

reason_feature_values = pd.read_parquet(
    clean_modeling_path,
    columns=[
        "SK_ID_CURR",
    ] + reason_feature_names,
)

reason_feature_values = (
    reason_feature_values.loc[
        reason_feature_values[
            "SK_ID_CURR"
        ].isin(reason_customer_ids)
    ]
    .set_index("SK_ID_CURR")
)

assert len(reason_feature_values) == (
    reason_customer_count
)


def format_feature_value(value):
    if pd.isna(value):
        return "Missing"

    if isinstance(
        value,
        (float, np.floating),
    ):
        return f"{value:.6g}"

    return str(value)


technical_reason_codes[
    "feature_value"
] = [
    format_feature_value(
        reason_feature_values.loc[
            customer_id,
            feature_name,
        ]
    )
    for customer_id, feature_name
    in zip(
        technical_reason_codes[
            "SK_ID_CURR"
        ],
        technical_reason_codes[
            "feature"
        ],
    )
]


# =========================================================
# 8. Human-readable technical labels
# =========================================================

reason_label_map = {
    "APP_EXT_SOURCE_MEAN":
        "Average external risk score",
    "APP_EXT_SOURCE_MIN":
        "Minimum external risk score",
    "APP_EXT_SOURCE_MAX":
        "Maximum external risk score",
    "BUREAU_MAX_DEBT_CREDIT_RATIO":
        "Maximum bureau debt-to-credit ratio",
    "BUREAU_MEAN_DEBT_CREDIT_RATIO":
        "Mean bureau debt-to-credit ratio",
    "CODE_GENDER":
        "Gender — protected attribute",
    "ORGANIZATION_TYPE":
        "Employer organization type",
    "NAME_EDUCATION_TYPE":
        "Education category",
    "AMT_ANNUITY":
        "Requested loan annuity",
    "OCCUPATION_TYPE":
        "Occupation category",
    "DAYS_BIRTH":
        "Applicant age",
    "DAYS_EMPLOYED":
        "Employment duration",
    "PREV_REFUSED_SHARE":
        "Share of previously refused applications",
    "INST_CUSTOMER_CREDITS_WITH_LATE_PAYMENT_SHARE":
        "Share of credits with late instalments",
}


def readable_reason_label(
    feature_name,
):
    if feature_name in reason_label_map:
        return reason_label_map[
            feature_name
        ]

    return (
        feature_name
        .replace("_", " ")
        .title()
    )


technical_reason_codes[
    "technical_reason"
] = (
    technical_reason_codes[
        "feature"
    ].map(
        readable_reason_label
    )
)


# =========================================================
# 9. Customer-level reason summary
# =========================================================

protected_reason_counts = (
    technical_reason_codes.groupby(
        "SK_ID_CURR"
    )["protected_or_sensitive"]
    .sum()
)

customer_reason_summary = (
    top_risk_customers[
        [
            "SK_ID_CURR",
            "TARGET",
            "SELECTED_PROBABILITY",
            "RAW_MODEL_SCORE",
        ]
    ]
    .copy()
)

customer_reason_summary[
    "risk_band"
] = (
    customer_reason_summary[
        "SELECTED_PROBABILITY"
    ].map(assign_risk_band)
)

customer_reason_summary[
    "protected_reason_count"
] = (
    customer_reason_summary[
        "SK_ID_CURR"
    ]
    .map(protected_reason_counts)
    .fillna(0)
    .astype("int8")
)

customer_reason_summary[
    "governance_warning"
] = (
    customer_reason_summary[
        "protected_reason_count"
    ] > 0
)

customer_reason_summary = (
    customer_reason_summary
    .sort_values(
        "SELECTED_PROBABILITY",
        ascending=False,
    )
    .reset_index(drop=True)
)

customer_reason_summary.insert(
    0,
    "risk_rank_in_sample",
    np.arange(
        1,
        len(customer_reason_summary) + 1,
    ),
)

customer_reason_summary[
    "calibrated_probability_pct"
] = (
    customer_reason_summary[
        "SELECTED_PROBABILITY"
    ] * 100
)


# =========================================================
# 10. Save analytical outputs
# =========================================================

directionality_summary_path = (
    explainability_path
    / "shap_top20_directionality_summary.csv"
)

numeric_bins_path = (
    explainability_path
    / "shap_numeric_effect_bins.csv"
)

categorical_effects_path = (
    explainability_path
    / "shap_categorical_effects.csv"
)

reason_codes_path = (
    explainability_path
    / "shap_top_risk_technical_reason_codes.csv"
)

customer_summary_path = (
    explainability_path
    / "shap_top_risk_customer_summary.csv"
)

directionality_summary.to_csv(
    directionality_summary_path,
    index=False,
)

numeric_effect_bins.to_csv(
    numeric_bins_path,
    index=False,
)

categorical_effects.to_csv(
    categorical_effects_path,
    index=False,
)

technical_reason_codes.to_csv(
    reason_codes_path,
    index=False,
)

customer_reason_summary.to_csv(
    customer_summary_path,
    index=False,
)


# =========================================================
# 11. Directionality chart for top numeric features
# =========================================================

top_numeric_features = (
    directionality_summary.loc[
        directionality_summary[
            "feature_type"
        ] == "Numeric",
        "feature",
    ]
    .head(6)
    .tolist()
)

figure, axes = plt.subplots(
    2,
    3,
    figsize=(18, 10),
)

axes = axes.flatten()

for axis, feature_name in zip(
    axes,
    top_numeric_features,
):
    feature_bins = (
        numeric_effect_bins.loc[
            numeric_effect_bins[
                "feature"
            ] == feature_name
        ]
        .sort_values("bin_number")
    )

    axis.plot(
        feature_bins["bin_number"],
        feature_bins["mean_shap"],
        marker="o",
        linewidth=2,
        color="#355C7D",
    )

    axis.axhline(
        0,
        color="black",
        linewidth=1,
        alpha=0.5,
    )

    axis.set_title(feature_name)
    axis.set_xlabel(
        "Feature-value decile"
    )
    axis.set_ylabel(
        "Mean SHAP contribution"
    )
    axis.grid(alpha=0.2)

for unused_axis in axes[
    len(top_numeric_features):
]:
    unused_axis.axis("off")

figure.suptitle(
    "FinGuard360 — Direction of Key Numeric Risk Effects",
    fontsize=16,
    fontweight="bold",
)

figure.tight_layout()

direction_chart_path = (
    explainability_path
    / "shap_numeric_directionality.png"
)

figure.savefig(
    direction_chart_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()


# =========================================================
# 12. Validation summary
# =========================================================

reason_code_validation = pd.DataFrame(
    {
        "metric": [
            "Directionality features",
            "Numeric directionality features",
            "Categorical directionality features",
            "Numeric effect-bin rows",
            "Categorical effect rows",
            "Reason-code customers",
            "Reason-code rows",
            "Expected reason-code rows",
            "Duplicate customer-reason ranks",
            "Protected/sensitive reason rows",
            "Missing calibrated probabilities",
            "Directionality file exists",
            "Reason-code file exists",
            "Direction chart exists",
        ],
        "value": [
            len(directionality_summary),
            (
                directionality_summary[
                    "feature_type"
                ] == "Numeric"
            ).sum(),
            (
                directionality_summary[
                    "feature_type"
                ] == "Categorical"
            ).sum(),
            len(numeric_effect_bins),
            len(categorical_effects),
            technical_reason_codes[
                "SK_ID_CURR"
            ].nunique(),
            len(technical_reason_codes),
            (
                reason_customer_count
                * reasons_per_customer
            ),
            technical_reason_codes.duplicated(
                subset=[
                    "SK_ID_CURR",
                    "reason_rank",
                ]
            ).sum(),
            technical_reason_codes[
                "protected_or_sensitive"
            ].sum(),
            reason_sample[
                "SELECTED_PROBABILITY"
            ].isna().sum(),
            directionality_summary_path.exists(),
            reason_codes_path.exists(),
            direction_chart_path.exists(),
        ],
    }
)

display(
    reason_code_validation
)

display(
    directionality_summary.round(5)
)

display(
    categorical_effects.loc[
        categorical_effects[
            "eligible_category"
        ]
    ]
    .sort_values(
        [
            "feature",
            "mean_shap",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .head(30)
    .round(5)
)

display(
    customer_reason_summary[
        [
            "risk_rank_in_sample",
            "SK_ID_CURR",
            "TARGET",
            "calibrated_probability_pct",
            "risk_band",
            "protected_reason_count",
            "governance_warning",
        ]
    ].round(4)
)

display(
    technical_reason_codes[
        [
            "SK_ID_CURR",
            "reason_rank",
            "technical_reason",
            "feature_value",
            "feature_source",
            "shap_contribution",
            "protected_or_sensitive",
        ]
    ]
    .head(25)
    .round(5)
)


# =========================================================
# 13. Memory cleanup
# =========================================================

del top_feature_values
del direction_sample
del reason_feature_values
del reason_sample
del shap_sample

gc.collect()

In [ ]:
# =========================================================
# FinGuard360 Power BI data package
# =========================================================

import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
    roc_curve,
)


# =========================================================
# 1. Paths
# =========================================================

project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

processed_data_path = (
    project_path
    / "data"
    / "processed"
)

modeling_output_path = (
    processed_data_path
    / "modeling"
)

explainability_path = (
    modeling_output_path
    / "explainability"
)

powerbi_path = (
    project_path
    / "powerbi"
)

powerbi_data_path = (
    powerbi_path
    / "data"
)

powerbi_data_path.mkdir(
    parents=True,
    exist_ok=True,
)

clean_modeling_path = (
    processed_data_path
    / "modeling_train_clean.parquet"
)

tuned_oof_path = (
    modeling_output_path
    / "lightgbm_tuned_oof.parquet"
)

calibrated_oof_path = (
    modeling_output_path
    / "lightgbm_calibrated_oof.parquet"
)

feature_importance_path = (
    explainability_path
    / "shap_global_feature_importance_corrected.csv"
)

source_importance_path = (
    explainability_path
    / "shap_source_importance_corrected.csv"
)

reason_codes_path = (
    explainability_path
    / "shap_top_risk_technical_reason_codes.csv"
)

governance_path = (
    modeling_output_path
    / "final_model_governance_decision.csv"
)

required_paths = [
    clean_modeling_path,
    tuned_oof_path,
    calibrated_oof_path,
    feature_importance_path,
    source_importance_path,
    reason_codes_path,
    governance_path,
]

for required_path in required_paths:
    assert required_path.exists(), required_path


# =========================================================
# 2. Load customer dimensions and model scores
# =========================================================

customer_dimensions = pd.read_parquet(
    clean_modeling_path,
    columns=[
        "SK_ID_CURR",
        "TARGET",
        "CODE_GENDER",
        "DAYS_BIRTH",
        "AMT_INCOME_TOTAL",
        "NAME_EDUCATION_TYPE",
    ],
)

tuned_oof = pd.read_parquet(
    tuned_oof_path,
    columns=[
        "SK_ID_CURR",
        "CV_FOLD",
        "LIGHTGBM_TUNED_OOF",
    ],
)

calibrated_oof = pd.read_parquet(
    calibrated_oof_path,
    columns=[
        "SK_ID_CURR",
        "SELECTED_PROBABILITY",
    ],
)

for frame in [
    customer_dimensions,
    tuned_oof,
    calibrated_oof,
]:
    assert (
        frame["SK_ID_CURR"]
        .duplicated()
        .sum()
        == 0
    )

customer_risk = (
    customer_dimensions
    .merge(
        tuned_oof,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one",
    )
    .merge(
        calibrated_oof,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one",
    )
)

assert len(customer_risk) == 307_511

assert (
    customer_risk[
        [
            "CV_FOLD",
            "LIGHTGBM_TUNED_OOF",
            "SELECTED_PROBABILITY",
        ]
    ].isna().sum().sum()
    == 0
)


# =========================================================
# 3. Dashboard dimensions
# =========================================================

customer_risk[
    "AGE_YEARS"
] = (
    -customer_risk["DAYS_BIRTH"]
    / 365.25
)

customer_risk[
    "AGE_BAND"
] = pd.cut(
    customer_risk["AGE_YEARS"],
    bins=[
        20,
        30,
        40,
        50,
        60,
        70,
    ],
    labels=[
        "20–29",
        "30–39",
        "40–49",
        "50–59",
        "60–69",
    ],
    right=False,
    include_lowest=True,
)

income_labels = [
    "Q1: Lowest",
    "Q2",
    "Q3",
    "Q4",
    "Q5: Highest",
]

customer_risk[
    "INCOME_QUINTILE"
] = pd.qcut(
    customer_risk[
        "AMT_INCOME_TOTAL"
    ].rank(method="first"),
    q=5,
    labels=income_labels,
)

customer_risk[
    "DEFAULT_LABEL"
] = np.where(
    customer_risk["TARGET"] == 1,
    "Default",
    "Non-default",
)

customer_risk[
    "PREDICTED_DEFAULT_PCT"
] = (
    customer_risk[
        "SELECTED_PROBABILITY"
    ] * 100
)


# =========================================================
# 4. Exact risk ranks, deciles and bands
# =========================================================

customer_count = len(customer_risk)

customer_risk[
    "RISK_RANK"
] = (
    customer_risk[
        "SELECTED_PROBABILITY"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype("int32")
)

customer_risk[
    "RISK_POPULATION_PERCENTILE"
] = (
    customer_risk["RISK_RANK"]
    / customer_count
    * 100
)

customer_risk[
    "RISK_DECILE"
] = pd.cut(
    customer_risk[
        "RISK_POPULATION_PERCENTILE"
    ],
    bins=[
        0,
        10,
        20,
        30,
        40,
        50,
        60,
        70,
        80,
        90,
        100.0001,
    ],
    labels=[
        1, 2, 3, 4, 5,
        6, 7, 8, 9, 10,
    ],
    include_lowest=True,
).astype("int8")

risk_percentile = customer_risk[
    "RISK_POPULATION_PERCENTILE"
]

customer_risk[
    "RISK_BAND"
] = np.select(
    [
        risk_percentile <= 5,
        risk_percentile <= 10,
        risk_percentile <= 20,
        risk_percentile <= 40,
    ],
    [
        "Very High: Top 5%",
        "High: 5–10%",
        "Elevated: 10–20%",
        "Moderate: 20–40%",
    ],
    default="Lower: Bottom 60%",
)

for review_share in [
    5,
    10,
    20,
    40,
]:
    review_count = int(
        np.ceil(
            customer_count
            * review_share
            / 100
        )
    )

    customer_risk[
        f"TOP_{review_share}_FLAG"
    ] = (
        customer_risk["RISK_RANK"]
        <= review_count
    ).astype("int8")

customer_risk[
    "SENSITIVE_DIMENSIONS_AUDIT_ONLY"
] = 1


# =========================================================
# 5. Risk-band summary
# =========================================================

risk_band_order = [
    "Very High: Top 5%",
    "High: 5–10%",
    "Elevated: 10–20%",
    "Moderate: 20–40%",
    "Lower: Bottom 60%",
]

customer_risk[
    "RISK_BAND"
] = pd.Categorical(
    customer_risk["RISK_BAND"],
    categories=risk_band_order,
    ordered=True,
)

overall_default_rate = (
    customer_risk["TARGET"].mean()
)

total_defaults = int(
    customer_risk["TARGET"].sum()
)

risk_band_summary = (
    customer_risk.groupby(
        "RISK_BAND",
        observed=True,
        sort=False,
    )
    .agg(
        customers=(
            "SK_ID_CURR",
            "size",
        ),
        defaults=(
            "TARGET",
            "sum",
        ),
        observed_default_rate=(
            "TARGET",
            "mean",
        ),
        mean_predicted_probability=(
            "SELECTED_PROBABILITY",
            "mean",
        ),
        minimum_probability=(
            "SELECTED_PROBABILITY",
            "min",
        ),
        maximum_probability=(
            "SELECTED_PROBABILITY",
            "max",
        ),
    )
    .reset_index()
)

risk_band_summary[
    "population_share_pct"
] = (
    risk_band_summary["customers"]
    / customer_count
    * 100
)

risk_band_summary[
    "observed_default_rate_pct"
] = (
    risk_band_summary[
        "observed_default_rate"
    ] * 100
)

risk_band_summary[
    "mean_predicted_probability_pct"
] = (
    risk_band_summary[
        "mean_predicted_probability"
    ] * 100
)

risk_band_summary[
    "minimum_probability_pct"
] = (
    risk_band_summary[
        "minimum_probability"
    ] * 100
)

risk_band_summary[
    "maximum_probability_pct"
] = (
    risk_band_summary[
        "maximum_probability"
    ] * 100
)

risk_band_summary[
    "default_capture_pct"
] = (
    risk_band_summary["defaults"]
    / total_defaults
    * 100
)

risk_band_summary[
    "cumulative_default_capture_pct"
] = (
    risk_band_summary[
        "default_capture_pct"
    ].cumsum()
)

risk_band_summary[
    "risk_lift"
] = (
    risk_band_summary[
        "observed_default_rate"
    ]
    / overall_default_rate
)


# =========================================================
# 6. Decile summary
# =========================================================

risk_decile_summary = (
    customer_risk.groupby(
        "RISK_DECILE",
        observed=True,
        sort=True,
    )
    .agg(
        customers=(
            "SK_ID_CURR",
            "size",
        ),
        defaults=(
            "TARGET",
            "sum",
        ),
        observed_default_rate=(
            "TARGET",
            "mean",
        ),
        mean_predicted_probability=(
            "SELECTED_PROBABILITY",
            "mean",
        ),
    )
    .reset_index()
)

risk_decile_summary[
    "observed_default_rate_pct"
] = (
    risk_decile_summary[
        "observed_default_rate"
    ] * 100
)

risk_decile_summary[
    "mean_predicted_probability_pct"
] = (
    risk_decile_summary[
        "mean_predicted_probability"
    ] * 100
)

risk_decile_summary[
    "default_capture_pct"
] = (
    risk_decile_summary["defaults"]
    / total_defaults
    * 100
)

risk_decile_summary[
    "cumulative_default_capture_pct"
] = (
    risk_decile_summary[
        "default_capture_pct"
    ].cumsum()
)

risk_decile_summary[
    "risk_lift"
] = (
    risk_decile_summary[
        "observed_default_rate"
    ]
    / overall_default_rate
)


# =========================================================
# 7. Review-threshold analysis
# =========================================================

threshold_rows = []

for review_share in [
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]:
    review_count = int(
        np.ceil(
            customer_count
            * review_share
            / 100
        )
    )

    selected = (
        customer_risk["RISK_RANK"]
        <= review_count
    )

    selected_target = (
        customer_risk.loc[
            selected,
            "TARGET",
        ]
    )

    remaining_target = (
        customer_risk.loc[
            ~selected,
            "TARGET",
        ]
    )

    true_positives = int(
        selected_target.sum()
    )

    false_positives = int(
        selected.sum()
        - true_positives
    )

    false_negatives = int(
        remaining_target.sum()
    )

    reviewed_default_rate = (
        selected_target.mean()
    )

    remaining_default_rate = (
        remaining_target.mean()
    )

    threshold_rows.append(
        {
            "review_share_pct":
                review_share,
            "score_threshold_pct": (
                customer_risk.loc[
                    selected,
                    "SELECTED_PROBABILITY",
                ].min()
                * 100
            ),
            "reviewed_customers":
                int(selected.sum()),
            "remaining_customers":
                int((~selected).sum()),
            "captured_defaults":
                true_positives,
            "flagged_nondefaults":
                false_positives,
            "remaining_defaults":
                false_negatives,
            "default_capture_pct": (
                true_positives
                / total_defaults
                * 100
            ),
            "reviewed_default_rate_pct": (
                reviewed_default_rate
                * 100
            ),
            "risk_lift": (
                reviewed_default_rate
                / overall_default_rate
            ),
            "nondefault_population_flagged_pct": (
                false_positives
                / int(
                    (
                        customer_risk[
                            "TARGET"
                        ] == 0
                    ).sum()
                )
                * 100
            ),
            "remaining_default_rate_pct": (
                remaining_default_rate
                * 100
            ),
            "remaining_risk_reduction_pct": (
                1
                - remaining_default_rate
                / overall_default_rate
            ) * 100,
            "flagged_nondefaults_per_default": (
                false_positives
                / true_positives
            ),
        }
    )

threshold_summary = pd.DataFrame(
    threshold_rows
)


# =========================================================
# 8. Model-performance summary
# =========================================================

def calculate_ks(
    actual,
    probability,
):
    false_positive_rate, true_positive_rate, _ = (
        roc_curve(
            actual,
            probability,
        )
    )

    return float(
        np.max(
            true_positive_rate
            - false_positive_rate
        )
    )


def model_metric_row(
    model_name,
    probability,
    model_role,
):
    return {
        "model": model_name,
        "model_role": model_role,
        "features": 639,
        "roc_auc": roc_auc_score(
            customer_risk["TARGET"],
            probability,
        ),
        "gini": (
            2
            * roc_auc_score(
                customer_risk["TARGET"],
                probability,
            )
            - 1
        ),
        "average_precision":
            average_precision_score(
                customer_risk["TARGET"],
                probability,
            ),
        "ks_statistic": calculate_ks(
            customer_risk["TARGET"],
            probability,
        ),
        "log_loss": log_loss(
            customer_risk["TARGET"],
            probability,
        ),
        "brier_score": brier_score_loss(
            customer_risk["TARGET"],
            probability,
        ),
        "observed_default_rate_pct":
            overall_default_rate * 100,
        "mean_predicted_default_pct":
            probability.mean() * 100,
    }


model_performance = pd.DataFrame(
    [
        model_metric_row(
            "Optuna-tuned LightGBM",
            customer_risk[
                "LIGHTGBM_TUNED_OOF"
            ],
            "Technical ranking champion",
        ),
        model_metric_row(
            "Optuna-tuned LightGBM + Platt",
            customer_risk[
                "SELECTED_PROBABILITY"
            ],
            "Probability champion",
        ),
    ]
)


# =========================================================
# 9. Responsible-AI subgroup audit
# =========================================================

audit_dimensions = {
    "Gender": "CODE_GENDER",
    "Age band": "AGE_BAND",
    "Income quintile": "INCOME_QUINTILE",
    "Education": "NAME_EDUCATION_TYPE",
}

subgroup_rows = []

for dimension_name, dimension_column in (
    audit_dimensions.items()
):
    for group_name, group in (
        customer_risk.groupby(
            dimension_column,
            observed=True,
            dropna=False,
        )
    ):
        defaults = int(
            group["TARGET"].sum()
        )

        nondefaults = int(
            len(group) - defaults
        )

        flagged = (
            group["TOP_20_FLAG"] == 1
        )

        captured_defaults = int(
            (
                flagged
                & (group["TARGET"] == 1)
            ).sum()
        )

        flagged_nondefaults = int(
            (
                flagged
                & (group["TARGET"] == 0)
            ).sum()
        )

        subgroup_auc = (
            roc_auc_score(
                group["TARGET"],
                group[
                    "SELECTED_PROBABILITY"
                ],
            )
            if group[
                "TARGET"
            ].nunique() == 2
            else np.nan
        )

        subgroup_rows.append(
            {
                "dimension":
                    dimension_name,
                "group": str(group_name),
                "customers": len(group),
                "defaults": defaults,
                "observed_default_rate_pct": (
                    group["TARGET"].mean()
                    * 100
                ),
                "mean_predicted_rate_pct": (
                    group[
                        "SELECTED_PROBABILITY"
                    ].mean()
                    * 100
                ),
                "calibration_gap_pp": (
                    group[
                        "SELECTED_PROBABILITY"
                    ].mean()
                    - group["TARGET"].mean()
                ) * 100,
                "roc_auc": subgroup_auc,
                "top_20_flag_rate_pct": (
                    flagged.mean() * 100
                ),
                "default_capture_pct": (
                    captured_defaults
                    / defaults
                    * 100
                    if defaults > 0
                    else np.nan
                ),
                "nondefault_flag_rate_pct": (
                    flagged_nondefaults
                    / nondefaults
                    * 100
                    if nondefaults > 0
                    else np.nan
                ),
                "eligible_comparison_group": (
                    len(group) >= 1000
                    and defaults >= 50
                    and nondefaults >= 50
                ),
            }
        )

subgroup_audit = pd.DataFrame(
    subgroup_rows
)


# =========================================================
# 10. Load explainability and governance tables
# =========================================================

feature_importance = pd.read_csv(
    feature_importance_path
).head(50)

source_importance = pd.read_csv(
    source_importance_path
)

reason_codes = pd.read_csv(
    reason_codes_path
)

governance_decision = pd.read_csv(
    governance_path
)


# =========================================================
# 11. Export Power BI tables
# =========================================================

fact_columns = [
    "SK_ID_CURR",
    "TARGET",
    "DEFAULT_LABEL",
    "CV_FOLD",
    "LIGHTGBM_TUNED_OOF",
    "SELECTED_PROBABILITY",
    "PREDICTED_DEFAULT_PCT",
    "RISK_RANK",
    "RISK_POPULATION_PERCENTILE",
    "RISK_DECILE",
    "RISK_BAND",
    "TOP_5_FLAG",
    "TOP_10_FLAG",
    "TOP_20_FLAG",
    "TOP_40_FLAG",
    "CODE_GENDER",
    "AGE_YEARS",
    "AGE_BAND",
    "AMT_INCOME_TOTAL",
    "INCOME_QUINTILE",
    "NAME_EDUCATION_TYPE",
    "SENSITIVE_DIMENSIONS_AUDIT_ONLY",
]

export_tables = {
    "fact_customer_risk.csv":
        customer_risk[fact_columns],
    "summary_risk_bands.csv":
        risk_band_summary,
    "summary_risk_deciles.csv":
        risk_decile_summary,
    "summary_review_thresholds.csv":
        threshold_summary,
    "summary_model_performance.csv":
        model_performance,
    "summary_subgroup_audit.csv":
        subgroup_audit,
    "shap_feature_importance_top50.csv":
        feature_importance,
    "shap_source_importance.csv":
        source_importance,
    "technical_reason_codes.csv":
        reason_codes,
    "model_governance_decision.csv":
        governance_decision,
}

for file_name, table in (
    export_tables.items()
):
    table.to_csv(
        powerbi_data_path
        / file_name,
        index=False,
        encoding="utf-8-sig",
    )


# =========================================================
# 12. Power BI data dictionary
# =========================================================

data_dictionary = pd.DataFrame(
    [
        [
            "fact_customer_risk.csv",
            "SK_ID_CURR",
            "Anonymized customer identifier",
        ],
        [
            "fact_customer_risk.csv",
            "TARGET",
            "Observed default outcome: 1 default, 0 non-default",
        ],
        [
            "fact_customer_risk.csv",
            "SELECTED_PROBABILITY",
            "Cross-fitted Platt-calibrated default probability",
        ],
        [
            "fact_customer_risk.csv",
            "RISK_DECILE",
            "Risk ranking decile; 1 is highest risk",
        ],
        [
            "fact_customer_risk.csv",
            "RISK_BAND",
            "Five-level operational risk segment",
        ],
        [
            "fact_customer_risk.csv",
            "TOP_20_FLAG",
            "Exact global top-20-percent review flag",
        ],
        [
            "fact_customer_risk.csv",
            "CODE_GENDER",
            "Protected attribute included for audit only",
        ],
        [
            "fact_customer_risk.csv",
            "AGE_BAND",
            "Applicant age range used for subgroup audit",
        ],
        [
            "fact_customer_risk.csv",
            "INCOME_QUINTILE",
            "Equal-population income group",
        ],
        [
            "summary_risk_bands.csv",
            "default_capture_pct",
            "Share of all defaults captured by the band",
        ],
        [
            "summary_review_thresholds.csv",
            "risk_lift",
            "Reviewed-group default rate divided by global rate",
        ],
        [
            "summary_subgroup_audit.csv",
            "calibration_gap_pp",
            "Mean predicted rate minus observed rate in percentage points",
        ],
        [
            "shap_feature_importance_top50.csv",
            "mean_absolute_shap",
            "Average absolute TreeSHAP contribution in raw log-odds",
        ],
        [
            "technical_reason_codes.csv",
            "shap_contribution",
            "Technical positive risk contribution in raw log-odds",
        ],
    ],
    columns=[
        "table_name",
        "column_name",
        "definition",
    ],
)

data_dictionary.to_csv(
    powerbi_data_path
    / "data_dictionary.csv",
    index=False,
    encoding="utf-8-sig",
)


# =========================================================
# 13. Package manifest
# =========================================================

manifest_rows = []

for csv_path in sorted(
    powerbi_data_path.glob("*.csv")
):
    verified_table = pd.read_csv(
        csv_path,
        low_memory=False,
    )

    manifest_rows.append(
        {
            "file_name": csv_path.name,
            "rows": len(verified_table),
            "columns":
                verified_table.shape[1],
            "file_size_mb": (
                csv_path.stat().st_size
                / 1024**2
            ),
        }
    )

package_manifest = pd.DataFrame(
    manifest_rows
)

package_manifest.to_csv(
    powerbi_data_path
    / "package_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)


# =========================================================
# 14. Create ZIP package
# =========================================================

zip_package_path = (
    powerbi_path
    / "FinGuard360_PowerBI_Package.zip"
)

with zipfile.ZipFile(
    zip_package_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as zip_file:
    for csv_path in sorted(
        powerbi_data_path.glob("*.csv")
    ):
        zip_file.write(
            csv_path,
            arcname=(
                f"data/{csv_path.name}"
            ),
        )


# =========================================================
# 15. Final validation
# =========================================================

powerbi_package_validation = pd.DataFrame(
    {
        "metric": [
            "Customer fact rows",
            "Unique customer IDs",
            "Duplicate customer IDs",
            "Target defaults",
            "Risk bands",
            "Risk deciles",
            "Exact top-20 customers",
            "Expected top-20 customers",
            "Subgroup audit rows",
            "Top SHAP features",
            "Technical reason-code rows",
            "Exported CSV files",
            "ZIP package exists",
            "ZIP package size (MB)",
        ],
        "value": [
            len(customer_risk),
            customer_risk[
                "SK_ID_CURR"
            ].nunique(),
            customer_risk[
                "SK_ID_CURR"
            ].duplicated().sum(),
            customer_risk[
                "TARGET"
            ].sum(),
            customer_risk[
                "RISK_BAND"
            ].nunique(),
            customer_risk[
                "RISK_DECILE"
            ].nunique(),
            customer_risk[
                "TOP_20_FLAG"
            ].sum(),
            int(
                np.ceil(
                    len(customer_risk)
                    * 0.20
                )
            ),
            len(subgroup_audit),
            len(feature_importance),
            len(reason_codes),
            len(
                list(
                    powerbi_data_path.glob(
                        "*.csv"
                    )
                )
            ),
            zip_package_path.exists(),
            (
                zip_package_path.stat().st_size
                / 1024**2
            ),
        ],
    }
)

display(
    powerbi_package_validation.round(4)
)

display(
    model_performance.round(6)
)

display(
    risk_band_summary[
        [
            "RISK_BAND",
            "customers",
            "defaults",
            "population_share_pct",
            "observed_default_rate_pct",
            "mean_predicted_probability_pct",
            "default_capture_pct",
            "cumulative_default_capture_pct",
            "risk_lift",
        ]
    ].round(4)
)

display(
    threshold_summary.round(4)
)

display(
    package_manifest.round(4)
)

print(
    "Power BI package:",
    zip_package_path,
)

In [ ]:
from pathlib import Path
import re
import zipfile

import numpy as np
import pandas as pd


# =========================================================
# 1. Paths
# =========================================================
project_path = Path(
    str((Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
)

powerbi_data_path = (
    project_path
    / "powerbi"
    / "data"
)

risk_band_path = (
    powerbi_data_path
    / "summary_risk_bands.csv"
)

manifest_path = (
    powerbi_data_path
    / "package_manifest.csv"
)

zip_path = (
    project_path
    / "powerbi"
    / "FinGuard360_PowerBI_Package.zip"
)

assert risk_band_path.exists(), risk_band_path


# =========================================================
# 2. Load summary and identify columns
# =========================================================
risk_band_summary = pd.read_csv(
    risk_band_path
)


def normalize_column_name(value):
    return re.sub(
        r"[^A-Z0-9]+",
        "_",
        str(value).upper(),
    ).strip("_")


def find_column(dataframe, *candidates):
    normalized_columns = {
        normalize_column_name(column): column
        for column in dataframe.columns
    }

    for candidate in candidates:
        normalized_candidate = (
            normalize_column_name(candidate)
        )

        if normalized_candidate in normalized_columns:
            return normalized_columns[
                normalized_candidate
            ]

    raise KeyError(
        f"Columns not found: {candidates}. "
        f"Available columns: "
        f"{dataframe.columns.tolist()}"
    )


band_column = find_column(
    risk_band_summary,
    "RISK_BAND",
)

customer_column = find_column(
    risk_band_summary,
    "CUSTOMERS",
    "CUSTOMER_COUNT",
)

default_column = find_column(
    risk_band_summary,
    "DEFAULTS",
    "DEFAULT_COUNT",
)


# =========================================================
# 3. Apply correct risk order
# =========================================================
risk_band_order = [
    "Very High: Top 5%",
    "High: 5–10%",
    "Elevated: 10–20%",
    "Moderate: 20–40%",
    "Lower: Bottom 60%",
]

risk_band_order_map = {
    band: position
    for position, band in enumerate(
        risk_band_order,
        start=1,
    )
}

risk_band_summary[
    "RISK_BAND_ORDER"
] = (
    risk_band_summary[band_column]
    .astype("string")
    .map(risk_band_order_map)
)

assert (
    risk_band_summary[
        "RISK_BAND_ORDER"
    ].notna().all()
), (
    "Unexpected risk-band labels: "
    f"{risk_band_summary[band_column].tolist()}"
)

risk_band_summary[
    "RISK_BAND_ORDER"
] = risk_band_summary[
    "RISK_BAND_ORDER"
].astype("int8")

risk_band_summary = (
    risk_band_summary
    .sort_values("RISK_BAND_ORDER")
    .reset_index(drop=True)
)


# =========================================================
# 4. Recalculate cumulative metrics
# =========================================================
total_customers = int(
    risk_band_summary[
        customer_column
    ].sum()
)

total_defaults = int(
    risk_band_summary[
        default_column
    ].sum()
)

population_share = (
    risk_band_summary[customer_column]
    / total_customers
    * 100
)

default_capture = (
    risk_band_summary[default_column]
    / total_defaults
    * 100
)

cumulative_population = (
    risk_band_summary[customer_column]
    .cumsum()
    / total_customers
    * 100
)

cumulative_default_capture = (
    risk_band_summary[default_column]
    .cumsum()
    / total_defaults
    * 100
)


def update_or_create_column(
    dataframe,
    values,
    *candidate_names,
):
    normalized_columns = {
        normalize_column_name(column): column
        for column in dataframe.columns
    }

    selected_column = None

    for candidate in candidate_names:
        normalized_candidate = (
            normalize_column_name(candidate)
        )

        if normalized_candidate in normalized_columns:
            selected_column = normalized_columns[
                normalized_candidate
            ]
            break

    if selected_column is None:
        selected_column = candidate_names[0]

    dataframe[selected_column] = values

    return selected_column


population_share_column = (
    update_or_create_column(
        risk_band_summary,
        population_share,
        "POPULATION_SHARE_PCT",
    )
)

default_capture_column = (
    update_or_create_column(
        risk_band_summary,
        default_capture,
        "DEFAULT_CAPTURE_PCT",
    )
)

cumulative_population_column = (
    update_or_create_column(
        risk_band_summary,
        cumulative_population,
        "CUMULATIVE_POPULATION_PCT",
        "CUMULATIVE_POPULATION_SHARE_PCT",
    )
)

cumulative_capture_column = (
    update_or_create_column(
        risk_band_summary,
        cumulative_default_capture,
        "CUMULATIVE_DEFAULT_CAPTURE_PCT",
    )
)


# =========================================================
# 5. Validate corrected summary
# =========================================================
assert total_customers == 307511
assert total_defaults == 24825

assert (
    risk_band_summary[band_column].tolist()
    == risk_band_order
)

assert np.isclose(
    risk_band_summary[
        cumulative_population_column
    ].iloc[-1],
    100.0,
)

assert np.isclose(
    risk_band_summary[
        cumulative_capture_column
    ].iloc[-1],
    100.0,
)

assert risk_band_summary[
    cumulative_capture_column
].is_monotonic_increasing

expected_cumulative_capture = np.array(
    [
        23.6536,
        38.2236,
        57.6798,
        79.6657,
        100.0000,
    ]
)

assert np.allclose(
    risk_band_summary[
        cumulative_capture_column
    ].to_numpy(),
    expected_cumulative_capture,
    atol=0.01,
)


# =========================================================
# 6. Save corrected file
# =========================================================
risk_band_summary.to_csv(
    risk_band_path,
    index=False,
)


# =========================================================
# 7. Rebuild package manifest
# =========================================================
manifest_records = []

data_files = sorted(
    path
    for path in powerbi_data_path.glob("*.csv")
    if path.name != manifest_path.name
)

for file_path in data_files:
    file_data = pd.read_csv(
        file_path,
        low_memory=False,
    )

    manifest_records.append(
        {
            "file_name": file_path.name,
            "rows": len(file_data),
            "columns": file_data.shape[1],
            "size_mb": (
                file_path.stat().st_size
                / 1024**2
            ),
        }
    )

package_manifest = pd.DataFrame(
    manifest_records
)

package_manifest.to_csv(
    manifest_path,
    index=False,
)


# =========================================================
# 8. Rebuild ZIP package
# =========================================================
with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=9,
) as package_zip:

    for file_path in sorted(
        powerbi_data_path.glob("*.csv")
    ):
        package_zip.write(
            file_path,
            arcname=file_path.name,
        )


# =========================================================
# 9. Final audit
# =========================================================
risk_band_package_fix_check = pd.DataFrame(
    {
        "metric": [
            "Corrected risk bands",
            "Reconciled customers",
            "Reconciled defaults",
            "Final cumulative population (%)",
            "Final cumulative capture (%)",
            "Top 5% cumulative capture (%)",
            "Top 10% cumulative capture (%)",
            "Top 20% cumulative capture (%)",
            "Top 40% cumulative capture (%)",
            "Manifest data files",
            "Total CSV files",
            "ZIP package exists",
            "ZIP package size (MB)",
        ],
        "value": [
            len(risk_band_summary),
            total_customers,
            total_defaults,
            risk_band_summary[
                cumulative_population_column
            ].iloc[-1],
            risk_band_summary[
                cumulative_capture_column
            ].iloc[-1],
            risk_band_summary[
                cumulative_capture_column
            ].iloc[0],
            risk_band_summary[
                cumulative_capture_column
            ].iloc[1],
            risk_band_summary[
                cumulative_capture_column
            ].iloc[2],
            risk_band_summary[
                cumulative_capture_column
            ].iloc[3],
            len(package_manifest),
            len(
                list(
                    powerbi_data_path.glob(
                        "*.csv"
                    )
                )
            ),
            zip_path.exists(),
            (
                zip_path.stat().st_size
                / 1024**2
            ),
        ],
    }
)

display(
    risk_band_package_fix_check.round(4)
)

display(
    risk_band_summary[
        [
            "RISK_BAND_ORDER",
            band_column,
            customer_column,
            default_column,
            population_share_column,
            default_capture_column,
            cumulative_capture_column,
        ]
    ].round(4)
)

print(
    f"Corrected Power BI package: {zip_path}"
)